In [1]:
# CELL 1: Upload the gold and Fed-signal Excel workbooks

from google.colab import files
from pathlib import Path
import pandas as pd
import numpy as np

uploaded = files.upload()

uploaded_files = [Path(filename) for filename in uploaded]

if len(uploaded_files) < 2:
    raise ValueError(
        "Please upload both the Gold_1 workbook and the Fed gold-ready workbook."
    )

# Identify the Fed workbook from its filename.
fed_candidates = [
    file for file in uploaded_files
    if "fed_gold_ready" in file.name.lower()
]

# Treat the remaining Excel workbook as Gold_1.
gold_candidates = [
    file for file in uploaded_files
    if file.suffix.lower() in {".xlsx", ".xls"}
    and file not in fed_candidates
]

if len(fed_candidates) != 1:
    raise ValueError(
        "Could not uniquely identify the Fed workbook. "
        "Its filename should contain 'Fed_Gold_Ready'."
    )

if len(gold_candidates) != 1:
    raise ValueError(
        "Could not uniquely identify the Gold_1 workbook. "
        "Please upload only the two required Excel files."
    )

FED_FILE = fed_candidates[0]
GOLD_FILE = gold_candidates[0]

print("Gold workbook:", GOLD_FILE.name)
print("Fed workbook:", FED_FILE.name)

print("\nGold workbook sheets:")
print(pd.ExcelFile(GOLD_FILE).sheet_names)

print("\nFed workbook sheets:")
print(pd.ExcelFile(FED_FILE).sheet_names)

Saving Fed_Gold_Ready_Variables_Jan2019_to_July2026.xlsx to Fed_Gold_Ready_Variables_Jan2019_to_July2026.xlsx
Saving Gold_1.xlsx to Gold_1.xlsx
Gold workbook: Gold_1.xlsx
Fed workbook: Fed_Gold_Ready_Variables_Jan2019_to_July2026.xlsx

Gold workbook sheets:
['Monthly Causal Data', 'Quarterly Demand Data', 'Regional ETF Panel']

Fed workbook sheets:
['Fed_Gold_Ready_2019_Onward']


In [2]:
# CELL 2: Load the monthly gold dataset and Fed-policy variables

GOLD_SHEET = "Monthly Causal Data"
FED_SHEET = "Fed_Gold_Ready_2019_Onward"

gold_monthly = pd.read_excel(
    GOLD_FILE,
    sheet_name=GOLD_SHEET
)

fed_monthly = pd.read_excel(
    FED_FILE,
    sheet_name=FED_SHEET
)

# Standardise date columns.
gold_monthly["Month"] = pd.to_datetime(
    gold_monthly["Month"],
    errors="coerce"
).dt.to_period("M").dt.to_timestamp()

fed_monthly["Month"] = pd.to_datetime(
    fed_monthly["Month"],
    errors="coerce"
).dt.to_period("M").dt.to_timestamp()

fed_monthly["Decision_Date"] = pd.to_datetime(
    fed_monthly["Decision_Date"],
    errors="coerce"
)

# Sort chronologically.
gold_monthly = (
    gold_monthly
    .sort_values("Month")
    .reset_index(drop=True)
)

fed_monthly = (
    fed_monthly
    .sort_values("Month")
    .reset_index(drop=True)
)

print("Gold monthly shape:", gold_monthly.shape)
print(
    "Gold period:",
    gold_monthly["Month"].min(),
    "to",
    gold_monthly["Month"].max()
)

print("\nFed monthly shape:", fed_monthly.shape)
print(
    "Fed period:",
    fed_monthly["Month"].min(),
    "to",
    fed_monthly["Month"].max()
)

print("\nGold columns:")
display(pd.DataFrame({"Gold_Variable": gold_monthly.columns}))

print("\nFed columns:")
display(pd.DataFrame({"Fed_Variable": fed_monthly.columns}))

print("\nFirst five Fed observations:")
display(fed_monthly.head())

Gold monthly shape: (100, 47)
Gold period: 2019-01-01 00:00:00 to 2026-07-01 00:00:00

Fed monthly shape: (91, 24)
Fed period: 2019-01-01 00:00:00 to 2026-07-01 00:00:00

Gold columns:


,Gold_Variable
0,Month
1,Year
2,Quarter
3,Month_ID
4,Gold_Monthly_Return_pct
5,ETF_Net_Demand_Global_WGC_tonnes
6,CB_Reported_Holdings_Change_Global_IMF_tonnes
7,ETF_plus_CB_IMF_tonnes
8,CB_Reported_Net_Change_Global_WGC_tonnes
9,ETF_plus_CB_WGC_tonnes



Fed columns:


,Fed_Variable
0,Month
1,Decision_Date
2,Actual
3,Actual_Action_Code
4,Primary_Raw_P_Cut
5,Primary_Raw_P_Hold
6,Primary_Raw_P_Hike
7,Primary_Fed_Direction
8,Primary_Fed_Uncertainty_Normalized
9,Primary_Fed_Confidence



First five Fed observations:


,Month,Decision_Date,Actual,Actual_Action_Code,Primary_Raw_P_Cut,Primary_Raw_P_Hold,Primary_Raw_P_Hike,Primary_Fed_Direction,Primary_Fed_Uncertainty_Normalized,Primary_Fed_Confidence,...,Robust_Raw_P_Hold,Robust_Raw_P_Hike,Robust_Fed_Direction,Robust_Fed_Uncertainty_Normalized,Robust_Fed_Confidence,Robust_Fed_Surprise_Signed,Robust_Fed_Surprise_Absolute,Robust_Unexpected_Action_Probability,FOMC_Meetings_In_Month,FOMC_Meeting_Flag
0,2019-01-01,2019-01-30,Hold,0.0,0.039173,0.542855,0.417972,0.378799,0.749275,0.542855,...,0.517008,0.440367,0.397742,0.761625,0.517008,-0.397742,0.397742,0.482992,1,1
1,2019-02-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,2019-03-01,2019-03-20,Hold,0.0,0.082873,0.758507,0.158620,0.075747,0.644543,0.758507,...,0.759603,0.164041,0.087686,0.638806,0.759603,-0.087686,0.087686,0.240397,1,1
3,2019-04-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,2019-05-01,2019-05-01,Hold,0.0,0.109331,0.734049,0.156620,0.047289,0.691150,0.734049,...,0.745445,0.165075,0.075596,0.666597,0.745445,-0.075596,0.075596,0.254555,1,1


In [3]:
# CELL 3: Convert missing non-meeting-month Fed values into event-flow zeros

fed_analysis = fed_monthly.copy()

fed_signal_columns = [
    "Primary_Fed_Direction",
    "Primary_Fed_Uncertainty_Normalized",
    "Primary_Fed_Confidence",
    "Primary_Fed_Surprise_Signed",
    "Primary_Fed_Surprise_Absolute",
    "Primary_Unexpected_Action_Probability",
    "Robust_Fed_Direction",
    "Robust_Fed_Uncertainty_Normalized",
    "Robust_Fed_Confidence",
    "Robust_Fed_Surprise_Signed",
    "Robust_Fed_Surprise_Absolute",
    "Robust_Unexpected_Action_Probability",
]

# Validate that missing values occur only in non-meeting months.
invalid_missing = fed_analysis.loc[
    fed_analysis["FOMC_Meeting_Flag"].eq(1),
    fed_signal_columns
].isna().any(axis=1)

if invalid_missing.any():
    problematic_months = fed_analysis.loc[
        fed_analysis["FOMC_Meeting_Flag"].eq(1)
        & invalid_missing,
        "Month"
    ]

    raise ValueError(
        "Some meeting months have missing Fed signals: "
        + ", ".join(problematic_months.dt.strftime("%Y-%m"))
    )

# Preserve the original columns and create explicitly named event variables.
for column in fed_signal_columns:
    event_column = f"{column}_Event"

    fed_analysis[event_column] = (
        fed_analysis[column]
        .where(fed_analysis["FOMC_Meeting_Flag"].eq(1), 0.0)
    )

# Non-meeting months must now equal zero in the event variables.
event_columns = [
    f"{column}_Event"
    for column in fed_signal_columns
]

assert (
    fed_analysis.loc[
        fed_analysis["FOMC_Meeting_Flag"].eq(0),
        event_columns
    ]
    .eq(0)
    .all()
    .all()
)

print("Total monthly observations:", len(fed_analysis))
print(
    "Meeting months:",
    fed_analysis["FOMC_Meeting_Flag"].sum()
)
print(
    "Non-meeting months retained:",
    fed_analysis["FOMC_Meeting_Flag"].eq(0).sum()
)

display(
    fed_analysis[
        [
            "Month",
            "FOMC_Meetings_In_Month",
            "FOMC_Meeting_Flag",
            "Primary_Fed_Direction_Event",
            "Primary_Fed_Uncertainty_Normalized_Event",
            "Primary_Fed_Surprise_Signed_Event",
        ]
    ].head(15)
)

Total monthly observations: 91
Meeting months: 62
Non-meeting months retained: 29


,Month,FOMC_Meetings_In_Month,FOMC_Meeting_Flag,Primary_Fed_Direction_Event,Primary_Fed_Uncertainty_Normalized_Event,Primary_Fed_Surprise_Signed_Event
0,2019-01-01,1,1,0.378799,0.749275,-0.378799
1,2019-02-01,0,0,0.000000,0.000000,0.000000
2,2019-03-01,1,1,0.075747,0.644543,-0.075747
3,2019-04-01,0,0,0.000000,0.000000,0.000000
4,2019-05-01,1,1,0.047289,0.691150,-0.047289
5,2019-06-01,1,1,0.093836,0.714753,-0.093836
6,2019-07-01,1,1,0.092682,0.642616,-1.092682
7,2019-08-01,0,0,0.000000,0.000000,0.000000
8,2019-09-01,1,1,-0.275047,0.896467,-0.724953
9,2019-10-01,1,1,-0.253850,0.911889,-0.746150


In [4]:
# CELL 4A: Diagnose duplicate months in the gold workbook

month_counts = (
    gold_monthly
    .groupby("Month", dropna=False)
    .size()
    .reset_index(name="Rows_Per_Month")
)

print("Total rows:", len(gold_monthly))
print("Unique months:", gold_monthly["Month"].nunique())
print("Maximum rows in one month:", month_counts["Rows_Per_Month"].max())

print("\nDistribution of rows per month:")
display(
    month_counts["Rows_Per_Month"]
    .value_counts()
    .sort_index()
    .rename_axis("Rows_Per_Month")
    .reset_index(name="Number_of_Months")
)

print("\nAvailable columns:")
display(
    pd.DataFrame({
        "Column_Number": range(len(gold_monthly.columns)),
        "Column_Name": gold_monthly.columns
    })
)

duplicate_months = month_counts.loc[
    month_counts["Rows_Per_Month"] > 1,
    "Month"
]

duplicate_rows = (
    gold_monthly[
        gold_monthly["Month"].isin(duplicate_months)
    ]
    .sort_values("Month")
    .reset_index(drop=True)
)

print("\nFirst duplicated monthly records:")
display(duplicate_rows.head(20))

# Identify columns that vary within the same month.
within_month_variation = []

for column in gold_monthly.columns:
    if column == "Month":
        continue

    maximum_unique_values = (
        gold_monthly
        .groupby("Month", dropna=False)[column]
        .nunique(dropna=False)
        .max()
    )

    if maximum_unique_values > 1:
        within_month_variation.append({
            "Column": column,
            "Maximum_Unique_Values_Within_Month":
                maximum_unique_values
        })

print("\nColumns that differ within at least one month:")
display(pd.DataFrame(within_month_variation))

Total rows: 100
Unique months: 91
Maximum rows in one month: 9

Distribution of rows per month:


,Rows_Per_Month,Number_of_Months
0,1,91
1,9,1



Available columns:


,Column_Number,Column_Name
0,0,Month
1,1,Year
2,2,Quarter
3,3,Month_ID
4,4,Gold_Monthly_Return_pct
5,5,ETF_Net_Demand_Global_WGC_tonnes
6,6,CB_Reported_Holdings_Change_Global_IMF_tonnes
7,7,ETF_plus_CB_IMF_tonnes
8,8,CB_Reported_Net_Change_Global_WGC_tonnes
9,9,ETF_plus_CB_WGC_tonnes



First duplicated monthly records:


,Month,Year,Quarter,Month_ID,Gold_Monthly_Return_pct,ETF_Net_Demand_Global_WGC_tonnes,CB_Reported_Holdings_Change_Global_IMF_tonnes,ETF_plus_CB_IMF_tonnes,CB_Reported_Net_Change_Global_WGC_tonnes,ETF_plus_CB_WGC_tonnes,...,CB_Global_Eligible_Countries_IMF,CB_Global_Adjacent_Coverage_pct,CB_IMF_Coverage_50pct_Flag,Core_Causal_Row_Complete_Flag,Post_2022_Dummy,Post_2025_Dummy,ETF_Net_Demand_North_America_WGC_tonnes,ETF_Net_Demand_Europe_WGC_tonnes,ETF_Net_Demand_Asia_WGC_tonnes,ETF_Net_Demand_Others_WGC_tonnes
0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaT,https://data.imf.org/en/Data-Explorer?datasetU...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaT,https://www.gold.org/goldhub/data,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaT,https://fred.stlouisfed.org/series/BAMLH0A0HYM2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaT,https://github.com/csaladenes/eco-archive/blob...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaT,https://equibles.com/economicdata/bamlh0a0hym2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaT,IMF values are adjacent-month changes in repor...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Columns that differ within at least one month:


,Column,Maximum_Unique_Values_Within_Month
0,Year,7


In [5]:
# CELL 4B: Remove source-note rows and merge the monthly Fed variables

# Preserve source-note rows separately for documentation.
gold_source_notes = gold_monthly.loc[
    gold_monthly["Month"].isna()
].copy()

# Retain only genuine monthly observations.
gold_monthly_clean = (
    gold_monthly
    .loc[gold_monthly["Month"].notna()]
    .copy()
    .sort_values("Month")
    .reset_index(drop=True)
)

# Confirm that valid monthly observations are unique.
if gold_monthly_clean["Month"].duplicated().any():
    duplicated_months = (
        gold_monthly_clean.loc[
            gold_monthly_clean["Month"].duplicated(keep=False),
            "Month"
        ]
        .dt.strftime("%Y-%m")
        .unique()
    )

    raise ValueError(
        "Actual duplicate months remain: "
        + ", ".join(duplicated_months)
    )

# Identify months with more than one FOMC decision.
fed_analysis["Multiple_FOMC_Meetings_Flag"] = (
    fed_analysis["FOMC_Meetings_In_Month"] > 1
).astype(int)

fed_merge_columns = [
    "Month",
    "Decision_Date",
    "Actual",
    "Actual_Action_Code",
    "FOMC_Meetings_In_Month",
    "FOMC_Meeting_Flag",
    "Multiple_FOMC_Meetings_Flag",

    # Primary macro-policy model
    "Primary_Fed_Direction_Event",
    "Primary_Fed_Uncertainty_Normalized_Event",
    "Primary_Fed_Confidence_Event",
    "Primary_Fed_Surprise_Signed_Event",
    "Primary_Fed_Surprise_Absolute_Event",
    "Primary_Unexpected_Action_Probability_Event",

    # Combined-model robustness specification
    "Robust_Fed_Direction_Event",
    "Robust_Fed_Uncertainty_Normalized_Event",
    "Robust_Fed_Confidence_Event",
    "Robust_Fed_Surprise_Signed_Event",
    "Robust_Fed_Surprise_Absolute_Event",
    "Robust_Unexpected_Action_Probability_Event",
]

# Merge without removing any valid gold month.
gold_analysis = (
    gold_monthly_clean
    .loc[gold_monthly_clean["Month"] >= "2019-01-01"]
    .merge(
        fed_analysis[fed_merge_columns],
        on="Month",
        how="left",
        validate="one_to_one",
        indicator=True
    )
    .sort_values("Month")
    .reset_index(drop=True)
)

merge_summary = pd.DataFrame({
    "Check": [
        "Source-note rows removed",
        "Valid gold months",
        "Unique gold months",
        "Matched Fed-calendar months",
        "Unmatched gold months",
        "Meeting months",
        "Non-meeting months",
        "Multiple-meeting months",
    ],
    "Value": [
        len(gold_source_notes),
        len(gold_monthly_clean),
        gold_monthly_clean["Month"].nunique(),
        gold_analysis["_merge"].eq("both").sum(),
        gold_analysis["_merge"].ne("both").sum(),
        gold_analysis["FOMC_Meeting_Flag"].eq(1).sum(),
        gold_analysis["FOMC_Meeting_Flag"].eq(0).sum(),
        gold_analysis["Multiple_FOMC_Meetings_Flag"].eq(1).sum(),
    ]
})

display(merge_summary)

# Remove the temporary merge-audit column after validation.
if not gold_analysis["_merge"].eq("both").all():
    display(
        gold_analysis.loc[
            gold_analysis["_merge"].ne("both"),
            ["Month", "_merge"]
        ]
    )
    raise ValueError("Some gold months failed to match the Fed calendar.")

gold_analysis = gold_analysis.drop(columns="_merge")

display(
    gold_analysis[
        [
            "Month",
            "Gold_Monthly_Return_pct",
            "ETF_Net_Demand_Global_WGC_tonnes",
            "CB_Reported_Holdings_Change_Global_IMF_tonnes",
            "FOMC_Meeting_Flag",
            "Primary_Fed_Direction_Event",
            "Primary_Fed_Uncertainty_Normalized_Event",
            "Primary_Fed_Surprise_Signed_Event",
        ]
    ].head(18)
)

,Check,Value
0,Source-note rows removed,9
1,Valid gold months,91
2,Unique gold months,91
3,Matched Fed-calendar months,91
4,Unmatched gold months,0
5,Meeting months,62
6,Non-meeting months,29
7,Multiple-meeting months,1


,Month,Gold_Monthly_Return_pct,ETF_Net_Demand_Global_WGC_tonnes,CB_Reported_Holdings_Change_Global_IMF_tonnes,FOMC_Meeting_Flag,Primary_Fed_Direction_Event,Primary_Fed_Uncertainty_Normalized_Event,Primary_Fed_Surprise_Signed_Event
0,2019-01-01,3.021073,71.8,68.164924,1,0.378799,0.749275,-0.378799
1,2019-02-01,-0.311335,-32.4,18.236201,0,0.000000,0.000000,0.000000
2,2019-03-01,-0.262267,1.7,46.992204,1,0.075747,0.644543,-0.075747
3,2019-04-01,-0.434646,-56.9,59.124482,0,0.000000,0.000000,0.000000
4,2019-05-01,1.904169,-2.4,43.595762,1,0.047289,0.691150,-0.047289
5,2019-06-01,6.577455,126.8,107.641113,1,0.093836,0.714753,-0.093836
6,2019-07-01,2.922922,52.2,18.269797,1,0.092682,0.642616,-1.092682
7,2019-08-01,6.911109,122.0,25.812958,0,0.000000,0.000000,0.000000
8,2019-09-01,-5.187921,78.6,49.205766,1,-0.275047,0.896467,-0.724953
9,2019-10-01,1.983807,44.4,53.998441,1,-0.253850,0.911889,-0.746150


In [6]:
# CELL 5: Define the primary Fed channels for monthly gold analysis

# Expected policy direction before the FOMC decision:
# Positive = expected tightening; negative = expected easing.
gold_analysis["Fed_Expected_Direction"] = (
    gold_analysis["Primary_Fed_Direction_Event"]
)

# Pre-meeting policy uncertainty:
# Higher value = probabilities more evenly distributed across Cut/Hold/Hike.
gold_analysis["Fed_Policy_Uncertainty"] = (
    gold_analysis["Primary_Fed_Uncertainty_Normalized_Event"]
)

# Signed policy surprise:
# Positive = decision was more hawkish than expected.
# Negative = decision was more dovish than expected.
gold_analysis["Fed_Surprise_Signed"] = (
    gold_analysis["Primary_Fed_Surprise_Signed_Event"]
)

# Absolute surprise magnitude, irrespective of direction.
gold_analysis["Fed_Surprise_Absolute"] = (
    gold_analysis["Primary_Fed_Surprise_Absolute_Event"]
)

# Probability assigned to outcomes other than the realized decision.
gold_analysis["Fed_Unexpected_Action_Probability"] = (
    gold_analysis["Primary_Unexpected_Action_Probability_Event"]
)

# Meeting indicators.
gold_analysis["Fed_Meeting_Month"] = (
    gold_analysis["FOMC_Meeting_Flag"].astype(int)
)

gold_analysis["Fed_Multiple_Meeting_Month"] = (
    gold_analysis["Multiple_FOMC_Meetings_Flag"].astype(int)
)

fed_model_variables = [
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
    "Fed_Surprise_Absolute",
    "Fed_Unexpected_Action_Probability",
    "Fed_Meeting_Month",
    "Fed_Multiple_Meeting_Month",
]

# Availability and descriptive audit.
fed_variable_audit = pd.DataFrame({
    "Variable": fed_model_variables,
    "Available": [
        gold_analysis[column].notna().sum()
        for column in fed_model_variables
    ],
    "Missing": [
        gold_analysis[column].isna().sum()
        for column in fed_model_variables
    ],
    "Minimum": [
        gold_analysis[column].min()
        for column in fed_model_variables
    ],
    "Maximum": [
        gold_analysis[column].max()
        for column in fed_model_variables
    ],
    "Mean": [
        gold_analysis[column].mean()
        for column in fed_model_variables
    ],
    "Standard_Deviation": [
        gold_analysis[column].std()
        for column in fed_model_variables
    ],
})

display(fed_variable_audit.round(4))

# Logical integrity checks.
fed_integrity_checks = pd.DataFrame({
    "Check": [
        "Expected direction lies between -1 and +1",
        "Policy uncertainty lies between 0 and 1",
        "Absolute surprise is non-negative",
        "Unexpected-action probability lies between 0 and 1",
        "Non-meeting direction equals zero",
        "Non-meeting uncertainty equals zero",
        "Non-meeting surprise equals zero",
    ],
    "Passed": [
        gold_analysis["Fed_Expected_Direction"].between(-1, 1).all(),
        gold_analysis["Fed_Policy_Uncertainty"].between(0, 1).all(),
        gold_analysis["Fed_Surprise_Absolute"].ge(0).all(),
        gold_analysis[
            "Fed_Unexpected_Action_Probability"
        ].between(0, 1).all(),

        gold_analysis.loc[
            gold_analysis["Fed_Meeting_Month"].eq(0),
            "Fed_Expected_Direction"
        ].eq(0).all(),

        gold_analysis.loc[
            gold_analysis["Fed_Meeting_Month"].eq(0),
            "Fed_Policy_Uncertainty"
        ].eq(0).all(),

        gold_analysis.loc[
            gold_analysis["Fed_Meeting_Month"].eq(0),
            "Fed_Surprise_Signed"
        ].eq(0).all(),
    ]
})

display(fed_integrity_checks)

,Variable,Available,Missing,Minimum,Maximum,Mean,Standard_Deviation
0,Fed_Expected_Direction,91,0,-0.8482,0.9009,0.0710,0.3267
1,Fed_Policy_Uncertainty,91,0,0.0000,0.9218,0.4398,0.3491
2,Fed_Surprise_Signed,91,0,-1.0927,0.9145,-0.0600,0.3271
3,Fed_Surprise_Absolute,91,0,0.0000,1.0927,0.1929,0.2702
4,Fed_Unexpected_Action_Probability,91,0,0.0000,0.9424,0.2522,0.2567
5,Fed_Meeting_Month,91,0,0.0000,1.0000,0.6813,0.4685
6,Fed_Multiple_Meeting_Month,91,0,0.0000,1.0000,0.0110,0.1048


,Check,Passed
0,Expected direction lies between -1 and +1,True
1,Policy uncertainty lies between 0 and 1,True
2,Absolute surprise is non-negative,True
3,Unexpected-action probability lies between 0 a...,True
4,Non-meeting direction equals zero,True
5,Non-meeting uncertainty equals zero,True
6,Non-meeting surprise equals zero,True


In [7]:
# CELL 6: ADF and KPSS stationarity tests for Fed event-flow variables

import warnings

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning

fed_stationarity_variables = [
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
    "Fed_Surprise_Absolute",
    "Fed_Unexpected_Action_Probability",
]

def stationarity_test(series, variable_name):
    x = (
        pd.to_numeric(series, errors="coerce")
        .dropna()
        .astype(float)
    )

    if x.nunique() <= 1:
        return {
            "Variable": variable_name,
            "N": len(x),
            "ADF_Statistic": np.nan,
            "ADF_p": np.nan,
            "ADF_Lag": np.nan,
            "KPSS_Statistic": np.nan,
            "KPSS_p": np.nan,
            "Assessment": "Constant series",
        }

    # ADF null hypothesis: unit root / non-stationarity.
    adf_result = adfuller(
        x,
        regression="c",
        autolag="AIC"
    )

    # KPSS null hypothesis: level stationarity.
    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore",
            category=InterpolationWarning
        )

        kpss_result = kpss(
            x,
            regression="c",
            nlags="auto"
        )

    adf_p = adf_result[1]
    kpss_p = kpss_result[1]

    if adf_p < 0.05 and kpss_p >= 0.05:
        assessment = "Stationary"

    elif adf_p >= 0.05 and kpss_p < 0.05:
        assessment = "Non-stationary"

    elif adf_p < 0.05 and kpss_p < 0.05:
        assessment = "Conflicting evidence"

    else:
        assessment = "Inconclusive / low power"

    return {
        "Variable": variable_name,
        "N": len(x),
        "ADF_Statistic": adf_result[0],
        "ADF_p": adf_p,
        "ADF_Lag": adf_result[2],
        "KPSS_Statistic": kpss_result[0],
        "KPSS_p": kpss_p,
        "Assessment": assessment,
    }

fed_stationarity_results = pd.DataFrame([
    stationarity_test(
        gold_analysis[column],
        column
    )
    for column in fed_stationarity_variables
])

fed_stationarity_results["ADF_p_Display"] = (
    fed_stationarity_results["ADF_p"]
    .map(lambda value: "< 0.0001" if value < 0.0001
         else f"{value:.4f}")
)

fed_stationarity_results["KPSS_p_Display"] = (
    fed_stationarity_results["KPSS_p"]
    .map(lambda value: "> 0.10" if value >= 0.10
         else "< 0.01" if value <= 0.01
         else f"{value:.4f}")
)

display(
    fed_stationarity_results[
        [
            "Variable",
            "N",
            "ADF_Statistic",
            "ADF_p_Display",
            "ADF_Lag",
            "KPSS_Statistic",
            "KPSS_p_Display",
            "Assessment",
        ]
    ].round(4)
)

,Variable,N,ADF_Statistic,ADF_p_Display,ADF_Lag,KPSS_Statistic,KPSS_p_Display,Assessment
0,Fed_Expected_Direction,91,-2.5504,0.1037,2,0.2670,> 0.10,Inconclusive / low power
1,Fed_Policy_Uncertainty,91,-2.0411,0.2688,8,0.4674,0.0490,Non-stationary
2,Fed_Surprise_Signed,91,-3.8313,0.0026,2,0.1713,> 0.10,Stationary
3,Fed_Surprise_Absolute,91,-5.1569,< 0.0001,4,0.0640,> 0.10,Stationary
4,Fed_Unexpected_Action_Probability,91,-2.8485,0.0517,5,0.3648,0.0923,Inconclusive / low power


In [8]:
# CELL 7: Test the underlying Fed signals using meeting months only

from statsmodels.tsa.stattools import zivot_andrews

meeting_data = (
    gold_analysis
    .loc[gold_analysis["Fed_Meeting_Month"].eq(1)]
    .copy()
    .sort_values("Month")
    .reset_index(drop=True)
)

meeting_stationarity_results = pd.DataFrame([
    stationarity_test(
        meeting_data[column],
        column
    )
    for column in fed_stationarity_variables
])

meeting_stationarity_results["ADF_p_Display"] = (
    meeting_stationarity_results["ADF_p"]
    .map(lambda value: "< 0.0001" if value < 0.0001
         else f"{value:.4f}")
)

meeting_stationarity_results["KPSS_p_Display"] = (
    meeting_stationarity_results["KPSS_p"]
    .map(lambda value: "> 0.10" if value >= 0.10
         else "< 0.01" if value <= 0.01
         else f"{value:.4f}")
)

print("Meeting-month observations:", len(meeting_data))

display(
    meeting_stationarity_results[
        [
            "Variable",
            "N",
            "ADF_Statistic",
            "ADF_p_Display",
            "ADF_Lag",
            "KPSS_Statistic",
            "KPSS_p_Display",
            "Assessment",
        ]
    ].round(4)
)

# Zivot-Andrews test allows one endogenous structural break.
break_test_variables = [
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Unexpected_Action_Probability",
]

break_results = []

for column in break_test_variables:
    x = meeting_data[column].astype(float)

    za_statistic, za_p, za_critical_values, used_lag, break_index = (
        zivot_andrews(
            x,
            trim=0.15,
            maxlag=6,
            regression="ct",
            autolag="AIC"
        )
    )

    break_results.append({
        "Variable": column,
        "N": len(x),
        "ZA_Statistic": za_statistic,
        "ZA_p": za_p,
        "Used_Lag": used_lag,
        "Estimated_Break_Month":
            meeting_data.loc[break_index, "Month"],
        "Critical_Value_1pct":
            za_critical_values["1%"],
        "Critical_Value_5pct":
            za_critical_values["5%"],
        "Critical_Value_10pct":
            za_critical_values["10%"],
        "Reject_Unit_Root_5pct":
            za_statistic < za_critical_values["5%"],
    })

break_results = pd.DataFrame(break_results)

display(break_results.round(4))

Meeting-month observations: 62


,Variable,N,ADF_Statistic,ADF_p_Display,ADF_Lag,KPSS_Statistic,KPSS_p_Display,Assessment
0,Fed_Expected_Direction,62,-2.7022,0.0737,0,0.2028,> 0.10,Inconclusive / low power
1,Fed_Policy_Uncertainty,62,-2.7059,0.0730,5,0.4079,0.0737,Inconclusive / low power
2,Fed_Surprise_Signed,62,-6.2289,< 0.0001,0,0.1457,> 0.10,Stationary
3,Fed_Surprise_Absolute,62,-2.4414,0.1304,8,0.0499,> 0.10,Inconclusive / low power
4,Fed_Unexpected_Action_Probability,62,-5.6616,< 0.0001,0,0.2586,> 0.10,Stationary


,Variable,N,ZA_Statistic,ZA_p,Used_Lag,Estimated_Break_Month,Critical_Value_1pct,Critical_Value_5pct,Critical_Value_10pct,Reject_Unit_Root_5pct
0,Fed_Expected_Direction,62,-4.4585,0.2311,0,2022-03-01,-5.5756,-5.0733,-4.8267,False
1,Fed_Policy_Uncertainty,62,-5.9524,0.0030,6,2020-06-01,-5.5756,-5.0733,-4.8267,True
2,Fed_Unexpected_Action_Probability,62,-7.0479,0.0010,0,2020-06-01,-5.5756,-5.0733,-4.8267,True


In [9]:
# CELL 8: Construct final Fed variables for the causal models

# Structural break identified by the Zivot-Andrews tests.
gold_analysis["Post_June_2020_Dummy"] = (
    gold_analysis["Month"] >= pd.Timestamp("2020-06-01")
).astype(int)

# COVID-period indicator for the acute policy-intervention period.
gold_analysis["COVID_Shock_Dummy"] = (
    gold_analysis["Month"].between(
        pd.Timestamp("2020-03-01"),
        pd.Timestamp("2021-06-01")
    )
).astype(int)

# ---------------------------------------------------------
# Meeting-to-meeting changes for robustness specifications
# ---------------------------------------------------------

meeting_changes = (
    gold_analysis.loc[
        gold_analysis["Fed_Meeting_Month"].eq(1),
        [
            "Month",
            "Fed_Expected_Direction",
            "Fed_Policy_Uncertainty",
        ]
    ]
    .copy()
    .sort_values("Month")
)

meeting_changes["Delta_Fed_Expected_Direction"] = (
    meeting_changes["Fed_Expected_Direction"].diff()
)

meeting_changes["Delta_Fed_Policy_Uncertainty"] = (
    meeting_changes["Fed_Policy_Uncertainty"].diff()
)

gold_analysis = gold_analysis.merge(
    meeting_changes[
        [
            "Month",
            "Delta_Fed_Expected_Direction",
            "Delta_Fed_Policy_Uncertainty",
        ]
    ],
    on="Month",
    how="left",
    validate="one_to_one"
)

# Non-meeting months have no new meeting-to-meeting information.
# The first meeting remains missing because no preceding meeting exists
# inside the January-2019 sample.
for column in [
    "Delta_Fed_Expected_Direction",
    "Delta_Fed_Policy_Uncertainty",
]:
    non_meeting_mask = gold_analysis["Fed_Meeting_Month"].eq(0)

    gold_analysis.loc[
        non_meeting_mask,
        column
    ] = 0.0

# Explicitly identify the first meeting, where changes are unavailable.
first_meeting_month = gold_analysis.loc[
    gold_analysis["Fed_Meeting_Month"].eq(1),
    "Month"
].min()

gold_analysis["Fed_Change_Available_Flag"] = (
    gold_analysis["Delta_Fed_Expected_Direction"].notna()
    & gold_analysis["Delta_Fed_Policy_Uncertainty"].notna()
).astype(int)

print("First meeting month:", first_meeting_month)

# ---------------------------------------------------------
# Define the model roles
# ---------------------------------------------------------

primary_fed_variables = [
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
    "Fed_Meeting_Month",
    "Fed_Multiple_Meeting_Month",
    "Post_June_2020_Dummy",
    "COVID_Shock_Dummy",
]

robustness_fed_variables = [
    "Delta_Fed_Expected_Direction",
    "Delta_Fed_Policy_Uncertainty",
    "Fed_Unexpected_Action_Probability",
    "Fed_Surprise_Absolute",
]

variable_roles = pd.DataFrame({
    "Variable": [
        "Fed_Expected_Direction",
        "Fed_Policy_Uncertainty",
        "Fed_Surprise_Signed",
        "Fed_Meeting_Month",
        "Fed_Multiple_Meeting_Month",
        "Post_June_2020_Dummy",
        "COVID_Shock_Dummy",
        "Delta_Fed_Expected_Direction",
        "Delta_Fed_Policy_Uncertainty",
        "Fed_Unexpected_Action_Probability",
        "Fed_Surprise_Absolute",
    ],
    "Role": [
        "Primary",
        "Primary",
        "Primary",
        "Required event control",
        "March-2020 sensitivity control",
        "Structural-break control",
        "Crisis control",
        "Robustness",
        "Robustness",
        "Alternative surprise measure",
        "Alternative surprise measure",
    ],
    "Interpretation": [
        "Expected tightening minus expected easing",
        "Pre-meeting policy uncertainty",
        "Hawkish or dovish realized policy surprise",
        "Distinguishes no meeting from zero-valued signal",
        "Identifies the only month containing two meetings",
        "Controls for the June-2020 uncertainty break",
        "Controls for the exceptional pandemic regime",
        "Change relative to the preceding FOMC meeting",
        "Change relative to the preceding FOMC meeting",
        "Probability not assigned to the realized action",
        "Magnitude of surprise without directional information",
    ],
})

display(variable_roles)

display(
    gold_analysis[
        [
            "Month",
            "Fed_Meeting_Month",
            "Fed_Expected_Direction",
            "Delta_Fed_Expected_Direction",
            "Fed_Policy_Uncertainty",
            "Delta_Fed_Policy_Uncertainty",
            "Fed_Surprise_Signed",
            "Post_June_2020_Dummy",
            "COVID_Shock_Dummy",
        ]
    ].head(18)
)

First meeting month: 2019-01-01 00:00:00


,Variable,Role,Interpretation
0,Fed_Expected_Direction,Primary,Expected tightening minus expected easing
1,Fed_Policy_Uncertainty,Primary,Pre-meeting policy uncertainty
2,Fed_Surprise_Signed,Primary,Hawkish or dovish realized policy surprise
3,Fed_Meeting_Month,Required event control,Distinguishes no meeting from zero-valued signal
4,Fed_Multiple_Meeting_Month,March-2020 sensitivity control,Identifies the only month containing two meetings
5,Post_June_2020_Dummy,Structural-break control,Controls for the June-2020 uncertainty break
6,COVID_Shock_Dummy,Crisis control,Controls for the exceptional pandemic regime
7,Delta_Fed_Expected_Direction,Robustness,Change relative to the preceding FOMC meeting
8,Delta_Fed_Policy_Uncertainty,Robustness,Change relative to the preceding FOMC meeting
9,Fed_Unexpected_Action_Probability,Alternative surprise measure,Probability not assigned to the realized action


,Month,Fed_Meeting_Month,Fed_Expected_Direction,Delta_Fed_Expected_Direction,Fed_Policy_Uncertainty,Delta_Fed_Policy_Uncertainty,Fed_Surprise_Signed,Post_June_2020_Dummy,COVID_Shock_Dummy
0,2019-01-01,1,0.378799,NaN,0.749275,NaN,-0.378799,0,0
1,2019-02-01,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
2,2019-03-01,1,0.075747,-0.303052,0.644543,-0.104732,-0.075747,0,0
3,2019-04-01,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
4,2019-05-01,1,0.047289,-0.028458,0.691150,0.046607,-0.047289,0,0
5,2019-06-01,1,0.093836,0.046547,0.714753,0.023603,-0.093836,0,0
6,2019-07-01,1,0.092682,-0.001154,0.642616,-0.072137,-1.092682,0,0
7,2019-08-01,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
8,2019-09-01,1,-0.275047,-0.367730,0.896467,0.253851,-0.724953,0,0
9,2019-10-01,1,-0.253850,0.021197,0.911889,0.015422,-0.746150,0,0


In [10]:
# CELL 9: Plotly time-series EDA of Fed channels and gold returns

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=[
        "Monthly Gold Return",
        "Expected Fed Direction and Realised Policy Surprise",
        "Fed Policy Uncertainty and FOMC Meeting Months",
    ],
    specs=[
        [{"secondary_y": False}],
        [{"secondary_y": False}],
        [{"secondary_y": True}],
    ],
)

# ---------------------------------------------------------
# Panel 1: Gold monthly return
# ---------------------------------------------------------

gold_colours = np.where(
    gold_analysis["Gold_Monthly_Return_pct"] >= 0,
    "#C89411",
    "#B42318"
)

fig.add_trace(
    go.Bar(
        x=gold_analysis["Month"],
        y=gold_analysis["Gold_Monthly_Return_pct"],
        name="Gold return",
        marker_color=gold_colours,
        hovertemplate=(
            "Month: %{x|%b %Y}"
            "<br>Gold return: %{y:.2f}%"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
    row=1,
    col=1,
)

# ---------------------------------------------------------
# Panel 2: Expected direction and signed surprise
# ---------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=gold_analysis["Month"],
        y=gold_analysis["Fed_Expected_Direction"],
        name="Expected Fed direction",
        mode="lines+markers",
        line=dict(color="#1565C0", width=2),
        marker=dict(size=6),
        customdata=gold_analysis[
            ["FOMC_Meeting_Flag", "Actual"]
        ].to_numpy(),
        hovertemplate=(
            "Month: %{x|%b %Y}"
            "<br>Expected direction: %{y:.3f}"
            "<br>Meeting flag: %{customdata[0]}"
            "<br>Actual decision: %{customdata[1]}"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=gold_analysis["Month"],
        y=gold_analysis["Fed_Surprise_Signed"],
        name="Signed Fed surprise",
        marker_color=np.where(
            gold_analysis["Fed_Surprise_Signed"] >= 0,
            "#7A5195",
            "#EF5675"
        ),
        opacity=0.55,
        hovertemplate=(
            "Month: %{x|%b %Y}"
            "<br>Signed surprise: %{y:.3f}"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
    row=2,
    col=1,
)

# ---------------------------------------------------------
# Panel 3: Uncertainty and meeting occurrence
# ---------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=gold_analysis["Month"],
        y=gold_analysis["Fed_Policy_Uncertainty"],
        name="Fed uncertainty",
        mode="lines+markers",
        line=dict(color="#2E7D32", width=2),
        marker=dict(size=6),
        hovertemplate=(
            "Month: %{x|%b %Y}"
            "<br>Uncertainty: %{y:.3f}"
            "<extra></extra>"
        ),
    ),
    row=3,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Bar(
        x=gold_analysis["Month"],
        y=gold_analysis["Fed_Meeting_Month"],
        name="FOMC meeting month",
        marker_color="#AAB7C4",
        opacity=0.30,
        hovertemplate=(
            "Month: %{x|%b %Y}"
            "<br>Meeting flag: %{y:.0f}"
            "<extra></extra>"
        ),
    ),
    row=3,
    col=1,
    secondary_y=True,
)

# Structural-break marker.
fig.add_vline(
    x=pd.Timestamp("2020-06-01").timestamp() * 1000,
    line_dash="dash",
    line_color="#B42318",
    line_width=1.5,
)

fig.add_annotation(
    x=pd.Timestamp("2020-06-01"),
    y=1.05,
    xref="x3",
    yref="y3",
    text="Estimated uncertainty break",
    showarrow=False,
    font=dict(color="#B42318", size=11),
)

fig.update_yaxes(
    title_text="Gold return (%)",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Direction / surprise",
    range=[-1.25, 1.25],
    row=2,
    col=1,
)

fig.update_yaxes(
    title_text="Normalized uncertainty",
    range=[0, 1.10],
    row=3,
    col=1,
    secondary_y=False,
)

fig.update_yaxes(
    title_text="Meeting flag",
    range=[0, 1.15],
    tickvals=[0, 1],
    row=3,
    col=1,
    secondary_y=True,
)

fig.update_layout(
    title=(
        "Fed Expectations, Policy Surprises and Gold Returns"
        "<br><sup>Monthly data: January 2019 onward</sup>"
    ),
    template="plotly_white",
    height=1000,
    width=1400,
    hovermode="x unified",
    barmode="overlay",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    ),
)

fig.show()

In [11]:
# CELL 10: Pearson and Spearman correlations for the Fed channels

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fed_corr_data = pd.DataFrame({
    "Gold_Return":
        gold_analysis["Gold_Monthly_Return_pct"],

    "ETF_Flow":
        gold_analysis["ETF_Net_Demand_Global_WGC_tonnes"],

    "CB_IMF_Change":
        gold_analysis[
            "CB_Reported_Holdings_Change_Global_IMF_tonnes"
        ],

    "Broad_USD_Change":
        gold_analysis["Broad_USD_Index_Avg"].diff(),

    "Real10Y_Change_pp":
        gold_analysis[
            "US_10Y_Real_Yield_MonthEnd_pct"
        ].diff(),

    "VIX_Change":
        gold_analysis["VIX_MonthEnd"].diff(),

    "Fed_Expected_Direction":
        gold_analysis["Fed_Expected_Direction"],

    "Fed_Policy_Uncertainty":
        gold_analysis["Fed_Policy_Uncertainty"],

    "Fed_Surprise_Signed":
        gold_analysis["Fed_Surprise_Signed"],

    "Fed_Meeting_Month":
        gold_analysis["Fed_Meeting_Month"],
})

pearson_corr = fed_corr_data.corr(
    method="pearson",
    min_periods=24
)

spearman_corr = fed_corr_data.corr(
    method="spearman",
    min_periods=24
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Pearson Correlations",
        "Spearman Correlations",
    ],
    horizontal_spacing=0.12,
)

for matrix, column_number in [
    (pearson_corr, 1),
    (spearman_corr, 2),
]:
    fig.add_trace(
        go.Heatmap(
            z=matrix.values,
            x=matrix.columns,
            y=matrix.index,
            zmin=-1,
            zmax=1,
            zmid=0,
            colorscale="RdBu_r",
            text=np.round(matrix.values, 2),
            texttemplate="%{text}",
            hovertemplate=(
                "%{y} versus %{x}"
                "<br>Correlation: %{z:.3f}"
                "<extra></extra>"
            ),
            colorbar=dict(
                title="Correlation",
                x=0.46 if column_number == 1 else 1.02,
            ),
        ),
        row=1,
        col=column_number,
    )

fig.update_layout(
    title=(
        "Fed Channels, Gold Returns and Related Market Variables"
        "<br><sup>Contemporaneous monthly correlations; "
        "not causal estimates</sup>"
    ),
    template="plotly_white",
    height=800,
    width=1550,
)

fig.update_xaxes(tickangle=55)

fig.show()

# Focused correlation table against gold returns.
fed_gold_correlations = pd.DataFrame({
    "Pearson_with_Gold_Return":
        pearson_corr["Gold_Return"],

    "Spearman_with_Gold_Return":
        spearman_corr["Gold_Return"],
})

fed_gold_correlations = (
    fed_gold_correlations
    .drop(index="Gold_Return")
    .assign(
        Absolute_Spearman=lambda frame:
            frame["Spearman_with_Gold_Return"].abs()
    )
    .sort_values(
        "Absolute_Spearman",
        ascending=False
    )
    .drop(columns="Absolute_Spearman")
)

print("Correlations with monthly gold returns:")
display(fed_gold_correlations.round(4))

# Correlations among the Fed measures themselves.
fed_internal_correlations = spearman_corr.loc[
    [
        "Fed_Expected_Direction",
        "Fed_Policy_Uncertainty",
        "Fed_Surprise_Signed",
        "Fed_Meeting_Month",
    ],
    [
        "Fed_Expected_Direction",
        "Fed_Policy_Uncertainty",
        "Fed_Surprise_Signed",
        "Fed_Meeting_Month",
    ],
]

print("\nSpearman correlations among Fed variables:")
display(fed_internal_correlations.round(4))

Correlations with monthly gold returns:


,Pearson_with_Gold_Return,Spearman_with_Gold_Return
ETF_Flow,0.5612,0.5715
Real10Y_Change_pp,-0.2896,-0.3672
Broad_USD_Change,-0.0656,-0.1909
Fed_Expected_Direction,-0.2656,-0.1883
CB_IMF_Change,-0.1647,-0.1759
VIX_Change,0.0829,0.1382
Fed_Meeting_Month,-0.1270,-0.1064
Fed_Surprise_Signed,-0.0043,0.0147
Fed_Policy_Uncertainty,-0.0366,0.0052



Spearman correlations among Fed variables:


,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed,Fed_Meeting_Month
Fed_Expected_Direction,1.0000,0.2238,-0.1342,0.3970
Fed_Policy_Uncertainty,0.2238,1.0000,-0.3887,0.8205
Fed_Surprise_Signed,-0.1342,-0.3887,1.0000,-0.2911
Fed_Meeting_Month,0.3970,0.8205,-0.2911,1.0000


In [12]:
# CELL 11: Orthogonalise the Fed signals relative to the meeting indicator

fed_variables_to_center = [
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
]

meeting_month_mask = gold_analysis["Fed_Meeting_Month"].eq(1)

meeting_means = {}

for column in fed_variables_to_center:
    meeting_mean = gold_analysis.loc[
        meeting_month_mask,
        column
    ].mean()

    meeting_means[column] = meeting_mean

    centered_column = f"{column}_Centered"

    # Meeting month:
    # observed value minus the average meeting-month value.
    #
    # Non-meeting month:
    # zero because no FOMC event occurred.
    gold_analysis[centered_column] = np.where(
        meeting_month_mask,
        gold_analysis[column] - meeting_mean,
        0.0
    )

meeting_means_table = pd.DataFrame({
    "Variable": meeting_means.keys(),
    "Meeting_Month_Mean": meeting_means.values(),
})

display(meeting_means_table.round(4))

centered_fed_variables = [
    "Fed_Expected_Direction_Centered",
    "Fed_Policy_Uncertainty_Centered",
    "Fed_Surprise_Signed_Centered",
]

# Verify that each centred signal has a zero mean
# within FOMC meeting months.
centering_checks = pd.DataFrame({
    "Variable": centered_fed_variables,
    "Meeting_Month_Mean_After_Centering": [
        gold_analysis.loc[
            meeting_month_mask,
            column
        ].mean()
        for column in centered_fed_variables
    ],
    "Correlation_With_Meeting_Flag": [
        gold_analysis[
            [column, "Fed_Meeting_Month"]
        ].corr(method="pearson").iloc[0, 1]
        for column in centered_fed_variables
    ],
})

display(centering_checks.round(8))

# Compare the new correlation structure.
centered_corr_variables = [
    "Fed_Meeting_Month",
    "Fed_Expected_Direction_Centered",
    "Fed_Policy_Uncertainty_Centered",
    "Fed_Surprise_Signed_Centered",
]

centered_fed_correlation = (
    gold_analysis[centered_corr_variables]
    .corr(method="spearman")
)

fig = go.Figure(
    data=go.Heatmap(
        z=centered_fed_correlation.values,
        x=centered_fed_correlation.columns,
        y=centered_fed_correlation.index,
        zmin=-1,
        zmax=1,
        zmid=0,
        colorscale="RdBu_r",
        text=np.round(centered_fed_correlation.values, 3),
        texttemplate="%{text}",
        hovertemplate=(
            "%{y} versus %{x}"
            "<br>Spearman correlation: %{z:.3f}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=(
        "Correlation Structure After Meeting-Month Centering"
    ),
    template="plotly_white",
    height=650,
    width=950,
)

fig.update_xaxes(tickangle=35)

fig.show()

,Variable,Meeting_Month_Mean
0,Fed_Expected_Direction,0.1042
1,Fed_Policy_Uncertainty,0.6456
2,Fed_Surprise_Signed,-0.0881


,Variable,Meeting_Month_Mean_After_Centering,Correlation_With_Meeting_Flag
0,Fed_Expected_Direction_Centered,0.0,0.0
1,Fed_Policy_Uncertainty_Centered,0.0,0.0
2,Fed_Surprise_Signed_Centered,0.0,0.0


In [13]:
# CELL 12: Create stationary variables for the Fed-augmented ADL analysis

fed_adl_df = gold_analysis[["Month"]].copy()

# ---------------------------------------------------------
# Dependent variable and demand channels
# ---------------------------------------------------------

fed_adl_df["Gold_Return"] = (
    gold_analysis["Gold_Monthly_Return_pct"]
)

fed_adl_df["ETF_Flow"] = (
    gold_analysis["ETF_Net_Demand_Global_WGC_tonnes"]
)

fed_adl_df["CB_IMF_Change"] = (
    gold_analysis[
        "CB_Reported_Holdings_Change_Global_IMF_tonnes"
    ]
)

# ---------------------------------------------------------
# Primary Fed variables
# ---------------------------------------------------------

fed_adl_df["Fed_Direction_Centered"] = (
    gold_analysis["Fed_Expected_Direction_Centered"]
)

fed_adl_df["Fed_Uncertainty_Centered"] = (
    gold_analysis["Fed_Policy_Uncertainty_Centered"]
)

fed_adl_df["Fed_Surprise_Centered"] = (
    gold_analysis["Fed_Surprise_Signed_Centered"]
)

fed_adl_df["Fed_Meeting_Flag"] = (
    gold_analysis["Fed_Meeting_Month"]
)

fed_adl_df["Fed_Multiple_Meeting_Flag"] = (
    gold_analysis["Fed_Multiple_Meeting_Month"]
)

# ---------------------------------------------------------
# Structural controls
# ---------------------------------------------------------

fed_adl_df["Post_June_2020_Dummy"] = (
    gold_analysis["Post_June_2020_Dummy"]
)

fed_adl_df["COVID_Shock_Dummy"] = (
    gold_analysis["COVID_Shock_Dummy"]
)

# ---------------------------------------------------------
# Core macro-financial controls
# ---------------------------------------------------------

# CPI is already expressed as a monthly inflation rate.
fed_adl_df["CPI_MoM"] = (
    gold_analysis["US_CPI_Inflation_MoM_pct"]
)

# Change in broad USD index.
fed_adl_df["Broad_USD_Change"] = (
    gold_analysis["Broad_USD_Index_Avg"]
    .diff()
)

# Prefer the existing source-calculated real-yield change.
fed_adl_df["Real10Y_Change_pp"] = (
    gold_analysis[
        "US_10Y_Real_Yield_Monthly_Change_pp"
    ]
)

# Existing monthly VIX change in index points.
fed_adl_df["VIX_Change"] = (
    gold_analysis["VIX_Monthly_Change_points"]
)

# High-yield credit-spread change.
fed_adl_df["HY_OAS_Change_pp"] = (
    gold_analysis[
        "US_High_Yield_OAS_Monthly_Change_pp"
    ]
)

# Brent monthly percentage return.
fed_adl_df["Brent_Return_pct"] = (
    gold_analysis[
        "Brent_MonthEnd_USD_per_Barrel"
    ].pct_change(fill_method=None) * 100
)

# ---------------------------------------------------------
# Robustness-only Fed transformations
# ---------------------------------------------------------

fed_adl_df["Delta_Fed_Direction"] = (
    gold_analysis["Delta_Fed_Expected_Direction"]
)

fed_adl_df["Delta_Fed_Uncertainty"] = (
    gold_analysis["Delta_Fed_Policy_Uncertainty"]
)

fed_adl_df["Fed_Unexpected_Probability"] = (
    gold_analysis["Fed_Unexpected_Action_Probability"]
)

# Sort chronologically.
fed_adl_df = (
    fed_adl_df
    .sort_values("Month")
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Availability audit
# ---------------------------------------------------------

availability = pd.DataFrame({
    "Variable": fed_adl_df.columns.drop("Month"),
    "Available": [
        fed_adl_df[column].notna().sum()
        for column in fed_adl_df.columns
        if column != "Month"
    ],
    "Missing": [
        fed_adl_df[column].isna().sum()
        for column in fed_adl_df.columns
        if column != "Month"
    ],
})

availability["Missing_Percent"] = (
    availability["Missing"]
    / len(fed_adl_df)
    * 100
)

display(availability.round(4))

print("Dataset observations:", len(fed_adl_df))
print(
    "Period:",
    fed_adl_df["Month"].min(),
    "to",
    fed_adl_df["Month"].max()
)

display(fed_adl_df.head())

,Variable,Available,Missing,Missing_Percent
0,Gold_Return,91,0,0.0000
1,ETF_Flow,91,0,0.0000
2,CB_IMF_Change,91,0,0.0000
3,Fed_Direction_Centered,91,0,0.0000
4,Fed_Uncertainty_Centered,91,0,0.0000
5,Fed_Surprise_Centered,91,0,0.0000
6,Fed_Meeting_Flag,91,0,0.0000
7,Fed_Multiple_Meeting_Flag,91,0,0.0000
8,Post_June_2020_Dummy,91,0,0.0000
9,COVID_Shock_Dummy,91,0,0.0000


Dataset observations: 91
Period: 2019-01-01 00:00:00 to 2026-07-01 00:00:00


,Month,Gold_Return,ETF_Flow,CB_IMF_Change,Fed_Direction_Centered,Fed_Uncertainty_Centered,Fed_Surprise_Centered,Fed_Meeting_Flag,Fed_Multiple_Meeting_Flag,Post_June_2020_Dummy,COVID_Shock_Dummy,CPI_MoM,Broad_USD_Change,Real10Y_Change_pp,VIX_Change,HY_OAS_Change_pp,Brent_Return_pct,Delta_Fed_Direction,Delta_Fed_Uncertainty,Fed_Unexpected_Probability
0,2019-01-01,3.021073,71.8,68.164924,0.274569,0.103703,-0.290698,1,0,0,0,-0.081498,NaN,-0.20,-8.85,-0.96,NaN,NaN,NaN,0.457145
1,2019-02-01,-0.311335,-32.4,18.236201,0.000000,0.000000,0.000000,0,0,0,0,0.300126,-1.066922,0.00,-1.79,-0.45,6.689287,0.000000,0.000000,0.000000
2,2019-03-01,-0.262267,1.7,46.992204,-0.028483,-0.001029,0.012354,1,0,0,0,0.378179,11.811814,-0.25,-1.07,0.13,3.574134,-0.303052,-0.104732,0.241493
3,2019-04-01,-0.434646,-56.9,59.124482,0.000000,0.000000,0.000000,0,0,0,0,0.375968,0.120235,0.03,-0.59,-0.32,6.448317,0.000000,0.000000,0.000000
4,2019-05-01,1.904169,-2.4,43.595762,-0.056941,0.045578,0.040812,1,0,0,0,0.024683,-3.987998,-0.16,5.59,0.86,-11.414842,-0.028458,0.046607,0.265951


In [14]:
# CELL 13: Replace the corrupted broad USD series using official FRED data

FRED_BROAD_USD_URL = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    "?id=DTWEXBGS"
)

usd_daily = pd.read_csv(FRED_BROAD_USD_URL)

usd_daily.columns = [
    "Date",
    "Broad_USD_Index"
]

usd_daily["Date"] = pd.to_datetime(
    usd_daily["Date"],
    errors="coerce"
)

usd_daily["Broad_USD_Index"] = pd.to_numeric(
    usd_daily["Broad_USD_Index"],
    errors="coerce"
)

usd_daily = (
    usd_daily
    .dropna(subset=["Date", "Broad_USD_Index"])
    .sort_values("Date")
    .reset_index(drop=True)
)

# Convert the official daily series into monthly averages.
usd_monthly_corrected = (
    usd_daily
    .assign(
        Month=lambda frame:
            frame["Date"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby("Month", as_index=False)
    .agg(
        Broad_USD_Index_Avg_Corrected=(
            "Broad_USD_Index",
            "mean"
        ),
        Broad_USD_Available_Days=(
            "Broad_USD_Index",
            "count"
        ),
    )
)

# Restrict the official series to the gold-analysis period.
usd_monthly_corrected = usd_monthly_corrected.loc[
    usd_monthly_corrected["Month"].between(
        gold_analysis["Month"].min(),
        gold_analysis["Month"].max()
    )
].copy()

# Merge the corrected series.
gold_analysis = gold_analysis.merge(
    usd_monthly_corrected,
    on="Month",
    how="left",
    validate="one_to_one"
)

# Compare the workbook values against official FRED values.
usd_comparison = gold_analysis[
    [
        "Month",
        "Broad_USD_Index_Avg",
        "Broad_USD_Index_Avg_Corrected",
        "Broad_USD_Available_Days",
    ]
].copy()

usd_comparison["Difference"] = (
    usd_comparison["Broad_USD_Index_Avg"]
    - usd_comparison["Broad_USD_Index_Avg_Corrected"]
)

usd_comparison["Absolute_Difference"] = (
    usd_comparison["Difference"].abs()
)

print(
    "Official FRED monthly observations:",
    usd_comparison["Broad_USD_Index_Avg_Corrected"]
    .notna()
    .sum()
)

print(
    "Months with absolute discrepancy above 0.01:",
    usd_comparison["Absolute_Difference"]
    .gt(0.01)
    .sum()
)

display(
    usd_comparison.sort_values(
        "Absolute_Difference",
        ascending=False
    ).head(20)
)

# Create the corrected stationary monthly change.
gold_analysis["Broad_USD_Change_Corrected"] = (
    gold_analysis[
        "Broad_USD_Index_Avg_Corrected"
    ].diff()
)

# Replace the incorrect ADL variable.
fed_adl_df["Broad_USD_Change"] = (
    gold_analysis["Broad_USD_Change_Corrected"]
)

display(
    gold_analysis[
        [
            "Month",
            "Broad_USD_Index_Avg",
            "Broad_USD_Index_Avg_Corrected",
            "Broad_USD_Change_Corrected",
        ]
    ].head(12)
)

print(
    "\nLargest corrected monthly USD changes:"
)

display(
    gold_analysis[
        [
            "Month",
            "Broad_USD_Index_Avg_Corrected",
            "Broad_USD_Change_Corrected",
        ]
    ]
    .dropna()
    .sort_values(
        "Broad_USD_Change_Corrected",
        key=np.abs,
        ascending=False
    )
    .head(10)
)

Official FRED monthly observations: 91
Months with absolute discrepancy above 0.01: 87


,Month,Broad_USD_Index_Avg,Broad_USD_Index_Avg_Corrected,Broad_USD_Available_Days,Difference,Absolute_Difference
1,2019-02-01,102.971705,114.413006,18,-11.441301,11.441301
10,2019-11-01,105.458567,116.559468,19,-11.100902,11.100902
11,2019-12-01,105.372373,115.909610,20,-10.537237,10.537237
0,2019-01-01,104.038627,114.442490,20,-10.403863,10.403863
8,2019-09-01,111.737067,117.323920,20,-5.586853,5.586853
90,2026-07-01,115.353696,120.597045,22,-5.243350,5.243350
9,2019-10-01,111.670335,116.746259,22,-5.075924,5.075924
4,2019-05-01,110.915757,115.957382,22,-5.041625,5.041625
6,2019-07-01,110.074217,115.077591,22,-5.003374,5.003374
39,2022-04-01,119.493300,117.168795,21,2.324505,2.324505


,Month,Broad_USD_Index_Avg,Broad_USD_Index_Avg_Corrected,Broad_USD_Change_Corrected
0,2019-01-01,104.038627,114.442490,NaN
1,2019-02-01,102.971705,114.413006,-0.029484
2,2019-03-01,114.783519,114.783519,0.370513
3,2019-04-01,114.903755,114.903755,0.120235
4,2019-05-01,110.915757,115.957382,1.053627
5,2019-06-01,115.427070,115.427070,-0.530312
6,2019-07-01,110.074217,115.077591,-0.349479
7,2019-08-01,117.100386,117.100386,2.022795
8,2019-09-01,111.737067,117.323920,0.223534
9,2019-10-01,111.670335,116.746259,-0.577661



Largest corrected monthly USD changes:


,Month,Broad_USD_Index_Avg_Corrected,Broad_USD_Change_Corrected
14,2020-03-01,121.022741,4.312199
44,2022-09-01,125.594600,3.276848
17,2020-06-01,119.676605,-2.938405
42,2022-07-01,122.751200,2.725490
70,2024-11-01,126.311337,2.716046
46,2022-11-01,124.852720,-2.627885
47,2022-12-01,122.234329,-2.618391
40,2022-05-01,119.696190,2.527395
48,2023-01-01,119.810065,-2.424264
58,2023-11-01,121.222120,-2.275028


In [15]:
# CELL 14: Cross-validate core controls against official FRED series

FRED_SERIES = {
    "CPIAUCSL": "CPI_Index",
    "DFII10": "Real10Y",
    "VIXCLS": "VIX",
    "BAMLH0A0HYM2": "HY_OAS",
}

def download_fred_series(series_id, value_name):
    url = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id={series_id}"
    )

    data = pd.read_csv(url)

    data.columns = ["Date", value_name]

    data["Date"] = pd.to_datetime(
        data["Date"],
        errors="coerce"
    )

    data[value_name] = pd.to_numeric(
        data[value_name],
        errors="coerce"
    )

    return (
        data
        .dropna(subset=["Date", value_name])
        .sort_values("Date")
        .reset_index(drop=True)
    )

# ---------------------------------------------------------
# 1. CPI: monthly seasonally adjusted index
# ---------------------------------------------------------

cpi = download_fred_series(
    "CPIAUCSL",
    "CPI_Index"
)

cpi["Month"] = (
    cpi["Date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

cpi_monthly = (
    cpi
    .groupby("Month", as_index=False)
    .agg(CPI_Index=("CPI_Index", "last"))
)

cpi_monthly["CPI_MoM_Corrected"] = (
    cpi_monthly["CPI_Index"]
    .pct_change(fill_method=None)
    * 100
)

# ---------------------------------------------------------
# 2. Daily series: monthly averages, month-end observations
# ---------------------------------------------------------

def create_daily_monthly_table(
    data,
    value_column,
    prefix
):
    monthly = (
        data
        .assign(
            Month=lambda frame:
                frame["Date"]
                .dt.to_period("M")
                .dt.to_timestamp()
        )
        .groupby("Month", as_index=False)
        .agg(
            **{
                f"{prefix}_Monthly_Avg_Corrected":
                    (value_column, "mean"),

                f"{prefix}_MonthEnd_Corrected":
                    (value_column, "last"),

                f"{prefix}_Available_Days":
                    (value_column, "count"),
            }
        )
    )

    monthly[f"{prefix}_Monthly_Change_Corrected"] = (
        monthly[f"{prefix}_MonthEnd_Corrected"]
        .diff()
    )

    return monthly

real10y = create_daily_monthly_table(
    download_fred_series("DFII10", "Real10Y"),
    "Real10Y",
    "Real10Y"
)

vix = create_daily_monthly_table(
    download_fred_series("VIXCLS", "VIX"),
    "VIX",
    "VIX"
)

hy_oas = create_daily_monthly_table(
    download_fred_series(
        "BAMLH0A0HYM2",
        "HY_OAS"
    ),
    "HY_OAS",
    "HY_OAS"
)

# ---------------------------------------------------------
# Merge official values into one validation table
# ---------------------------------------------------------

official_controls = (
    gold_analysis[["Month"]]
    .merge(
        cpi_monthly[
            ["Month", "CPI_MoM_Corrected"]
        ],
        on="Month",
        how="left"
    )
    .merge(
        real10y[
            [
                "Month",
                "Real10Y_MonthEnd_Corrected",
                "Real10Y_Monthly_Change_Corrected",
            ]
        ],
        on="Month",
        how="left"
    )
    .merge(
        vix[
            [
                "Month",
                "VIX_MonthEnd_Corrected",
                "VIX_Monthly_Change_Corrected",
            ]
        ],
        on="Month",
        how="left"
    )
    .merge(
        hy_oas[
            [
                "Month",
                "HY_OAS_MonthEnd_Corrected",
                "HY_OAS_Monthly_Change_Corrected",
            ]
        ],
        on="Month",
        how="left"
    )
)

macro_validation = gold_analysis[
    [
        "Month",
        "US_CPI_Inflation_MoM_pct",
        "US_10Y_Real_Yield_MonthEnd_pct",
        "US_10Y_Real_Yield_Monthly_Change_pp",
        "VIX_MonthEnd",
        "VIX_Monthly_Change_points",
        "US_High_Yield_OAS_MonthEnd_pct",
        "US_High_Yield_OAS_Monthly_Change_pp",
    ]
].merge(
    official_controls,
    on="Month",
    how="left",
    validate="one_to_one"
)

comparison_pairs = {
    "CPI_MoM": (
        "US_CPI_Inflation_MoM_pct",
        "CPI_MoM_Corrected"
    ),

    "Real10Y_MonthEnd": (
        "US_10Y_Real_Yield_MonthEnd_pct",
        "Real10Y_MonthEnd_Corrected"
    ),

    "Real10Y_Change": (
        "US_10Y_Real_Yield_Monthly_Change_pp",
        "Real10Y_Monthly_Change_Corrected"
    ),

    "VIX_MonthEnd": (
        "VIX_MonthEnd",
        "VIX_MonthEnd_Corrected"
    ),

    "VIX_Change": (
        "VIX_Monthly_Change_points",
        "VIX_Monthly_Change_Corrected"
    ),

    "HY_OAS_MonthEnd": (
        "US_High_Yield_OAS_MonthEnd_pct",
        "HY_OAS_MonthEnd_Corrected"
    ),

    "HY_OAS_Change": (
        "US_High_Yield_OAS_Monthly_Change_pp",
        "HY_OAS_Monthly_Change_Corrected"
    ),
}

audit_rows = []

for variable, (
    workbook_column,
    corrected_column
) in comparison_pairs.items():

    difference = (
        macro_validation[workbook_column]
        - macro_validation[corrected_column]
    )

    valid_difference = difference.dropna()

    audit_rows.append({
        "Variable": variable,
        "Comparable_Observations":
            valid_difference.size,

        "Maximum_Absolute_Difference":
            valid_difference.abs().max(),

        "Mean_Absolute_Difference":
            valid_difference.abs().mean(),

        "Months_Above_0_01":
            valid_difference.abs().gt(0.01).sum(),

        "Correlation":
            macro_validation[
                [workbook_column, corrected_column]
            ].corr().iloc[0, 1],
    })

macro_audit_summary = pd.DataFrame(audit_rows)

display(macro_audit_summary.round(6))

,Variable,Comparable_Observations,Maximum_Absolute_Difference,Mean_Absolute_Difference,Months_Above_0_01,Correlation
0,CPI_MoM,90,0.380523,0.144903,73,0.864815
1,Real10Y_MonthEnd,91,0.000000,0.000000,0,1.000000
2,Real10Y_Change,91,0.000000,0.000000,0,1.000000
3,VIX_MonthEnd,91,0.000000,0.000000,0,1.000000
4,VIX_Change,91,0.000000,0.000000,0,1.000000
5,HY_OAS_MonthEnd,35,0.000000,0.000000,0,1.000000
6,HY_OAS_Change,34,0.000000,0.000000,0,1.000000


In [16]:
# CELL 15: Compare the workbook CPI with SA and NSA official CPI definitions

# Seasonally adjusted CPI: CPIAUCSL
cpi_sa = download_fred_series(
    "CPIAUCSL",
    "CPI_SA_Index"
)

cpi_sa["Month"] = (
    cpi_sa["Date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

cpi_sa = (
    cpi_sa
    .groupby("Month", as_index=False)
    .agg(CPI_SA_Index=("CPI_SA_Index", "last"))
)

cpi_sa["CPI_SA_MoM_pct"] = (
    cpi_sa["CPI_SA_Index"]
    .pct_change(fill_method=None)
    * 100
)

# Non-seasonally adjusted CPI: CPIAUCNS
cpi_nsa = download_fred_series(
    "CPIAUCNS",
    "CPI_NSA_Index"
)

cpi_nsa["Month"] = (
    cpi_nsa["Date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

cpi_nsa = (
    cpi_nsa
    .groupby("Month", as_index=False)
    .agg(CPI_NSA_Index=("CPI_NSA_Index", "last"))
)

cpi_nsa["CPI_NSA_MoM_pct"] = (
    cpi_nsa["CPI_NSA_Index"]
    .pct_change(fill_method=None)
    * 100
)

cpi_definition_check = (
    gold_analysis[
        [
            "Month",
            "US_CPI_Inflation_MoM_pct",
        ]
    ]
    .merge(
        cpi_sa[
            [
                "Month",
                "CPI_SA_MoM_pct",
            ]
        ],
        on="Month",
        how="left",
        validate="one_to_one"
    )
    .merge(
        cpi_nsa[
            [
                "Month",
                "CPI_NSA_MoM_pct",
            ]
        ],
        on="Month",
        how="left",
        validate="one_to_one"
    )
)

cpi_definition_check["Error_vs_SA"] = (
    cpi_definition_check["US_CPI_Inflation_MoM_pct"]
    - cpi_definition_check["CPI_SA_MoM_pct"]
)

cpi_definition_check["Error_vs_NSA"] = (
    cpi_definition_check["US_CPI_Inflation_MoM_pct"]
    - cpi_definition_check["CPI_NSA_MoM_pct"]
)

def comparison_statistics(
    frame,
    workbook_column,
    candidate_column,
    label
):
    valid = frame[
        [workbook_column, candidate_column]
    ].dropna()

    error = (
        valid[workbook_column]
        - valid[candidate_column]
    )

    return {
        "Candidate_Definition": label,
        "N": len(valid),
        "Correlation":
            valid.corr().iloc[0, 1],
        "Mean_Absolute_Error":
            error.abs().mean(),
        "Root_Mean_Squared_Error":
            np.sqrt(np.mean(error ** 2)),
        "Maximum_Absolute_Error":
            error.abs().max(),
        "Months_Above_0_01":
            error.abs().gt(0.01).sum(),
    }

cpi_candidate_summary = pd.DataFrame([
    comparison_statistics(
        cpi_definition_check,
        "US_CPI_Inflation_MoM_pct",
        "CPI_SA_MoM_pct",
        "Seasonally adjusted CPIAUCSL"
    ),

    comparison_statistics(
        cpi_definition_check,
        "US_CPI_Inflation_MoM_pct",
        "CPI_NSA_MoM_pct",
        "Non-seasonally adjusted CPIAUCNS"
    ),
])

display(cpi_candidate_summary.round(6))

print("\nFirst 18 monthly observations:")

display(
    cpi_definition_check[
        [
            "Month",
            "US_CPI_Inflation_MoM_pct",
            "CPI_SA_MoM_pct",
            "CPI_NSA_MoM_pct",
            "Error_vs_SA",
            "Error_vs_NSA",
        ]
    ].head(18).round(6)
)

print("\nLargest discrepancies against seasonally adjusted CPI:")

display(
    cpi_definition_check[
        [
            "Month",
            "US_CPI_Inflation_MoM_pct",
            "CPI_SA_MoM_pct",
            "CPI_NSA_MoM_pct",
            "Error_vs_SA",
        ]
    ]
    .sort_values(
        "Error_vs_SA",
        key=np.abs,
        ascending=False
    )
    .head(15)
    .round(6)
)

,Candidate_Definition,N,Correlation,Mean_Absolute_Error,Root_Mean_Squared_Error,Maximum_Absolute_Error,Months_Above_0_01
0,Seasonally adjusted CPIAUCSL,90,0.864815,0.144903,0.180607,0.380523,73
1,Non-seasonally adjusted CPIAUCNS,90,0.975524,0.033173,0.080201,0.382199,20



First 18 monthly observations:


,Month,US_CPI_Inflation_MoM_pct,CPI_SA_MoM_pct,CPI_NSA_MoM_pct,Error_vs_SA,Error_vs_NSA
0,2019-01-01,-0.081498,-0.081498,0.190660,0.000000,-0.272158
1,2019-02-01,0.300126,0.300126,0.422705,0.000000,-0.122580
2,2019-03-01,0.378179,0.378179,0.564136,0.000000,-0.185957
3,2019-04-01,0.375968,0.375968,0.529500,0.000000,-0.153532
4,2019-05-01,0.024683,0.024683,0.212876,0.000000,-0.188193
5,2019-06-01,-0.032511,-0.032511,0.019915,0.000000,-0.052426
6,2019-07-01,0.230788,0.230788,0.167094,0.000000,0.063693
7,2019-08-01,0.091477,0.091477,-0.005067,0.000000,0.096544
8,2019-09-01,0.153885,0.153885,0.078345,0.000000,0.075540
9,2019-10-01,0.282728,0.282728,0.228619,0.000000,0.054109



Largest discrepancies against seasonally adjusted CPI:


,Month,US_CPI_Inflation_MoM_pct,CPI_SA_MoM_pct,CPI_NSA_MoM_pct,Error_vs_SA
35,2021-12-01,0.31,0.690523,0.307252,-0.380523
46,2022-11-01,-0.11,0.261403,-0.101003,-0.371403
58,2023-11-01,-0.21,0.146898,-0.201514,-0.356898
34,2021-11-01,0.50,0.856626,0.491343,-0.356626
70,2024-11-01,-0.06,0.284193,-0.054172,-0.344193
23,2020-12-01,0.10,0.434631,0.094148,-0.334631
83,2025-12-01,-0.03,0.297788,-0.020980,-0.327788
47,2022-12-01,-0.31,0.015396,-0.307009,-0.325396
59,2023-12-01,-0.11,0.192440,-0.099332,-0.302440
71,2024-12-01,0.04,0.339938,0.035500,-0.299938


In [17]:
# CELL 16: Install the consistent seasonally adjusted CPI series

cpi_corrected = cpi_sa[
    [
        "Month",
        "CPI_SA_MoM_pct",
    ]
].rename(
    columns={
        "CPI_SA_MoM_pct":
            "CPI_MoM_SA_Corrected"
    }
)

# Avoid duplicate columns if this cell is rerun.
gold_analysis = gold_analysis.drop(
    columns=["CPI_MoM_SA_Corrected"],
    errors="ignore"
)

gold_analysis = gold_analysis.merge(
    cpi_corrected,
    on="Month",
    how="left",
    validate="one_to_one"
)

# Preserve the mixed workbook series but exclude it from modelling.
gold_analysis[
    "CPI_MoM_Original_Mixed_Definition"
] = gold_analysis[
    "US_CPI_Inflation_MoM_pct"
]

# Replace the CPI variable in the model-ready dataframe.
fed_adl_df["CPI_MoM"] = (
    gold_analysis["CPI_MoM_SA_Corrected"]
)

# Ensure the corrected broad USD variable remains installed.
fed_adl_df["Broad_USD_Change"] = (
    gold_analysis["Broad_USD_Change_Corrected"]
)

# Identify any unavailable corrected CPI months.
missing_corrected_cpi = gold_analysis.loc[
    gold_analysis["CPI_MoM_SA_Corrected"].isna(),
    "Month"
]

print(
    "Corrected CPI observations:",
    gold_analysis["CPI_MoM_SA_Corrected"]
    .notna()
    .sum()
)

print(
    "Missing corrected CPI months:",
    missing_corrected_cpi
    .dt.strftime("%Y-%m")
    .tolist()
)

# Final CPI comparison.
display(
    gold_analysis[
        [
            "Month",
            "CPI_MoM_Original_Mixed_Definition",
            "CPI_MoM_SA_Corrected",
        ]
    ].head(18).round(6)
)

# Retest the corrected CPI series.
corrected_cpi_stationarity = pd.DataFrame([
    stationarity_test(
        gold_analysis["CPI_MoM_SA_Corrected"],
        "CPI_MoM_SA_Corrected"
    )
])

corrected_cpi_stationarity["ADF_p_Display"] = (
    corrected_cpi_stationarity["ADF_p"]
    .map(
        lambda value:
            "< 0.0001"
            if value < 0.0001
            else f"{value:.4f}"
    )
)

corrected_cpi_stationarity["KPSS_p_Display"] = (
    corrected_cpi_stationarity["KPSS_p"]
    .map(
        lambda value:
            "> 0.10"
            if value >= 0.10
            else "< 0.01"
            if value <= 0.01
            else f"{value:.4f}"
    )
)

display(
    corrected_cpi_stationarity[
        [
            "Variable",
            "N",
            "ADF_Statistic",
            "ADF_p_Display",
            "ADF_Lag",
            "KPSS_Statistic",
            "KPSS_p_Display",
            "Assessment",
        ]
    ].round(4)
)

# Confirm which versions are authorised for modelling.
data_version_table = pd.DataFrame({
    "Model_Variable": [
        "CPI_MoM",
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
        "HY_OAS_Change_pp",
    ],
    "Approved_Source": [
        "FRED CPIAUCSL, seasonally adjusted MoM change",
        "FRED DTWEXBGS, monthly-average change",
        "FRED DFII10, month-end change",
        "FRED VIXCLS, month-end change",
        "FRED/archived BAMLH0A0HYM2, month-end change",
    ],
    "Original_Workbook_Status": [
        "Rejected: mixed SA/NSA definition",
        "Rejected: corrupted monthly levels",
        "Verified",
        "Verified",
        "Verified where overlapping data are available",
    ],
})

display(data_version_table)

Corrected CPI observations: 90
Missing corrected CPI months: ['2025-10']


,Month,CPI_MoM_Original_Mixed_Definition,CPI_MoM_SA_Corrected
0,2019-01-01,-0.081498,-0.081498
1,2019-02-01,0.300126,0.300126
2,2019-03-01,0.378179,0.378179
3,2019-04-01,0.375968,0.375968
4,2019-05-01,0.024683,0.024683
5,2019-06-01,-0.032511,-0.032511
6,2019-07-01,0.230788,0.230788
7,2019-08-01,0.091477,0.091477
8,2019-09-01,0.153885,0.153885
9,2019-10-01,0.282728,0.282728


,Variable,N,ADF_Statistic,ADF_p_Display,ADF_Lag,KPSS_Statistic,KPSS_p_Display,Assessment
0,CPI_MoM_SA_Corrected,90,-3.5649,0.0065,2,0.2094,> 0.10,Stationary


,Model_Variable,Approved_Source,Original_Workbook_Status
0,CPI_MoM,"FRED CPIAUCSL, seasonally adjusted MoM change",Rejected: mixed SA/NSA definition
1,Broad_USD_Change,"FRED DTWEXBGS, monthly-average change",Rejected: corrupted monthly levels
2,Real10Y_Change_pp,"FRED DFII10, month-end change",Verified
3,VIX_Change,"FRED VIXCLS, month-end change",Verified
4,HY_OAS_Change_pp,"FRED/archived BAMLH0A0HYM2, month-end change",Verified where overlapping data are available


In [18]:
# CELL 17: Construct the Fed-augmented ADL(1,2,1) sample
# and diagnose multicollinearity

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)
from sklearn.preprocessing import StandardScaler

# Work on a fresh, chronologically ordered copy.
fed_adl_model = (
    fed_adl_df
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

# Confirm that the corrected variables are installed.
fed_adl_model["CPI_MoM"] = (
    gold_analysis["CPI_MoM_SA_Corrected"].to_numpy()
)

fed_adl_model["Broad_USD_Change"] = (
    gold_analysis["Broad_USD_Change_Corrected"].to_numpy()
)

# -----------------------------------------------------------
# ADL(1,2,1) dynamic structure
# p = 1: one lag of gold return
# q = 2: two lags of ETF flow
# r = 1: one lag of IMF central-bank change
# -----------------------------------------------------------

fed_adl_model["Gold_Return_L1"] = (
    fed_adl_model["Gold_Return"].shift(1)
)

fed_adl_model["ETF_Flow_L1"] = (
    fed_adl_model["ETF_Flow"].shift(1)
)

fed_adl_model["ETF_Flow_L2"] = (
    fed_adl_model["ETF_Flow"].shift(2)
)

fed_adl_model["CB_IMF_Change_L1"] = (
    fed_adl_model["CB_IMF_Change"].shift(1)
)

# Rename the centered Fed signals more compactly.
fed_adl_model["Fed_Direction_Centered"] = (
    fed_adl_model["Fed_Direction_Centered"]
)

fed_adl_model["Fed_Uncertainty_Centered"] = (
    fed_adl_model["Fed_Uncertainty_Centered"]
)

fed_adl_model["Fed_Surprise_Centered"] = (
    fed_adl_model["Fed_Surprise_Centered"]
)

# Core ADL demand-channel regressors.
dynamic_regressors = [
    "Gold_Return_L1",
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
    "CB_IMF_Change_L1",
]

# Primary Fed-information regressors.
fed_regressors = [
    "Fed_Meeting_Flag",
    "Fed_Direction_Centered",
    "Fed_Uncertainty_Centered",
    "Fed_Surprise_Centered",
]

# Macro-financial controls retained for the nested comparison.
macro_controls = [
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "VIX_Change",
    "CPI_MoM",
    "HY_OAS_Change_pp",
    "Brent_Return_pct",
]

# Structural-break variable suggested by the uncertainty tests.
structural_controls = [
    "Post_June_2020_Dummy",
]

full_regressors = (
    dynamic_regressors
    + fed_regressors
    + macro_controls
    + structural_controls
)

required_columns = [
    "Month",
    "Gold_Return",
] + full_regressors

missing_columns = [
    column
    for column in required_columns
    if column not in fed_adl_model.columns
]

if missing_columns:
    raise KeyError(
        f"Required model columns are missing: {missing_columns}"
    )

# One common sample for every subsequent nested specification.
# October 2025 is omitted because corrected CPI is unavailable.
fed_adl_common = (
    fed_adl_model[required_columns]
    .dropna()
    .reset_index(drop=True)
)

print(
    "Original monthly observations:",
    len(fed_adl_model)
)

print(
    "Common-sample observations:",
    len(fed_adl_common)
)

print(
    "Common-sample period:",
    fed_adl_common["Month"].min(),
    "to",
    fed_adl_common["Month"].max()
)

excluded_months = (
    fed_adl_model.loc[
        ~fed_adl_model["Month"].isin(
            fed_adl_common["Month"]
        ),
        "Month"
    ]
    .dt.strftime("%Y-%m")
    .tolist()
)

print("Months excluded from common sample:", excluded_months)

# -----------------------------------------------------------
# Correlation diagnostic
# -----------------------------------------------------------

X_diagnostic = fed_adl_common[
    full_regressors
].astype(float)

correlation_matrix = X_diagnostic.corr()

high_correlation_pairs = []

for i, variable_1 in enumerate(correlation_matrix.columns):
    for j in range(i + 1, len(correlation_matrix.columns)):
        variable_2 = correlation_matrix.columns[j]
        correlation = correlation_matrix.loc[
            variable_1,
            variable_2
        ]

        if abs(correlation) >= 0.70:
            high_correlation_pairs.append({
                "Variable_1": variable_1,
                "Variable_2": variable_2,
                "Correlation": correlation,
                "Absolute_Correlation": abs(correlation),
            })

high_correlation_table = pd.DataFrame(
    high_correlation_pairs
)

if not high_correlation_table.empty:
    high_correlation_table = (
        high_correlation_table
        .sort_values(
            "Absolute_Correlation",
            ascending=False
        )
        .reset_index(drop=True)
    )

print("\nHighly correlated pairs: |correlation| >= 0.70")
display(high_correlation_table.round(4))

# -----------------------------------------------------------
# Variance inflation factors
# -----------------------------------------------------------

X_vif = X_diagnostic.loc[
    :,
    X_diagnostic.std(ddof=0) > 0
].copy()

X_vif_standardized = pd.DataFrame(
    StandardScaler().fit_transform(X_vif),
    columns=X_vif.columns,
    index=X_vif.index,
)

vif_table = pd.DataFrame({
    "Variable": X_vif_standardized.columns,
    "VIF": [
        variance_inflation_factor(
            X_vif_standardized.values,
            position
        )
        for position in range(
            X_vif_standardized.shape[1]
        )
    ],
})

vif_table["Assessment"] = pd.cut(
    vif_table["VIF"],
    bins=[-np.inf, 5, 10, np.inf],
    labels=[
        "Acceptable",
        "Moderate",
        "High",
    ],
)

vif_table = (
    vif_table
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

# Condition number calculated after standardisation.
standardised_condition_number = np.linalg.cond(
    X_vif_standardized.to_numpy()
)

print(
    "\nStandardised condition number:",
    round(standardised_condition_number, 4)
)

display(vif_table.round(4))

# Plotly VIF chart.
fig = px.bar(
    vif_table.sort_values("VIF"),
    x="VIF",
    y="Variable",
    orientation="h",
    color="Assessment",
    color_discrete_map={
        "Acceptable": "#2ca02c",
        "Moderate": "#ffbf00",
        "High": "#d62728",
    },
    title=(
        "Multicollinearity Diagnostic for the "
        "Fed-Augmented ADL(1,2,1)"
    ),
)

fig.add_vline(
    x=5,
    line_dash="dash",
    line_color="orange",
    annotation_text="VIF = 5",
)

fig.add_vline(
    x=10,
    line_dash="dash",
    line_color="red",
    annotation_text="VIF = 10",
)

fig.update_layout(
    template="plotly_white",
    height=720,
    width=1050,
    xaxis_title="Variance Inflation Factor",
    yaxis_title="",
)

fig.show()

Original monthly observations: 91
Common-sample observations: 88
Common-sample period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00
Months excluded from common sample: ['2019-01', '2019-02', '2025-10']

Highly correlated pairs: |correlation| >= 0.70


""



Standardised condition number: 3.8578


,Variable,VIF,Assessment
0,ETF_Flow_L1,3.1001,Acceptable
1,HY_OAS_Change_pp,2.6054,Acceptable
2,ETF_Flow,2.1653,Acceptable
3,VIX_Change,2.1547,Acceptable
4,ETF_Flow_L2,1.8400,Acceptable
5,Brent_Return_pct,1.7705,Acceptable
6,Gold_Return_L1,1.6138,Acceptable
7,Fed_Direction_Centered,1.5456,Acceptable
8,Post_June_2020_Dummy,1.5419,Acceptable
9,Real10Y_Change_pp,1.4988,Acceptable


NameError: name 'px' is not defined

In [19]:
# CELL 17A: Import Plotly Express and display the VIF chart

import plotly.express as px

fig = px.bar(
    vif_table.sort_values("VIF"),
    x="VIF",
    y="Variable",
    orientation="h",
    color="Assessment",
    color_discrete_map={
        "Acceptable": "#2ca02c",
        "Moderate": "#ffbf00",
        "High": "#d62728",
    },
    title=(
        "Multicollinearity Diagnostic for the "
        "Fed-Augmented ADL(1,2,1)"
    ),
)

fig.add_vline(
    x=5,
    line_dash="dash",
    line_color="orange",
    annotation_text="VIF = 5",
    annotation_position="top",
)

fig.add_vline(
    x=10,
    line_dash="dash",
    line_color="red",
    annotation_text="VIF = 10",
    annotation_position="top",
)

fig.update_layout(
    template="plotly_white",
    height=720,
    width=1050,
    xaxis_title="Variance Inflation Factor",
    yaxis_title="",
    showlegend=True,
)

fig.show()

In [20]:
# CELL 18: Estimate nested Fed-augmented ADL(1,2,1) models

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------------------------------------
# Fixed dynamic demand-channel structure
# -----------------------------------------------------------

demand_regressors = [
    "Gold_Return_L1",
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
    "CB_IMF_Change_L1",
]

# -----------------------------------------------------------
# Nested specifications
# -----------------------------------------------------------

fed_model_specifications = {
    # Demand-channel benchmark
    "F0_Demand_Only": demand_regressors,

    # Average effect of an FOMC meeting month
    "F1_Meeting": demand_regressors + [
        "Fed_Meeting_Flag",
    ],

    # Add expected hawkish/dovish direction
    "F2_Direction": demand_regressors + [
        "Fed_Meeting_Flag",
        "Fed_Direction_Centered",
    ],

    # Add pre-meeting policy uncertainty
    "F3_Uncertainty": demand_regressors + [
        "Fed_Meeting_Flag",
        "Fed_Direction_Centered",
        "Fed_Uncertainty_Centered",
    ],

    # Complete primary Fed-information block
    "F4_Fed_Complete": demand_regressors + [
        "Fed_Meeting_Flag",
        "Fed_Direction_Centered",
        "Fed_Uncertainty_Centered",
        "Fed_Surprise_Centered",
    ],

    # Add the three central macro-financial controls
    "F5_Core_Macro": demand_regressors + [
        "Fed_Meeting_Flag",
        "Fed_Direction_Centered",
        "Fed_Uncertainty_Centered",
        "Fed_Surprise_Centered",
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
    ],

    # Add inflation and credit-market stress
    "F6_Expanded_Macro": demand_regressors + [
        "Fed_Meeting_Flag",
        "Fed_Direction_Centered",
        "Fed_Uncertainty_Centered",
        "Fed_Surprise_Centered",
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
        "CPI_MoM",
        "HY_OAS_Change_pp",
    ],

    # Complete model with oil and structural-break control
    "F7_Full": demand_regressors + [
        "Fed_Meeting_Flag",
        "Fed_Direction_Centered",
        "Fed_Uncertainty_Centered",
        "Fed_Surprise_Centered",
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
        "CPI_MoM",
        "HY_OAS_Change_pp",
        "Brent_Return_pct",
        "Post_June_2020_Dummy",
    ],
}

# -----------------------------------------------------------
# Helper functions
# -----------------------------------------------------------

def safe_wald_pvalue(result, restrictions):
    """
    HAC-robust Wald test with scalar output.
    """
    test = result.wald_test(
        restrictions,
        use_f=False,
        scalar=True,
    )
    return float(test.pvalue)


def cumulative_effect(result, variables):
    """
    Sum coefficients and test whether their sum equals zero.
    """
    restriction = " + ".join(variables) + " = 0"

    test = result.t_test(restriction)

    coefficient = float(
        np.asarray(test.effect).reshape(-1)[0]
    )

    p_value = float(
        np.asarray(test.pvalue).reshape(-1)[0]
    )

    return coefficient, p_value


# -----------------------------------------------------------
# Estimate every specification using the SAME observations
# -----------------------------------------------------------

fed_adl_results = {}
model_summary_rows = []
coefficient_rows = []

y = fed_adl_common["Gold_Return"].astype(float)

for model_name, regressors in fed_model_specifications.items():

    X = sm.add_constant(
        fed_adl_common[regressors].astype(float),
        has_constant="add",
    )

    ols_result = sm.OLS(y, X).fit()

    # Monthly time-series HAC inference.
    hac_result = ols_result.get_robustcov_results(
        cov_type="HAC",
        maxlags=3,
        use_correction=True,
    )

    # Restore labelled coefficient output.
    hac_result.model.data.xnames = X.columns.tolist()

    parameter_names = X.columns.tolist()

    coefficient_table = pd.DataFrame({
        "Variable": parameter_names,
        "Coefficient": hac_result.params,
        "HAC_SE": hac_result.bse,
        "HAC_t": hac_result.tvalues,
        "HAC_p": hac_result.pvalues,
        "CI_Lower_95pct": hac_result.conf_int()[:, 0],
        "CI_Upper_95pct": hac_result.conf_int()[:, 1],
    })

    coefficient_table["Model"] = model_name
    coefficient_table["Significant_5pct"] = (
        coefficient_table["HAC_p"] < 0.05
    )
    coefficient_table["Significant_10pct"] = (
        coefficient_table["HAC_p"] < 0.10
    )

    coefficient_rows.append(coefficient_table)

    # ETF channel tests
    etf_lag_p = safe_wald_pvalue(
        hac_result,
        [
            "ETF_Flow_L1 = 0",
            "ETF_Flow_L2 = 0",
        ],
    )

    complete_etf_p = safe_wald_pvalue(
        hac_result,
        [
            "ETF_Flow = 0",
            "ETF_Flow_L1 = 0",
            "ETF_Flow_L2 = 0",
        ],
    )

    cumulative_etf, cumulative_etf_p = cumulative_effect(
        hac_result,
        [
            "ETF_Flow",
            "ETF_Flow_L1",
            "ETF_Flow_L2",
        ],
    )

    # Central-bank channel tests
    cb_lag_p = float(
        hac_result.pvalues[
            parameter_names.index("CB_IMF_Change_L1")
        ]
    )

    complete_cb_p = safe_wald_pvalue(
        hac_result,
        [
            "CB_IMF_Change = 0",
            "CB_IMF_Change_L1 = 0",
        ],
    )

    cumulative_cb, cumulative_cb_p = cumulative_effect(
        hac_result,
        [
            "CB_IMF_Change",
            "CB_IMF_Change_L1",
        ],
    )

    # Fed block test—only over Fed variables in this model.
    possible_fed_variables = [
        "Fed_Meeting_Flag",
        "Fed_Direction_Centered",
        "Fed_Uncertainty_Centered",
        "Fed_Surprise_Centered",
    ]

    included_fed_variables = [
        variable
        for variable in possible_fed_variables
        if variable in regressors
    ]

    if included_fed_variables:
        fed_block_p = safe_wald_pvalue(
            hac_result,
            [
                f"{variable} = 0"
                for variable in included_fed_variables
            ],
        )
    else:
        fed_block_p = np.nan

    model_summary_rows.append({
        "Model": model_name,
        "Controls_Added": (
            len(regressors) - len(demand_regressors)
        ),
        "Parameters": int(hac_result.df_model + 1),
        "N": int(hac_result.nobs),
        "R2": hac_result.rsquared,
        "Adjusted_R2": hac_result.rsquared_adj,
        "AIC": hac_result.aic,
        "BIC": hac_result.bic,
        "ETF_Lag_Block_p": etf_lag_p,
        "Complete_ETF_Channel_p": complete_etf_p,
        "Cumulative_ETF_Coefficient": cumulative_etf,
        "Cumulative_ETF_p": cumulative_etf_p,
        "CB_L1_p": cb_lag_p,
        "Complete_CB_Channel_p": complete_cb_p,
        "Cumulative_CB_Coefficient": cumulative_cb,
        "Cumulative_CB_p": cumulative_cb_p,
        "Fed_Block_p": fed_block_p,
    })

    fed_adl_results[model_name] = {
        "OLS": ols_result,
        "HAC": hac_result,
        "Regressors": regressors,
        "Coefficient_Table": coefficient_table,
    }

# -----------------------------------------------------------
# Model-comparison results
# -----------------------------------------------------------

fed_adl_comparison = pd.DataFrame(model_summary_rows)

display(
    fed_adl_comparison.round(4)
)

# Combined coefficient table for later diagnostics.
fed_adl_coefficients = pd.concat(
    coefficient_rows,
    ignore_index=True,
)

# Focused coefficients across specifications.
focus_variables = [
    "Gold_Return_L1",
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
    "CB_IMF_Change_L1",
    "Fed_Meeting_Flag",
    "Fed_Direction_Centered",
    "Fed_Uncertainty_Centered",
    "Fed_Surprise_Centered",
]

focused_coefficients = (
    fed_adl_coefficients[
        fed_adl_coefficients["Variable"].isin(
            focus_variables
        )
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="Coefficient",
    )
    .reindex(fed_model_specifications.keys())
)

focused_pvalues = (
    fed_adl_coefficients[
        fed_adl_coefficients["Variable"].isin(
            focus_variables
        )
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="HAC_p",
    )
    .reindex(fed_model_specifications.keys())
)

print("\nPrimary coefficient comparison")
display(focused_coefficients.round(4))

print("\nHAC p-value comparison")
display(focused_pvalues.round(4))

# -----------------------------------------------------------
# Plot model fit and information criteria
# -----------------------------------------------------------

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Adjusted R²",
        "AIC and BIC",
    ],
    horizontal_spacing=0.15,
)

fig.add_trace(
    go.Bar(
        x=fed_adl_comparison["Model"],
        y=fed_adl_comparison["Adjusted_R2"],
        name="Adjusted R²",
        marker_color="#2a6fbb",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=fed_adl_comparison["Model"],
        y=fed_adl_comparison["AIC"],
        mode="lines+markers",
        name="AIC",
        line=dict(color="#d62728"),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=fed_adl_comparison["Model"],
        y=fed_adl_comparison["BIC"],
        mode="lines+markers",
        name="BIC",
        line=dict(color="#2ca02c"),
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title="Nested Fed-Augmented ADL(1,2,1) Comparison",
    template="plotly_white",
    width=1450,
    height=650,
)

fig.update_xaxes(tickangle=45)
fig.show()

,Model,Controls_Added,Parameters,N,R2,Adjusted_R2,AIC,BIC,ETF_Lag_Block_p,Complete_ETF_Channel_p,Cumulative_ETF_Coefficient,Cumulative_ETF_p,CB_L1_p,Complete_CB_Channel_p,Cumulative_CB_Coefficient,Cumulative_CB_p,Fed_Block_p
0,F0_Demand_Only,0,7,88,0.4062,0.3623,448.8844,466.2258,0.0009,0.0,0.0180,0.0105,0.6489,0.1251,-0.0088,0.3902,NaN
1,F1_Meeting,1,8,88,0.4283,0.3782,447.5580,467.3767,0.0012,0.0,0.0183,0.0084,0.5760,0.1011,-0.0085,0.3935,0.0559
2,F2_Direction,2,9,88,0.4283,0.3704,449.5566,471.8526,0.0012,0.0,0.0181,0.0092,0.5910,0.1290,-0.0085,0.3914,0.1515
3,F3_Uncertainty,3,10,88,0.4287,0.3627,451.4955,476.2689,0.0048,0.0,0.0184,0.0082,0.5851,0.1581,-0.0083,0.4235,0.2351
4,F4_Fed_Complete,4,11,88,0.4348,0.3614,452.5447,479.7954,0.0050,0.0,0.0189,0.0058,0.5729,0.1236,-0.0096,0.3782,0.1082
5,F5_Core_Macro,7,14,88,0.5437,0.4635,439.7098,474.3925,0.0048,0.0,0.0151,0.0186,0.3811,0.1733,-0.0058,0.5355,0.0316
6,F6_Expanded_Macro,9,16,88,0.5462,0.4517,443.2232,482.8606,0.0066,0.0,0.0156,0.0172,0.3891,0.1547,-0.0047,0.6132,0.0337
7,F7_Full,11,18,88,0.5485,0.4389,446.7786,491.3707,0.0298,0.0,0.0169,0.0148,0.2893,0.1133,-0.0043,0.6581,0.0325



Primary coefficient comparison


Variable,CB_IMF_Change,CB_IMF_Change_L1,ETF_Flow,ETF_Flow_L1,ETF_Flow_L2,Fed_Direction_Centered,Fed_Meeting_Flag,Fed_Surprise_Centered,Fed_Uncertainty_Centered,Gold_Return_L1
Model,,,,,,,,,,
F0_Demand_Only,-0.0116,0.0029,0.0378,-0.0043,-0.0154,NaN,NaN,NaN,NaN,0.1314
F1_Meeting,-0.0120,0.0034,0.0390,-0.0077,-0.0130,NaN,-1.2503,NaN,NaN,0.1249
F2_Direction,-0.0119,0.0034,0.0389,-0.0078,-0.0130,-0.0410,-1.2502,NaN,NaN,0.1252
F3_Uncertainty,-0.0117,0.0035,0.0389,-0.0078,-0.0128,0.0335,-1.2518,NaN,0.4508,0.1255
F4_Fed_Complete,-0.0133,0.0037,0.0404,-0.0092,-0.0123,0.1841,-1.2810,-0.9618,0.3088,0.1275
F5_Core_Macro,-0.0118,0.0060,0.0363,-0.0094,-0.0118,-0.2566,-1.3705,-0.7922,-0.2459,0.1160
F6_Expanded_Macro,-0.0110,0.0062,0.0364,-0.0085,-0.0123,-0.2994,-1.3510,-0.7836,0.0228,0.1171
F7_Full,-0.0117,0.0073,0.0365,-0.0083,-0.0113,-0.2632,-1.3679,-0.7976,0.2224,0.1162



HAC p-value comparison


Variable,CB_IMF_Change,CB_IMF_Change_L1,ETF_Flow,ETF_Flow_L1,ETF_Flow_L2,Fed_Direction_Centered,Fed_Meeting_Flag,Fed_Surprise_Centered,Fed_Uncertainty_Centered,Gold_Return_L1
Model,,,,,,,,,,
F0_Demand_Only,0.0776,0.6489,0.0,0.5191,0.0040,NaN,NaN,NaN,NaN,0.1461
F1_Meeting,0.0664,0.5760,0.0,0.2752,0.0153,NaN,0.0595,NaN,NaN,0.1778
F2_Direction,0.0731,0.5910,0.0,0.2788,0.0155,0.9673,0.0611,NaN,NaN,0.1811
F3_Uncertainty,0.0936,0.5851,0.0,0.2785,0.0259,0.9734,0.0619,NaN,0.8329,0.1784
F4_Fed_Complete,0.0744,0.5729,0.0,0.2041,0.0306,0.8451,0.0547,0.4159,0.8888,0.1748
F5_Core_Macro,0.0914,0.3811,0.0,0.1459,0.0488,0.7611,0.0288,0.4826,0.8972,0.2175
F6_Expanded_Macro,0.0834,0.3891,0.0,0.2259,0.0463,0.7264,0.0360,0.4969,0.9900,0.2208
F7_Full,0.0835,0.2893,0.0,0.2398,0.0916,0.7653,0.0345,0.4868,0.9108,0.2424


In [21]:
# CELL 19: Formal nested comparisons among F0, F1, F4 and F5

from scipy import stats
import pandas as pd
import numpy as np

# -----------------------------------------------------------
# Define the three required nested comparisons
# -----------------------------------------------------------

nested_comparisons = [
    {
        "Comparison": "F0 → F1",
        "Restricted_Model": "F0_Demand_Only",
        "Unrestricted_Model": "F1_Meeting",
        "Added_Block": "FOMC meeting-month indicator",
        "Restrictions": [
            "Fed_Meeting_Flag = 0",
        ],
    },
    {
        "Comparison": "F1 → F4",
        "Restricted_Model": "F1_Meeting",
        "Unrestricted_Model": "F4_Fed_Complete",
        "Added_Block": (
            "Fed direction, uncertainty and signed surprise"
        ),
        "Restrictions": [
            "Fed_Direction_Centered = 0",
            "Fed_Uncertainty_Centered = 0",
            "Fed_Surprise_Centered = 0",
        ],
    },
    {
        "Comparison": "F4 → F5",
        "Restricted_Model": "F4_Fed_Complete",
        "Unrestricted_Model": "F5_Core_Macro",
        "Added_Block": (
            "Broad USD, real 10Y yield and VIX"
        ),
        "Restrictions": [
            "Broad_USD_Change = 0",
            "Real10Y_Change_pp = 0",
            "VIX_Change = 0",
        ],
    },
]

comparison_rows = []

# -----------------------------------------------------------
# Run the nested comparisons
# -----------------------------------------------------------

for specification in nested_comparisons:

    restricted_name = specification["Restricted_Model"]
    unrestricted_name = specification["Unrestricted_Model"]
    restrictions = specification["Restrictions"]

    number_restrictions = len(restrictions)

    # Retrieve previously estimated models.
    restricted_ols = (
        fed_adl_results[restricted_name]["OLS"]
    )

    unrestricted_ols = (
        fed_adl_results[unrestricted_name]["OLS"]
    )

    unrestricted_hac = (
        fed_adl_results[unrestricted_name]["HAC"]
    )

    # Confirm identical observations.
    if int(restricted_ols.nobs) != int(unrestricted_ols.nobs):
        raise ValueError(
            f"{restricted_name} and {unrestricted_name} "
            "do not use the same number of observations."
        )

    # -------------------------------------------------------
    # Primary inference: HAC-robust Wald chi-square test
    # -------------------------------------------------------

    hac_wald = unrestricted_hac.wald_test(
        restrictions,
        use_f=False,
        scalar=True,
    )

    hac_wald_statistic = float(
        np.asarray(hac_wald.statistic).reshape(-1)[0]
    )

    hac_wald_p = float(
        np.asarray(hac_wald.pvalue).reshape(-1)[0]
    )

    # -------------------------------------------------------
    # Reference inference: conventional partial-F test
    # -------------------------------------------------------

    rss_restricted = float(
        np.sum(np.square(restricted_ols.resid))
    )

    rss_unrestricted = float(
        np.sum(np.square(unrestricted_ols.resid))
    )

    numerator = (
        rss_restricted - rss_unrestricted
    ) / number_restrictions

    denominator = (
        rss_unrestricted
        / float(unrestricted_ols.df_resid)
    )

    partial_f_statistic = numerator / denominator

    partial_f_p = float(
        stats.f.sf(
            partial_f_statistic,
            number_restrictions,
            unrestricted_ols.df_resid,
        )
    )

    # -------------------------------------------------------
    # Retrieve model-fit measures
    # -------------------------------------------------------

    restricted_summary = (
        fed_adl_comparison.loc[
            fed_adl_comparison["Model"]
            == restricted_name
        ]
        .iloc[0]
    )

    unrestricted_summary = (
        fed_adl_comparison.loc[
            fed_adl_comparison["Model"]
            == unrestricted_name
        ]
        .iloc[0]
    )

    delta_r2 = (
        unrestricted_summary["R2"]
        - restricted_summary["R2"]
    )

    delta_adjusted_r2 = (
        unrestricted_summary["Adjusted_R2"]
        - restricted_summary["Adjusted_R2"]
    )

    delta_aic = (
        unrestricted_summary["AIC"]
        - restricted_summary["AIC"]
    )

    delta_bic = (
        unrestricted_summary["BIC"]
        - restricted_summary["BIC"]
    )

    comparison_rows.append({
        "Comparison": specification["Comparison"],
        "Restricted_Model": restricted_name,
        "Unrestricted_Model": unrestricted_name,
        "Added_Block": specification["Added_Block"],
        "Restrictions": number_restrictions,
        "N": int(unrestricted_ols.nobs),
        "HAC_Wald_Statistic": hac_wald_statistic,
        "HAC_Wald_p": hac_wald_p,
        "HAC_Reject_5pct": hac_wald_p < 0.05,
        "HAC_Reject_10pct": hac_wald_p < 0.10,
        "Conventional_Partial_F": partial_f_statistic,
        "Conventional_F_p": partial_f_p,
        "Delta_R2": delta_r2,
        "Delta_Adjusted_R2": delta_adjusted_r2,
        "Delta_AIC": delta_aic,
        "Delta_BIC": delta_bic,
    })

formal_nested_tests = pd.DataFrame(comparison_rows)

# -----------------------------------------------------------
# Add interpretation
# -----------------------------------------------------------

def nested_conclusion(row):

    if row["HAC_Wald_p"] < 0.05:
        significance = "Significant at 5%"
    elif row["HAC_Wald_p"] < 0.10:
        significance = "Significant only at 10%"
    else:
        significance = "Not jointly significant"

    if row["Delta_Adjusted_R2"] > 0:
        adjusted_r2_result = "Adjusted R² improves"
    else:
        adjusted_r2_result = "Adjusted R² worsens"

    if row["Delta_AIC"] < 0:
        aic_result = "AIC improves"
    else:
        aic_result = "AIC worsens"

    if row["Delta_BIC"] < 0:
        bic_result = "BIC improves"
    else:
        bic_result = "BIC worsens"

    return (
        f"{significance}; "
        f"{adjusted_r2_result}; "
        f"{aic_result}; "
        f"{bic_result}"
    )

formal_nested_tests["Conclusion"] = (
    formal_nested_tests.apply(
        nested_conclusion,
        axis=1,
    )
)

# -----------------------------------------------------------
# Display formal results
# -----------------------------------------------------------

display_columns = [
    "Comparison",
    "Added_Block",
    "Restrictions",
    "N",
    "HAC_Wald_Statistic",
    "HAC_Wald_p",
    "HAC_Reject_5pct",
    "HAC_Reject_10pct",
    "Conventional_Partial_F",
    "Conventional_F_p",
    "Delta_R2",
    "Delta_Adjusted_R2",
    "Delta_AIC",
    "Delta_BIC",
    "Conclusion",
]

display(
    formal_nested_tests[
        display_columns
    ].round(4)
)

,Comparison,Added_Block,Restrictions,N,HAC_Wald_Statistic,HAC_Wald_p,HAC_Reject_5pct,HAC_Reject_10pct,Conventional_Partial_F,Conventional_F_p,Delta_R2,Delta_Adjusted_R2,Delta_AIC,Delta_BIC,Conclusion
0,F0 → F1,FOMC meeting-month indicator,1,88,3.6550,0.0559,False,True,3.0819,0.0830,0.0220,0.0160,-1.3264,1.1509,Significant only at 10%; Adjusted R² improves;...
1,F1 → F4,"Fed direction, uncertainty and signed surprise",3,88,0.8739,0.8317,False,False,0.2972,0.8273,0.0065,-0.0168,4.9867,12.4187,Not jointly significant; Adjusted R² worsens; ...
2,F4 → F5,"Broad USD, real 10Y yield and VIX",3,88,37.8715,0.0000,True,True,5.8870,0.0012,0.1089,0.1021,-12.8349,-5.4029,Significant at 5%; Adjusted R² improves; AIC i...


In [22]:
# CELL 19A: Plot the HAC-robust nested-test results

import plotly.express as px

nested_plot = formal_nested_tests.copy()

nested_plot["Negative_Log10_p"] = (
    -np.log10(
        nested_plot["HAC_Wald_p"].clip(lower=1e-12)
    )
)

nested_plot["Decision"] = np.where(
    nested_plot["HAC_Wald_p"] < 0.05,
    "Significant at 5%",
    np.where(
        nested_plot["HAC_Wald_p"] < 0.10,
        "Significant at 10% only",
        "Not significant",
    ),
)

fig = px.bar(
    nested_plot,
    x="Comparison",
    y="Negative_Log10_p",
    color="Decision",
    text="HAC_Wald_p",
    hover_data={
        "Added_Block": True,
        "HAC_Wald_Statistic": ":.4f",
        "HAC_Wald_p": ":.4f",
        "Delta_R2": ":.4f",
        "Delta_Adjusted_R2": ":.4f",
        "Delta_AIC": ":.4f",
        "Delta_BIC": ":.4f",
        "Negative_Log10_p": False,
    },
    color_discrete_map={
        "Significant at 5%": "#2ca02c",
        "Significant at 10% only": "#ffbf00",
        "Not significant": "#d62728",
    },
    title=(
        "HAC-Robust Nested Tests for the "
        "Gold-Return ADL(1,2,1)"
    ),
)

fig.add_hline(
    y=-np.log10(0.05),
    line_dash="dash",
    line_color="red",
    annotation_text="5% significance threshold",
    annotation_position="top left",
)

fig.add_hline(
    y=-np.log10(0.10),
    line_dash="dot",
    line_color="orange",
    annotation_text="10% significance threshold",
    annotation_position="bottom left",
)

fig.update_traces(
    texttemplate="p=%{text:.4f}",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=620,
    xaxis_title="Nested model comparison",
    yaxis_title="−log₁₀(HAC p-value)",
    legend_title="HAC decision",
)

fig.show()

# The next analysis stream: FOMC information → ETF flows

In [23]:
# CELL 20: Construct the FOMC-meeting-only ETF dataset

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------------------------------------
# Start from the complete monthly dataset
# -----------------------------------------------------------

etf_monthly = (
    fed_adl_model
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

# -----------------------------------------------------------
# Restore the original, non-centred event signals
# -----------------------------------------------------------

raw_fed_columns = [
    "Month",
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
    "Fed_Surprise_Absolute",
    "Fed_Unexpected_Action_Probability",
]

available_raw_fed_columns = [
    column
    for column in raw_fed_columns
    if column in gold_analysis.columns
]

if "Month" not in available_raw_fed_columns:
    raise KeyError(
        "The Month column is missing from gold_analysis."
    )

required_raw_signals = [
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
]

missing_raw_signals = [
    column
    for column in required_raw_signals
    if column not in gold_analysis.columns
]

if missing_raw_signals:
    raise KeyError(
        "Raw Fed signals are missing from gold_analysis: "
        f"{missing_raw_signals}"
    )

# Avoid duplicate columns if the cell is rerun.
etf_monthly = etf_monthly.drop(
    columns=[
        column
        for column in raw_fed_columns
        if column != "Month"
        and column in etf_monthly.columns
    ],
    errors="ignore",
)

etf_monthly = etf_monthly.merge(
    gold_analysis[available_raw_fed_columns],
    on="Month",
    how="left",
    validate="one_to_one",
)

# -----------------------------------------------------------
# Create calendar-month ETF dynamics BEFORE filtering
# -----------------------------------------------------------

etf_monthly["ETF_Flow_L1_Calendar"] = (
    etf_monthly["ETF_Flow"].shift(1)
)

etf_monthly["ETF_Flow_L2_Calendar"] = (
    etf_monthly["ETF_Flow"].shift(2)
)

etf_monthly["ETF_Flow_Next_Month"] = (
    etf_monthly["ETF_Flow"].shift(-1)
)

etf_monthly["ETF_Flow_Next_2_Months_Avg"] = (
    (
        etf_monthly["ETF_Flow"].shift(-1)
        + etf_monthly["ETF_Flow"].shift(-2)
    ) / 2
)

# Month-to-month change in ETF net demand.
# This is a change in the flow, not the ETF flow itself.
etf_monthly["Delta_ETF_Flow"] = (
    etf_monthly["ETF_Flow"].diff()
)

# -----------------------------------------------------------
# Retain only genuine FOMC meeting months
# -----------------------------------------------------------

etf_meeting_df = (
    etf_monthly.loc[
        etf_monthly["Fed_Meeting_Flag"].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)

# Verify that zero-filled non-meeting observations are absent.
if not etf_meeting_df["Fed_Meeting_Flag"].eq(1).all():
    raise ValueError(
        "The ETF event sample contains non-meeting months."
    )

# The meeting flag is constant and must not enter regressions.
print(
    "FOMC meeting-month observations:",
    len(etf_meeting_df)
)

print(
    "Period:",
    etf_meeting_df["Month"].min(),
    "to",
    etf_meeting_df["Month"].max()
)

print(
    "Unique value of meeting flag:",
    etf_meeting_df["Fed_Meeting_Flag"].unique()
)

print(
    "Multiple-meeting months:",
    etf_meeting_df.loc[
        etf_meeting_df[
            "Fed_Multiple_Meeting_Flag"
        ].eq(1),
        "Month"
    ].dt.strftime("%Y-%m").tolist()
)

# -----------------------------------------------------------
# Check that the Fed signals are actual event observations
# -----------------------------------------------------------

meeting_signal_check = pd.DataFrame({
    "Variable": [
        "Fed_Expected_Direction",
        "Fed_Policy_Uncertainty",
        "Fed_Surprise_Signed",
        "Fed_Surprise_Absolute",
        "Fed_Unexpected_Action_Probability",
    ]
})

meeting_signal_check["Available"] = (
    meeting_signal_check["Variable"]
    .map(etf_meeting_df.notna().sum())
)

meeting_signal_check["Missing"] = (
    len(etf_meeting_df)
    - meeting_signal_check["Available"]
)

display(meeting_signal_check)

# -----------------------------------------------------------
# Variables retained for ETF-flow modelling
# -----------------------------------------------------------

etf_analysis_variables = [
    "Month",
    "ETF_Flow",
    "ETF_Flow_L1_Calendar",
    "ETF_Flow_L2_Calendar",
    "Delta_ETF_Flow",
    "ETF_Flow_Next_Month",
    "ETF_Flow_Next_2_Months_Avg",
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
    "Fed_Surprise_Absolute",
    "Fed_Unexpected_Action_Probability",
    "Fed_Multiple_Meeting_Flag",
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "VIX_Change",
    "CPI_MoM",
    "HY_OAS_Change_pp",
    "Brent_Return_pct",
    "Post_June_2020_Dummy",
    "COVID_Shock_Dummy",
]

available_etf_analysis_variables = [
    column
    for column in etf_analysis_variables
    if column in etf_meeting_df.columns
]

etf_meeting_analysis = etf_meeting_df[
    available_etf_analysis_variables
].copy()

# -----------------------------------------------------------
# Availability report
# -----------------------------------------------------------

availability_rows = []

for column in etf_meeting_analysis.columns:

    if column == "Month":
        continue

    available = int(
        etf_meeting_analysis[column].notna().sum()
    )

    missing = int(
        etf_meeting_analysis[column].isna().sum()
    )

    availability_rows.append({
        "Variable": column,
        "Available": available,
        "Missing": missing,
        "Missing_Percent": (
            missing / len(etf_meeting_analysis) * 100
        ),
    })

etf_meeting_availability = pd.DataFrame(
    availability_rows
)

display(
    etf_meeting_availability.round(4)
)

# Display the actual meeting observations.
display(
    etf_meeting_analysis[
        [
            "Month",
            "ETF_Flow",
            "ETF_Flow_L1_Calendar",
            "ETF_Flow_Next_Month",
            "Fed_Expected_Direction",
            "Fed_Policy_Uncertainty",
            "Fed_Surprise_Signed",
            "Broad_USD_Change",
            "Real10Y_Change_pp",
            "VIX_Change",
        ]
    ].head(15).round(6)
)

FOMC meeting-month observations: 62
Period: 2019-01-01 00:00:00 to 2026-07-01 00:00:00
Unique value of meeting flag: [1]
Multiple-meeting months: ['2020-03']


,Variable,Available,Missing
0,Fed_Expected_Direction,62,0
1,Fed_Policy_Uncertainty,62,0
2,Fed_Surprise_Signed,62,0
3,Fed_Surprise_Absolute,62,0
4,Fed_Unexpected_Action_Probability,62,0


,Variable,Available,Missing,Missing_Percent
0,ETF_Flow,62,0,0.0000
1,ETF_Flow_L1_Calendar,61,1,1.6129
2,ETF_Flow_L2_Calendar,61,1,1.6129
3,Delta_ETF_Flow,61,1,1.6129
4,ETF_Flow_Next_Month,61,1,1.6129
5,ETF_Flow_Next_2_Months_Avg,60,2,3.2258
6,Fed_Expected_Direction,62,0,0.0000
7,Fed_Policy_Uncertainty,62,0,0.0000
8,Fed_Surprise_Signed,62,0,0.0000
9,Fed_Surprise_Absolute,62,0,0.0000


,Month,ETF_Flow,ETF_Flow_L1_Calendar,ETF_Flow_Next_Month,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed,Broad_USD_Change,Real10Y_Change_pp,VIX_Change
0,2019-01-01,71.8,NaN,-32.4,0.378799,0.749275,-0.378799,NaN,-0.20,-8.85
1,2019-03-01,1.7,-32.4,-56.9,0.075747,0.644543,-0.075747,0.370513,-0.25,-1.07
2,2019-05-01,-2.4,-56.9,126.8,0.047289,0.691150,-0.047289,1.053627,-0.16,5.59
3,2019-06-01,126.8,-2.4,52.2,0.093836,0.714753,-0.093836,-0.530312,-0.09,-3.63
4,2019-07-01,52.2,126.8,122.0,0.092682,0.642616,-1.092682,-0.349479,-0.05,1.04
5,2019-09-01,78.6,122.0,44.4,-0.275047,0.896467,-0.724953,0.223534,0.20,-2.74
6,2019-10-01,44.4,78.6,-30.1,-0.253850,0.911889,-0.746150,-0.577661,0.00,-3.02
7,2019-12-01,25.8,-30.1,61.7,-0.318866,0.892772,0.318866,-0.649858,-0.02,1.16
8,2020-01-01,61.7,25.8,84.5,0.115311,0.778939,-0.115311,-0.630224,-0.29,5.06
9,2020-03-01,150.9,84.5,170.1,-0.848228,0.371185,-0.151772,4.312199,0.11,13.43


In [24]:
# CELL 21: Plot ETF flows and Fed information in meeting months

plot_df = etf_meeting_analysis.dropna(
    subset=[
        "ETF_Flow",
        "Fed_Expected_Direction",
        "Fed_Policy_Uncertainty",
        "Fed_Surprise_Signed",
    ]
).copy()

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.055,
    subplot_titles=[
        "Global Gold ETF Net Demand",
        "Expected Fed Direction",
        "Fed Policy Uncertainty",
        "Signed Fed Surprise",
    ],
)

fig.add_trace(
    go.Bar(
        x=plot_df["Month"],
        y=plot_df["ETF_Flow"],
        name="ETF flow",
        marker_color=np.where(
            plot_df["ETF_Flow"] >= 0,
            "#2ca02c",
            "#d62728",
        ),
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "ETF net demand: %{y:.2f} tonnes"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=plot_df["Month"],
        y=plot_df["Fed_Expected_Direction"],
        mode="lines+markers",
        name="Expected direction",
        line=dict(color="#1f77b4"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=plot_df["Month"],
        y=plot_df["Fed_Policy_Uncertainty"],
        mode="lines+markers",
        name="Policy uncertainty",
        line=dict(color="#ff7f0e"),
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=plot_df["Month"],
        y=plot_df["Fed_Surprise_Signed"],
        name="Signed surprise",
        marker_color=np.where(
            plot_df["Fed_Surprise_Signed"] >= 0,
            "#7b2cbf",
            "#17a2b8",
        ),
    ),
    row=4,
    col=1,
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
    row=1,
    col=1,
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
    row=2,
    col=1,
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
    row=4,
    col=1,
)

fig.update_layout(
    title=(
        "Gold ETF Flows and Fed Information: "
        "FOMC Meeting Months Only"
    ),
    template="plotly_white",
    width=1250,
    height=1050,
    showlegend=False,
)

fig.update_yaxes(
    title_text="Tonnes",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Hawkish ↔ dovish",
    row=2,
    col=1,
)

fig.update_yaxes(
    title_text="Normalized uncertainty",
    row=3,
    col=1,
)

fig.update_yaxes(
    title_text="Surprise",
    row=4,
    col=1,
)

fig.update_xaxes(
    title_text="FOMC meeting month",
    row=4,
    col=1,
)

fig.show()

In [25]:
# CELL 22: Recover realized Fed actions and examine ETF flows
# across hike, hold and cut meeting months

import pandas as pd

In [26]:
# CELL 22: Recover realized Fed decisions and summarize ETF flows

etf_event_df = etf_meeting_analysis.copy()

# -----------------------------------------------------------
# Reconstruct the realized policy action
# -----------------------------------------------------------

etf_event_df["Fed_Realized_Action_Raw"] = (
    etf_event_df["Fed_Expected_Direction"]
    + etf_event_df["Fed_Surprise_Signed"]
)

# Numerical probabilities can create extremely small deviations
# from exactly -1, 0 or +1.
etf_event_df["Fed_Realized_Action"] = (
    etf_event_df["Fed_Realized_Action_Raw"]
    .round()
    .astype(int)
)

action_labels = {
    -1: "Cut",
     0: "Hold",
     1: "Hike",
}

etf_event_df["Fed_Action"] = (
    etf_event_df["Fed_Realized_Action"]
    .map(action_labels)
)

# Validate the reconstructed action.
valid_actions = {-1, 0, 1}

observed_actions = set(
    etf_event_df["Fed_Realized_Action"].unique()
)

if not observed_actions.issubset(valid_actions):
    raise ValueError(
        "Unexpected reconstructed Fed-action values: "
        f"{observed_actions}"
    )

etf_event_df["Action_Reconstruction_Error"] = (
    etf_event_df["Fed_Realized_Action_Raw"]
    - etf_event_df["Fed_Realized_Action"]
).abs()

print(
    "Maximum action-reconstruction error:",
    etf_event_df[
        "Action_Reconstruction_Error"
    ].max()
)

# -----------------------------------------------------------
# Create action indicators
# Hold is the omitted/reference category in regressions.
# -----------------------------------------------------------

etf_event_df["Fed_Hike"] = (
    etf_event_df["Fed_Realized_Action"] == 1
).astype(int)

etf_event_df["Fed_Cut"] = (
    etf_event_df["Fed_Realized_Action"] == -1
).astype(int)

# -----------------------------------------------------------
# Action-frequency table
# -----------------------------------------------------------

action_frequency = (
    etf_event_df
    .groupby("Fed_Action", observed=True)
    .agg(
        Meetings=("Month", "count"),
        First_Month=("Month", "min"),
        Last_Month=("Month", "max"),
    )
    .reset_index()
)

action_frequency["Share_Percent"] = (
    action_frequency["Meetings"]
    / len(etf_event_df)
    * 100
)

action_order = ["Cut", "Hold", "Hike"]

action_frequency["Fed_Action"] = pd.Categorical(
    action_frequency["Fed_Action"],
    categories=action_order,
    ordered=True,
)

action_frequency = (
    action_frequency
    .sort_values("Fed_Action")
    .reset_index(drop=True)
)

print("\nFOMC decision frequencies")
display(action_frequency.round(4))

# -----------------------------------------------------------
# ETF-flow behaviour by realized Fed decision
# -----------------------------------------------------------

etf_by_action = (
    etf_event_df
    .groupby("Fed_Action", observed=True)
    .agg(
        Meetings=("ETF_Flow", "count"),
        Mean_ETF_Flow=("ETF_Flow", "mean"),
        Median_ETF_Flow=("ETF_Flow", "median"),
        Std_ETF_Flow=("ETF_Flow", "std"),
        Positive_Flow_Share=(
            "ETF_Flow",
            lambda series: (series > 0).mean()
        ),
        Mean_Delta_ETF_Flow=(
            "Delta_ETF_Flow",
            "mean"
        ),
        Mean_Next_Month_ETF_Flow=(
            "ETF_Flow_Next_Month",
            "mean"
        ),
        Mean_Next_2_Month_ETF_Flow=(
            "ETF_Flow_Next_2_Months_Avg",
            "mean"
        ),
        Mean_Expected_Direction=(
            "Fed_Expected_Direction",
            "mean"
        ),
        Mean_Uncertainty=(
            "Fed_Policy_Uncertainty",
            "mean"
        ),
        Mean_Signed_Surprise=(
            "Fed_Surprise_Signed",
            "mean"
        ),
    )
    .reset_index()
)

etf_by_action["Fed_Action"] = pd.Categorical(
    etf_by_action["Fed_Action"],
    categories=action_order,
    ordered=True,
)

etf_by_action = (
    etf_by_action
    .sort_values("Fed_Action")
    .reset_index(drop=True)
)

etf_by_action["Positive_Flow_Share"] *= 100

print("\nETF behaviour by realized FOMC decision")
display(etf_by_action.round(4))

# -----------------------------------------------------------
# Individual meeting observations
# -----------------------------------------------------------

meeting_event_table = etf_event_df[
    [
        "Month",
        "Fed_Action",
        "Fed_Expected_Direction",
        "Fed_Policy_Uncertainty",
        "Fed_Surprise_Signed",
        "ETF_Flow",
        "Delta_ETF_Flow",
        "ETF_Flow_Next_Month",
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
        "Fed_Multiple_Meeting_Flag",
    ]
].copy()

display(
    meeting_event_table.round(6)
)

Maximum action-reconstruction error: 1.1102230246251565e-16

FOMC decision frequencies


,Fed_Action,Meetings,First_Month,Last_Month,Share_Percent
0,Cut,10,2019-07-01,2025-12-01,16.1290
1,Hold,41,2019-01-01,2026-07-01,66.1290
2,Hike,11,2022-03-01,2023-07-01,17.7419



ETF behaviour by realized FOMC decision


,Fed_Action,Meetings,Mean_ETF_Flow,Median_ETF_Flow,Std_ETF_Flow,Positive_Flow_Share,Mean_Delta_ETF_Flow,Mean_Next_Month_ETF_Flow,Mean_Next_2_Month_ETF_Flow,Mean_Expected_Direction,Mean_Uncertainty,Mean_Signed_Surprise
0,Cut,10,47.1500,48.3,49.3226,90.0000,-9.5400,60.1300,53.8550,-0.3111,0.7075,-0.6889
1,Hold,41,18.2939,13.7,65.7422,60.9756,13.3812,23.2787,19.6718,0.0520,0.6632,-0.0520
2,Hike,11,-11.5273,-34.0,76.0603,27.2727,11.1000,-23.7636,-31.6182,0.6765,0.5236,0.3235


,Month,Fed_Action,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed,ETF_Flow,Delta_ETF_Flow,ETF_Flow_Next_Month,Broad_USD_Change,Real10Y_Change_pp,VIX_Change,Fed_Multiple_Meeting_Flag
0,2019-01-01,Hold,0.378799,0.749275,-0.378799,71.800000,NaN,-32.400000,NaN,-0.20,-8.85,0
1,2019-03-01,Hold,0.075747,0.644543,-0.075747,1.700000,34.100000,-56.900000,0.370513,-0.25,-1.07,0
2,2019-05-01,Hold,0.047289,0.691150,-0.047289,-2.400000,54.500000,126.800000,1.053627,-0.16,5.59,0
3,2019-06-01,Hold,0.093836,0.714753,-0.093836,126.800000,129.200000,52.200000,-0.530312,-0.09,-3.63,0
4,2019-07-01,Cut,0.092682,0.642616,-1.092682,52.200000,-74.600000,122.000000,-0.349479,-0.05,1.04,0
...,...,...,...,...,...,...,...,...,...,...,...,...
57,2026-01-01,Hold,-0.409311,0.859499,0.409311,120.100000,43.300000,26.200000,-0.958513,-0.03,2.49,0
58,2026-03-01,Hold,-0.011241,0.774599,0.011241,-84.800000,-111.000000,45.200000,2.013833,0.28,5.39,0
59,2026-04-01,Hold,0.130148,0.769602,-0.130148,45.200000,130.000000,-16.200000,-0.883536,-0.06,-8.36,0
60,2026-06-01,Hold,0.051644,0.887675,-0.051644,-74.300000,-58.100000,23.549252,1.304263,0.13,1.13,0


In [27]:
# CELL 23: Plot ETF flows across hike, hold and cut meetings

action_color_map = {
    "Cut": "#2ca02c",
    "Hold": "#7f7f7f",
    "Hike": "#d62728",
}

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Distribution of Meeting-Month ETF Flows",
        "Mean Current and Forward ETF Flows",
    ],
    horizontal_spacing=0.14,
)

for action in action_order:

    action_data = etf_event_df.loc[
        etf_event_df["Fed_Action"] == action
    ]

    fig.add_trace(
        go.Box(
            x=[action] * len(action_data),
            y=action_data["ETF_Flow"],
            name=action,
            marker_color=action_color_map[action],
            boxpoints="all",
            jitter=0.30,
            pointpos=0,
            hovertemplate=(
                f"Action: {action}<br>"
                "ETF flow: %{y:.2f} tonnes"
                "<extra></extra>"
            ),
            showlegend=False,
        ),
        row=1,
        col=1,
    )

flow_long = etf_by_action.melt(
    id_vars="Fed_Action",
    value_vars=[
        "Mean_ETF_Flow",
        "Mean_Next_Month_ETF_Flow",
        "Mean_Next_2_Month_ETF_Flow",
    ],
    var_name="ETF_Horizon",
    value_name="Mean_ETF_Flow_tonnes",
)

horizon_labels = {
    "Mean_ETF_Flow":
        "Meeting month",
    "Mean_Next_Month_ETF_Flow":
        "Next month",
    "Mean_Next_2_Month_ETF_Flow":
        "Next two months average",
}

flow_long["ETF_Horizon"] = (
    flow_long["ETF_Horizon"]
    .map(horizon_labels)
)

for horizon, colour in {
    "Meeting month": "#1f77b4",
    "Next month": "#ff7f0e",
    "Next two months average": "#9467bd",
}.items():

    horizon_data = flow_long.loc[
        flow_long["ETF_Horizon"] == horizon
    ]

    fig.add_trace(
        go.Bar(
            x=horizon_data["Fed_Action"],
            y=horizon_data["Mean_ETF_Flow_tonnes"],
            name=horizon,
            marker_color=colour,
            hovertemplate=(
                "Action: %{x}<br>"
                f"Horizon: {horizon}<br>"
                "Mean ETF flow: %{y:.2f} tonnes"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
    row=1,
    col=1,
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
    row=1,
    col=2,
)

fig.update_layout(
    title=(
        "Gold ETF Net Demand Around FOMC Decisions: "
        "Meeting Months Only"
    ),
    template="plotly_white",
    width=1350,
    height=650,
    boxmode="group",
    barmode="group",
    legend_title="ETF-flow horizon",
)

fig.update_yaxes(
    title_text="ETF net demand (tonnes)",
)

fig.update_xaxes(
    title_text="Realized FOMC decision",
)

fig.show()

In [28]:
# CELL 24: Correlations and ETF autoregressive lag selection

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------------------------------------
# Variables for the primary ETF-flow model
# -----------------------------------------------------------

etf_primary_variables = [
    "Month",
    "ETF_Flow",
    "ETF_Flow_L1_Calendar",
    "ETF_Flow_L2_Calendar",
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "VIX_Change",
]

# Common sample so all lag-order comparisons use identical months.
etf_primary_common = (
    etf_event_df[etf_primary_variables]
    .dropna()
    .sort_values("Month")
    .reset_index(drop=True)
)

print(
    "Primary ETF meeting-sample observations:",
    len(etf_primary_common)
)

print(
    "Period:",
    etf_primary_common["Month"].min(),
    "to",
    etf_primary_common["Month"].max()
)

excluded_meeting_months = (
    etf_event_df.loc[
        ~etf_event_df["Month"].isin(
            etf_primary_common["Month"]
        ),
        "Month"
    ]
    .dt.strftime("%Y-%m")
    .tolist()
)

print(
    "Meeting months excluded:",
    excluded_meeting_months
)

# -----------------------------------------------------------
# Pearson and Spearman correlations
# -----------------------------------------------------------

correlation_variables = [
    "ETF_Flow",
    "ETF_Flow_L1_Calendar",
    "ETF_Flow_L2_Calendar",
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "VIX_Change",
]

etf_corr_data = etf_primary_common[
    correlation_variables
]

etf_pearson_corr = etf_corr_data.corr(
    method="pearson"
)

etf_spearman_corr = etf_corr_data.corr(
    method="spearman"
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Pearson Correlations",
        "Spearman Correlations",
    ],
    horizontal_spacing=0.12,
)

for matrix, column_number in [
    (etf_pearson_corr, 1),
    (etf_spearman_corr, 2),
]:
    fig.add_trace(
        go.Heatmap(
            z=matrix.values,
            x=matrix.columns,
            y=matrix.index,
            zmin=-1,
            zmax=1,
            zmid=0,
            colorscale="RdBu_r",
            colorbar=dict(
                title="Correlation",
                x=0.46 if column_number == 1 else 1.02,
            ),
            hovertemplate=(
                "%{y} vs %{x}<br>"
                "Correlation: %{z:.3f}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=column_number,
    )

fig.update_layout(
    title=(
        "ETF Flows, Fed Information and Macro Variables: "
        "FOMC Meeting Months"
    ),
    template="plotly_white",
    width=1500,
    height=760,
)

fig.update_xaxes(tickangle=55)
fig.show()

etf_correlations = pd.DataFrame({
    "Pearson_with_ETF_Flow":
        etf_pearson_corr["ETF_Flow"],
    "Spearman_with_ETF_Flow":
        etf_spearman_corr["ETF_Flow"],
})

etf_correlations = (
    etf_correlations
    .drop(index="ETF_Flow")
    .sort_values(
        "Spearman_with_ETF_Flow",
        key=np.abs,
        ascending=False,
    )
)

print("\nCorrelations with meeting-month ETF flows")
display(etf_correlations.round(4))

# -----------------------------------------------------------
# Select ETF calendar-lag order: AR(0), AR(1), or AR(2)
# -----------------------------------------------------------

fed_information_variables = [
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
]

etf_ar_specifications = {
    "ETF_AR0": [],
    "ETF_AR1": [
        "ETF_Flow_L1_Calendar",
    ],
    "ETF_AR2": [
        "ETF_Flow_L1_Calendar",
        "ETF_Flow_L2_Calendar",
    ],
}

etf_ar_results = {}
etf_ar_rows = []

for model_name, etf_lags in etf_ar_specifications.items():

    regressors = (
        etf_lags
        + fed_information_variables
    )

    X = sm.add_constant(
        etf_primary_common[regressors].astype(float),
        has_constant="add",
    )

    y = etf_primary_common["ETF_Flow"].astype(float)

    ols_result = sm.OLS(y, X).fit()

    hac_result = ols_result.get_robustcov_results(
        cov_type="HAC",
        maxlags=3,
        use_correction=True,
    )

    hac_result.model.data.xnames = X.columns.tolist()

    etf_ar_results[model_name] = {
        "OLS": ols_result,
        "HAC": hac_result,
        "Regressors": regressors,
    }

    etf_ar_rows.append({
        "Model": model_name,
        "ETF_Calendar_Lags": len(etf_lags),
        "Parameters": int(ols_result.df_model + 1),
        "N": int(ols_result.nobs),
        "R2": ols_result.rsquared,
        "Adjusted_R2": ols_result.rsquared_adj,
        "AIC": ols_result.aic,
        "BIC": ols_result.bic,
    })

etf_ar_comparison = pd.DataFrame(etf_ar_rows)

etf_ar_comparison["Delta_AIC_From_Minimum"] = (
    etf_ar_comparison["AIC"]
    - etf_ar_comparison["AIC"].min()
)

etf_ar_comparison["Delta_BIC_From_Minimum"] = (
    etf_ar_comparison["BIC"]
    - etf_ar_comparison["BIC"].min()
)

display(etf_ar_comparison.round(4))

aic_selected_etf_ar = (
    etf_ar_comparison
    .sort_values("AIC")
    .iloc[0]["Model"]
)

bic_selected_etf_ar = (
    etf_ar_comparison
    .sort_values("BIC")
    .iloc[0]["Model"]
)

print(
    "AIC-selected ETF persistence model:",
    aic_selected_etf_ar
)

print(
    "BIC-selected ETF persistence model:",
    bic_selected_etf_ar
)

Primary ETF meeting-sample observations: 61
Period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00
Meeting months excluded: ['2019-01']



Correlations with meeting-month ETF flows


,Pearson_with_ETF_Flow,Spearman_with_ETF_Flow
ETF_Flow_L1_Calendar,0.5515,0.5188
Fed_Expected_Direction,-0.5299,-0.4952
ETF_Flow_L2_Calendar,0.2968,0.3016
Broad_USD_Change,-0.1100,-0.1945
Real10Y_Change_pp,-0.2105,-0.1873
Fed_Policy_Uncertainty,0.1168,0.1697
VIX_Change,0.0074,0.0247
Fed_Surprise_Signed,0.1398,-0.0116


,Model,ETF_Calendar_Lags,Parameters,N,R2,Adjusted_R2,AIC,BIC,Delta_AIC_From_Minimum,Delta_BIC_From_Minimum
0,ETF_AR0,0,4,61,0.3234,0.2878,668.8780,677.3215,10.2685,8.1576
1,ETF_AR1,1,5,61,0.4467,0.4071,658.6095,669.1639,0.0000,0.0000
2,ETF_AR2,2,6,61,0.4496,0.3996,660.2825,672.9478,1.6730,3.7839


AIC-selected ETF persistence model: ETF_AR1
BIC-selected ETF persistence model: ETF_AR1


In [29]:
# CELL 25: Nested FOMC-information models for meeting-month ETF flows

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------------------------------------
# Freeze the ETF persistence structure selected by AIC and BIC
# -----------------------------------------------------------

etf_base_regressors = [
    "ETF_Flow_L1_Calendar",
]

etf_model_specifications = {
    # Persistence-only benchmark
    "E0_AR1_Only": etf_base_regressors,

    # Add expected hawkish/dovish direction
    "E1_Direction": etf_base_regressors + [
        "Fed_Expected_Direction",
    ],

    # Add policy uncertainty
    "E2_Uncertainty": etf_base_regressors + [
        "Fed_Expected_Direction",
        "Fed_Policy_Uncertainty",
    ],

    # Add signed policy surprise
    "E3_Fed_Complete": etf_base_regressors + [
        "Fed_Expected_Direction",
        "Fed_Policy_Uncertainty",
        "Fed_Surprise_Signed",
    ],

    # Add the main financial-market transmission variables
    "E4_Core_Macro": etf_base_regressors + [
        "Fed_Expected_Direction",
        "Fed_Policy_Uncertainty",
        "Fed_Surprise_Signed",
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
    ],
}

# -----------------------------------------------------------
# Use one identical 61-meeting sample
# -----------------------------------------------------------

all_etf_primary_regressors = list(dict.fromkeys(
    variable
    for regressors in etf_model_specifications.values()
    for variable in regressors
))

etf_model_common = (
    etf_event_df[
        [
            "Month",
            "ETF_Flow",
        ] + all_etf_primary_regressors
    ]
    .dropna()
    .sort_values("Month")
    .reset_index(drop=True)
)

print(
    "ETF-model observations:",
    len(etf_model_common)
)

print(
    "Period:",
    etf_model_common["Month"].min(),
    "to",
    etf_model_common["Month"].max()
)

print(
    "Excluded meeting months:",
    etf_event_df.loc[
        ~etf_event_df["Month"].isin(
            etf_model_common["Month"]
        ),
        "Month"
    ].dt.strftime("%Y-%m").tolist()
)

# -----------------------------------------------------------
# Helper functions
# -----------------------------------------------------------

def extract_scalar(value):
    return float(
        np.asarray(value).reshape(-1)[0]
    )


def robust_wald_pvalue(result, restrictions):

    test = result.wald_test(
        restrictions,
        use_f=False,
        scalar=True,
    )

    return (
        extract_scalar(test.statistic),
        extract_scalar(test.pvalue),
    )


# -----------------------------------------------------------
# Estimate nested models
# -----------------------------------------------------------

etf_nested_results = {}
etf_model_rows = []
etf_coefficient_rows = []

y_etf = etf_model_common["ETF_Flow"].astype(float)

for model_name, regressors in etf_model_specifications.items():

    X_etf = sm.add_constant(
        etf_model_common[regressors].astype(float),
        has_constant="add",
    )

    etf_ols = sm.OLS(
        y_etf,
        X_etf,
    ).fit()

    # Event-time HAC:
    # maxlags=2 means dependence across up to two successive
    # FOMC meeting observations, not two calendar months.
    etf_hac = etf_ols.get_robustcov_results(
        cov_type="HAC",
        maxlags=2,
        use_correction=True,
    )

    parameter_names = X_etf.columns.tolist()
    etf_hac.model.data.xnames = parameter_names

    coefficient_table = pd.DataFrame({
        "Model": model_name,
        "Variable": parameter_names,
        "Coefficient": etf_hac.params,
        "HAC_SE": etf_hac.bse,
        "HAC_t": etf_hac.tvalues,
        "HAC_p": etf_hac.pvalues,
        "CI_Lower_95pct": etf_hac.conf_int()[:, 0],
        "CI_Upper_95pct": etf_hac.conf_int()[:, 1],
    })

    coefficient_table["Significant_5pct"] = (
        coefficient_table["HAC_p"] < 0.05
    )

    coefficient_table["Significant_10pct"] = (
        coefficient_table["HAC_p"] < 0.10
    )

    etf_coefficient_rows.append(coefficient_table)

    # Full Fed-information block, conditional on ETF persistence.
    included_fed_variables = [
        variable
        for variable in [
            "Fed_Expected_Direction",
            "Fed_Policy_Uncertainty",
            "Fed_Surprise_Signed",
        ]
        if variable in regressors
    ]

    if included_fed_variables:

        fed_block_stat, fed_block_p = (
            robust_wald_pvalue(
                etf_hac,
                [
                    f"{variable} = 0"
                    for variable in included_fed_variables
                ],
            )
        )

    else:
        fed_block_stat = np.nan
        fed_block_p = np.nan

    # Core financial-transmission block.
    core_macro_variables = [
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
    ]

    if all(
        variable in regressors
        for variable in core_macro_variables
    ):

        macro_block_stat, macro_block_p = (
            robust_wald_pvalue(
                etf_hac,
                [
                    f"{variable} = 0"
                    for variable in core_macro_variables
                ],
            )
        )

    else:
        macro_block_stat = np.nan
        macro_block_p = np.nan

    etf_model_rows.append({
        "Model": model_name,
        "Fed_Variables": len(included_fed_variables),
        "Parameters": int(etf_ols.df_model + 1),
        "N": int(etf_ols.nobs),
        "R2": etf_ols.rsquared,
        "Adjusted_R2": etf_ols.rsquared_adj,
        "AIC": etf_ols.aic,
        "BIC": etf_ols.bic,
        "Fed_Block_Wald": fed_block_stat,
        "Fed_Block_p": fed_block_p,
        "Core_Macro_Block_Wald": macro_block_stat,
        "Core_Macro_Block_p": macro_block_p,
    })

    etf_nested_results[model_name] = {
        "OLS": etf_ols,
        "HAC": etf_hac,
        "Regressors": regressors,
        "Coefficient_Table": coefficient_table,
    }

etf_model_comparison = pd.DataFrame(
    etf_model_rows
)

etf_nested_coefficients = pd.concat(
    etf_coefficient_rows,
    ignore_index=True,
)

# -----------------------------------------------------------
# Add changes relative to the preceding nested model
# -----------------------------------------------------------

etf_model_comparison["Delta_Adjusted_R2"] = (
    etf_model_comparison["Adjusted_R2"].diff()
)

etf_model_comparison["Delta_AIC"] = (
    etf_model_comparison["AIC"].diff()
)

etf_model_comparison["Delta_BIC"] = (
    etf_model_comparison["BIC"].diff()
)

display(
    etf_model_comparison.round(4)
)

# -----------------------------------------------------------
# Coefficient and HAC p-value comparisons
# -----------------------------------------------------------

etf_focus_variables = [
    "ETF_Flow_L1_Calendar",
    "Fed_Expected_Direction",
    "Fed_Policy_Uncertainty",
    "Fed_Surprise_Signed",
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "VIX_Change",
]

etf_focused_coefficients = (
    etf_nested_coefficients.loc[
        etf_nested_coefficients["Variable"].isin(
            etf_focus_variables
        )
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="Coefficient",
    )
    .reindex(etf_model_specifications.keys())
)

etf_focused_pvalues = (
    etf_nested_coefficients.loc[
        etf_nested_coefficients["Variable"].isin(
            etf_focus_variables
        )
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="HAC_p",
    )
    .reindex(etf_model_specifications.keys())
)

print("\nETF-model coefficient comparison")
display(etf_focused_coefficients.round(4))

print("\nETF-model HAC p-value comparison")
display(etf_focused_pvalues.round(4))

# -----------------------------------------------------------
# Plot model comparison
# -----------------------------------------------------------

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Adjusted R²",
        "Information Criteria",
    ],
    horizontal_spacing=0.15,
)

fig.add_trace(
    go.Bar(
        x=etf_model_comparison["Model"],
        y=etf_model_comparison["Adjusted_R2"],
        name="Adjusted R²",
        marker_color="#2a6fbb",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=etf_model_comparison["Model"],
        y=etf_model_comparison["AIC"],
        mode="lines+markers",
        name="AIC",
        line=dict(color="#d62728"),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=etf_model_comparison["Model"],
        y=etf_model_comparison["BIC"],
        mode="lines+markers",
        name="BIC",
        line=dict(color="#2ca02c"),
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title=(
        "Nested FOMC-Information Models for "
        "Meeting-Month Gold ETF Flows"
    ),
    template="plotly_white",
    width=1400,
    height=650,
)

fig.update_xaxes(tickangle=35)
fig.show()

ETF-model observations: 61
Period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00
Excluded meeting months: ['2019-01']


,Model,Fed_Variables,Parameters,N,R2,Adjusted_R2,AIC,BIC,Fed_Block_Wald,Fed_Block_p,Core_Macro_Block_Wald,Core_Macro_Block_p,Delta_Adjusted_R2,Delta_AIC,Delta_BIC
0,E0_AR1_Only,0,2,61,0.3041,0.2923,666.5925,670.8142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E1_Direction,1,3,61,0.3976,0.3768,659.7938,666.1264,13.6957,0.0002,NaN,NaN,0.0845,-6.7987,-4.6878
2,E2_Uncertainty,2,4,61,0.3976,0.3659,661.7881,670.2316,13.8037,0.0010,NaN,NaN,-0.0109,1.9944,4.1052
3,E3_Fed_Complete,3,5,61,0.4467,0.4071,658.6095,669.1639,21.6265,0.0001,NaN,NaN,0.0412,-3.1786,-1.0677
4,E4_Core_Macro,3,8,61,0.4864,0.4185,660.0664,676.9533,18.5470,0.0003,3.3974,0.3343,0.0114,1.4568,7.7895



ETF-model coefficient comparison


Variable,Broad_USD_Change,ETF_Flow_L1_Calendar,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed,Real10Y_Change_pp,VIX_Change
Model,,,,,,,
E0_AR1_Only,NaN,0.5982,NaN,NaN,NaN,NaN,NaN
E1_Direction,NaN,0.4206,-58.8016,NaN,NaN,NaN,NaN
E2_Uncertainty,NaN,0.4217,-58.3515,2.4542,NaN,NaN,NaN
E3_Fed_Complete,NaN,0.4355,-61.0304,8.6295,37.8361,NaN,NaN
E4_Core_Macro,-4.5049,0.4633,-58.1403,-3.3908,41.1509,-31.1353,1.7363



ETF-model HAC p-value comparison


Variable,Broad_USD_Change,ETF_Flow_L1_Calendar,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed,Real10Y_Change_pp,VIX_Change
Model,,,,,,,
E0_AR1_Only,NaN,0.0000,NaN,NaN,NaN,NaN,NaN
E1_Direction,NaN,0.0002,0.0005,NaN,NaN,NaN,NaN
E2_Uncertainty,NaN,0.0003,0.0021,0.9462,NaN,NaN,NaN
E3_Fed_Complete,NaN,0.0001,0.0004,0.8115,0.0143,NaN,NaN
E4_Core_Macro,0.3952,0.0003,0.0004,0.9105,0.0354,0.2836,0.2904


In [30]:
# CELL 26: Formal HAC-robust nested tests for ETF models

from scipy import stats
import pandas as pd
import numpy as np

etf_nested_comparisons = [
    {
        "Comparison": "E0 → E1",
        "Restricted_Model": "E0_AR1_Only",
        "Unrestricted_Model": "E1_Direction",
        "Added_Block": "Expected Fed direction",
        "Restrictions": [
            "Fed_Expected_Direction = 0",
        ],
    },
    {
        "Comparison": "E1 → E2",
        "Restricted_Model": "E1_Direction",
        "Unrestricted_Model": "E2_Uncertainty",
        "Added_Block": "Fed policy uncertainty",
        "Restrictions": [
            "Fed_Policy_Uncertainty = 0",
        ],
    },
    {
        "Comparison": "E2 → E3",
        "Restricted_Model": "E2_Uncertainty",
        "Unrestricted_Model": "E3_Fed_Complete",
        "Added_Block": "Signed Fed surprise",
        "Restrictions": [
            "Fed_Surprise_Signed = 0",
        ],
    },
    {
        "Comparison": "E3 → E4",
        "Restricted_Model": "E3_Fed_Complete",
        "Unrestricted_Model": "E4_Core_Macro",
        "Added_Block": (
            "Broad USD, real 10Y yield and VIX"
        ),
        "Restrictions": [
            "Broad_USD_Change = 0",
            "Real10Y_Change_pp = 0",
            "VIX_Change = 0",
        ],
    },
]

etf_nested_test_rows = []

for specification in etf_nested_comparisons:

    restricted_name = specification["Restricted_Model"]
    unrestricted_name = specification["Unrestricted_Model"]
    restrictions = specification["Restrictions"]

    restricted_ols = (
        etf_nested_results[restricted_name]["OLS"]
    )

    unrestricted_ols = (
        etf_nested_results[unrestricted_name]["OLS"]
    )

    unrestricted_hac = (
        etf_nested_results[unrestricted_name]["HAC"]
    )

    number_restrictions = len(restrictions)

    if int(restricted_ols.nobs) != int(unrestricted_ols.nobs):
        raise ValueError(
            f"{restricted_name} and {unrestricted_name} "
            "do not use the same observations."
        )

    # -------------------------------------------------------
    # Primary HAC-robust Wald test
    # -------------------------------------------------------

    hac_test = unrestricted_hac.wald_test(
        restrictions,
        use_f=False,
        scalar=True,
    )

    hac_statistic = float(
        np.asarray(
            hac_test.statistic
        ).reshape(-1)[0]
    )

    hac_p_value = float(
        np.asarray(
            hac_test.pvalue
        ).reshape(-1)[0]
    )

    # -------------------------------------------------------
    # Conventional partial-F test for reference
    # -------------------------------------------------------

    restricted_rss = float(
        np.sum(
            np.square(restricted_ols.resid)
        )
    )

    unrestricted_rss = float(
        np.sum(
            np.square(unrestricted_ols.resid)
        )
    )

    partial_f = (
        (
            restricted_rss
            - unrestricted_rss
        ) / number_restrictions
    ) / (
        unrestricted_rss
        / unrestricted_ols.df_resid
    )

    partial_f_p = float(
        stats.f.sf(
            partial_f,
            number_restrictions,
            unrestricted_ols.df_resid,
        )
    )

    restricted_row = (
        etf_model_comparison.loc[
            etf_model_comparison["Model"]
            == restricted_name
        ]
        .iloc[0]
    )

    unrestricted_row = (
        etf_model_comparison.loc[
            etf_model_comparison["Model"]
            == unrestricted_name
        ]
        .iloc[0]
    )

    etf_nested_test_rows.append({
        "Comparison":
            specification["Comparison"],
        "Added_Block":
            specification["Added_Block"],
        "Restrictions":
            number_restrictions,
        "N":
            int(unrestricted_ols.nobs),
        "HAC_Wald_Statistic":
            hac_statistic,
        "HAC_Wald_p":
            hac_p_value,
        "HAC_Reject_5pct":
            hac_p_value < 0.05,
        "HAC_Reject_10pct":
            hac_p_value < 0.10,
        "Conventional_Partial_F":
            partial_f,
        "Conventional_F_p":
            partial_f_p,
        "Delta_R2":
            unrestricted_row["R2"]
            - restricted_row["R2"],
        "Delta_Adjusted_R2":
            unrestricted_row["Adjusted_R2"]
            - restricted_row["Adjusted_R2"],
        "Delta_AIC":
            unrestricted_row["AIC"]
            - restricted_row["AIC"],
        "Delta_BIC":
            unrestricted_row["BIC"]
            - restricted_row["BIC"],
    })

etf_formal_nested_tests = pd.DataFrame(
    etf_nested_test_rows
)

# -----------------------------------------------------------
# Automated interpretation
# -----------------------------------------------------------

def interpret_etf_nested_test(row):

    if row["HAC_Wald_p"] < 0.05:
        significance = "Significant at 5%"
    elif row["HAC_Wald_p"] < 0.10:
        significance = "Significant only at 10%"
    else:
        significance = "Not significant"

    adjusted_r2 = (
        "Adjusted R² improves"
        if row["Delta_Adjusted_R2"] > 0
        else "Adjusted R² worsens"
    )

    aic = (
        "AIC improves"
        if row["Delta_AIC"] < 0
        else "AIC worsens"
    )

    bic = (
        "BIC improves"
        if row["Delta_BIC"] < 0
        else "BIC worsens"
    )

    return (
        f"{significance}; "
        f"{adjusted_r2}; {aic}; {bic}"
    )

etf_formal_nested_tests["Conclusion"] = (
    etf_formal_nested_tests.apply(
        interpret_etf_nested_test,
        axis=1,
    )
)

display(
    etf_formal_nested_tests.round(4)
)

# -----------------------------------------------------------
# Plot the formal HAC results
# -----------------------------------------------------------

etf_nested_plot = (
    etf_formal_nested_tests.copy()
)

etf_nested_plot["Negative_Log10_p"] = (
    -np.log10(
        etf_nested_plot[
            "HAC_Wald_p"
        ].clip(lower=1e-12)
    )
)

etf_nested_plot["Decision"] = np.select(
    [
        etf_nested_plot["HAC_Wald_p"] < 0.05,
        etf_nested_plot["HAC_Wald_p"] < 0.10,
    ],
    [
        "Significant at 5%",
        "Significant at 10% only",
    ],
    default="Not significant",
)

fig = px.bar(
    etf_nested_plot,
    x="Comparison",
    y="Negative_Log10_p",
    color="Decision",
    text="HAC_Wald_p",
    hover_data={
        "Added_Block": True,
        "HAC_Wald_Statistic": ":.4f",
        "HAC_Wald_p": ":.4f",
        "Delta_Adjusted_R2": ":.4f",
        "Delta_AIC": ":.4f",
        "Delta_BIC": ":.4f",
        "Negative_Log10_p": False,
    },
    color_discrete_map={
        "Significant at 5%": "#2ca02c",
        "Significant at 10% only": "#ffbf00",
        "Not significant": "#d62728",
    },
    title=(
        "Formal Nested Tests for the "
        "FOMC-to-ETF Flow Models"
    ),
)

fig.add_hline(
    y=-np.log10(0.05),
    line_dash="dash",
    line_color="red",
    annotation_text="5% threshold",
)

fig.add_hline(
    y=-np.log10(0.10),
    line_dash="dot",
    line_color="orange",
    annotation_text="10% threshold",
)

fig.update_traces(
    texttemplate="p=%{text:.4f}",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=620,
    xaxis_title="Nested-model comparison",
    yaxis_title="−log₁₀(HAC p-value)",
    legend_title="Decision",
)

fig.show()

,Comparison,Added_Block,Restrictions,N,HAC_Wald_Statistic,HAC_Wald_p,HAC_Reject_5pct,HAC_Reject_10pct,Conventional_Partial_F,Conventional_F_p,Delta_R2,Delta_Adjusted_R2,Delta_AIC,Delta_BIC,Conclusion
0,E0 → E1,Expected Fed direction,1,61,13.6957,0.0002,True,True,8.9994,0.0040,0.0935,0.0845,-6.7987,-4.6878,Significant at 5%; Adjusted R² improves; AIC i...
1,E1 → E2,Fed policy uncertainty,1,61,0.0046,0.9460,False,False,0.0053,0.9424,0.0001,-0.0109,1.9944,4.1052,Not significant; Adjusted R² worsens; AIC wors...
2,E2 → E3,Signed Fed surprise,1,61,6.3939,0.0115,True,True,4.9618,0.0299,0.0490,0.0412,-3.1786,-1.0677,Significant at 5%; Adjusted R² improves; AIC i...
3,E3 → E4,"Broad USD, real 10Y yield and VIX",3,61,3.3974,0.3343,False,False,1.3660,0.2632,0.0397,0.0114,1.4568,7.7895,Not significant; Adjusted R² improves; AIC wor...


In [31]:
# CELL 27: Estimate the parsimonious direction-and-surprise model

parsimonious_regressors = [
    "ETF_Flow_L1_Calendar",
    "Fed_Expected_Direction",
    "Fed_Surprise_Signed",
]

X_parsimonious = sm.add_constant(
    etf_model_common[
        parsimonious_regressors
    ].astype(float),
    has_constant="add",
)

y_parsimonious = (
    etf_model_common["ETF_Flow"]
    .astype(float)
)

etf_parsimonious_ols = sm.OLS(
    y_parsimonious,
    X_parsimonious,
).fit()

etf_parsimonious_hac = (
    etf_parsimonious_ols
    .get_robustcov_results(
        cov_type="HAC",
        maxlags=2,
        use_correction=True,
    )
)

parameter_names = X_parsimonious.columns.tolist()

etf_parsimonious_hac.model.data.xnames = (
    parameter_names
)

# -----------------------------------------------------------
# Coefficient table
# -----------------------------------------------------------

etf_parsimonious_coefficients = pd.DataFrame({
    "Variable": parameter_names,
    "Coefficient": etf_parsimonious_hac.params,
    "HAC_SE": etf_parsimonious_hac.bse,
    "HAC_t": etf_parsimonious_hac.tvalues,
    "HAC_p": etf_parsimonious_hac.pvalues,
    "CI_Lower_95pct":
        etf_parsimonious_hac.conf_int()[:, 0],
    "CI_Upper_95pct":
        etf_parsimonious_hac.conf_int()[:, 1],
})

etf_parsimonious_coefficients[
    "Significant_5pct"
] = (
    etf_parsimonious_coefficients["HAC_p"]
    < 0.05
)

etf_parsimonious_coefficients[
    "Significant_10pct"
] = (
    etf_parsimonious_coefficients["HAC_p"]
    < 0.10
)

print("Parsimonious ETF model: Direction + Surprise")
print(
    "Observations:",
    int(etf_parsimonious_ols.nobs)
)
print(
    "Period:",
    etf_model_common["Month"].min(),
    "to",
    etf_model_common["Month"].max()
)
print(
    "R-squared:",
    round(etf_parsimonious_ols.rsquared, 4)
)
print(
    "Adjusted R-squared:",
    round(
        etf_parsimonious_ols.rsquared_adj,
        4
    )
)
print(
    "AIC:",
    round(etf_parsimonious_ols.aic, 4)
)
print(
    "BIC:",
    round(etf_parsimonious_ols.bic, 4)
)

display(
    etf_parsimonious_coefficients.round(4)
)

# -----------------------------------------------------------
# Joint Fed-information test
# -----------------------------------------------------------

joint_fed_test = (
    etf_parsimonious_hac.wald_test(
        [
            "Fed_Expected_Direction = 0",
            "Fed_Surprise_Signed = 0",
        ],
        use_f=False,
        scalar=True,
    )
)

joint_fed_statistic = float(
    np.asarray(
        joint_fed_test.statistic
    ).reshape(-1)[0]
)

joint_fed_p = float(
    np.asarray(
        joint_fed_test.pvalue
    ).reshape(-1)[0]
)

# -----------------------------------------------------------
# Cumulative policy-information effect
#
# Because:
# Realized Action = Expected Direction + Surprise,
# equality of the two coefficients has a useful interpretation.
# -----------------------------------------------------------

coefficient_equality_test = (
    etf_parsimonious_hac.t_test(
        "Fed_Expected_Direction "
        "- Fed_Surprise_Signed = 0"
    )
)

equality_statistic = float(
    np.asarray(
        coefficient_equality_test.tvalue
    ).reshape(-1)[0]
)

equality_p = float(
    np.asarray(
        coefficient_equality_test.pvalue
    ).reshape(-1)[0]
)

fed_channel_tests = pd.DataFrame({
    "Test": [
        "Direction and surprise jointly equal zero",
        "Direction coefficient equals surprise coefficient",
    ],
    "Statistic": [
        joint_fed_statistic,
        equality_statistic,
    ],
    "p_value": [
        joint_fed_p,
        equality_p,
    ],
})

fed_channel_tests["Reject_5pct"] = (
    fed_channel_tests["p_value"] < 0.05
)

display(fed_channel_tests.round(4))

# -----------------------------------------------------------
# Compare E1, parsimonious model and E3
# -----------------------------------------------------------

comparison_models = {
    "E1_Direction": (
        etf_nested_results[
            "E1_Direction"
        ]["OLS"]
    ),
    "E1S_Direction_Surprise": (
        etf_parsimonious_ols
    ),
    "E3_Fed_Complete": (
        etf_nested_results[
            "E3_Fed_Complete"
        ]["OLS"]
    ),
}

parsimonious_comparison_rows = []

for model_name, result in comparison_models.items():

    parsimonious_comparison_rows.append({
        "Model": model_name,
        "Parameters": int(
            result.df_model + 1
        ),
        "N": int(result.nobs),
        "R2": result.rsquared,
        "Adjusted_R2": result.rsquared_adj,
        "AIC": result.aic,
        "BIC": result.bic,
    })

parsimonious_etf_comparison = pd.DataFrame(
    parsimonious_comparison_rows
)

parsimonious_etf_comparison[
    "Delta_AIC_From_Best"
] = (
    parsimonious_etf_comparison["AIC"]
    - parsimonious_etf_comparison["AIC"].min()
)

parsimonious_etf_comparison[
    "Delta_BIC_From_Best"
] = (
    parsimonious_etf_comparison["BIC"]
    - parsimonious_etf_comparison["BIC"].min()
)

display(
    parsimonious_etf_comparison.round(4)
)

# -----------------------------------------------------------
# Does uncertainty add anything after both direction
# and surprise are included?
# -----------------------------------------------------------

uncertainty_test = (
    etf_nested_results[
        "E3_Fed_Complete"
    ]["HAC"]
    .t_test(
        "Fed_Policy_Uncertainty = 0"
    )
)

uncertainty_incremental_result = pd.DataFrame({
    "Test": [
        (
            "Uncertainty adds information after "
            "direction and surprise"
        )
    ],
    "Coefficient": [
        float(
            np.asarray(
                uncertainty_test.effect
            ).reshape(-1)[0]
        )
    ],
    "HAC_t": [
        float(
            np.asarray(
                uncertainty_test.tvalue
            ).reshape(-1)[0]
        )
    ],
    "HAC_p": [
        float(
            np.asarray(
                uncertainty_test.pvalue
            ).reshape(-1)[0]
        )
    ],
})

display(
    uncertainty_incremental_result.round(4)
)

Parsimonious ETF model: Direction + Surprise
Observations: 61
Period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00
R-squared: 0.446
Adjusted R-squared: 0.4168
AIC: 656.6848
BIC: 665.1283


,Variable,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct,Significant_5pct,Significant_10pct
0,const,22.8698,6.4470,3.5473,0.0008,9.9598,35.7797,True,True
1,ETF_Flow_L1_Calendar,0.4316,0.0981,4.4007,0.0000,0.2352,0.6280,True,True
2,Fed_Expected_Direction,-62.5750,14.9736,-4.1790,0.0001,-92.5591,-32.5909,True,True
3,Fed_Surprise_Signed,37.4587,14.6691,2.5536,0.0134,8.0844,66.8330,True,True


,Test,Statistic,p_value,Reject_5pct
0,Direction and surprise jointly equal zero,21.6031,0.0,True
1,Direction coefficient equals surprise coefficient,-4.4929,0.0,True


,Model,Parameters,N,R2,Adjusted_R2,AIC,BIC,Delta_AIC_From_Best,Delta_BIC_From_Best
0,E1_Direction,3,61,0.3976,0.3768,659.7938,666.1264,3.1090,0.9981
1,E1S_Direction_Surprise,4,61,0.4460,0.4168,656.6848,665.1283,0.0000,0.0000
2,E3_Fed_Complete,5,61,0.4467,0.4071,658.6095,669.1639,1.9247,4.0356


,Test,Coefficient,HAC_t,HAC_p
0,Uncertainty adds information after direction a...,8.6295,0.2396,0.8115


In [32]:
# CELL 28: Diagnostics for the preferred ETF model

from statsmodels.stats.diagnostic import (
    acorr_breusch_godfrey,
    acorr_ljungbox,
    het_breuschpagan,
    het_arch,
    linear_reset,
    breaks_cusumolsresid,
)
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.outliers_influence import OLSInfluence
import pandas as pd
import numpy as np
import statsmodels.api as sm

preferred_etf_ols = etf_parsimonious_ols
preferred_etf_hac = etf_parsimonious_hac

diagnostic_rows = []

# -----------------------------------------------------------
# Residual serial correlation
# -----------------------------------------------------------

for lag in [1, 2, 3, 6]:

    lm_stat, lm_p, f_stat, f_p = (
        acorr_breusch_godfrey(
            preferred_etf_ols,
            nlags=lag,
        )
    )

    diagnostic_rows.append({
        "Category": "Serial correlation",
        "Test": f"Breusch-Godfrey LM({lag})",
        "Statistic": lm_stat,
        "p_value": lm_p,
        "Null_Hypothesis": (
            f"No serial correlation through lag {lag}"
        ),
    })

ljung_box = acorr_ljungbox(
    preferred_etf_ols.resid,
    lags=[3, 6, 12],
    return_df=True,
)

for lag in [3, 6, 12]:

    diagnostic_rows.append({
        "Category": "Serial correlation",
        "Test": f"Ljung-Box Q({lag})",
        "Statistic": ljung_box.loc[
            lag, "lb_stat"
        ],
        "p_value": ljung_box.loc[
            lag, "lb_pvalue"
        ],
        "Null_Hypothesis": (
            f"No residual autocorrelation through lag {lag}"
        ),
    })

# -----------------------------------------------------------
# Heteroskedasticity
# -----------------------------------------------------------

bp_lm, bp_lm_p, bp_f, bp_f_p = (
    het_breuschpagan(
        preferred_etf_ols.resid,
        preferred_etf_ols.model.exog,
        robust=True,
    )
)

diagnostic_rows.append({
    "Category": "Heteroskedasticity",
    "Test": "Koenker-Breusch-Pagan",
    "Statistic": bp_lm,
    "p_value": bp_lm_p,
    "Null_Hypothesis": "Homoskedastic residuals",
})

# -----------------------------------------------------------
# Conditional heteroskedasticity
# -----------------------------------------------------------

for lag in [1, 2, 3, 6]:

    arch_lm, arch_p, arch_f, arch_f_p = het_arch(
        preferred_etf_ols.resid,
        nlags=lag,
    )

    diagnostic_rows.append({
        "Category": "Conditional variance",
        "Test": f"ARCH LM({lag})",
        "Statistic": arch_lm,
        "p_value": arch_p,
        "Null_Hypothesis": (
            f"No ARCH effects through lag {lag}"
        ),
    })

# -----------------------------------------------------------
# Functional form
# -----------------------------------------------------------

for power in [2, 3]:

    reset_result = linear_reset(
        preferred_etf_ols,
        power=power,
        use_f=True,
    )

    diagnostic_rows.append({
        "Category": "Functional form",
        "Test": f"Ramsey RESET power {power}",
        "Statistic": float(
            np.asarray(
                reset_result.fvalue
            ).reshape(-1)[0]
        ),
        "p_value": float(
            np.asarray(
                reset_result.pvalue
            ).reshape(-1)[0]
        ),
        "Null_Hypothesis": (
            "Linear functional form is correctly specified"
        ),
    })

# -----------------------------------------------------------
# Parameter stability
# -----------------------------------------------------------

cusum_stat, cusum_p, cusum_critical = (
    breaks_cusumolsresid(
        preferred_etf_ols.resid,
        ddof=int(
            preferred_etf_ols.df_model + 1
        ),
    )
)

diagnostic_rows.append({
    "Category": "Parameter stability",
    "Test": "CUSUM",
    "Statistic": cusum_stat,
    "p_value": cusum_p,
    "Null_Hypothesis": (
        "Regression parameters are stable"
    ),
})

# -----------------------------------------------------------
# Residual normality
# -----------------------------------------------------------

jb_stat, jb_p, residual_skew, residual_kurtosis = (
    jarque_bera(
        preferred_etf_ols.resid
    )
)

diagnostic_rows.append({
    "Category": "Residual distribution",
    "Test": "Jarque-Bera",
    "Statistic": jb_stat,
    "p_value": jb_p,
    "Null_Hypothesis": (
        "Residuals are normally distributed"
    ),
})

etf_diagnostics = pd.DataFrame(
    diagnostic_rows
)

etf_diagnostics["Reject_Null_5pct"] = (
    etf_diagnostics["p_value"] < 0.05
)

etf_diagnostics["Conclusion"] = np.where(
    etf_diagnostics["Reject_Null_5pct"],
    "Potential problem detected",
    "No evidence of problem",
)

display(etf_diagnostics.round(4))

print(
    "Residual skewness:",
    round(residual_skew, 4)
)

print(
    "Residual kurtosis:",
    round(residual_kurtosis, 4)
)

# -----------------------------------------------------------
# Influence diagnostics
# -----------------------------------------------------------

etf_influence = OLSInfluence(
    preferred_etf_ols
)

influence_frame = (
    etf_influence.summary_frame()
)

number_observations = int(
    preferred_etf_ols.nobs
)

number_parameters = int(
    preferred_etf_ols.df_model + 1
)

cooks_threshold = 4 / number_observations
leverage_threshold = (
    2 * number_parameters
    / number_observations
)

influence_table = pd.DataFrame({
    "Month": etf_model_common["Month"],
    "ETF_Flow": y_parsimonious.to_numpy(),
    "Fitted_ETF_Flow":
        preferred_etf_ols.fittedvalues.to_numpy(),
    "Residual":
        preferred_etf_ols.resid.to_numpy(),
    "Studentised_Residual":
        influence_frame[
            "student_resid"
        ].to_numpy(),
    "Leverage":
        influence_frame[
            "hat_diag"
        ].to_numpy(),
    "Cooks_Distance":
        influence_frame[
            "cooks_d"
        ].to_numpy(),
})

influence_table[
    "Large_Studentised_Residual"
] = (
    influence_table[
        "Studentised_Residual"
    ].abs() > 2
)

influence_table[
    "High_Leverage"
] = (
    influence_table["Leverage"]
    > leverage_threshold
)

influence_table[
    "High_Cooks_Distance"
] = (
    influence_table["Cooks_Distance"]
    > cooks_threshold
)

influence_table[
    "Potentially_Influential"
] = (
    influence_table[
        [
            "Large_Studentised_Residual",
            "High_Leverage",
            "High_Cooks_Distance",
        ]
    ].any(axis=1)
)

influential_meetings = (
    influence_table.loc[
        influence_table[
            "Potentially_Influential"
        ]
    ]
    .sort_values(
        "Cooks_Distance",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "Cook's-distance threshold:",
    round(cooks_threshold, 4)
)

print(
    "Leverage threshold:",
    round(leverage_threshold, 4)
)

print(
    "Potentially influential meetings:",
    len(influential_meetings)
)

display(
    influential_meetings.round(4)
)

,Category,Test,Statistic,p_value,Null_Hypothesis,Reject_Null_5pct,Conclusion
0,Serial correlation,Breusch-Godfrey LM(1),6.9307,0.0085,No serial correlation through lag 1,True,Potential problem detected
1,Serial correlation,Breusch-Godfrey LM(2),7.2873,0.0262,No serial correlation through lag 2,True,Potential problem detected
2,Serial correlation,Breusch-Godfrey LM(3),7.3159,0.0625,No serial correlation through lag 3,False,No evidence of problem
3,Serial correlation,Breusch-Godfrey LM(6),8.4681,0.2058,No serial correlation through lag 6,False,No evidence of problem
4,Serial correlation,Ljung-Box Q(3),4.8158,0.1858,No residual autocorrelation through lag 3,False,No evidence of problem
5,Serial correlation,Ljung-Box Q(6),6.9147,0.3288,No residual autocorrelation through lag 6,False,No evidence of problem
6,Serial correlation,Ljung-Box Q(12),18.3470,0.1056,No residual autocorrelation through lag 12,False,No evidence of problem
7,Heteroskedasticity,Koenker-Breusch-Pagan,3.9553,0.2663,Homoskedastic residuals,False,No evidence of problem
8,Conditional variance,ARCH LM(1),0.1074,0.7431,No ARCH effects through lag 1,False,No evidence of problem
9,Conditional variance,ARCH LM(2),0.3843,0.8252,No ARCH effects through lag 2,False,No evidence of problem


Residual skewness: -0.3251
Residual kurtosis: 3.7959
Cook's-distance threshold: 0.0656
Leverage threshold: 0.1311
Potentially influential meetings: 8


,Month,ETF_Flow,Fitted_ETF_Flow,Residual,Studentised_Residual,Leverage,Cooks_Distance,Large_Studentised_Residual,High_Leverage,High_Cooks_Distance,Potentially_Influential
0,2022-03-01,187.3,67.0075,120.2925,2.6600,0.1303,0.2396,True,False,True,True
1,2020-06-01,104.3,154.3643,-50.0643,-1.1009,0.2023,0.0765,False,True,True,True
2,2020-11-01,-107.1,32.0014,-139.1014,-2.9303,0.0188,0.0363,True,False,False,True
3,2022-05-01,-53.3,-8.8040,-44.4960,-0.9351,0.1318,0.0332,False,True,False,True
4,2026-03-01,-84.8,35.3032,-120.1032,-2.4834,0.0197,0.0284,True,False,False,True
5,2019-06-01,126.8,12.4471,114.3529,2.3489,0.0170,0.0221,True,False,False,True
6,2020-04-01,170.1,145.9477,24.1523,0.5194,0.1798,0.0150,False,True,False,True
7,2019-07-01,52.2,30.8712,21.3288,0.4639,0.1987,0.0135,False,True,False,True


In [33]:
# CELL 29: Covariance and leave-one-meeting-out robustness

# -----------------------------------------------------------
# Compare conventional, HC3 and HAC inference
# -----------------------------------------------------------

etf_hc3 = preferred_etf_ols.get_robustcov_results(
    cov_type="HC3"
)

etf_hc3.model.data.xnames = (
    X_parsimonious.columns.tolist()
)

covariance_results = {
    "Conventional_OLS": preferred_etf_ols,
    "HC3": etf_hc3,
    "Event_Time_HAC_2": preferred_etf_hac,
}

covariance_rows = []

for covariance_name, result in covariance_results.items():

    names = result.model.exog_names

    for variable in [
        "ETF_Flow_L1_Calendar",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
    ]:

        position = names.index(variable)

        covariance_rows.append({
            "Covariance_Estimator":
                covariance_name,
            "Variable":
                variable,
            "Coefficient":
                result.params[position],
            "Standard_Error":
                result.bse[position],
            "t_Statistic":
                result.tvalues[position],
            "p_value":
                result.pvalues[position],
        })

covariance_comparison = pd.DataFrame(
    covariance_rows
)

display(
    covariance_comparison.round(4)
)

# -----------------------------------------------------------
# Leave one meeting out at a time
# -----------------------------------------------------------

leave_one_out_rows = []

for removed_position in range(
    len(etf_model_common)
):

    reduced_data = (
        etf_model_common
        .drop(index=removed_position)
        .reset_index(drop=True)
    )

    X_reduced = sm.add_constant(
        reduced_data[
            parsimonious_regressors
        ].astype(float),
        has_constant="add",
    )

    y_reduced = (
        reduced_data["ETF_Flow"]
        .astype(float)
    )

    reduced_ols = sm.OLS(
        y_reduced,
        X_reduced,
    ).fit()

    reduced_hac = (
        reduced_ols.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    reduced_names = X_reduced.columns.tolist()
    reduced_hac.model.data.xnames = reduced_names

    direction_position = reduced_names.index(
        "Fed_Expected_Direction"
    )

    surprise_position = reduced_names.index(
        "Fed_Surprise_Signed"
    )

    joint_test = reduced_hac.wald_test(
        [
            "Fed_Expected_Direction = 0",
            "Fed_Surprise_Signed = 0",
        ],
        use_f=False,
        scalar=True,
    )

    leave_one_out_rows.append({
        "Removed_Month":
            etf_model_common.loc[
                removed_position,
                "Month"
            ],
        "Direction_Coefficient":
            reduced_hac.params[
                direction_position
            ],
        "Direction_HAC_p":
            reduced_hac.pvalues[
                direction_position
            ],
        "Surprise_Coefficient":
            reduced_hac.params[
                surprise_position
            ],
        "Surprise_HAC_p":
            reduced_hac.pvalues[
                surprise_position
            ],
        "Joint_Fed_Block_p":
            float(
                np.asarray(
                    joint_test.pvalue
                ).reshape(-1)[0]
            ),
        "Adjusted_R2":
            reduced_ols.rsquared_adj,
        "AIC":
            reduced_ols.aic,
        "BIC":
            reduced_ols.bic,
    })

etf_leave_one_out = pd.DataFrame(
    leave_one_out_rows
)

# -----------------------------------------------------------
# Stability summary
# -----------------------------------------------------------

etf_loo_stability = pd.DataFrame({
    "Variable": [
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
    ],
    "Minimum_Coefficient": [
        etf_leave_one_out[
            "Direction_Coefficient"
        ].min(),
        etf_leave_one_out[
            "Surprise_Coefficient"
        ].min(),
    ],
    "Maximum_Coefficient": [
        etf_leave_one_out[
            "Direction_Coefficient"
        ].max(),
        etf_leave_one_out[
            "Surprise_Coefficient"
        ].max(),
    ],
    "Median_Coefficient": [
        etf_leave_one_out[
            "Direction_Coefficient"
        ].median(),
        etf_leave_one_out[
            "Surprise_Coefficient"
        ].median(),
    ],
    "Minimum_p": [
        etf_leave_one_out[
            "Direction_HAC_p"
        ].min(),
        etf_leave_one_out[
            "Surprise_HAC_p"
        ].min(),
    ],
    "Maximum_p": [
        etf_leave_one_out[
            "Direction_HAC_p"
        ].max(),
        etf_leave_one_out[
            "Surprise_HAC_p"
        ].max(),
    ],
    "Median_p": [
        etf_leave_one_out[
            "Direction_HAC_p"
        ].median(),
        etf_leave_one_out[
            "Surprise_HAC_p"
        ].median(),
    ],
    "Significant_5pct_Share": [
        (
            etf_leave_one_out[
                "Direction_HAC_p"
            ] < 0.05
        ).mean(),
        (
            etf_leave_one_out[
                "Surprise_HAC_p"
            ] < 0.05
        ).mean(),
    ],
    "Sign_Stable": [
        (
            etf_leave_one_out[
                "Direction_Coefficient"
            ] < 0
        ).all(),
        (
            etf_leave_one_out[
                "Surprise_Coefficient"
            ] > 0
        ).all(),
    ],
})

display(
    etf_loo_stability.round(4)
)

print(
    "\nLeast favourable direction results"
)

display(
    etf_leave_one_out[
        [
            "Removed_Month",
            "Direction_Coefficient",
            "Direction_HAC_p",
        ]
    ]
    .sort_values(
        "Direction_HAC_p",
        ascending=False,
    )
    .head(10)
    .round(4)
)

print(
    "\nLeast favourable surprise results"
)

display(
    etf_leave_one_out[
        [
            "Removed_Month",
            "Surprise_Coefficient",
            "Surprise_HAC_p",
        ]
    ]
    .sort_values(
        "Surprise_HAC_p",
        ascending=False,
    )
    .head(10)
    .round(4)
)

print(
    "\nLeast favourable joint Fed-block results"
)

display(
    etf_leave_one_out[
        [
            "Removed_Month",
            "Joint_Fed_Block_p",
        ]
    ]
    .sort_values(
        "Joint_Fed_Block_p",
        ascending=False,
    )
    .head(10)
    .round(4)
)

/tmp/ipykernel_1468/489916799.py:41: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipykernel_1468/489916799.py:43: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipykernel_1468/489916799.py:45: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipykernel_1468/489916799.py:47: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consi

,Covariance_Estimator,Variable,Coefficient,Standard_Error,t_Statistic,p_value
0,Conventional_OLS,ETF_Flow_L1_Calendar,0.4316,0.1214,3.5552,0.0008
1,Conventional_OLS,Fed_Expected_Direction,-62.5750,19.0367,-3.2871,0.0017
2,Conventional_OLS,Fed_Surprise_Signed,37.4587,16.7864,2.2315,0.0296
3,HC3,ETF_Flow_L1_Calendar,0.4316,0.1235,3.4941,0.0009
4,HC3,Fed_Expected_Direction,-62.5750,16.0885,-3.8894,0.0003
5,HC3,Fed_Surprise_Signed,37.4587,18.6134,2.0125,0.0489
6,Event_Time_HAC_2,ETF_Flow_L1_Calendar,0.4316,0.0981,4.4007,0.0000
7,Event_Time_HAC_2,Fed_Expected_Direction,-62.5750,14.9736,-4.1790,0.0001
8,Event_Time_HAC_2,Fed_Surprise_Signed,37.4587,14.6691,2.5536,0.0134


,Variable,Minimum_Coefficient,Maximum_Coefficient,Median_Coefficient,Minimum_p,Maximum_p,Median_p,Significant_5pct_Share,Sign_Stable
0,Fed_Expected_Direction,-69.0818,-56.3990,-62.5756,0.000,0.0007,0.0001,1.0,True
1,Fed_Surprise_Signed,22.3701,43.2393,37.5144,0.005,0.0380,0.0136,1.0,True



Least favourable direction results


,Removed_Month,Direction_Coefficient,Direction_HAC_p
25,2022-05-01,-56.3990,0.0007
8,2020-03-01,-57.6229,0.0006
34,2023-06-01,-60.5805,0.0005
26,2022-06-01,-64.6563,0.0004
31,2023-02-01,-61.4807,0.0003
27,2022-07-01,-58.5316,0.0003
28,2022-09-01,-58.4894,0.0002
46,2024-12-01,-63.4029,0.0002
49,2025-05-01,-58.6815,0.0002
55,2025-12-01,-60.8753,0.0002



Least favourable surprise results


,Removed_Month,Surprise_Coefficient,Surprise_HAC_p
9,2020-04-01,34.9934,0.0380
11,2020-07-01,35.7218,0.0271
56,2026-01-01,35.9147,0.0264
45,2024-11-01,34.3997,0.0250
32,2023-03-01,35.2302,0.0217
34,2023-06-01,35.9313,0.0204
36,2023-09-01,36.6192,0.0175
5,2019-10-01,37.4350,0.0172
30,2022-12-01,36.7238,0.0171
53,2025-09-01,38.8427,0.0170



Least favourable joint Fed-block results


,Removed_Month,Joint_Fed_Block_p
34,2023-06-01,0.0003
8,2020-03-01,0.0002
26,2022-06-01,0.0001
56,2026-01-01,0.0001
27,2022-07-01,0.0001
6,2019-12-01,0.0001
28,2022-09-01,0.0001
9,2020-04-01,0.0001
3,2019-07-01,0.0000
17,2021-04-01,0.0000


In [34]:
# CELL 30: Compare calendar-month and previous-meeting ETF dynamics

import statsmodels.api as sm
from statsmodels.stats.diagnostic import (
    acorr_breusch_godfrey,
    acorr_ljungbox,
)
import pandas as pd
import numpy as np

# -----------------------------------------------------------
# Construct event-time lags AFTER selecting meeting months
# -----------------------------------------------------------

etf_dynamic_df = (
    etf_event_df
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

# Previous FOMC meeting's ETF flow.
etf_dynamic_df["ETF_Flow_L1_Meeting"] = (
    etf_dynamic_df["ETF_Flow"].shift(1)
)

etf_dynamic_df["ETF_Flow_L2_Meeting"] = (
    etf_dynamic_df["ETF_Flow"].shift(2)
)

# Number of calendar months since the preceding meeting.
etf_dynamic_df["Months_Since_Previous_Meeting"] = (
    etf_dynamic_df["Month"]
    .dt.to_period("M")
    .astype(int)
    .diff()
)

# -----------------------------------------------------------
# Dynamic specifications
# -----------------------------------------------------------

etf_dynamic_specifications = {
    "D1_Calendar_L1": [
        "ETF_Flow_L1_Calendar",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
    ],

    "D2_Meeting_L1": [
        "ETF_Flow_L1_Meeting",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
    ],

    "D3_Calendar_and_Meeting_L1": [
        "ETF_Flow_L1_Calendar",
        "ETF_Flow_L1_Meeting",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
    ],

    "D4_Meeting_L1_L2": [
        "ETF_Flow_L1_Meeting",
        "ETF_Flow_L2_Meeting",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
    ],
}

# -----------------------------------------------------------
# Identical sample for valid information-criterion comparison
# -----------------------------------------------------------

all_dynamic_regressors = list(dict.fromkeys(
    variable
    for regressors in etf_dynamic_specifications.values()
    for variable in regressors
))

etf_dynamic_common = (
    etf_dynamic_df[
        [
            "Month",
            "ETF_Flow",
            "Months_Since_Previous_Meeting",
        ] + all_dynamic_regressors
    ]
    .dropna()
    .reset_index(drop=True)
)

print(
    "Common dynamic-sample observations:",
    len(etf_dynamic_common)
)

print(
    "Period:",
    etf_dynamic_common["Month"].min(),
    "to",
    etf_dynamic_common["Month"].max()
)

print("\nSpacing between successive FOMC meeting months")

display(
    etf_dynamic_common[
        "Months_Since_Previous_Meeting"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("Calendar_Month_Gap")
    .reset_index(name="Observations")
)

# -----------------------------------------------------------
# Estimate all dynamic models
# -----------------------------------------------------------

etf_dynamic_results = {}
dynamic_model_rows = []
dynamic_coefficient_rows = []

y_dynamic = (
    etf_dynamic_common["ETF_Flow"]
    .astype(float)
)

for model_name, regressors in (
    etf_dynamic_specifications.items()
):

    X_dynamic = sm.add_constant(
        etf_dynamic_common[regressors]
        .astype(float),
        has_constant="add",
    )

    dynamic_ols = sm.OLS(
        y_dynamic,
        X_dynamic,
    ).fit()

    dynamic_hac = (
        dynamic_ols.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    parameter_names = X_dynamic.columns.tolist()
    dynamic_hac.model.data.xnames = parameter_names

    # Serial-correlation tests
    bg1 = acorr_breusch_godfrey(
        dynamic_ols,
        nlags=1,
    )

    bg2 = acorr_breusch_godfrey(
        dynamic_ols,
        nlags=2,
    )

    ljung_box = acorr_ljungbox(
        dynamic_ols.resid,
        lags=[2],
        return_df=True,
    )

    direction_position = parameter_names.index(
        "Fed_Expected_Direction"
    )

    surprise_position = parameter_names.index(
        "Fed_Surprise_Signed"
    )

    fed_joint_test = dynamic_hac.wald_test(
        [
            "Fed_Expected_Direction = 0",
            "Fed_Surprise_Signed = 0",
        ],
        use_f=False,
        scalar=True,
    )

    dynamic_model_rows.append({
        "Model": model_name,
        "Parameters": int(
            dynamic_ols.df_model + 1
        ),
        "N": int(dynamic_ols.nobs),
        "R2": dynamic_ols.rsquared,
        "Adjusted_R2": dynamic_ols.rsquared_adj,
        "AIC": dynamic_ols.aic,
        "BIC": dynamic_ols.bic,
        "Direction_Coefficient":
            dynamic_hac.params[
                direction_position
            ],
        "Direction_HAC_p":
            dynamic_hac.pvalues[
                direction_position
            ],
        "Surprise_Coefficient":
            dynamic_hac.params[
                surprise_position
            ],
        "Surprise_HAC_p":
            dynamic_hac.pvalues[
                surprise_position
            ],
        "Joint_Fed_Block_p":
            float(
                np.asarray(
                    fed_joint_test.pvalue
                ).reshape(-1)[0]
            ),
        "BG_LM1_p": float(bg1[1]),
        "BG_LM2_p": float(bg2[1]),
        "Ljung_Box_Q2_p": float(
            ljung_box.loc[2, "lb_pvalue"]
        ),
    })

    for position, variable in enumerate(
        parameter_names
    ):
        dynamic_coefficient_rows.append({
            "Model": model_name,
            "Variable": variable,
            "Coefficient":
                dynamic_hac.params[position],
            "HAC_SE":
                dynamic_hac.bse[position],
            "HAC_t":
                dynamic_hac.tvalues[position],
            "HAC_p":
                dynamic_hac.pvalues[position],
        })

    etf_dynamic_results[model_name] = {
        "OLS": dynamic_ols,
        "HAC": dynamic_hac,
        "Regressors": regressors,
    }

etf_dynamic_comparison = pd.DataFrame(
    dynamic_model_rows
)

etf_dynamic_comparison[
    "Serial_Correlation_Passes_5pct"
] = (
    (etf_dynamic_comparison["BG_LM1_p"] >= 0.05)
    & (etf_dynamic_comparison["BG_LM2_p"] >= 0.05)
)

display(
    etf_dynamic_comparison
    .sort_values("BIC")
    .round(4)
)

# -----------------------------------------------------------
# Complete coefficient comparison
# -----------------------------------------------------------

etf_dynamic_coefficients = pd.DataFrame(
    dynamic_coefficient_rows
)

dynamic_coefficient_comparison = (
    etf_dynamic_coefficients
    .pivot(
        index="Model",
        columns="Variable",
        values="Coefficient",
    )
    .reindex(etf_dynamic_specifications.keys())
)

dynamic_pvalue_comparison = (
    etf_dynamic_coefficients
    .pivot(
        index="Model",
        columns="Variable",
        values="HAC_p",
    )
    .reindex(etf_dynamic_specifications.keys())
)

print("\nDynamic-model coefficients")
display(
    dynamic_coefficient_comparison.round(4)
)

print("\nDynamic-model HAC p-values")
display(
    dynamic_pvalue_comparison.round(4)
)

Common dynamic-sample observations: 60
Period: 2019-05-01 00:00:00 to 2026-07-01 00:00:00

Spacing between successive FOMC meeting months


,Calendar_Month_Gap,Observations
0,1.0,32
1,2.0,28


,Model,Parameters,N,R2,Adjusted_R2,AIC,BIC,Direction_Coefficient,Direction_HAC_p,Surprise_Coefficient,Surprise_HAC_p,Joint_Fed_Block_p,BG_LM1_p,BG_LM2_p,Ljung_Box_Q2_p,Serial_Correlation_Passes_5pct
2,D3_Calendar_and_Meeting_L1,5,60,0.4880,0.4508,644.2537,654.7254,-63.9174,0.0002,42.6366,0.0074,0.0000,0.1672,0.3846,0.6347,True
0,D1_Calendar_L1,4,60,0.4455,0.4158,647.0423,655.4197,-62.5669,0.0001,37.4588,0.0135,0.0000,0.0088,0.0269,0.1012,False
1,D2_Meeting_L1,4,60,0.3454,0.3103,657.0006,665.3780,-83.0902,0.0000,33.5684,0.0760,0.0000,0.1968,0.4337,0.7000,True
3,D4_Meeting_L1_L2,5,60,0.3527,0.3056,658.3292,668.8009,-80.0944,0.0001,33.3684,0.0860,0.0001,0.3195,0.6060,0.8887,True



Dynamic-model coefficients


Variable,ETF_Flow_L1_Calendar,ETF_Flow_L1_Meeting,ETF_Flow_L2_Meeting,Fed_Expected_Direction,Fed_Surprise_Signed,const
Model,,,,,,
D1_Calendar_L1,0.4317,NaN,NaN,-62.5669,37.4588,22.8617
D2_Meeting_L1,NaN,0.1608,NaN,-83.0902,33.5684,25.4632
D3_Calendar_and_Meeting_L1,0.7569,-0.3650,NaN,-63.9174,42.6366,26.8348
D4_Meeting_L1_L2,NaN,0.1343,0.0941,-80.0944,33.3684,23.7921



Dynamic-model HAC p-values


Variable,ETF_Flow_L1_Calendar,ETF_Flow_L1_Meeting,ETF_Flow_L2_Meeting,Fed_Expected_Direction,Fed_Surprise_Signed,const
Model,,,,,,
D1_Calendar_L1,0.0001,NaN,NaN,0.0001,0.0135,0.0010
D2_Meeting_L1,NaN,0.1726,NaN,0.0000,0.0760,0.0033
D3_Calendar_and_Meeting_L1,0.0000,0.0469,NaN,0.0002,0.0074,0.0005
D4_Meeting_L1_L2,NaN,0.2533,0.4726,0.0001,0.0860,0.0075


In [35]:
# CELL 31: Validate the selected D3 ETF-flow model

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import pandas as pd
import numpy as np

selected_dynamic_name = (
    "D3_Calendar_and_Meeting_L1"
)

selected_dynamic_ols = (
    etf_dynamic_results[
        selected_dynamic_name
    ]["OLS"]
)

selected_dynamic_hac = (
    etf_dynamic_results[
        selected_dynamic_name
    ]["HAC"]
)

selected_dynamic_regressors = (
    etf_dynamic_results[
        selected_dynamic_name
    ]["Regressors"]
)

parameter_names = (
    selected_dynamic_ols.model.exog_names
)

print("Selected model:", selected_dynamic_name)
print(
    "Observations:",
    int(selected_dynamic_ols.nobs)
)
print(
    "Period:",
    etf_dynamic_common["Month"].min(),
    "to",
    etf_dynamic_common["Month"].max()
)
print(
    "Adjusted R-squared:",
    round(selected_dynamic_ols.rsquared_adj, 4)
)
print(
    "AIC:",
    round(selected_dynamic_ols.aic, 4)
)
print(
    "BIC:",
    round(selected_dynamic_ols.bic, 4)
)

# -----------------------------------------------------------
# HAC coefficient table
# -----------------------------------------------------------

selected_dynamic_coefficients = pd.DataFrame({
    "Variable": parameter_names,
    "Coefficient": selected_dynamic_hac.params,
    "HAC_SE": selected_dynamic_hac.bse,
    "HAC_t": selected_dynamic_hac.tvalues,
    "HAC_p": selected_dynamic_hac.pvalues,
    "CI_Lower_95pct":
        selected_dynamic_hac.conf_int()[:, 0],
    "CI_Upper_95pct":
        selected_dynamic_hac.conf_int()[:, 1],
})

selected_dynamic_coefficients[
    "Significant_5pct"
] = (
    selected_dynamic_coefficients["HAC_p"]
    < 0.05
)

selected_dynamic_coefficients[
    "Significant_10pct"
] = (
    selected_dynamic_coefficients["HAC_p"]
    < 0.10
)

display(
    selected_dynamic_coefficients.round(4)
)

# -----------------------------------------------------------
# Multicollinearity diagnostics
# -----------------------------------------------------------

X_selected = (
    etf_dynamic_common[
        selected_dynamic_regressors
    ]
    .astype(float)
)

selected_correlation = X_selected.corr()

print("\nRegressor correlation matrix")
display(selected_correlation.round(4))

X_selected_standardized = pd.DataFrame(
    StandardScaler().fit_transform(
        X_selected
    ),
    columns=X_selected.columns,
    index=X_selected.index,
)

selected_vif = pd.DataFrame({
    "Variable": X_selected_standardized.columns,
    "VIF": [
        variance_inflation_factor(
            X_selected_standardized.values,
            position,
        )
        for position in range(
            X_selected_standardized.shape[1]
        )
    ],
})

selected_vif["Assessment"] = pd.cut(
    selected_vif["VIF"],
    bins=[
        -np.inf,
        5,
        10,
        np.inf,
    ],
    labels=[
        "Acceptable",
        "Moderate",
        "High",
    ],
)

selected_vif = (
    selected_vif
    .sort_values(
        "VIF",
        ascending=False,
    )
    .reset_index(drop=True)
)

selected_condition_number = np.linalg.cond(
    X_selected_standardized.to_numpy()
)

print(
    "Standardised condition number:",
    round(selected_condition_number, 4)
)

display(selected_vif.round(4))

# -----------------------------------------------------------
# Test the persistence block
# -----------------------------------------------------------

persistence_joint_test = (
    selected_dynamic_hac.wald_test(
        [
            "ETF_Flow_L1_Calendar = 0",
            "ETF_Flow_L1_Meeting = 0",
        ],
        use_f=False,
        scalar=True,
    )
)

# Combined persistence for consecutive-month meetings.
combined_persistence_test = (
    selected_dynamic_hac.t_test(
        "ETF_Flow_L1_Calendar "
        "+ ETF_Flow_L1_Meeting = 0"
    )
)

# Is combined persistence equal to one?
unit_persistence_test = (
    selected_dynamic_hac.t_test(
        "ETF_Flow_L1_Calendar "
        "+ ETF_Flow_L1_Meeting = 1"
    )
)

def get_test_value(test_object, attribute):

    return float(
        np.asarray(
            getattr(test_object, attribute)
        ).reshape(-1)[0]
    )

dynamic_restriction_tests = pd.DataFrame({
    "Test": [
        "Both persistence coefficients equal zero",
        "Combined consecutive-month persistence equals zero",
        "Combined consecutive-month persistence equals one",
    ],
    "Statistic": [
        get_test_value(
            persistence_joint_test,
            "statistic",
        ),
        get_test_value(
            combined_persistence_test,
            "tvalue",
        ),
        get_test_value(
            unit_persistence_test,
            "tvalue",
        ),
    ],
    "p_value": [
        get_test_value(
            persistence_joint_test,
            "pvalue",
        ),
        get_test_value(
            combined_persistence_test,
            "pvalue",
        ),
        get_test_value(
            unit_persistence_test,
            "pvalue",
        ),
    ],
})

dynamic_restriction_tests[
    "Reject_5pct"
] = (
    dynamic_restriction_tests["p_value"]
    < 0.05
)

display(
    dynamic_restriction_tests.round(4)
)

# Report the combined persistence estimate directly.
calendar_lag_position = parameter_names.index(
    "ETF_Flow_L1_Calendar"
)

meeting_lag_position = parameter_names.index(
    "ETF_Flow_L1_Meeting"
)

combined_persistence = (
    selected_dynamic_hac.params[
        calendar_lag_position
    ]
    + selected_dynamic_hac.params[
        meeting_lag_position
    ]
)

print(
    "Combined persistence for consecutive-month meetings:",
    round(combined_persistence, 4)
)

# -----------------------------------------------------------
# Compare conventional, HC3 and event-time HAC inference
# -----------------------------------------------------------

selected_dynamic_hc3 = (
    selected_dynamic_ols
    .get_robustcov_results(
        cov_type="HC3"
    )
)

selected_dynamic_hc3.model.data.xnames = (
    parameter_names
)

covariance_estimators = {
    "Conventional_OLS":
        selected_dynamic_ols,
    "HC3":
        selected_dynamic_hc3,
    "Event_Time_HAC_2":
        selected_dynamic_hac,
}

selected_covariance_rows = []

for covariance_name, result in (
    covariance_estimators.items()
):

    for variable in parameter_names:

        if variable == "const":
            continue

        position = parameter_names.index(
            variable
        )

        selected_covariance_rows.append({
            "Covariance_Estimator":
                covariance_name,
            "Variable":
                variable,
            "Coefficient":
                result.params[position],
            "Standard_Error":
                result.bse[position],
            "t_Statistic":
                result.tvalues[position],
            "p_value":
                result.pvalues[position],
        })

selected_covariance_comparison = pd.DataFrame(
    selected_covariance_rows
)

display(
    selected_covariance_comparison.round(4)
)

Selected model: D3_Calendar_and_Meeting_L1
Observations: 60
Period: 2019-05-01 00:00:00 to 2026-07-01 00:00:00
Adjusted R-squared: 0.4508
AIC: 644.2537
BIC: 654.7254


,Variable,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct,Significant_5pct,Significant_10pct
0,const,26.8348,7.2188,3.7174,0.0005,12.3681,41.3015,True,True
1,ETF_Flow_L1_Calendar,0.7569,0.1677,4.5128,0.0000,0.4208,1.0930,True,True
2,ETF_Flow_L1_Meeting,-0.3650,0.1796,-2.0327,0.0469,-0.7248,-0.0051,True,True
3,Fed_Expected_Direction,-63.9174,15.7584,-4.0561,0.0002,-95.4979,-32.3368,True,True
4,Fed_Surprise_Signed,42.6366,15.3399,2.7795,0.0074,11.8948,73.3785,True,True



Regressor correlation matrix


,ETF_Flow_L1_Calendar,ETF_Flow_L1_Meeting,Fed_Expected_Direction,Fed_Surprise_Signed
ETF_Flow_L1_Calendar,1.0000,0.8209,-0.4744,-0.0933
ETF_Flow_L1_Meeting,0.8209,1.0000,-0.3998,0.0050
Fed_Expected_Direction,-0.4744,-0.3998,1.0000,0.1220
Fed_Surprise_Signed,-0.0933,0.0050,0.1220,1.0000


Standardised condition number: 3.5688


,Variable,VIF,Assessment
0,ETF_Flow_L1_Calendar,3.3917,Acceptable
1,ETF_Flow_L1_Meeting,3.1342,Acceptable
2,Fed_Expected_Direction,1.3022,Acceptable
3,Fed_Surprise_Signed,1.0389,Acceptable


,Test,Statistic,p_value,Reject_5pct
0,Both persistence coefficients equal zero,27.6374,0.0000,True
1,Combined consecutive-month persistence equals ...,3.5942,0.0007,True
2,Combined consecutive-month persistence equals one,-5.5774,0.0000,True


Combined persistence for consecutive-month meetings: 0.3919


/tmp/ipykernel_1468/2946220939.py:321: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipykernel_1468/2946220939.py:323: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipykernel_1468/2946220939.py:325: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipykernel_1468/2946220939.py:327: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as label

,Covariance_Estimator,Variable,Coefficient,Standard_Error,t_Statistic,p_value
0,Conventional_OLS,ETF_Flow_L1_Calendar,0.7569,0.1933,3.9146,0.0003
1,Conventional_OLS,ETF_Flow_L1_Meeting,-0.3650,0.1707,-2.1376,0.0370
2,Conventional_OLS,Fed_Expected_Direction,-63.9174,18.6596,-3.4254,0.0012
3,Conventional_OLS,Fed_Surprise_Signed,42.6366,16.5981,2.5688,0.0129
4,HC3,ETF_Flow_L1_Calendar,0.7569,0.2309,3.2773,0.0018
5,HC3,ETF_Flow_L1_Meeting,-0.3650,0.2127,-1.7159,0.0918
6,HC3,Fed_Expected_Direction,-63.9174,15.3363,-4.1677,0.0001
7,HC3,Fed_Surprise_Signed,42.6366,19.0333,2.2401,0.0291
8,Event_Time_HAC_2,ETF_Flow_L1_Calendar,0.7569,0.1677,4.5128,0.0000
9,Event_Time_HAC_2,ETF_Flow_L1_Meeting,-0.3650,0.1796,-2.0327,0.0469


In [36]:
# CELL 32: Final diagnostics and influence robustness for D3

from statsmodels.stats.diagnostic import (
    acorr_breusch_godfrey,
    acorr_ljungbox,
    het_breuschpagan,
    het_arch,
    linear_reset,
    breaks_cusumolsresid,
)
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.outliers_influence import OLSInfluence
import statsmodels.api as sm
import pandas as pd
import numpy as np

d3_ols = selected_dynamic_ols
d3_hac = selected_dynamic_hac

d3_regressors = [
    "ETF_Flow_L1_Calendar",
    "ETF_Flow_L1_Meeting",
    "Fed_Expected_Direction",
    "Fed_Surprise_Signed",
]

# -----------------------------------------------------------
# Complete residual diagnostic table
# -----------------------------------------------------------

d3_diagnostic_rows = []

for lag in [1, 2, 3, 6]:

    bg_result = acorr_breusch_godfrey(
        d3_ols,
        nlags=lag,
    )

    d3_diagnostic_rows.append({
        "Category": "Serial correlation",
        "Test": f"Breusch-Godfrey LM({lag})",
        "Statistic": float(bg_result[0]),
        "p_value": float(bg_result[1]),
        "Null_Hypothesis": (
            f"No residual serial correlation through lag {lag}"
        ),
    })

ljung_box_result = acorr_ljungbox(
    d3_ols.resid,
    lags=[3, 6, 12],
    return_df=True,
)

for lag in [3, 6, 12]:

    d3_diagnostic_rows.append({
        "Category": "Serial correlation",
        "Test": f"Ljung-Box Q({lag})",
        "Statistic": float(
            ljung_box_result.loc[
                lag,
                "lb_stat",
            ]
        ),
        "p_value": float(
            ljung_box_result.loc[
                lag,
                "lb_pvalue",
            ]
        ),
        "Null_Hypothesis": (
            f"No residual autocorrelation through lag {lag}"
        ),
    })

# Heteroskedasticity
bp_result = het_breuschpagan(
    d3_ols.resid,
    d3_ols.model.exog,
    robust=True,
)

d3_diagnostic_rows.append({
    "Category": "Heteroskedasticity",
    "Test": "Koenker-Breusch-Pagan",
    "Statistic": float(bp_result[0]),
    "p_value": float(bp_result[1]),
    "Null_Hypothesis": "Homoskedastic residuals",
})

# ARCH effects
for lag in [1, 2, 3, 6]:

    arch_result = het_arch(
        d3_ols.resid,
        nlags=lag,
    )

    d3_diagnostic_rows.append({
        "Category": "Conditional variance",
        "Test": f"ARCH LM({lag})",
        "Statistic": float(arch_result[0]),
        "p_value": float(arch_result[1]),
        "Null_Hypothesis": (
            f"No ARCH effects through lag {lag}"
        ),
    })

# Functional form
for power in [2, 3]:

    reset_result = linear_reset(
        d3_ols,
        power=power,
        use_f=True,
    )

    d3_diagnostic_rows.append({
        "Category": "Functional form",
        "Test": f"Ramsey RESET power {power}",
        "Statistic": float(
            np.asarray(
                reset_result.fvalue
            ).reshape(-1)[0]
        ),
        "p_value": float(
            np.asarray(
                reset_result.pvalue
            ).reshape(-1)[0]
        ),
        "Null_Hypothesis": (
            "Linear functional form is correctly specified"
        ),
    })

# Parameter stability
cusum_result = breaks_cusumolsresid(
    d3_ols.resid,
    ddof=int(d3_ols.df_model + 1),
)

d3_diagnostic_rows.append({
    "Category": "Parameter stability",
    "Test": "CUSUM",
    "Statistic": float(cusum_result[0]),
    "p_value": float(cusum_result[1]),
    "Null_Hypothesis": (
        "Regression parameters are stable"
    ),
})

# Residual normality
jb_result = jarque_bera(
    d3_ols.resid
)

d3_diagnostic_rows.append({
    "Category": "Residual distribution",
    "Test": "Jarque-Bera",
    "Statistic": float(jb_result[0]),
    "p_value": float(jb_result[1]),
    "Null_Hypothesis": (
        "Residuals are normally distributed"
    ),
})

d3_diagnostics = pd.DataFrame(
    d3_diagnostic_rows
)

d3_diagnostics["Reject_Null_5pct"] = (
    d3_diagnostics["p_value"] < 0.05
)

d3_diagnostics["Conclusion"] = np.where(
    d3_diagnostics["Reject_Null_5pct"],
    "Potential problem detected",
    "No evidence of problem",
)

display(d3_diagnostics.round(4))

print(
    "Residual skewness:",
    round(float(jb_result[2]), 4)
)

print(
    "Residual kurtosis:",
    round(float(jb_result[3]), 4)
)

# -----------------------------------------------------------
# Influence statistics
# -----------------------------------------------------------

d3_influence_object = OLSInfluence(
    d3_ols
)

d3_influence_summary = (
    d3_influence_object.summary_frame()
)

d3_n = int(d3_ols.nobs)
d3_k = int(d3_ols.df_model + 1)

d3_cooks_threshold = 4 / d3_n
d3_leverage_threshold = 2 * d3_k / d3_n

d3_influence_table = pd.DataFrame({
    "Month":
        etf_dynamic_common["Month"].to_numpy(),
    "ETF_Flow":
        etf_dynamic_common["ETF_Flow"].to_numpy(),
    "Fitted_ETF_Flow":
        np.asarray(d3_ols.fittedvalues),
    "Residual":
        np.asarray(d3_ols.resid),
    "Studentised_Residual":
        d3_influence_summary[
            "student_resid"
        ].to_numpy(),
    "Leverage":
        d3_influence_summary[
            "hat_diag"
        ].to_numpy(),
    "Cooks_Distance":
        d3_influence_summary[
            "cooks_d"
        ].to_numpy(),
})

d3_influence_table[
    "Large_Studentised_Residual"
] = (
    d3_influence_table[
        "Studentised_Residual"
    ].abs() > 2
)

d3_influence_table["High_Leverage"] = (
    d3_influence_table["Leverage"]
    > d3_leverage_threshold
)

d3_influence_table[
    "High_Cooks_Distance"
] = (
    d3_influence_table["Cooks_Distance"]
    > d3_cooks_threshold
)

d3_influence_table[
    "Potentially_Influential"
] = (
    d3_influence_table[
        [
            "Large_Studentised_Residual",
            "High_Leverage",
            "High_Cooks_Distance",
        ]
    ].any(axis=1)
)

d3_influential_meetings = (
    d3_influence_table.loc[
        d3_influence_table[
            "Potentially_Influential"
        ]
    ]
    .sort_values(
        "Cooks_Distance",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "\nCook's-distance threshold:",
    round(d3_cooks_threshold, 4)
)

print(
    "Leverage threshold:",
    round(d3_leverage_threshold, 4)
)

print(
    "Potentially influential meetings:",
    len(d3_influential_meetings)
)

display(
    d3_influential_meetings.round(4)
)

# -----------------------------------------------------------
# Leave-one-meeting-out estimation
# -----------------------------------------------------------

d3_leave_one_out_rows = []

for removed_position in range(
    len(etf_dynamic_common)
):

    reduced_df = (
        etf_dynamic_common
        .drop(index=removed_position)
        .reset_index(drop=True)
    )

    X_reduced = sm.add_constant(
        reduced_df[d3_regressors].astype(float),
        has_constant="add",
    )

    y_reduced = (
        reduced_df["ETF_Flow"].astype(float)
    )

    reduced_ols = sm.OLS(
        y_reduced,
        X_reduced,
    ).fit()

    reduced_hac = (
        reduced_ols.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    reduced_names = X_reduced.columns.tolist()
    reduced_hac.model.data.xnames = reduced_names

    reduced_params = np.asarray(
        reduced_hac.params
    )

    reduced_pvalues = np.asarray(
        reduced_hac.pvalues
    )

    def coefficient_and_p(variable):

        position = reduced_names.index(variable)

        return (
            float(reduced_params[position]),
            float(reduced_pvalues[position]),
        )

    calendar_coef, calendar_p = (
        coefficient_and_p(
            "ETF_Flow_L1_Calendar"
        )
    )

    meeting_coef, meeting_p = (
        coefficient_and_p(
            "ETF_Flow_L1_Meeting"
        )
    )

    direction_coef, direction_p = (
        coefficient_and_p(
            "Fed_Expected_Direction"
        )
    )

    surprise_coef, surprise_p = (
        coefficient_and_p(
            "Fed_Surprise_Signed"
        )
    )

    joint_fed_test = reduced_hac.wald_test(
        [
            "Fed_Expected_Direction = 0",
            "Fed_Surprise_Signed = 0",
        ],
        use_f=False,
        scalar=True,
    )

    d3_leave_one_out_rows.append({
        "Removed_Month":
            etf_dynamic_common.loc[
                removed_position,
                "Month",
            ],
        "Calendar_L1_Coefficient":
            calendar_coef,
        "Calendar_L1_p":
            calendar_p,
        "Meeting_L1_Coefficient":
            meeting_coef,
        "Meeting_L1_p":
            meeting_p,
        "Direction_Coefficient":
            direction_coef,
        "Direction_p":
            direction_p,
        "Surprise_Coefficient":
            surprise_coef,
        "Surprise_p":
            surprise_p,
        "Joint_Fed_Block_p":
            float(
                np.asarray(
                    joint_fed_test.pvalue
                ).reshape(-1)[0]
            ),
    })

d3_leave_one_out = pd.DataFrame(
    d3_leave_one_out_rows
)

# -----------------------------------------------------------
# Stability summary
# -----------------------------------------------------------

stability_definitions = [
    (
        "ETF_Flow_L1_Calendar",
        "Calendar_L1_Coefficient",
        "Calendar_L1_p",
        1,
    ),
    (
        "ETF_Flow_L1_Meeting",
        "Meeting_L1_Coefficient",
        "Meeting_L1_p",
        -1,
    ),
    (
        "Fed_Expected_Direction",
        "Direction_Coefficient",
        "Direction_p",
        -1,
    ),
    (
        "Fed_Surprise_Signed",
        "Surprise_Coefficient",
        "Surprise_p",
        1,
    ),
]

d3_stability_rows = []

for (
    variable,
    coefficient_column,
    pvalue_column,
    expected_sign,
) in stability_definitions:

    coefficients = d3_leave_one_out[
        coefficient_column
    ]

    pvalues = d3_leave_one_out[
        pvalue_column
    ]

    if expected_sign > 0:
        sign_stable = bool(
            (coefficients > 0).all()
        )
    else:
        sign_stable = bool(
            (coefficients < 0).all()
        )

    d3_stability_rows.append({
        "Variable": variable,
        "Minimum_Coefficient":
            coefficients.min(),
        "Maximum_Coefficient":
            coefficients.max(),
        "Median_Coefficient":
            coefficients.median(),
        "Minimum_p":
            pvalues.min(),
        "Maximum_p":
            pvalues.max(),
        "Median_p":
            pvalues.median(),
        "Significant_5pct_Share":
            (pvalues < 0.05).mean(),
        "Significant_10pct_Share":
            (pvalues < 0.10).mean(),
        "Sign_Stable":
            sign_stable,
    })

d3_stability_summary = pd.DataFrame(
    d3_stability_rows
)

display(
    d3_stability_summary.round(4)
)

print(
    "\nLeast favourable direction results"
)

display(
    d3_leave_one_out[
        [
            "Removed_Month",
            "Direction_Coefficient",
            "Direction_p",
        ]
    ]
    .sort_values(
        "Direction_p",
        ascending=False,
    )
    .head(10)
    .round(4)
)

print(
    "\nLeast favourable surprise results"
)

display(
    d3_leave_one_out[
        [
            "Removed_Month",
            "Surprise_Coefficient",
            "Surprise_p",
        ]
    ]
    .sort_values(
        "Surprise_p",
        ascending=False,
    )
    .head(10)
    .round(4)
)

print(
    "\nLeast favourable joint Fed-block results"
)

display(
    d3_leave_one_out[
        [
            "Removed_Month",
            "Joint_Fed_Block_p",
        ]
    ]
    .sort_values(
        "Joint_Fed_Block_p",
        ascending=False,
    )
    .head(10)
    .round(4)
)

,Category,Test,Statistic,p_value,Null_Hypothesis,Reject_Null_5pct,Conclusion
0,Serial correlation,Breusch-Godfrey LM(1),1.9082,0.1672,No residual serial correlation through lag 1,False,No evidence of problem
1,Serial correlation,Breusch-Godfrey LM(2),1.9109,0.3846,No residual serial correlation through lag 2,False,No evidence of problem
2,Serial correlation,Breusch-Godfrey LM(3),1.9545,0.5819,No residual serial correlation through lag 3,False,No evidence of problem
3,Serial correlation,Breusch-Godfrey LM(6),2.7723,0.8368,No residual serial correlation through lag 6,False,No evidence of problem
4,Serial correlation,Ljung-Box Q(3),0.9426,0.8151,No residual autocorrelation through lag 3,False,No evidence of problem
5,Serial correlation,Ljung-Box Q(6),1.6605,0.9481,No residual autocorrelation through lag 6,False,No evidence of problem
6,Serial correlation,Ljung-Box Q(12),12.7215,0.3896,No residual autocorrelation through lag 12,False,No evidence of problem
7,Heteroskedasticity,Koenker-Breusch-Pagan,5.4957,0.2401,Homoskedastic residuals,False,No evidence of problem
8,Conditional variance,ARCH LM(1),0.0015,0.9689,No ARCH effects through lag 1,False,No evidence of problem
9,Conditional variance,ARCH LM(2),0.1874,0.9106,No ARCH effects through lag 2,False,No evidence of problem


Residual skewness: -0.1277
Residual kurtosis: 3.3662

Cook's-distance threshold: 0.0667
Leverage threshold: 0.1667
Potentially influential meetings: 12


,Month,ETF_Flow,Fitted_ETF_Flow,Residual,Studentised_Residual,Leverage,Cooks_Distance,Large_Studentised_Residual,High_Leverage,High_Cooks_Distance,Potentially_Influential
0,2022-03-01,187.3,70.1746,117.1254,2.6530,0.1313,0.1918,True,False,True,True
1,2020-09-01,68.0,-4.1094,72.1094,1.6327,0.1929,0.1236,False,True,True,True
2,2026-03-01,-84.8,4.0272,-88.8272,-1.9285,0.1058,0.0839,False,False,True,True
3,2024-11-01,-28.5,67.6972,-96.1972,-2.0684,0.0795,0.0698,True,False,True,True
4,2025-05-01,-19.2,71.1232,-90.3232,-1.9403,0.0860,0.0674,False,False,True,True
5,2020-06-01,104.3,150.6587,-46.3587,-1.0418,0.2035,0.0554,False,True,False,True
6,2020-07-01,166.3,71.1937,95.1063,2.0245,0.0638,0.0529,True,False,False,True
7,2020-11-01,-107.1,17.7734,-124.8734,-2.6906,0.0368,0.0497,True,False,False,True
8,2019-07-01,52.2,24.0135,28.1865,0.6292,0.2030,0.0204,False,True,False,True
9,2019-06-01,126.8,15.8957,110.9043,2.3321,0.0184,0.0189,True,False,False,True


,Variable,Minimum_Coefficient,Maximum_Coefficient,Median_Coefficient,Minimum_p,Maximum_p,Median_p,Significant_5pct_Share,Significant_10pct_Share,Sign_Stable
0,ETF_Flow_L1_Calendar,0.6592,0.8574,0.7569,0.0000,0.0005,0.0000,1.0000,1.00,True
1,ETF_Flow_L1_Meeting,-0.4913,-0.2653,-0.3649,0.0192,0.1361,0.0477,0.6667,0.95,True
2,Fed_Expected_Direction,-71.3871,-58.3738,-63.9373,0.0000,0.0020,0.0002,1.0000,1.00,True
3,Fed_Surprise_Signed,27.7343,48.0502,42.6769,0.0008,0.0220,0.0075,1.0000,1.00,True



Least favourable direction results


,Removed_Month,Direction_Coefficient,Direction_p
24,2022-05-01,-65.2018,0.0020
7,2020-03-01,-59.9131,0.0007
33,2023-06-01,-62.0472,0.0006
30,2023-02-01,-62.9582,0.0004
17,2021-06-01,-62.2043,0.0004
45,2024-12-01,-64.9825,0.0003
54,2025-12-01,-61.8173,0.0003
11,2020-09-01,-64.1496,0.0003
48,2025-05-01,-59.7252,0.0003
5,2019-12-01,-63.5720,0.0003



Least favourable surprise results


,Removed_Month,Surprise_Coefficient,Surprise_p
8,2020-04-01,40.3268,0.0220
10,2020-07-01,40.8987,0.0152
44,2024-11-01,39.6636,0.0149
55,2026-01-01,41.2950,0.0138
31,2023-03-01,40.5558,0.0123
33,2023-06-01,41.1947,0.0108
45,2024-12-01,41.9646,0.0105
32,2023-05-01,41.7976,0.0099
35,2023-09-01,41.8388,0.0098
29,2022-12-01,41.9921,0.0097



Least favourable joint Fed-block results


,Removed_Month,Joint_Fed_Block_p
24,2022-05-01,0.0001
33,2023-06-01,0.0001
7,2020-03-01,0.0000
8,2020-04-01,0.0000
55,2026-01-01,0.0000
5,2019-12-01,0.0000
25,2022-06-01,0.0000
2,2019-07-01,0.0000
45,2024-12-01,0.0000
35,2023-09-01,0.0000


In [37]:
# CELL 32: Final diagnostics and influence robustness for D3

from statsmodels.stats.diagnostic import (
    acorr_breusch_godfrey,
    acorr_ljungbox,
    het_breuschpagan,
    het_arch,
    linear_reset,
    breaks_cusumolsresid,
)
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.outliers_influence import OLSInfluence
import statsmodels.api as sm
import pandas as pd
import numpy as np

d3_ols = selected_dynamic_ols
d3_hac = selected_dynamic_hac

d3_regressors = [
    "ETF_Flow_L1_Calendar",
    "ETF_Flow_L1_Meeting",
    "Fed_Expected_Direction",
    "Fed_Surprise_Signed",
]

# -----------------------------------------------------------
# Complete residual diagnostic table
# -----------------------------------------------------------

d3_diagnostic_rows = []

for lag in [1, 2, 3, 6]:

    bg_result = acorr_breusch_godfrey(
        d3_ols,
        nlags=lag,
    )

    d3_diagnostic_rows.append({
        "Category": "Serial correlation",
        "Test": f"Breusch-Godfrey LM({lag})",
        "Statistic": float(bg_result[0]),
        "p_value": float(bg_result[1]),
        "Null_Hypothesis": (
            f"No residual serial correlation through lag {lag}"
        ),
    })

ljung_box_result = acorr_ljungbox(
    d3_ols.resid,
    lags=[3, 6, 12],
    return_df=True,
)

for lag in [3, 6, 12]:

    d3_diagnostic_rows.append({
        "Category": "Serial correlation",
        "Test": f"Ljung-Box Q({lag})",
        "Statistic": float(
            ljung_box_result.loc[
                lag,
                "lb_stat",
            ]
        ),
        "p_value": float(
            ljung_box_result.loc[
                lag,
                "lb_pvalue",
            ]
        ),
        "Null_Hypothesis": (
            f"No residual autocorrelation through lag {lag}"
        ),
    })

# Heteroskedasticity
bp_result = het_breuschpagan(
    d3_ols.resid,
    d3_ols.model.exog,
    robust=True,
)

d3_diagnostic_rows.append({
    "Category": "Heteroskedasticity",
    "Test": "Koenker-Breusch-Pagan",
    "Statistic": float(bp_result[0]),
    "p_value": float(bp_result[1]),
    "Null_Hypothesis": "Homoskedastic residuals",
})

# ARCH effects
for lag in [1, 2, 3, 6]:

    arch_result = het_arch(
        d3_ols.resid,
        nlags=lag,
    )

    d3_diagnostic_rows.append({
        "Category": "Conditional variance",
        "Test": f"ARCH LM({lag})",
        "Statistic": float(arch_result[0]),
        "p_value": float(arch_result[1]),
        "Null_Hypothesis": (
            f"No ARCH effects through lag {lag}"
        ),
    })

# Functional form
for power in [2, 3]:

    reset_result = linear_reset(
        d3_ols,
        power=power,
        use_f=True,
    )

    d3_diagnostic_rows.append({
        "Category": "Functional form",
        "Test": f"Ramsey RESET power {power}",
        "Statistic": float(
            np.asarray(
                reset_result.fvalue
            ).reshape(-1)[0]
        ),
        "p_value": float(
            np.asarray(
                reset_result.pvalue
            ).reshape(-1)[0]
        ),
        "Null_Hypothesis": (
            "Linear functional form is correctly specified"
        ),
    })

# Parameter stability
cusum_result = breaks_cusumolsresid(
    d3_ols.resid,
    ddof=int(d3_ols.df_model + 1),
)

d3_diagnostic_rows.append({
    "Category": "Parameter stability",
    "Test": "CUSUM",
    "Statistic": float(cusum_result[0]),
    "p_value": float(cusum_result[1]),
    "Null_Hypothesis": (
        "Regression parameters are stable"
    ),
})

# Residual normality
jb_result = jarque_bera(
    d3_ols.resid
)

d3_diagnostic_rows.append({
    "Category": "Residual distribution",
    "Test": "Jarque-Bera",
    "Statistic": float(jb_result[0]),
    "p_value": float(jb_result[1]),
    "Null_Hypothesis": (
        "Residuals are normally distributed"
    ),
})

d3_diagnostics = pd.DataFrame(
    d3_diagnostic_rows
)

d3_diagnostics["Reject_Null_5pct"] = (
    d3_diagnostics["p_value"] < 0.05
)

d3_diagnostics["Conclusion"] = np.where(
    d3_diagnostics["Reject_Null_5pct"],
    "Potential problem detected",
    "No evidence of problem",
)

display(d3_diagnostics.round(4))

print(
    "Residual skewness:",
    round(float(jb_result[2]), 4)
)

print(
    "Residual kurtosis:",
    round(float(jb_result[3]), 4)
)

# -----------------------------------------------------------
# Influence statistics
# -----------------------------------------------------------

d3_influence_object = OLSInfluence(
    d3_ols
)

d3_influence_summary = (
    d3_influence_object.summary_frame()
)

d3_n = int(d3_ols.nobs)
d3_k = int(d3_ols.df_model + 1)

d3_cooks_threshold = 4 / d3_n
d3_leverage_threshold = 2 * d3_k / d3_n

d3_influence_table = pd.DataFrame({
    "Month":
        etf_dynamic_common["Month"].to_numpy(),
    "ETF_Flow":
        etf_dynamic_common["ETF_Flow"].to_numpy(),
    "Fitted_ETF_Flow":
        np.asarray(d3_ols.fittedvalues),
    "Residual":
        np.asarray(d3_ols.resid),
    "Studentised_Residual":
        d3_influence_summary[
            "student_resid"
        ].to_numpy(),
    "Leverage":
        d3_influence_summary[
            "hat_diag"
        ].to_numpy(),
    "Cooks_Distance":
        d3_influence_summary[
            "cooks_d"
        ].to_numpy(),
})

d3_influence_table[
    "Large_Studentised_Residual"
] = (
    d3_influence_table[
        "Studentised_Residual"
    ].abs() > 2
)

d3_influence_table["High_Leverage"] = (
    d3_influence_table["Leverage"]
    > d3_leverage_threshold
)

d3_influence_table[
    "High_Cooks_Distance"
] = (
    d3_influence_table["Cooks_Distance"]
    > d3_cooks_threshold
)

d3_influence_table[
    "Potentially_Influential"
] = (
    d3_influence_table[
        [
            "Large_Studentised_Residual",
            "High_Leverage",
            "High_Cooks_Distance",
        ]
    ].any(axis=1)
)

d3_influential_meetings = (
    d3_influence_table.loc[
        d3_influence_table[
            "Potentially_Influential"
        ]
    ]
    .sort_values(
        "Cooks_Distance",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "\nCook's-distance threshold:",
    round(d3_cooks_threshold, 4)
)

print(
    "Leverage threshold:",
    round(d3_leverage_threshold, 4)
)

print(
    "Potentially influential meetings:",
    len(d3_influential_meetings)
)

display(
    d3_influential_meetings.round(4)
)

# -----------------------------------------------------------
# Leave-one-meeting-out estimation
# -----------------------------------------------------------

d3_leave_one_out_rows = []

for removed_position in range(
    len(etf_dynamic_common)
):

    reduced_df = (
        etf_dynamic_common
        .drop(index=removed_position)
        .reset_index(drop=True)
    )

    X_reduced = sm.add_constant(
        reduced_df[d3_regressors].astype(float),
        has_constant="add",
    )

    y_reduced = (
        reduced_df["ETF_Flow"].astype(float)
    )

    reduced_ols = sm.OLS(
        y_reduced,
        X_reduced,
    ).fit()

    reduced_hac = (
        reduced_ols.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    reduced_names = X_reduced.columns.tolist()
    reduced_hac.model.data.xnames = reduced_names

    reduced_params = np.asarray(
        reduced_hac.params
    )

    reduced_pvalues = np.asarray(
        reduced_hac.pvalues
    )

    def coefficient_and_p(variable):

        position = reduced_names.index(variable)

        return (
            float(reduced_params[position]),
            float(reduced_pvalues[position]),
        )

    calendar_coef, calendar_p = (
        coefficient_and_p(
            "ETF_Flow_L1_Calendar"
        )
    )

    meeting_coef, meeting_p = (
        coefficient_and_p(
            "ETF_Flow_L1_Meeting"
        )
    )

    direction_coef, direction_p = (
        coefficient_and_p(
            "Fed_Expected_Direction"
        )
    )

    surprise_coef, surprise_p = (
        coefficient_and_p(
            "Fed_Surprise_Signed"
        )
    )

    joint_fed_test = reduced_hac.wald_test(
        [
            "Fed_Expected_Direction = 0",
            "Fed_Surprise_Signed = 0",
        ],
        use_f=False,
        scalar=True,
    )

    d3_leave_one_out_rows.append({
        "Removed_Month":
            etf_dynamic_common.loc[
                removed_position,
                "Month",
            ],
        "Calendar_L1_Coefficient":
            calendar_coef,
        "Calendar_L1_p":
            calendar_p,
        "Meeting_L1_Coefficient":
            meeting_coef,
        "Meeting_L1_p":
            meeting_p,
        "Direction_Coefficient":
            direction_coef,
        "Direction_p":
            direction_p,
        "Surprise_Coefficient":
            surprise_coef,
        "Surprise_p":
            surprise_p,
        "Joint_Fed_Block_p":
            float(
                np.asarray(
                    joint_fed_test.pvalue
                ).reshape(-1)[0]
            ),
    })

d3_leave_one_out = pd.DataFrame(
    d3_leave_one_out_rows
)

# -----------------------------------------------------------
# Stability summary
# -----------------------------------------------------------

stability_definitions = [
    (
        "ETF_Flow_L1_Calendar",
        "Calendar_L1_Coefficient",
        "Calendar_L1_p",
        1,
    ),
    (
        "ETF_Flow_L1_Meeting",
        "Meeting_L1_Coefficient",
        "Meeting_L1_p",
        -1,
    ),
    (
        "Fed_Expected_Direction",
        "Direction_Coefficient",
        "Direction_p",
        -1,
    ),
    (
        "Fed_Surprise_Signed",
        "Surprise_Coefficient",
        "Surprise_p",
        1,
    ),
]

d3_stability_rows = []

for (
    variable,
    coefficient_column,
    pvalue_column,
    expected_sign,
) in stability_definitions:

    coefficients = d3_leave_one_out[
        coefficient_column
    ]

    pvalues = d3_leave_one_out[
        pvalue_column
    ]

    if expected_sign > 0:
        sign_stable = bool(
            (coefficients > 0).all()
        )
    else:
        sign_stable = bool(
            (coefficients < 0).all()
        )

    d3_stability_rows.append({
        "Variable": variable,
        "Minimum_Coefficient":
            coefficients.min(),
        "Maximum_Coefficient":
            coefficients.max(),
        "Median_Coefficient":
            coefficients.median(),
        "Minimum_p":
            pvalues.min(),
        "Maximum_p":
            pvalues.max(),
        "Median_p":
            pvalues.median(),
        "Significant_5pct_Share":
            (pvalues < 0.05).mean(),
        "Significant_10pct_Share":
            (pvalues < 0.10).mean(),
        "Sign_Stable":
            sign_stable,
    })

d3_stability_summary = pd.DataFrame(
    d3_stability_rows
)

display(
    d3_stability_summary.round(4)
)

print(
    "\nLeast favourable direction results"
)

display(
    d3_leave_one_out[
        [
            "Removed_Month",
            "Direction_Coefficient",
            "Direction_p",
        ]
    ]
    .sort_values(
        "Direction_p",
        ascending=False,
    )
    .head(10)
    .round(4)
)

print(
    "\nLeast favourable surprise results"
)

display(
    d3_leave_one_out[
        [
            "Removed_Month",
            "Surprise_Coefficient",
            "Surprise_p",
        ]
    ]
    .sort_values(
        "Surprise_p",
        ascending=False,
    )
    .head(10)
    .round(4)
)

print(
    "\nLeast favourable joint Fed-block results"
)

display(
    d3_leave_one_out[
        [
            "Removed_Month",
            "Joint_Fed_Block_p",
        ]
    ]
    .sort_values(
        "Joint_Fed_Block_p",
        ascending=False,
    )
    .head(10)
    .round(4)
)

,Category,Test,Statistic,p_value,Null_Hypothesis,Reject_Null_5pct,Conclusion
0,Serial correlation,Breusch-Godfrey LM(1),1.9082,0.1672,No residual serial correlation through lag 1,False,No evidence of problem
1,Serial correlation,Breusch-Godfrey LM(2),1.9109,0.3846,No residual serial correlation through lag 2,False,No evidence of problem
2,Serial correlation,Breusch-Godfrey LM(3),1.9545,0.5819,No residual serial correlation through lag 3,False,No evidence of problem
3,Serial correlation,Breusch-Godfrey LM(6),2.7723,0.8368,No residual serial correlation through lag 6,False,No evidence of problem
4,Serial correlation,Ljung-Box Q(3),0.9426,0.8151,No residual autocorrelation through lag 3,False,No evidence of problem
5,Serial correlation,Ljung-Box Q(6),1.6605,0.9481,No residual autocorrelation through lag 6,False,No evidence of problem
6,Serial correlation,Ljung-Box Q(12),12.7215,0.3896,No residual autocorrelation through lag 12,False,No evidence of problem
7,Heteroskedasticity,Koenker-Breusch-Pagan,5.4957,0.2401,Homoskedastic residuals,False,No evidence of problem
8,Conditional variance,ARCH LM(1),0.0015,0.9689,No ARCH effects through lag 1,False,No evidence of problem
9,Conditional variance,ARCH LM(2),0.1874,0.9106,No ARCH effects through lag 2,False,No evidence of problem


Residual skewness: -0.1277
Residual kurtosis: 3.3662

Cook's-distance threshold: 0.0667
Leverage threshold: 0.1667
Potentially influential meetings: 12


,Month,ETF_Flow,Fitted_ETF_Flow,Residual,Studentised_Residual,Leverage,Cooks_Distance,Large_Studentised_Residual,High_Leverage,High_Cooks_Distance,Potentially_Influential
0,2022-03-01,187.3,70.1746,117.1254,2.6530,0.1313,0.1918,True,False,True,True
1,2020-09-01,68.0,-4.1094,72.1094,1.6327,0.1929,0.1236,False,True,True,True
2,2026-03-01,-84.8,4.0272,-88.8272,-1.9285,0.1058,0.0839,False,False,True,True
3,2024-11-01,-28.5,67.6972,-96.1972,-2.0684,0.0795,0.0698,True,False,True,True
4,2025-05-01,-19.2,71.1232,-90.3232,-1.9403,0.0860,0.0674,False,False,True,True
5,2020-06-01,104.3,150.6587,-46.3587,-1.0418,0.2035,0.0554,False,True,False,True
6,2020-07-01,166.3,71.1937,95.1063,2.0245,0.0638,0.0529,True,False,False,True
7,2020-11-01,-107.1,17.7734,-124.8734,-2.6906,0.0368,0.0497,True,False,False,True
8,2019-07-01,52.2,24.0135,28.1865,0.6292,0.2030,0.0204,False,True,False,True
9,2019-06-01,126.8,15.8957,110.9043,2.3321,0.0184,0.0189,True,False,False,True


,Variable,Minimum_Coefficient,Maximum_Coefficient,Median_Coefficient,Minimum_p,Maximum_p,Median_p,Significant_5pct_Share,Significant_10pct_Share,Sign_Stable
0,ETF_Flow_L1_Calendar,0.6592,0.8574,0.7569,0.0000,0.0005,0.0000,1.0000,1.00,True
1,ETF_Flow_L1_Meeting,-0.4913,-0.2653,-0.3649,0.0192,0.1361,0.0477,0.6667,0.95,True
2,Fed_Expected_Direction,-71.3871,-58.3738,-63.9373,0.0000,0.0020,0.0002,1.0000,1.00,True
3,Fed_Surprise_Signed,27.7343,48.0502,42.6769,0.0008,0.0220,0.0075,1.0000,1.00,True



Least favourable direction results


,Removed_Month,Direction_Coefficient,Direction_p
24,2022-05-01,-65.2018,0.0020
7,2020-03-01,-59.9131,0.0007
33,2023-06-01,-62.0472,0.0006
30,2023-02-01,-62.9582,0.0004
17,2021-06-01,-62.2043,0.0004
45,2024-12-01,-64.9825,0.0003
54,2025-12-01,-61.8173,0.0003
11,2020-09-01,-64.1496,0.0003
48,2025-05-01,-59.7252,0.0003
5,2019-12-01,-63.5720,0.0003



Least favourable surprise results


,Removed_Month,Surprise_Coefficient,Surprise_p
8,2020-04-01,40.3268,0.0220
10,2020-07-01,40.8987,0.0152
44,2024-11-01,39.6636,0.0149
55,2026-01-01,41.2950,0.0138
31,2023-03-01,40.5558,0.0123
33,2023-06-01,41.1947,0.0108
45,2024-12-01,41.9646,0.0105
32,2023-05-01,41.7976,0.0099
35,2023-09-01,41.8388,0.0098
29,2022-12-01,41.9921,0.0097



Least favourable joint Fed-block results


,Removed_Month,Joint_Fed_Block_p
24,2022-05-01,0.0001
33,2023-06-01,0.0001
7,2020-03-01,0.0000
8,2020-04-01,0.0000
55,2026-01-01,0.0000
5,2019-12-01,0.0000
25,2022-06-01,0.0000
2,2019-07-01,0.0000
45,2024-12-01,0.0000
35,2023-09-01,0.0000


In [38]:
# CELL 33: Forward ETF-flow models

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------------------------------------
# Forward-model specifications
# -----------------------------------------------------------

forward_etf_specifications = {
    # Current ETF flow provides the persistence benchmark.
    "P0_ETF_Persistence": [
        "ETF_Flow",
    ],

    # Add expected policy direction.
    "P1_Direction": [
        "ETF_Flow",
        "Fed_Expected_Direction",
    ],

    # Add realized signed surprise.
    "P2_Direction_Surprise": [
        "ETF_Flow",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
    ],

    # Retain uncertainty as a conceptual robustness variable.
    "P3_Complete_Fed": [
        "ETF_Flow",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
        "Fed_Policy_Uncertainty",
    ],

    # Add the principal financial transmission controls.
    "P4_Core_Macro": [
        "ETF_Flow",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
        "Fed_Policy_Uncertainty",
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
    ],
}

all_forward_regressors = list(dict.fromkeys(
    variable
    for regressors in forward_etf_specifications.values()
    for variable in regressors
))

# -----------------------------------------------------------
# Common sample for every nested specification
# -----------------------------------------------------------

forward_etf_common = (
    etf_event_df[
        [
            "Month",
            "ETF_Flow_Next_Month",
        ] + all_forward_regressors
    ]
    .dropna()
    .sort_values("Month")
    .reset_index(drop=True)
)

print(
    "Forward ETF-model observations:",
    len(forward_etf_common)
)

print(
    "FOMC-information period:",
    forward_etf_common["Month"].min(),
    "to",
    forward_etf_common["Month"].max()
)

print(
    "Outcome period:",
    (
        forward_etf_common["Month"].min()
        + pd.offsets.MonthBegin(1)
    ),
    "to",
    (
        forward_etf_common["Month"].max()
        + pd.offsets.MonthBegin(1)
    )
)

print(
    "Meeting months excluded:",
    etf_event_df.loc[
        ~etf_event_df["Month"].isin(
            forward_etf_common["Month"]
        ),
        "Month"
    ].dt.strftime("%Y-%m").tolist()
)

# -----------------------------------------------------------
# Estimate nested forward models
# -----------------------------------------------------------

forward_etf_results = {}
forward_model_rows = []
forward_coefficient_rows = []

y_forward = (
    forward_etf_common[
        "ETF_Flow_Next_Month"
    ].astype(float)
)

for model_name, regressors in (
    forward_etf_specifications.items()
):

    X_forward = sm.add_constant(
        forward_etf_common[
            regressors
        ].astype(float),
        has_constant="add",
    )

    forward_ols = sm.OLS(
        y_forward,
        X_forward,
    ).fit()

    forward_hac = (
        forward_ols.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    parameter_names = X_forward.columns.tolist()
    forward_hac.model.data.xnames = parameter_names

    coefficient_table = pd.DataFrame({
        "Model": model_name,
        "Variable": parameter_names,
        "Coefficient": np.asarray(
            forward_hac.params
        ),
        "HAC_SE": np.asarray(
            forward_hac.bse
        ),
        "HAC_t": np.asarray(
            forward_hac.tvalues
        ),
        "HAC_p": np.asarray(
            forward_hac.pvalues
        ),
        "CI_Lower_95pct":
            forward_hac.conf_int()[:, 0],
        "CI_Upper_95pct":
            forward_hac.conf_int()[:, 1],
    })

    coefficient_table[
        "Significant_5pct"
    ] = (
        coefficient_table["HAC_p"] < 0.05
    )

    coefficient_table[
        "Significant_10pct"
    ] = (
        coefficient_table["HAC_p"] < 0.10
    )

    forward_coefficient_rows.append(
        coefficient_table
    )

    # Fed block included in the particular model.
    included_fed_variables = [
        variable
        for variable in [
            "Fed_Expected_Direction",
            "Fed_Surprise_Signed",
            "Fed_Policy_Uncertainty",
        ]
        if variable in regressors
    ]

    if included_fed_variables:

        fed_block_test = (
            forward_hac.wald_test(
                [
                    f"{variable} = 0"
                    for variable
                    in included_fed_variables
                ],
                use_f=False,
                scalar=True,
            )
        )

        fed_block_statistic = float(
            np.asarray(
                fed_block_test.statistic
            ).reshape(-1)[0]
        )

        fed_block_p = float(
            np.asarray(
                fed_block_test.pvalue
            ).reshape(-1)[0]
        )

    else:
        fed_block_statistic = np.nan
        fed_block_p = np.nan

    # Core macro block.
    core_macro_variables = [
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
    ]

    if all(
        variable in regressors
        for variable in core_macro_variables
    ):

        macro_block_test = (
            forward_hac.wald_test(
                [
                    f"{variable} = 0"
                    for variable
                    in core_macro_variables
                ],
                use_f=False,
                scalar=True,
            )
        )

        macro_block_statistic = float(
            np.asarray(
                macro_block_test.statistic
            ).reshape(-1)[0]
        )

        macro_block_p = float(
            np.asarray(
                macro_block_test.pvalue
            ).reshape(-1)[0]
        )

    else:
        macro_block_statistic = np.nan
        macro_block_p = np.nan

    forward_model_rows.append({
        "Model": model_name,
        "Fed_Variables":
            len(included_fed_variables),
        "Parameters": int(
            forward_ols.df_model + 1
        ),
        "N": int(forward_ols.nobs),
        "R2": forward_ols.rsquared,
        "Adjusted_R2":
            forward_ols.rsquared_adj,
        "AIC": forward_ols.aic,
        "BIC": forward_ols.bic,
        "Fed_Block_Wald":
            fed_block_statistic,
        "Fed_Block_p":
            fed_block_p,
        "Core_Macro_Block_Wald":
            macro_block_statistic,
        "Core_Macro_Block_p":
            macro_block_p,
    })

    forward_etf_results[model_name] = {
        "OLS": forward_ols,
        "HAC": forward_hac,
        "Regressors": regressors,
        "Coefficient_Table": coefficient_table,
    }

forward_etf_comparison = pd.DataFrame(
    forward_model_rows
)

forward_etf_coefficients = pd.concat(
    forward_coefficient_rows,
    ignore_index=True,
)

forward_etf_comparison[
    "Delta_Adjusted_R2"
] = (
    forward_etf_comparison[
        "Adjusted_R2"
    ].diff()
)

forward_etf_comparison["Delta_AIC"] = (
    forward_etf_comparison["AIC"].diff()
)

forward_etf_comparison["Delta_BIC"] = (
    forward_etf_comparison["BIC"].diff()
)

display(
    forward_etf_comparison.round(4)
)

# -----------------------------------------------------------
# Coefficient comparison
# -----------------------------------------------------------

forward_focus_variables = [
    "ETF_Flow",
    "Fed_Expected_Direction",
    "Fed_Surprise_Signed",
    "Fed_Policy_Uncertainty",
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "VIX_Change",
]

forward_coefficient_comparison = (
    forward_etf_coefficients.loc[
        forward_etf_coefficients[
            "Variable"
        ].isin(forward_focus_variables)
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="Coefficient",
    )
    .reindex(
        forward_etf_specifications.keys()
    )
)

forward_pvalue_comparison = (
    forward_etf_coefficients.loc[
        forward_etf_coefficients[
            "Variable"
        ].isin(forward_focus_variables)
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="HAC_p",
    )
    .reindex(
        forward_etf_specifications.keys()
    )
)

print("\nForward-model coefficients")
display(
    forward_coefficient_comparison.round(4)
)

print("\nForward-model HAC p-values")
display(
    forward_pvalue_comparison.round(4)
)

# -----------------------------------------------------------
# Plot model fit
# -----------------------------------------------------------

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Adjusted R²",
        "Information Criteria",
    ],
    horizontal_spacing=0.15,
)

fig.add_trace(
    go.Bar(
        x=forward_etf_comparison["Model"],
        y=forward_etf_comparison["Adjusted_R2"],
        name="Adjusted R²",
        marker_color="#2a6fbb",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=forward_etf_comparison["Model"],
        y=forward_etf_comparison["AIC"],
        mode="lines+markers",
        name="AIC",
        line=dict(color="#d62728"),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=forward_etf_comparison["Model"],
        y=forward_etf_comparison["BIC"],
        mode="lines+markers",
        name="BIC",
        line=dict(color="#2ca02c"),
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title=(
        "FOMC Information and Next-Month "
        "Gold ETF Net Demand"
    ),
    template="plotly_white",
    width=1400,
    height=650,
)

fig.update_xaxes(tickangle=35)
fig.show()

Forward ETF-model observations: 60
FOMC-information period: 2019-03-01 00:00:00 to 2026-06-01 00:00:00
Outcome period: 2019-04-01 00:00:00 to 2026-07-01 00:00:00
Meeting months excluded: ['2019-01', '2026-07']


,Model,Fed_Variables,Parameters,N,R2,Adjusted_R2,AIC,BIC,Fed_Block_Wald,Fed_Block_p,Core_Macro_Block_Wald,Core_Macro_Block_p,Delta_Adjusted_R2,Delta_AIC,Delta_BIC
0,P0_ETF_Persistence,0,2,60,0.3409,0.3296,640.3864,644.5751,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P1_Direction,1,3,60,0.4848,0.4667,627.6103,633.8933,20.2662,0.0000,NaN,NaN,0.1372,-12.7761,-10.6818
2,P2_Direction_Surprise,2,4,60,0.4850,0.4574,629.5892,637.9666,20.6110,0.0000,NaN,NaN,-0.0093,1.9790,4.0733
3,P3_Complete_Fed,3,5,60,0.4855,0.4480,631.5323,642.0040,20.4957,0.0001,NaN,NaN,-0.0093,1.9431,4.0374
4,P4_Core_Macro,3,8,60,0.4882,0.4193,637.2162,653.9709,20.5119,0.0001,0.2738,0.9649,-0.0288,5.6839,11.9669



Forward-model coefficients


Variable,Broad_USD_Change,ETF_Flow,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed,Real10Y_Change_pp,VIX_Change
Model,,,,,,,
P0_ETF_Persistence,NaN,0.5236,NaN,NaN,NaN,NaN,NaN
P1_Direction,NaN,0.3107,-68.0747,NaN,NaN,NaN,NaN
P2_Direction_Surprise,NaN,0.3071,-68.6596,NaN,2.1164,NaN,NaN
P3_Complete_Fed,NaN,0.3064,-69.7074,-6.593,1.8931,NaN,NaN
P4_Core_Macro,1.0192,0.3046,-69.5656,-8.809,2.9397,-5.6887,0.4983



Forward-model HAC p-values


Variable,Broad_USD_Change,ETF_Flow,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed,Real10Y_Change_pp,VIX_Change
Model,,,,,,,
P0_ETF_Persistence,NaN,0.0000,NaN,NaN,NaN,NaN,NaN
P1_Direction,NaN,0.0010,0.0000,NaN,NaN,NaN,NaN
P2_Direction_Surprise,NaN,0.0014,0.0001,NaN,0.8947,NaN,NaN
P3_Complete_Fed,NaN,0.0016,0.0001,0.8176,0.9072,NaN,NaN
P4_Core_Macro,0.7564,0.0020,0.0001,0.7594,0.8656,0.7872,0.6481


In [39]:
# CELL 34: Diagnostics and robustness for P1_Direction

from statsmodels.stats.diagnostic import (
    acorr_breusch_godfrey,
    acorr_ljungbox,
    het_breuschpagan,
    het_arch,
    linear_reset,
    breaks_cusumolsresid,
)
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.outliers_influence import OLSInfluence
import statsmodels.api as sm
import pandas as pd
import numpy as np

p1_name = "P1_Direction"

p1_ols = forward_etf_results[
    p1_name
]["OLS"]

p1_hac = forward_etf_results[
    p1_name
]["HAC"]

p1_regressors = forward_etf_results[
    p1_name
]["Regressors"]

p1_names = p1_ols.model.exog_names

# -----------------------------------------------------------
# Coefficients under alternative covariance estimators
# -----------------------------------------------------------

p1_hc3 = p1_ols.get_robustcov_results(
    cov_type="HC3"
)

p1_hc3.model.data.xnames = p1_names

p1_covariance_rows = []

for covariance_name, result in {
    "Conventional_OLS": p1_ols,
    "HC3": p1_hc3,
    "Event_Time_HAC_2": p1_hac,
}.items():

    parameters = np.asarray(result.params)
    standard_errors = np.asarray(result.bse)
    statistics = np.asarray(result.tvalues)
    pvalues = np.asarray(result.pvalues)

    for position, variable in enumerate(p1_names):

        p1_covariance_rows.append({
            "Covariance_Estimator":
                covariance_name,
            "Variable":
                variable,
            "Coefficient":
                parameters[position],
            "Standard_Error":
                standard_errors[position],
            "t_Statistic":
                statistics[position],
            "p_value":
                pvalues[position],
        })

p1_covariance_comparison = pd.DataFrame(
    p1_covariance_rows
)

display(
    p1_covariance_comparison.round(4)
)

# -----------------------------------------------------------
# Residual diagnostics
# -----------------------------------------------------------

p1_diagnostic_rows = []

for lag in [1, 2, 3, 6]:

    bg = acorr_breusch_godfrey(
        p1_ols,
        nlags=lag,
    )

    p1_diagnostic_rows.append({
        "Category": "Serial correlation",
        "Test": f"Breusch-Godfrey LM({lag})",
        "Statistic": float(bg[0]),
        "p_value": float(bg[1]),
        "Null_Hypothesis": (
            f"No residual serial correlation through lag {lag}"
        ),
    })

ljung_box = acorr_ljungbox(
    p1_ols.resid,
    lags=[3, 6, 12],
    return_df=True,
)

for lag in [3, 6, 12]:

    p1_diagnostic_rows.append({
        "Category": "Serial correlation",
        "Test": f"Ljung-Box Q({lag})",
        "Statistic": float(
            ljung_box.loc[lag, "lb_stat"]
        ),
        "p_value": float(
            ljung_box.loc[lag, "lb_pvalue"]
        ),
        "Null_Hypothesis": (
            f"No residual autocorrelation through lag {lag}"
        ),
    })

bp = het_breuschpagan(
    p1_ols.resid,
    p1_ols.model.exog,
    robust=True,
)

p1_diagnostic_rows.append({
    "Category": "Heteroskedasticity",
    "Test": "Koenker-Breusch-Pagan",
    "Statistic": float(bp[0]),
    "p_value": float(bp[1]),
    "Null_Hypothesis": "Homoskedastic residuals",
})

for lag in [1, 2, 3, 6]:

    arch = het_arch(
        p1_ols.resid,
        nlags=lag,
    )

    p1_diagnostic_rows.append({
        "Category": "Conditional variance",
        "Test": f"ARCH LM({lag})",
        "Statistic": float(arch[0]),
        "p_value": float(arch[1]),
        "Null_Hypothesis": (
            f"No ARCH effects through lag {lag}"
        ),
    })

for power in [2, 3]:

    reset = linear_reset(
        p1_ols,
        power=power,
        use_f=True,
    )

    p1_diagnostic_rows.append({
        "Category": "Functional form",
        "Test": f"Ramsey RESET power {power}",
        "Statistic": float(
            np.asarray(
                reset.fvalue
            ).reshape(-1)[0]
        ),
        "p_value": float(
            np.asarray(
                reset.pvalue
            ).reshape(-1)[0]
        ),
        "Null_Hypothesis": (
            "Linear functional form is correctly specified"
        ),
    })

cusum = breaks_cusumolsresid(
    p1_ols.resid,
    ddof=int(p1_ols.df_model + 1),
)

p1_diagnostic_rows.append({
    "Category": "Parameter stability",
    "Test": "CUSUM",
    "Statistic": float(cusum[0]),
    "p_value": float(cusum[1]),
    "Null_Hypothesis": (
        "Regression parameters are stable"
    ),
})

jb = jarque_bera(
    p1_ols.resid
)

p1_diagnostic_rows.append({
    "Category": "Residual distribution",
    "Test": "Jarque-Bera",
    "Statistic": float(jb[0]),
    "p_value": float(jb[1]),
    "Null_Hypothesis": (
        "Residuals are normally distributed"
    ),
})

p1_diagnostics = pd.DataFrame(
    p1_diagnostic_rows
)

p1_diagnostics["Reject_Null_5pct"] = (
    p1_diagnostics["p_value"] < 0.05
)

p1_diagnostics["Conclusion"] = np.where(
    p1_diagnostics["Reject_Null_5pct"],
    "Potential problem detected",
    "No evidence of problem",
)

display(p1_diagnostics.round(4))

print(
    "Residual skewness:",
    round(float(jb[2]), 4)
)

print(
    "Residual kurtosis:",
    round(float(jb[3]), 4)
)

# -----------------------------------------------------------
# Influence diagnostics
# -----------------------------------------------------------

p1_influence = OLSInfluence(
    p1_ols
)

p1_influence_frame = (
    p1_influence.summary_frame()
)

p1_n = int(p1_ols.nobs)
p1_k = int(p1_ols.df_model + 1)

p1_cooks_threshold = 4 / p1_n
p1_leverage_threshold = 2 * p1_k / p1_n

p1_influence_table = pd.DataFrame({
    "Meeting_Month":
        forward_etf_common[
            "Month"
        ].to_numpy(),
    "Outcome_Month":
        (
            forward_etf_common["Month"]
            + pd.offsets.MonthBegin(1)
        ).to_numpy(),
    "ETF_Flow_Next_Month":
        forward_etf_common[
            "ETF_Flow_Next_Month"
        ].to_numpy(),
    "Fitted_ETF_Flow":
        np.asarray(p1_ols.fittedvalues),
    "Residual":
        np.asarray(p1_ols.resid),
    "Studentised_Residual":
        p1_influence_frame[
            "student_resid"
        ].to_numpy(),
    "Leverage":
        p1_influence_frame[
            "hat_diag"
        ].to_numpy(),
    "Cooks_Distance":
        p1_influence_frame[
            "cooks_d"
        ].to_numpy(),
})

p1_influence_table[
    "Large_Studentised_Residual"
] = (
    p1_influence_table[
        "Studentised_Residual"
    ].abs() > 2
)

p1_influence_table["High_Leverage"] = (
    p1_influence_table["Leverage"]
    > p1_leverage_threshold
)

p1_influence_table[
    "High_Cooks_Distance"
] = (
    p1_influence_table["Cooks_Distance"]
    > p1_cooks_threshold
)

p1_influence_table[
    "Potentially_Influential"
] = (
    p1_influence_table[
        [
            "Large_Studentised_Residual",
            "High_Leverage",
            "High_Cooks_Distance",
        ]
    ].any(axis=1)
)

p1_influential_observations = (
    p1_influence_table.loc[
        p1_influence_table[
            "Potentially_Influential"
        ]
    ]
    .sort_values(
        "Cooks_Distance",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "\nCook's-distance threshold:",
    round(p1_cooks_threshold, 4)
)

print(
    "Leverage threshold:",
    round(p1_leverage_threshold, 4)
)

print(
    "Potentially influential observations:",
    len(p1_influential_observations)
)

display(
    p1_influential_observations.round(4)
)

# -----------------------------------------------------------
# Leave-one-meeting-out robustness
# -----------------------------------------------------------

p1_leave_one_out_rows = []

for removed_position in range(
    len(forward_etf_common)
):

    reduced_df = (
        forward_etf_common
        .drop(index=removed_position)
        .reset_index(drop=True)
    )

    X_reduced = sm.add_constant(
        reduced_df[p1_regressors].astype(float),
        has_constant="add",
    )

    y_reduced = (
        reduced_df[
            "ETF_Flow_Next_Month"
        ].astype(float)
    )

    reduced_ols = sm.OLS(
        y_reduced,
        X_reduced,
    ).fit()

    reduced_hac = (
        reduced_ols.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    reduced_names = X_reduced.columns.tolist()
    reduced_params = np.asarray(
        reduced_hac.params
    )
    reduced_pvalues = np.asarray(
        reduced_hac.pvalues
    )

    persistence_position = (
        reduced_names.index("ETF_Flow")
    )

    direction_position = (
        reduced_names.index(
            "Fed_Expected_Direction"
        )
    )

    p1_leave_one_out_rows.append({
        "Removed_Meeting_Month":
            forward_etf_common.loc[
                removed_position,
                "Month",
            ],
        "Removed_Outcome_Month":
            forward_etf_common.loc[
                removed_position,
                "Month",
            ] + pd.offsets.MonthBegin(1),
        "Persistence_Coefficient":
            reduced_params[
                persistence_position
            ],
        "Persistence_HAC_p":
            reduced_pvalues[
                persistence_position
            ],
        "Direction_Coefficient":
            reduced_params[
                direction_position
            ],
        "Direction_HAC_p":
            reduced_pvalues[
                direction_position
            ],
    })

p1_leave_one_out = pd.DataFrame(
    p1_leave_one_out_rows
)

p1_stability_summary = pd.DataFrame({
    "Variable": [
        "Current_ETF_Flow",
        "Fed_Expected_Direction",
    ],
    "Minimum_Coefficient": [
        p1_leave_one_out[
            "Persistence_Coefficient"
        ].min(),
        p1_leave_one_out[
            "Direction_Coefficient"
        ].min(),
    ],
    "Maximum_Coefficient": [
        p1_leave_one_out[
            "Persistence_Coefficient"
        ].max(),
        p1_leave_one_out[
            "Direction_Coefficient"
        ].max(),
    ],
    "Median_Coefficient": [
        p1_leave_one_out[
            "Persistence_Coefficient"
        ].median(),
        p1_leave_one_out[
            "Direction_Coefficient"
        ].median(),
    ],
    "Minimum_p": [
        p1_leave_one_out[
            "Persistence_HAC_p"
        ].min(),
        p1_leave_one_out[
            "Direction_HAC_p"
        ].min(),
    ],
    "Maximum_p": [
        p1_leave_one_out[
            "Persistence_HAC_p"
        ].max(),
        p1_leave_one_out[
            "Direction_HAC_p"
        ].max(),
    ],
    "Median_p": [
        p1_leave_one_out[
            "Persistence_HAC_p"
        ].median(),
        p1_leave_one_out[
            "Direction_HAC_p"
        ].median(),
    ],
    "Significant_5pct_Share": [
        (
            p1_leave_one_out[
                "Persistence_HAC_p"
            ] < 0.05
        ).mean(),
        (
            p1_leave_one_out[
                "Direction_HAC_p"
            ] < 0.05
        ).mean(),
    ],
    "Sign_Stable": [
        (
            p1_leave_one_out[
                "Persistence_Coefficient"
            ] > 0
        ).all(),
        (
            p1_leave_one_out[
                "Direction_Coefficient"
            ] < 0
        ).all(),
    ],
})

display(
    p1_stability_summary.round(4)
)

print(
    "\nLeast favourable direction results"
)

display(
    p1_leave_one_out[
        [
            "Removed_Meeting_Month",
            "Removed_Outcome_Month",
            "Direction_Coefficient",
            "Direction_HAC_p",
        ]
    ]
    .sort_values(
        "Direction_HAC_p",
        ascending=False,
    )
    .head(10)
    .round(4)
)

,Covariance_Estimator,Variable,Coefficient,Standard_Error,t_Statistic,p_value
0,Conventional_OLS,const,23.2274,6.4048,3.6266,0.0006
1,Conventional_OLS,ETF_Flow,0.3107,0.1006,3.0887,0.0031
2,Conventional_OLS,Fed_Expected_Direction,-68.0747,17.0631,-3.9896,0.0002
3,HC3,const,23.2274,6.5649,3.5381,0.0008
4,HC3,ETF_Flow,0.3107,0.0910,3.4149,0.0012
5,HC3,Fed_Expected_Direction,-68.0747,15.4666,-4.4014,0.0000
6,Event_Time_HAC_2,const,23.2274,6.4598,3.5957,0.0007
7,Event_Time_HAC_2,ETF_Flow,0.3107,0.0892,3.4851,0.0010
8,Event_Time_HAC_2,Fed_Expected_Direction,-68.0747,15.1217,-4.5018,0.0000


,Category,Test,Statistic,p_value,Null_Hypothesis,Reject_Null_5pct,Conclusion
0,Serial correlation,Breusch-Godfrey LM(1),0.4955,0.4815,No residual serial correlation through lag 1,False,No evidence of problem
1,Serial correlation,Breusch-Godfrey LM(2),1.3076,0.5201,No residual serial correlation through lag 2,False,No evidence of problem
2,Serial correlation,Breusch-Godfrey LM(3),3.6031,0.3076,No residual serial correlation through lag 3,False,No evidence of problem
3,Serial correlation,Breusch-Godfrey LM(6),4.1441,0.6572,No residual serial correlation through lag 6,False,No evidence of problem
4,Serial correlation,Ljung-Box Q(3),4.0374,0.2575,No residual autocorrelation through lag 3,False,No evidence of problem
5,Serial correlation,Ljung-Box Q(6),4.8010,0.5696,No residual autocorrelation through lag 6,False,No evidence of problem
6,Serial correlation,Ljung-Box Q(12),11.0476,0.5248,No residual autocorrelation through lag 12,False,No evidence of problem
7,Heteroskedasticity,Koenker-Breusch-Pagan,1.3505,0.5090,Homoskedastic residuals,False,No evidence of problem
8,Conditional variance,ARCH LM(1),0.2136,0.6439,No ARCH effects through lag 1,False,No evidence of problem
9,Conditional variance,ARCH LM(2),9.7736,0.0075,No ARCH effects through lag 2,True,Potential problem detected


Residual skewness: 0.0692
Residual kurtosis: 2.7243

Cook's-distance threshold: 0.0667
Leverage threshold: 0.1
Potentially influential observations: 11


,Meeting_Month,Outcome_Month,ETF_Flow_Next_Month,Fitted_ETF_Flow,Residual,Studentised_Residual,Leverage,Cooks_Distance,Large_Studentised_Residual,High_Leverage,High_Cooks_Distance,Potentially_Influential
0,2020-06-01,2020-07-01,166.3,99.7950,66.5050,1.5917,0.0788,0.0703,False,False,True,True
1,2024-11-01,2024-12-01,3.7,51.0222,-47.3222,-1.1432,0.1148,0.0562,False,True,False,True
2,2020-03-01,2020-04-01,170.1,127.8559,42.2441,1.0244,0.1254,0.0501,False,True,False,True
3,2022-03-01,2022-04-01,42.9,75.5990,-32.6990,-0.8092,0.1661,0.0437,False,True,False,True
4,2019-05-01,2019-06-01,126.8,19.2625,107.5375,2.5817,0.0198,0.0409,True,False,False,True
5,2020-07-01,2020-08-01,38.8,77.1245,-38.3245,-0.9235,0.1173,0.0379,False,True,False,True
6,2020-04-01,2020-05-01,154.3,115.5105,38.7895,0.9312,0.1105,0.0360,False,True,False,True
7,2019-07-01,2019-08-01,122.0,33.1369,88.8631,2.0979,0.0231,0.0327,True,False,False,True
8,2021-01-01,2021-02-01,-84.7,17.7551,-102.4551,-2.4422,0.0169,0.0314,True,False,False,True
9,2020-11-01,2020-12-01,-40.1,-9.7979,-30.3021,-0.7249,0.1096,0.0218,False,True,False,True


,Variable,Minimum_Coefficient,Maximum_Coefficient,Median_Coefficient,Minimum_p,Maximum_p,Median_p,Significant_5pct_Share,Sign_Stable
0,Current_ETF_Flow,0.2822,0.3453,0.3108,0.0002,0.0051,0.001,1.0,True
1,Fed_Expected_Direction,-74.2948,-62.9697,-68.0272,0.0000,0.0002,0.000,1.0,True



Least favourable direction results


,Removed_Meeting_Month,Removed_Outcome_Month,Direction_Coefficient,Direction_HAC_p
47,2025-01-01,2025-02-01,-65.3083,0.0002
33,2023-05-01,2023-06-01,-64.4217,0.0002
26,2022-06-01,2022-07-01,-64.3626,0.0001
30,2022-12-01,2023-01-01,-68.4851,0.0001
55,2025-12-01,2026-01-01,-65.5080,0.0001
25,2022-05-01,2022-06-01,-70.3053,0.0001
24,2022-03-01,2022-04-01,-65.0171,0.0001
1,2019-05-01,2019-06-01,-66.2622,0.0001
57,2026-03-01,2026-04-01,-64.7664,0.0001
16,2021-03-01,2021-04-01,-68.4585,0.0001


In [40]:
# CELL 35: Expanding-window forecasting comparison
# P0: current ETF flow only
# P1: current ETF flow + expected Fed direction

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy import stats

forecast_df = (
    forward_etf_common[
        [
            "Month",
            "ETF_Flow_Next_Month",
            "ETF_Flow",
            "Fed_Expected_Direction",
        ]
    ]
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

# Minimum number of completed meeting observations used
# before producing the first pseudo-out-of-sample forecast.
initial_training_observations = 30

if initial_training_observations >= len(forecast_df):
    raise ValueError(
        "Initial training window is too large."
    )

recursive_forecast_rows = []

for forecast_position in range(
    initial_training_observations,
    len(forecast_df)
):

    # Only observations preceding the forecast event
    # are used to estimate the coefficients.
    training_data = forecast_df.iloc[
        :forecast_position
    ].copy()

    forecast_observation = forecast_df.iloc[
        [forecast_position]
    ].copy()

    # Persistence-only benchmark.
    X_train_p0 = sm.add_constant(
        training_data[
            ["ETF_Flow"]
        ].astype(float),
        has_constant="add",
    )

    X_test_p0 = sm.add_constant(
        forecast_observation[
            ["ETF_Flow"]
        ].astype(float),
        has_constant="add",
    )

    X_test_p0 = X_test_p0.reindex(
        columns=X_train_p0.columns,
        fill_value=1.0,
    )

    p0_recursive_model = sm.OLS(
        training_data[
            "ETF_Flow_Next_Month"
        ].astype(float),
        X_train_p0,
    ).fit()

    p0_forecast = float(
        p0_recursive_model.predict(
            X_test_p0
        ).iloc[0]
    )

    # Fed-direction model.
    X_train_p1 = sm.add_constant(
        training_data[
            [
                "ETF_Flow",
                "Fed_Expected_Direction",
            ]
        ].astype(float),
        has_constant="add",
    )

    X_test_p1 = sm.add_constant(
        forecast_observation[
            [
                "ETF_Flow",
                "Fed_Expected_Direction",
            ]
        ].astype(float),
        has_constant="add",
    )

    X_test_p1 = X_test_p1.reindex(
        columns=X_train_p1.columns,
        fill_value=1.0,
    )

    p1_recursive_model = sm.OLS(
        training_data[
            "ETF_Flow_Next_Month"
        ].astype(float),
        X_train_p1,
    ).fit()

    p1_forecast = float(
        p1_recursive_model.predict(
            X_test_p1
        ).iloc[0]
    )

    actual_value = float(
        forecast_observation[
            "ETF_Flow_Next_Month"
        ].iloc[0]
    )

    meeting_month = forecast_observation[
        "Month"
    ].iloc[0]

    recursive_forecast_rows.append({
        "Meeting_Month": meeting_month,
        "Outcome_Month": (
            meeting_month
            + pd.offsets.MonthBegin(1)
        ),
        "Training_Observations":
            len(training_data),
        "Actual_ETF_Flow":
            actual_value,
        "P0_Persistence_Forecast":
            p0_forecast,
        "P1_Fed_Direction_Forecast":
            p1_forecast,
        "P0_Error":
            actual_value - p0_forecast,
        "P1_Error":
            actual_value - p1_forecast,
        "Recursive_Direction_Coefficient":
            p1_recursive_model.params[
                "Fed_Expected_Direction"
            ],
    })

recursive_etf_forecasts = pd.DataFrame(
    recursive_forecast_rows
)

print(
    "Pseudo-out-of-sample forecasts:",
    len(recursive_etf_forecasts)
)

print(
    "First forecasted outcome month:",
    recursive_etf_forecasts[
        "Outcome_Month"
    ].min()
)

print(
    "Final forecasted outcome month:",
    recursive_etf_forecasts[
        "Outcome_Month"
    ].max()
)

# -----------------------------------------------------------
# Forecast evaluation
# -----------------------------------------------------------

def forecast_metrics(
    actual,
    forecast,
):

    error = actual - forecast

    rmse = np.sqrt(
        np.mean(np.square(error))
    )

    mae = np.mean(np.abs(error))

    bias = np.mean(error)

    correlation = np.corrcoef(
        actual,
        forecast,
    )[0, 1]

    sign_accuracy = np.mean(
        np.sign(actual)
        == np.sign(forecast)
    )

    return {
        "RMSE": rmse,
        "MAE": mae,
        "Bias": bias,
        "Forecast_Actual_Correlation":
            correlation,
        "Flow_Sign_Accuracy":
            sign_accuracy,
    }

actual = recursive_etf_forecasts[
    "Actual_ETF_Flow"
].to_numpy()

p0_predictions = recursive_etf_forecasts[
    "P0_Persistence_Forecast"
].to_numpy()

p1_predictions = recursive_etf_forecasts[
    "P1_Fed_Direction_Forecast"
].to_numpy()

p0_metrics = forecast_metrics(
    actual,
    p0_predictions,
)

p1_metrics = forecast_metrics(
    actual,
    p1_predictions,
)

recursive_metric_table = pd.DataFrame([
    {
        "Model": "P0_Persistence",
        **p0_metrics,
    },
    {
        "Model": "P1_Fed_Direction",
        **p1_metrics,
    },
])

benchmark_rmse = recursive_metric_table.loc[
    recursive_metric_table["Model"]
    == "P0_Persistence",
    "RMSE",
].iloc[0]

benchmark_mae = recursive_metric_table.loc[
    recursive_metric_table["Model"]
    == "P0_Persistence",
    "MAE",
].iloc[0]

recursive_metric_table[
    "RMSE_Improvement_vs_P0_pct"
] = (
    (
        benchmark_rmse
        - recursive_metric_table["RMSE"]
    )
    / benchmark_rmse
    * 100
)

recursive_metric_table[
    "MAE_Improvement_vs_P0_pct"
] = (
    (
        benchmark_mae
        - recursive_metric_table["MAE"]
    )
    / benchmark_mae
    * 100
)

display(
    recursive_metric_table.round(4)
)

# -----------------------------------------------------------
# Clark-West test for nested forecast models
#
# H0: P1 does not improve prediction over P0.
# H1: P1 improves prediction over P0.
# -----------------------------------------------------------

p0_errors = recursive_etf_forecasts[
    "P0_Error"
].to_numpy()

p1_errors = recursive_etf_forecasts[
    "P1_Error"
].to_numpy()

clark_west_adjusted_difference = (
    np.square(p0_errors)
    - (
        np.square(p1_errors)
        - np.square(
            p0_predictions
            - p1_predictions
        )
    )
)

cw_regression = sm.OLS(
    clark_west_adjusted_difference,
    np.ones(
        len(clark_west_adjusted_difference)
    ),
).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 2,
        "use_correction": True,
    },
)

cw_statistic = float(
    np.asarray(
        cw_regression.tvalues
    ).reshape(-1)[0]
)

# One-sided p-value:
# positive statistic supports the larger P1 model.
cw_p_one_sided = float(
    1 - stats.t.cdf(
        cw_statistic,
        df=cw_regression.df_resid,
    )
)

clark_west_result = pd.DataFrame({
    "Test": [
        "Clark-West: P1 improves over P0"
    ],
    "Adjusted_Loss_Difference_Mean": [
        np.mean(
            clark_west_adjusted_difference
        )
    ],
    "Statistic": [
        cw_statistic
    ],
    "One_Sided_p": [
        cw_p_one_sided
    ],
    "Reject_5pct": [
        cw_p_one_sided < 0.05
    ],
    "Reject_10pct": [
        cw_p_one_sided < 0.10
    ],
})

display(
    clark_west_result.round(4)
)

# -----------------------------------------------------------
# Recursive coefficient stability
# -----------------------------------------------------------

recursive_coefficient_summary = pd.DataFrame({
    "Minimum_Direction_Coefficient": [
        recursive_etf_forecasts[
            "Recursive_Direction_Coefficient"
        ].min()
    ],
    "Maximum_Direction_Coefficient": [
        recursive_etf_forecasts[
            "Recursive_Direction_Coefficient"
        ].max()
    ],
    "Median_Direction_Coefficient": [
        recursive_etf_forecasts[
            "Recursive_Direction_Coefficient"
        ].median()
    ],
    "Final_Direction_Coefficient": [
        recursive_etf_forecasts[
            "Recursive_Direction_Coefficient"
        ].iloc[-1]
    ],
    "Negative_Coefficient_Share": [
        (
            recursive_etf_forecasts[
                "Recursive_Direction_Coefficient"
            ] < 0
        ).mean()
    ],
})

display(
    recursive_coefficient_summary.round(4)
)

# -----------------------------------------------------------
# Cumulative squared-error comparison
# -----------------------------------------------------------

recursive_etf_forecasts[
    "P0_Cumulative_Squared_Error"
] = (
    np.square(
        recursive_etf_forecasts["P0_Error"]
    ).cumsum()
)

recursive_etf_forecasts[
    "P1_Cumulative_Squared_Error"
] = (
    np.square(
        recursive_etf_forecasts["P1_Error"]
    ).cumsum()
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=recursive_etf_forecasts[
            "Outcome_Month"
        ],
        y=recursive_etf_forecasts[
            "Actual_ETF_Flow"
        ],
        mode="lines+markers",
        name="Actual ETF flow",
        line=dict(
            color="black",
            width=3,
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=recursive_etf_forecasts[
            "Outcome_Month"
        ],
        y=recursive_etf_forecasts[
            "P0_Persistence_Forecast"
        ],
        mode="lines+markers",
        name="P0 persistence forecast",
        line=dict(
            color="#7f7f7f",
            dash="dot",
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=recursive_etf_forecasts[
            "Outcome_Month"
        ],
        y=recursive_etf_forecasts[
            "P1_Fed_Direction_Forecast"
        ],
        mode="lines+markers",
        name="P1 Fed-direction forecast",
        line=dict(
            color="#1f77b4",
        ),
    )
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title=(
        "Expanding-Window Forecasts of "
        "Next-Month Gold ETF Demand"
    ),
    template="plotly_white",
    width=1300,
    height=650,
    xaxis_title="ETF-flow outcome month",
    yaxis_title="ETF net demand (tonnes)",
)

fig.show()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=recursive_etf_forecasts[
            "Outcome_Month"
        ],
        y=recursive_etf_forecasts[
            "P0_Cumulative_Squared_Error"
        ],
        mode="lines",
        name="P0 persistence",
        line=dict(
            color="#7f7f7f",
            width=3,
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=recursive_etf_forecasts[
            "Outcome_Month"
        ],
        y=recursive_etf_forecasts[
            "P1_Cumulative_Squared_Error"
        ],
        mode="lines",
        name="P1 Fed direction",
        line=dict(
            color="#1f77b4",
            width=3,
        ),
    )
)

fig.update_layout(
    title=(
        "Cumulative Squared Forecast Error: "
        "Persistence versus Fed Direction"
    ),
    template="plotly_white",
    width=1200,
    height=600,
    xaxis_title="ETF-flow outcome month",
    yaxis_title="Cumulative squared error",
)

fig.show()

Pseudo-out-of-sample forecasts: 30
First forecasted outcome month: 2023-01-01 00:00:00
Final forecasted outcome month: 2026-07-01 00:00:00


,Model,RMSE,MAE,Bias,Forecast_Actual_Correlation,Flow_Sign_Accuracy,RMSE_Improvement_vs_P0_pct,MAE_Improvement_vs_P0_pct
0,P0_Persistence,43.1613,36.6175,5.0086,0.4520,0.6667,0.0000,0.0000
1,P1_Fed_Direction,39.4161,33.9431,-0.8270,0.5693,0.7333,8.6772,7.3034


,Test,Adjusted_Loss_Difference_Mean,Statistic,One_Sided_p,Reject_5pct,Reject_10pct
0,Clark-West: P1 improves over P0,819.1693,2.4387,0.0105,True,True


,Minimum_Direction_Coefficient,Maximum_Direction_Coefficient,Median_Direction_Coefficient,Final_Direction_Coefficient,Negative_Coefficient_Share
0,-79.2597,-58.0517,-64.9781,-66.6505,1.0


In [41]:
# CELL 36: Forecast robustness across initial training-window sizes

import statsmodels.api as sm
from scipy import stats
import pandas as pd
import numpy as np
import plotly.express as px

initial_window_candidates = [
    18,
    24,
    30,
    36,
]

window_result_rows = []
all_window_forecasts = []

for initial_window in initial_window_candidates:

    if initial_window >= len(forecast_df):
        continue

    recursive_rows = []

    for forecast_position in range(
        initial_window,
        len(forecast_df)
    ):

        training_data = forecast_df.iloc[
            :forecast_position
        ].copy()

        test_data = forecast_df.iloc[
            [forecast_position]
        ].copy()

        y_train = training_data[
            "ETF_Flow_Next_Month"
        ].astype(float)

        # -----------------------------------------------
        # P0: ETF persistence
        # -----------------------------------------------

        X0_train = sm.add_constant(
            training_data[
                ["ETF_Flow"]
            ].astype(float),
            has_constant="add",
        )

        X0_test = sm.add_constant(
            test_data[
                ["ETF_Flow"]
            ].astype(float),
            has_constant="add",
        ).reindex(
            columns=X0_train.columns,
            fill_value=1.0,
        )

        model_0 = sm.OLS(
            y_train,
            X0_train,
        ).fit()

        forecast_0 = float(
            model_0.predict(
                X0_test
            ).iloc[0]
        )

        # -----------------------------------------------
        # P1: ETF persistence + expected Fed direction
        # -----------------------------------------------

        X1_train = sm.add_constant(
            training_data[
                [
                    "ETF_Flow",
                    "Fed_Expected_Direction",
                ]
            ].astype(float),
            has_constant="add",
        )

        X1_test = sm.add_constant(
            test_data[
                [
                    "ETF_Flow",
                    "Fed_Expected_Direction",
                ]
            ].astype(float),
            has_constant="add",
        ).reindex(
            columns=X1_train.columns,
            fill_value=1.0,
        )

        model_1 = sm.OLS(
            y_train,
            X1_train,
        ).fit()

        forecast_1 = float(
            model_1.predict(
                X1_test
            ).iloc[0]
        )

        actual = float(
            test_data[
                "ETF_Flow_Next_Month"
            ].iloc[0]
        )

        meeting_month = test_data[
            "Month"
        ].iloc[0]

        recursive_rows.append({
            "Initial_Window": initial_window,
            "Meeting_Month": meeting_month,
            "Outcome_Month": (
                meeting_month
                + pd.offsets.MonthBegin(1)
            ),
            "Actual": actual,
            "P0_Forecast": forecast_0,
            "P1_Forecast": forecast_1,
            "P0_Error": actual - forecast_0,
            "P1_Error": actual - forecast_1,
            "Direction_Coefficient": (
                model_1.params[
                    "Fed_Expected_Direction"
                ]
            ),
        })

    window_forecasts = pd.DataFrame(
        recursive_rows
    )

    all_window_forecasts.append(
        window_forecasts
    )

    actual = window_forecasts[
        "Actual"
    ].to_numpy()

    forecast_0 = window_forecasts[
        "P0_Forecast"
    ].to_numpy()

    forecast_1 = window_forecasts[
        "P1_Forecast"
    ].to_numpy()

    error_0 = actual - forecast_0
    error_1 = actual - forecast_1

    rmse_0 = np.sqrt(
        np.mean(error_0 ** 2)
    )

    rmse_1 = np.sqrt(
        np.mean(error_1 ** 2)
    )

    mae_0 = np.mean(
        np.abs(error_0)
    )

    mae_1 = np.mean(
        np.abs(error_1)
    )

    sign_accuracy_0 = np.mean(
        np.sign(actual)
        == np.sign(forecast_0)
    )

    sign_accuracy_1 = np.mean(
        np.sign(actual)
        == np.sign(forecast_1)
    )

    # -----------------------------------------------
    # Clark-West nested-model test
    # -----------------------------------------------

    cw_adjusted_loss = (
        error_0 ** 2
        - (
            error_1 ** 2
            - (
                forecast_0
                - forecast_1
            ) ** 2
        )
    )

    cw_model = sm.OLS(
        cw_adjusted_loss,
        np.ones(len(cw_adjusted_loss)),
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 2,
            "use_correction": True,
        },
    )

    cw_statistic = float(
        np.asarray(
            cw_model.tvalues
        ).reshape(-1)[0]
    )

    cw_one_sided_p = float(
        1 - stats.t.cdf(
            cw_statistic,
            df=cw_model.df_resid,
        )
    )

    window_result_rows.append({
        "Initial_Training_Window":
            initial_window,
        "Forecasts":
            len(window_forecasts),
        "First_Outcome_Month":
            window_forecasts[
                "Outcome_Month"
            ].min(),
        "P0_RMSE":
            rmse_0,
        "P1_RMSE":
            rmse_1,
        "RMSE_Improvement_pct":
            (rmse_0 - rmse_1)
            / rmse_0 * 100,
        "P0_MAE":
            mae_0,
        "P1_MAE":
            mae_1,
        "MAE_Improvement_pct":
            (mae_0 - mae_1)
            / mae_0 * 100,
        "P0_Sign_Accuracy":
            sign_accuracy_0,
        "P1_Sign_Accuracy":
            sign_accuracy_1,
        "Clark_West_Statistic":
            cw_statistic,
        "Clark_West_One_Sided_p":
            cw_one_sided_p,
        "Clark_West_Reject_5pct":
            cw_one_sided_p < 0.05,
        "Minimum_Direction_Coefficient":
            window_forecasts[
                "Direction_Coefficient"
            ].min(),
        "Maximum_Direction_Coefficient":
            window_forecasts[
                "Direction_Coefficient"
            ].max(),
        "Negative_Direction_Share":
            (
                window_forecasts[
                    "Direction_Coefficient"
                ] < 0
            ).mean(),
    })

training_window_robustness = pd.DataFrame(
    window_result_rows
)

all_training_window_forecasts = pd.concat(
    all_window_forecasts,
    ignore_index=True,
)

display(
    training_window_robustness.round(4)
)

# -----------------------------------------------------------
# Plot sensitivity results
# -----------------------------------------------------------

fig = px.bar(
    training_window_robustness,
    x="Initial_Training_Window",
    y="RMSE_Improvement_pct",
    color="Clark_West_Reject_5pct",
    text="RMSE_Improvement_pct",
    color_discrete_map={
        True: "#2ca02c",
        False: "#d62728",
    },
    hover_data={
        "Forecasts": True,
        "First_Outcome_Month": True,
        "MAE_Improvement_pct": ":.2f",
        "P0_Sign_Accuracy": ":.3f",
        "P1_Sign_Accuracy": ":.3f",
        "Clark_West_One_Sided_p": ":.4f",
        "Negative_Direction_Share": ":.3f",
    },
    title=(
        "Sensitivity of Fed-Direction ETF Forecast Gains "
        "to the Initial Training Window"
    ),
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=620,
    xaxis_title="Initial training observations",
    yaxis_title="P1 RMSE improvement over P0 (%)",
    legend_title="Clark–West significant at 5%",
)

fig.show()

,Initial_Training_Window,Forecasts,First_Outcome_Month,P0_RMSE,P1_RMSE,RMSE_Improvement_pct,P0_MAE,P1_MAE,MAE_Improvement_pct,P0_Sign_Accuracy,P1_Sign_Accuracy,Clark_West_Statistic,Clark_West_One_Sided_p,Clark_West_Reject_5pct,Minimum_Direction_Coefficient,Maximum_Direction_Coefficient,Negative_Direction_Share
0,18,42,2021-07-01,43.0367,37.5851,12.6673,36.0164,32.2517,10.4529,0.6667,0.7381,2.8864,0.0031,True,-113.5450,-58.0517,1.0
1,24,36,2022-04-01,44.4077,39.5716,10.8903,36.9287,34.4041,6.8364,0.6944,0.7778,2.5344,0.0079,True,-113.5450,-58.0517,1.0
2,30,30,2023-01-01,43.1613,39.4161,8.6772,36.6175,33.9431,7.3034,0.6667,0.7333,2.4387,0.0105,True,-79.2597,-58.0517,1.0
3,36,24,2023-10-01,43.4859,39.5546,9.0402,36.8127,34.4209,6.4971,0.7083,0.7500,1.9074,0.0345,True,-72.8577,-58.0517,1.0


In [42]:
# CELL 37A: Load the Regional ETF Panel from the gold workbook

import pandas as pd
import numpy as np
import glob
import os

# -----------------------------------------------------------
# Locate the uploaded Gold_1 workbook
# -----------------------------------------------------------

possible_gold_paths = []

# Reuse an existing path variable if available.
for variable_name in [
    "gold_workbook_path",
    "gold_file",
    "gold_path",
]:

    if variable_name in globals():

        candidate = globals()[variable_name]

        if isinstance(candidate, str):
            possible_gold_paths.append(candidate)

# Also search the current Colab directory.
possible_gold_paths.extend(
    glob.glob("Gold_1*.xlsx")
)

# Keep only valid files and remove duplicates.
possible_gold_paths = list(dict.fromkeys(
    path
    for path in possible_gold_paths
    if os.path.exists(path)
))

if not possible_gold_paths:
    raise FileNotFoundError(
        "Gold_1.xlsx was not found. "
        "Upload the gold workbook before running this cell."
    )

# Prefer the exact filename when available.
exact_matches = [
    path
    for path in possible_gold_paths
    if os.path.basename(path) == "Gold_1.xlsx"
]

if exact_matches:
    gold_panel_workbook = exact_matches[0]
else:
    gold_panel_workbook = possible_gold_paths[0]

print(
    "Gold workbook used:",
    gold_panel_workbook
)

# -----------------------------------------------------------
# Confirm that the regional panel sheet exists
# -----------------------------------------------------------

gold_sheet_names = pd.ExcelFile(
    gold_panel_workbook
).sheet_names

print(
    "Available sheets:",
    gold_sheet_names
)

regional_sheet_name = "Regional ETF Panel"

if regional_sheet_name not in gold_sheet_names:
    raise KeyError(
        f"Sheet '{regional_sheet_name}' was not found. "
        f"Available sheets: {gold_sheet_names}"
    )

# -----------------------------------------------------------
# Load the regional panel
# -----------------------------------------------------------

panel_df = pd.read_excel(
    gold_panel_workbook,
    sheet_name=regional_sheet_name,
)

# Clean column labels.
panel_df.columns = (
    panel_df.columns
    .astype(str)
    .str.strip()
)

print(
    "Original panel rows:",
    len(panel_df)
)

print(
    "Panel columns:",
    panel_df.columns.tolist()
)

# -----------------------------------------------------------
# Identify the required columns
# -----------------------------------------------------------

month_candidates = [
    "Month",
    "Date",
    "month",
]

region_candidates = [
    "Region",
    "ETF_Region",
    "region",
]

flow_candidates = [
    "ETF_Net_Demand_Regional_WGC_tonnes",
    "ETF_Net_Demand_tonnes",
    "Regional_ETF_Net_Demand_tonnes",
]

def select_existing_column(
    dataframe,
    candidates,
    variable_description,
):

    matches = [
        column
        for column in candidates
        if column in dataframe.columns
    ]

    if not matches:
        raise KeyError(
            f"No {variable_description} column found. "
            f"Available columns: "
            f"{dataframe.columns.tolist()}"
        )

    return matches[0]

month_column = select_existing_column(
    panel_df,
    month_candidates,
    "month",
)

region_column = select_existing_column(
    panel_df,
    region_candidates,
    "region",
)

regional_flow_source_column = (
    select_existing_column(
        panel_df,
        flow_candidates,
        "regional ETF-flow",
    )
)

print(
    "Month column:",
    month_column
)

print(
    "Region column:",
    region_column
)

print(
    "Regional ETF-flow column:",
    regional_flow_source_column
)

# -----------------------------------------------------------
# Standardize the required names
# -----------------------------------------------------------

panel_df = panel_df.rename(
    columns={
        month_column: "Month",
        region_column: "Region",
        regional_flow_source_column:
            "ETF_Net_Demand_Regional_WGC_tonnes",
    }
)

panel_df["Month"] = pd.to_datetime(
    panel_df["Month"],
    errors="coerce",
)

panel_df["Region"] = (
    panel_df["Region"]
    .astype("string")
    .str.strip()
)

panel_df[
    "ETF_Net_Demand_Regional_WGC_tonnes"
] = pd.to_numeric(
    panel_df[
        "ETF_Net_Demand_Regional_WGC_tonnes"
    ],
    errors="coerce",
)

# Remove footer/source-note rows.
panel_df = (
    panel_df
    .dropna(
        subset=[
            "Month",
            "Region",
            "ETF_Net_Demand_Regional_WGC_tonnes",
        ]
    )
    .sort_values(
        ["Region", "Month"]
    )
    .reset_index(drop=True)
)

# -----------------------------------------------------------
# Validate panel structure
# -----------------------------------------------------------

duplicate_region_months = (
    panel_df.duplicated(
        subset=["Region", "Month"]
    )
)

if duplicate_region_months.any():

    display(
        panel_df.loc[
            duplicate_region_months,
            [
                "Month",
                "Region",
                "ETF_Net_Demand_Regional_WGC_tonnes",
            ],
        ]
    )

    raise ValueError(
        "Duplicate region-month observations exist."
    )

panel_structure = (
    panel_df
    .groupby(
        "Region",
        observed=True,
    )
    .agg(
        Observations=("Month", "count"),
        First_Month=("Month", "min"),
        Final_Month=("Month", "max"),
        Missing_ETF_Flow=(
            "ETF_Net_Demand_Regional_WGC_tonnes",
            lambda series: series.isna().sum(),
        ),
    )
    .reset_index()
)

print(
    "Clean panel rows:",
    len(panel_df)
)

print(
    "Regions:",
    panel_df["Region"].unique().tolist()
)

display(panel_structure)

display(
    panel_df[
        [
            "Month",
            "Region",
            "ETF_Net_Demand_Regional_WGC_tonnes",
        ]
    ].head(12)
)

Gold workbook used: Gold_1.xlsx
Available sheets: ['Monthly Causal Data', 'Quarterly Demand Data', 'Regional ETF Panel']
Original panel rows: 371
Panel columns: ['Month', 'Year', 'Quarter', 'Month_ID', 'Region_ID', 'Region', 'Panel_Row_ID', 'ETF_Net_Demand_Regional_WGC_tonnes', 'ETF_Region_Share_of_Global', 'ETF_Regional_Deviation_From_Monthly_Mean_tonnes', 'Gold_Monthly_Return_pct', 'ETF_Net_Demand_Global_WGC_tonnes', 'CB_Reported_Holdings_Change_Global_IMF_tonnes', 'ETF_plus_CB_IMF_tonnes', 'CB_Reported_Net_Change_Global_WGC_tonnes', 'ETF_plus_CB_WGC_tonnes', 'Jewellery_Quarterly_Control_tonnes', 'Jewellery_QoQ_Growth_pct', 'Jewellery_YoY_Growth_pct', 'US_CPI_Inflation_MoM_pct', 'SP500_Monthly_Return_pct', 'Broad_USD_Index_Avg', 'US_10Y_Yield_MonthEnd_pct', 'Brent_MonthEnd_USD_per_Barrel', 'Effective_Fed_Funds_Avg_pct', 'US_10Y_Real_Yield_Monthly_Avg_pct', 'US_10Y_Real_Yield_MonthEnd_pct', 'US_10Y_Real_Yield_Monthly_Change_pp', 'VIX_Monthly_Avg', 'VIX_MonthEnd', 'VIX_Monthly_Change_p

,Region,Observations,First_Month,Final_Month,Missing_ETF_Flow
0,Asia,91,2019-01-01,2026-07-01,0
1,Europe,91,2019-01-01,2026-07-01,0
2,North America,91,2019-01-01,2026-07-01,0
3,Others,91,2019-01-01,2026-07-01,0


,Month,Region,ETF_Net_Demand_Regional_WGC_tonnes
0,2019-01-01,Asia,0.1
1,2019-02-01,Asia,-3.0
2,2019-03-01,Asia,-1.2
3,2019-04-01,Asia,-2.5
4,2019-05-01,Asia,-4.1
5,2019-06-01,Asia,2.4
6,2019-07-01,Asia,0.8
7,2019-08-01,Asia,9.0
8,2019-09-01,Asia,4.2
9,2019-10-01,Asia,-0.8


In [43]:
# CELL 37: FOMC direction and next-month regional ETF flows

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.express as px

regional_flow_column = (
    "ETF_Net_Demand_Regional_WGC_tonnes"
)

required_panel_columns = [
    "Month",
    "Region",
    regional_flow_column,
]

missing_panel_columns = [
    column
    for column in required_panel_columns
    if column not in panel_df.columns
]

if missing_panel_columns:
    raise KeyError(
        "Required regional-panel columns are missing: "
        f"{missing_panel_columns}"
    )

# -----------------------------------------------------------
# Prepare the complete regular monthly regional panel
# -----------------------------------------------------------

regional_forward_df = (
    panel_df[required_panel_columns]
    .copy()
)

regional_forward_df["Month"] = pd.to_datetime(
    regional_forward_df["Month"],
    errors="coerce",
)

regional_forward_df[
    regional_flow_column
] = pd.to_numeric(
    regional_forward_df[
        regional_flow_column
    ],
    errors="coerce",
)

regional_forward_df = (
    regional_forward_df
    .dropna(
        subset=[
            "Month",
            "Region",
            regional_flow_column,
        ]
    )
    .sort_values(
        ["Region", "Month"]
    )
    .reset_index(drop=True)
)

# Check for duplicated region-month observations.
duplicate_region_months = (
    regional_forward_df.duplicated(
        subset=["Region", "Month"]
    )
)

if duplicate_region_months.any():

    duplicated_rows = regional_forward_df.loc[
        duplicate_region_months,
        ["Region", "Month"],
    ]

    raise ValueError(
        "Duplicate region-month observations detected:\n"
        f"{duplicated_rows}"
    )

# Create next-calendar-month regional ETF flow before
# filtering to FOMC meeting months.
regional_forward_df[
    "Regional_ETF_Flow_Next_Month"
] = (
    regional_forward_df
    .groupby(
        "Region",
        observed=True,
    )[regional_flow_column]
    .shift(-1)
)

# Merge the meeting-level expected Fed direction.
fed_direction_events = (
    etf_event_df[
        [
            "Month",
            "Fed_Expected_Direction",
        ]
    ]
    .drop_duplicates("Month")
    .copy()
)

regional_forward_df = (
    regional_forward_df
    .merge(
        fed_direction_events,
        on="Month",
        how="left",
        validate="many_to_one",
    )
)

# Use exactly the same meeting months as the global
# forward comparison.
global_forward_months = set(
    forward_etf_common["Month"]
)

regional_forward_common = (
    regional_forward_df.loc[
        regional_forward_df["Month"].isin(
            global_forward_months
        )
    ]
    .dropna(
        subset=[
            regional_flow_column,
            "Regional_ETF_Flow_Next_Month",
            "Fed_Expected_Direction",
        ]
    )
    .sort_values(
        ["Region", "Month"]
    )
    .reset_index(drop=True)
)

print(
    "Regions:",
    regional_forward_common[
        "Region"
    ].unique().tolist()
)

regional_sample_check = (
    regional_forward_common
    .groupby(
        "Region",
        observed=True,
    )
    .agg(
        Observations=("Month", "count"),
        First_Meeting_Month=("Month", "min"),
        Final_Meeting_Month=("Month", "max"),
    )
    .reset_index()
)

display(regional_sample_check)

# -----------------------------------------------------------
# Estimate persistence-only and Fed-direction models
# separately for every ETF region
# -----------------------------------------------------------

regional_forward_results = {}
regional_model_rows = []
regional_coefficient_rows = []

for region in regional_forward_common[
    "Region"
].unique():

    region_data = (
        regional_forward_common.loc[
            regional_forward_common[
                "Region"
            ] == region
        ]
        .sort_values("Month")
        .reset_index(drop=True)
        .copy()
    )

    y_region = (
        region_data[
            "Regional_ETF_Flow_Next_Month"
        ].astype(float)
    )

    region_specifications = {
        "R0_Persistence": [
            regional_flow_column,
        ],
        "R1_Fed_Direction": [
            regional_flow_column,
            "Fed_Expected_Direction",
        ],
    }

    regional_forward_results[region] = {}

    region_fit_rows = {}

    for model_name, regressors in (
        region_specifications.items()
    ):

        X_region = sm.add_constant(
            region_data[
                regressors
            ].astype(float),
            has_constant="add",
        )

        region_ols = sm.OLS(
            y_region,
            X_region,
        ).fit()

        region_hac = (
            region_ols
            .get_robustcov_results(
                cov_type="HAC",
                maxlags=2,
                use_correction=True,
            )
        )

        region_names = X_region.columns.tolist()
        region_hac.model.data.xnames = region_names

        parameters = np.asarray(
            region_hac.params
        )

        standard_errors = np.asarray(
            region_hac.bse
        )

        pvalues = np.asarray(
            region_hac.pvalues
        )

        for position, variable in enumerate(
            region_names
        ):

            regional_coefficient_rows.append({
                "Region": region,
                "Model": model_name,
                "Variable": variable,
                "Coefficient":
                    parameters[position],
                "HAC_SE":
                    standard_errors[position],
                "HAC_p":
                    pvalues[position],
            })

        region_fit_rows[model_name] = {
            "R2": region_ols.rsquared,
            "Adjusted_R2":
                region_ols.rsquared_adj,
            "AIC": region_ols.aic,
            "BIC": region_ols.bic,
        }

        regional_forward_results[
            region
        ][model_name] = {
            "OLS": region_ols,
            "HAC": region_hac,
            "Data": region_data,
            "Regressors": regressors,
        }

    r1_hac = regional_forward_results[
        region
    ]["R1_Fed_Direction"]["HAC"]

    r1_names = (
        regional_forward_results[
            region
        ]["R1_Fed_Direction"]["OLS"]
        .model.exog_names
    )

    direction_position = r1_names.index(
        "Fed_Expected_Direction"
    )

    direction_coefficient = float(
        np.asarray(
            r1_hac.params
        )[direction_position]
    )

    direction_standard_error = float(
        np.asarray(
            r1_hac.bse
        )[direction_position]
    )

    direction_p = float(
        np.asarray(
            r1_hac.pvalues
        )[direction_position]
    )

    regional_model_rows.append({
        "Region": region,
        "N": len(region_data),
        "Direction_Coefficient":
            direction_coefficient,
        "Direction_HAC_SE":
            direction_standard_error,
        "Direction_HAC_p":
            direction_p,
        "Direction_Effect_for_0_10":
            direction_coefficient * 0.10,
        "Direction_Significant_5pct":
            direction_p < 0.05,
        "Direction_Significant_10pct":
            direction_p < 0.10,
        "R0_Adjusted_R2":
            region_fit_rows[
                "R0_Persistence"
            ]["Adjusted_R2"],
        "R1_Adjusted_R2":
            region_fit_rows[
                "R1_Fed_Direction"
            ]["Adjusted_R2"],
        "Delta_Adjusted_R2":
            region_fit_rows[
                "R1_Fed_Direction"
            ]["Adjusted_R2"]
            - region_fit_rows[
                "R0_Persistence"
            ]["Adjusted_R2"],
        "R0_AIC":
            region_fit_rows[
                "R0_Persistence"
            ]["AIC"],
        "R1_AIC":
            region_fit_rows[
                "R1_Fed_Direction"
            ]["AIC"],
        "Delta_AIC":
            region_fit_rows[
                "R1_Fed_Direction"
            ]["AIC"]
            - region_fit_rows[
                "R0_Persistence"
            ]["AIC"],
        "R0_BIC":
            region_fit_rows[
                "R0_Persistence"
            ]["BIC"],
        "R1_BIC":
            region_fit_rows[
                "R1_Fed_Direction"
            ]["BIC"],
        "Delta_BIC":
            region_fit_rows[
                "R1_Fed_Direction"
            ]["BIC"]
            - region_fit_rows[
                "R0_Persistence"
            ]["BIC"],
    })

regional_forward_summary = pd.DataFrame(
    regional_model_rows
)

regional_forward_summary = (
    regional_forward_summary
    .sort_values(
        "Direction_HAC_p"
    )
    .reset_index(drop=True)
)

regional_forward_coefficients = pd.DataFrame(
    regional_coefficient_rows
)

display(
    regional_forward_summary.round(4)
)

# -----------------------------------------------------------
# Plot regional direction coefficients with 95% intervals
# -----------------------------------------------------------

regional_plot = (
    regional_forward_summary.copy()
)

regional_plot["CI_Lower_95pct"] = (
    regional_plot["Direction_Coefficient"]
    - 1.96
    * regional_plot["Direction_HAC_SE"]
)

regional_plot["CI_Upper_95pct"] = (
    regional_plot["Direction_Coefficient"]
    + 1.96
    * regional_plot["Direction_HAC_SE"]
)

regional_plot["Significance"] = np.where(
    regional_plot[
        "Direction_HAC_p"
    ] < 0.05,
    "Significant at 5%",
    np.where(
        regional_plot[
            "Direction_HAC_p"
        ] < 0.10,
        "Significant at 10% only",
        "Not significant",
    ),
)

fig = px.scatter(
    regional_plot,
    x="Direction_Coefficient",
    y="Region",
    color="Significance",
    error_x=(
        regional_plot[
            "CI_Upper_95pct"
        ]
        - regional_plot[
            "Direction_Coefficient"
        ]
    ),
    hover_data={
        "Direction_HAC_p": ":.4f",
        "Direction_Effect_for_0_10": ":.3f",
        "Delta_Adjusted_R2": ":.4f",
        "Delta_AIC": ":.4f",
        "Delta_BIC": ":.4f",
    },
    color_discrete_map={
        "Significant at 5%": "#2ca02c",
        "Significant at 10% only": "#ffbf00",
        "Not significant": "#d62728",
    },
    title=(
        "Expected Fed Direction and "
        "Next-Month Regional Gold ETF Demand"
    ),
)

fig.add_vline(
    x=0,
    line_color="black",
    line_dash="dash",
)

fig.update_traces(
    marker=dict(size=13),
)

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=600,
    xaxis_title=(
        "Direction coefficient "
        "(tonnes for a full −1 to +1 change)"
    ),
    yaxis_title="ETF region",
    legend_title="HAC inference",
)

fig.show()

Regions: ['Asia', 'Europe', 'North America', 'Others']


,Region,Observations,First_Meeting_Month,Final_Meeting_Month
0,Asia,60,2019-03-01,2026-06-01
1,Europe,60,2019-03-01,2026-06-01
2,North America,60,2019-03-01,2026-06-01
3,Others,60,2019-03-01,2026-06-01


,Region,N,Direction_Coefficient,Direction_HAC_SE,Direction_HAC_p,Direction_Effect_for_0_10,Direction_Significant_5pct,Direction_Significant_10pct,R0_Adjusted_R2,R1_Adjusted_R2,Delta_Adjusted_R2,R0_AIC,R1_AIC,Delta_AIC,R0_BIC,R1_BIC,Delta_BIC
0,Europe,60,-19.2992,4.9753,0.0003,-1.9299,True,True,0.1167,0.1993,0.0826,538.3744,533.4406,-4.9339,542.5631,539.7236,-2.8395
1,North America,60,-41.9825,14.2779,0.0047,-4.1982,True,True,0.2972,0.4164,0.1192,597.8742,587.6768,-10.1974,602.0629,593.9599,-8.1030
2,Others,60,-0.9833,0.4655,0.0390,-0.0983,True,True,0.1122,0.1576,0.0454,203.8865,201.6912,-2.1953,208.0752,207.9743,-0.1009
3,Asia,60,-8.0266,3.9561,0.0471,-0.8027,True,True,0.0658,0.0970,0.0312,487.2887,486.2084,-1.0803,491.4774,492.4915,1.0141


In [44]:
# CELL 38: Multiple-testing correction and pooled regional model

from statsmodels.stats.multitest import (
    multipletests
)
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np

# -----------------------------------------------------------
# 1. Correct the four regional p-values
# -----------------------------------------------------------

regional_inference = (
    regional_forward_summary[
        [
            "Region",
            "Direction_Coefficient",
            "Direction_HAC_SE",
            "Direction_HAC_p",
            "Direction_Effect_for_0_10",
            "Delta_Adjusted_R2",
            "Delta_AIC",
            "Delta_BIC",
        ]
    ]
    .copy()
)

regional_inference[
    "Holm_Adjusted_p"
] = multipletests(
    regional_inference[
        "Direction_HAC_p"
    ],
    alpha=0.05,
    method="holm",
)[1]

regional_inference[
    "BH_FDR_Adjusted_p"
] = multipletests(
    regional_inference[
        "Direction_HAC_p"
    ],
    alpha=0.05,
    method="fdr_bh",
)[1]

regional_inference[
    "Holm_Significant_5pct"
] = (
    regional_inference[
        "Holm_Adjusted_p"
    ] < 0.05
)

regional_inference[
    "BH_FDR_Significant_5pct"
] = (
    regional_inference[
        "BH_FDR_Adjusted_p"
    ] < 0.05
)

regional_inference = (
    regional_inference
    .sort_values("Direction_HAC_p")
    .reset_index(drop=True)
)

print("Regional inference after multiple-testing correction")

display(
    regional_inference.round(4)
)

# -----------------------------------------------------------
# 2. Compare the regional sum with the global coefficient
# -----------------------------------------------------------

regional_coefficient_sum = (
    regional_inference[
        "Direction_Coefficient"
    ].sum()
)

global_direction_coefficient = float(
    forward_etf_coefficients.loc[
        (
            forward_etf_coefficients["Model"]
            == "P1_Direction"
        )
        & (
            forward_etf_coefficients["Variable"]
            == "Fed_Expected_Direction"
        ),
        "Coefficient",
    ].iloc[0]
)

coefficient_reconciliation = pd.DataFrame({
    "Quantity": [
        "Sum of four separate regional coefficients",
        "Global P1 direction coefficient",
        "Regional sum minus global coefficient",
    ],
    "Value": [
        regional_coefficient_sum,
        global_direction_coefficient,
        (
            regional_coefficient_sum
            - global_direction_coefficient
        ),
    ],
})

display(
    coefficient_reconciliation.round(4)
)

# -----------------------------------------------------------
# 3. Pooled regional model
#
# Region-specific intercepts, persistence coefficients and
# Fed-direction coefficients.
#
# Standard errors are clustered by meeting month to account
# for contemporaneous dependence across the four regions.
# -----------------------------------------------------------

pooled_regional_df = (
    regional_forward_common[
        [
            "Month",
            "Region",
            regional_flow_column,
            "Regional_ETF_Flow_Next_Month",
            "Fed_Expected_Direction",
        ]
    ]
    .copy()
)

pooled_regional_df["Region"] = pd.Categorical(
    pooled_regional_df["Region"],
    categories=[
        "North America",
        "Europe",
        "Asia",
        "Others",
    ],
)

pooled_formula = (
    "Regional_ETF_Flow_Next_Month ~ "
    "0 + C(Region) "
    "+ C(Region):"
    "ETF_Net_Demand_Regional_WGC_tonnes "
    "+ C(Region):Fed_Expected_Direction"
)

pooled_regional_ols = smf.ols(
    formula=pooled_formula,
    data=pooled_regional_df,
).fit()

pooled_regional_clustered = (
    pooled_regional_ols.get_robustcov_results(
        cov_type="cluster",
        groups=pooled_regional_df["Month"],
        use_correction=True,
    )
)

pooled_parameter_names = (
    pooled_regional_ols.model.exog_names
)

pooled_parameters = np.asarray(
    pooled_regional_clustered.params
)

pooled_standard_errors = np.asarray(
    pooled_regional_clustered.bse
)

pooled_pvalues = np.asarray(
    pooled_regional_clustered.pvalues
)

pooled_coefficient_table = pd.DataFrame({
    "Variable": pooled_parameter_names,
    "Coefficient": pooled_parameters,
    "Clustered_SE": pooled_standard_errors,
    "Clustered_p": pooled_pvalues,
})

display(
    pooled_coefficient_table.round(4)
)

# -----------------------------------------------------------
# 4. Extract regional Fed-direction coefficients
# -----------------------------------------------------------

pooled_direction_rows = []

for region in [
    "North America",
    "Europe",
    "Asia",
    "Others",
]:

    matching_names = [
        name
        for name in pooled_parameter_names
        if (
            "Fed_Expected_Direction" in name
            and f"[{region}]" in name
        )
    ]

    if len(matching_names) != 1:
        raise ValueError(
            "Could not uniquely identify the pooled "
            f"direction coefficient for {region}. "
            f"Matches: {matching_names}"
        )

    coefficient_name = matching_names[0]
    position = pooled_parameter_names.index(
        coefficient_name
    )

    coefficient = float(
        pooled_parameters[position]
    )

    standard_error = float(
        pooled_standard_errors[position]
    )

    p_value = float(
        pooled_pvalues[position]
    )

    pooled_direction_rows.append({
        "Region": region,
        "Direction_Coefficient":
            coefficient,
        "Clustered_SE":
            standard_error,
        "Clustered_p":
            p_value,
        "Effect_for_0_10_Change":
            coefficient * 0.10,
        "CI_Lower_95pct":
            coefficient
            - 1.96 * standard_error,
        "CI_Upper_95pct":
            coefficient
            + 1.96 * standard_error,
        "Significant_5pct":
            p_value < 0.05,
    })

pooled_direction_results = pd.DataFrame(
    pooled_direction_rows
)

display(
    pooled_direction_results.round(4)
)

# -----------------------------------------------------------
# 5. Test whether direction effects are equal across regions
# -----------------------------------------------------------

direction_parameter_names = {}

for region in [
    "North America",
    "Europe",
    "Asia",
    "Others",
]:

    direction_parameter_names[region] = [
        name
        for name in pooled_parameter_names
        if (
            "Fed_Expected_Direction" in name
            and f"[{region}]" in name
        )
    ][0]

north_america_name = (
    direction_parameter_names[
        "North America"
    ]
)

europe_name = (
    direction_parameter_names["Europe"]
)

asia_name = (
    direction_parameter_names["Asia"]
)

others_name = (
    direction_parameter_names["Others"]
)

regional_equality_test = (
    pooled_regional_clustered.wald_test(
        [
            f"{north_america_name} - {europe_name} = 0",
            f"{north_america_name} - {asia_name} = 0",
            f"{north_america_name} - {others_name} = 0",
        ],
        use_f=False,
        scalar=True,
    )
)

regional_equality_result = pd.DataFrame({
    "Test": [
        "Fed-direction coefficients equal across regions"
    ],
    "Wald_Statistic": [
        float(
            np.asarray(
                regional_equality_test.statistic
            ).reshape(-1)[0]
        )
    ],
    "Restrictions": [3],
    "p_value": [
        float(
            np.asarray(
                regional_equality_test.pvalue
            ).reshape(-1)[0]
        )
    ],
})

regional_equality_result[
    "Reject_Equality_5pct"
] = (
    regional_equality_result[
        "p_value"
    ] < 0.05
)

display(
    regional_equality_result.round(4)
)

Regional inference after multiple-testing correction


,Region,Direction_Coefficient,Direction_HAC_SE,Direction_HAC_p,Direction_Effect_for_0_10,Delta_Adjusted_R2,Delta_AIC,Delta_BIC,Holm_Adjusted_p,BH_FDR_Adjusted_p,Holm_Significant_5pct,BH_FDR_Significant_5pct
0,Europe,-19.2992,4.9753,0.0003,-1.9299,0.0826,-4.9339,-2.8395,0.0011,0.0011,True,True
1,North America,-41.9825,14.2779,0.0047,-4.1982,0.1192,-10.1974,-8.1030,0.0142,0.0095,True,True
2,Others,-0.9833,0.4655,0.0390,-0.0983,0.0454,-2.1953,-0.1009,0.0781,0.0471,False,True
3,Asia,-8.0266,3.9561,0.0471,-0.8027,0.0312,-1.0803,1.0141,0.0781,0.0471,False,True


,Quantity,Value
0,Sum of four separate regional coefficients,-70.2915
1,Global P1 direction coefficient,-68.0747
2,Regional sum minus global coefficient,-2.2168


,Variable,Coefficient,Clustered_SE,Clustered_p
0,C(Region)[North America],11.6858,4.9442,0.0214
1,C(Region)[Europe],5.6958,2.8593,0.0510
2,C(Region)[Asia],5.8590,1.7403,0.0013
3,C(Region)[Others],0.5001,0.1488,0.0014
4,C(Region)[North America]:ETF_Net_Demand_Region...,0.3470,0.0825,0.0001
5,C(Region)[Europe]:ETF_Net_Demand_Regional_WGC_...,0.1808,0.0847,0.0370
6,C(Region)[Asia]:ETF_Net_Demand_Regional_WGC_to...,0.2964,0.1811,0.1071
7,C(Region)[Others]:ETF_Net_Demand_Regional_WGC_...,0.2035,0.1030,0.0528
8,C(Region)[North America]:Fed_Expected_Direction,-41.9825,12.9213,0.0019
9,C(Region)[Europe]:Fed_Expected_Direction,-19.2992,6.2771,0.0032


,Region,Direction_Coefficient,Clustered_SE,Clustered_p,Effect_for_0_10_Change,CI_Lower_95pct,CI_Upper_95pct,Significant_5pct
0,North America,-41.9825,12.9213,0.0019,-4.1982,-67.3082,-16.6567,True
1,Europe,-19.2992,6.2771,0.0032,-1.9299,-31.6023,-6.9961,True
2,Asia,-8.0266,3.7044,0.0343,-0.8027,-15.2872,-0.7660,True
3,Others,-0.9833,0.4943,0.0513,-0.0983,-1.9521,-0.0145,False


,Test,Wald_Statistic,Restrictions,p_value,Reject_Equality_5pct
0,Fed-direction coefficients equal across regions,31.0784,3,0.0,True


In [45]:
# CELL 39: Expanding-window regional ETF forecasting

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from scipy import stats
import pandas as pd
import numpy as np
import plotly.express as px

regional_initial_window = 30

regional_recursive_rows = []
regional_forecast_summary_rows = []

for region in [
    "North America",
    "Europe",
    "Asia",
    "Others",
]:

    region_data = (
        regional_forward_common.loc[
            regional_forward_common["Region"] == region,
            [
                "Month",
                regional_flow_column,
                "Regional_ETF_Flow_Next_Month",
                "Fed_Expected_Direction",
            ],
        ]
        .sort_values("Month")
        .reset_index(drop=True)
        .copy()
    )

    if regional_initial_window >= len(region_data):
        raise ValueError(
            f"Initial window is too large for {region}."
        )

    region_forecast_rows = []

    for forecast_position in range(
        regional_initial_window,
        len(region_data)
    ):

        training_data = region_data.iloc[
            :forecast_position
        ].copy()

        test_data = region_data.iloc[
            [forecast_position]
        ].copy()

        y_train = training_data[
            "Regional_ETF_Flow_Next_Month"
        ].astype(float)

        # -----------------------------------------------
        # R0: regional ETF persistence only
        # -----------------------------------------------

        X0_train = sm.add_constant(
            training_data[
                [regional_flow_column]
            ].astype(float),
            has_constant="add",
        )

        X0_test = sm.add_constant(
            test_data[
                [regional_flow_column]
            ].astype(float),
            has_constant="add",
        ).reindex(
            columns=X0_train.columns,
            fill_value=1.0,
        )

        model_0 = sm.OLS(
            y_train,
            X0_train,
        ).fit()

        forecast_0 = float(
            model_0.predict(
                X0_test
            ).iloc[0]
        )

        # -----------------------------------------------
        # R1: persistence plus expected Fed direction
        # -----------------------------------------------

        X1_train = sm.add_constant(
            training_data[
                [
                    regional_flow_column,
                    "Fed_Expected_Direction",
                ]
            ].astype(float),
            has_constant="add",
        )

        X1_test = sm.add_constant(
            test_data[
                [
                    regional_flow_column,
                    "Fed_Expected_Direction",
                ]
            ].astype(float),
            has_constant="add",
        ).reindex(
            columns=X1_train.columns,
            fill_value=1.0,
        )

        model_1 = sm.OLS(
            y_train,
            X1_train,
        ).fit()

        forecast_1 = float(
            model_1.predict(
                X1_test
            ).iloc[0]
        )

        actual = float(
            test_data[
                "Regional_ETF_Flow_Next_Month"
            ].iloc[0]
        )

        meeting_month = test_data[
            "Month"
        ].iloc[0]

        direction_coefficient = float(
            model_1.params[
                "Fed_Expected_Direction"
            ]
        )

        observation_result = {
            "Region": region,
            "Meeting_Month": meeting_month,
            "Outcome_Month": (
                meeting_month
                + pd.offsets.MonthBegin(1)
            ),
            "Training_Observations":
                len(training_data),
            "Actual_Regional_ETF_Flow":
                actual,
            "R0_Persistence_Forecast":
                forecast_0,
            "R1_Fed_Direction_Forecast":
                forecast_1,
            "R0_Error":
                actual - forecast_0,
            "R1_Error":
                actual - forecast_1,
            "Recursive_Direction_Coefficient":
                direction_coefficient,
        }

        region_forecast_rows.append(
            observation_result
        )

        regional_recursive_rows.append(
            observation_result
        )

    region_forecasts = pd.DataFrame(
        region_forecast_rows
    )

    actual = region_forecasts[
        "Actual_Regional_ETF_Flow"
    ].to_numpy()

    forecast_0 = region_forecasts[
        "R0_Persistence_Forecast"
    ].to_numpy()

    forecast_1 = region_forecasts[
        "R1_Fed_Direction_Forecast"
    ].to_numpy()

    error_0 = actual - forecast_0
    error_1 = actual - forecast_1

    rmse_0 = np.sqrt(
        np.mean(error_0 ** 2)
    )

    rmse_1 = np.sqrt(
        np.mean(error_1 ** 2)
    )

    mae_0 = np.mean(
        np.abs(error_0)
    )

    mae_1 = np.mean(
        np.abs(error_1)
    )

    sign_accuracy_0 = np.mean(
        np.sign(actual)
        == np.sign(forecast_0)
    )

    sign_accuracy_1 = np.mean(
        np.sign(actual)
        == np.sign(forecast_1)
    )

    actual_standard_deviation = np.std(
        actual,
        ddof=1,
    )

    # -----------------------------------------------
    # Clark-West nested forecast test
    # -----------------------------------------------

    cw_adjusted_loss = (
        error_0 ** 2
        - (
            error_1 ** 2
            - (
                forecast_0
                - forecast_1
            ) ** 2
        )
    )

    cw_regression = sm.OLS(
        cw_adjusted_loss,
        np.ones(len(cw_adjusted_loss)),
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 2,
            "use_correction": True,
        },
    )

    cw_statistic = float(
        np.asarray(
            cw_regression.tvalues
        ).reshape(-1)[0]
    )

    cw_one_sided_p = float(
        1 - stats.t.cdf(
            cw_statistic,
            df=cw_regression.df_resid,
        )
    )

    # -----------------------------------------------
    # In-sample standardized direction coefficient
    # -----------------------------------------------

    full_region_summary = (
        regional_forward_summary.loc[
            regional_forward_summary["Region"]
            == region
        ]
        .iloc[0]
    )

    direction_sd = region_data[
        "Fed_Expected_Direction"
    ].std(ddof=1)

    outcome_sd = region_data[
        "Regional_ETF_Flow_Next_Month"
    ].std(ddof=1)

    standardized_direction_effect = (
        full_region_summary[
            "Direction_Coefficient"
        ]
        * direction_sd
        / outcome_sd
    )

    regional_forecast_summary_rows.append({
        "Region": region,
        "Forecasts": len(region_forecasts),
        "First_Outcome_Month":
            region_forecasts[
                "Outcome_Month"
            ].min(),
        "Final_Outcome_Month":
            region_forecasts[
                "Outcome_Month"
            ].max(),
        "R0_RMSE": rmse_0,
        "R1_RMSE": rmse_1,
        "RMSE_Improvement_pct":
            (rmse_0 - rmse_1)
            / rmse_0 * 100,
        "R0_MAE": mae_0,
        "R1_MAE": mae_1,
        "MAE_Improvement_pct":
            (mae_0 - mae_1)
            / mae_0 * 100,
        "R0_Normalized_RMSE":
            rmse_0 / actual_standard_deviation,
        "R1_Normalized_RMSE":
            rmse_1 / actual_standard_deviation,
        "R0_Sign_Accuracy":
            sign_accuracy_0,
        "R1_Sign_Accuracy":
            sign_accuracy_1,
        "Clark_West_Statistic":
            cw_statistic,
        "Clark_West_One_Sided_p":
            cw_one_sided_p,
        "Minimum_Recursive_Direction":
            region_forecasts[
                "Recursive_Direction_Coefficient"
            ].min(),
        "Maximum_Recursive_Direction":
            region_forecasts[
                "Recursive_Direction_Coefficient"
            ].max(),
        "Negative_Direction_Share":
            (
                region_forecasts[
                    "Recursive_Direction_Coefficient"
                ] < 0
            ).mean(),
        "Standardized_Direction_Effect":
            standardized_direction_effect,
    })

regional_recursive_forecasts = pd.DataFrame(
    regional_recursive_rows
)

regional_forecast_summary = pd.DataFrame(
    regional_forecast_summary_rows
)

# -----------------------------------------------------------
# Multiple-testing correction for regional Clark-West tests
# -----------------------------------------------------------

regional_forecast_summary[
    "Clark_West_Holm_p"
] = multipletests(
    regional_forecast_summary[
        "Clark_West_One_Sided_p"
    ],
    alpha=0.05,
    method="holm",
)[1]

regional_forecast_summary[
    "Clark_West_BH_FDR_p"
] = multipletests(
    regional_forecast_summary[
        "Clark_West_One_Sided_p"
    ],
    alpha=0.05,
    method="fdr_bh",
)[1]

regional_forecast_summary[
    "CW_Holm_Significant_5pct"
] = (
    regional_forecast_summary[
        "Clark_West_Holm_p"
    ] < 0.05
)

regional_forecast_summary[
    "CW_BH_FDR_Significant_5pct"
] = (
    regional_forecast_summary[
        "Clark_West_BH_FDR_p"
    ] < 0.05
)

regional_forecast_summary = (
    regional_forecast_summary
    .sort_values(
        "RMSE_Improvement_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    regional_forecast_summary.round(4)
)

# -----------------------------------------------------------
# Plot regional out-of-sample improvement
# -----------------------------------------------------------

regional_forecast_summary[
    "Forecast_Evidence"
] = np.where(
    regional_forecast_summary[
        "CW_Holm_Significant_5pct"
    ],
    "Holm significant",
    np.where(
        regional_forecast_summary[
            "CW_BH_FDR_Significant_5pct"
        ],
        "FDR significant",
        "Not significant",
    ),
)

fig = px.bar(
    regional_forecast_summary,
    x="Region",
    y="RMSE_Improvement_pct",
    color="Forecast_Evidence",
    text="RMSE_Improvement_pct",
    hover_data={
        "MAE_Improvement_pct": ":.2f",
        "R0_Sign_Accuracy": ":.3f",
        "R1_Sign_Accuracy": ":.3f",
        "Clark_West_One_Sided_p": ":.4f",
        "Clark_West_Holm_p": ":.4f",
        "Standardized_Direction_Effect": ":.4f",
        "Negative_Direction_Share": ":.3f",
    },
    color_discrete_map={
        "Holm significant": "#2ca02c",
        "FDR significant": "#ffbf00",
        "Not significant": "#d62728",
    },
    title=(
        "Out-of-Sample Value of Expected Fed Direction "
        "for Regional ETF Flows"
    ),
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    width=1150,
    height=650,
    xaxis_title="ETF region",
    yaxis_title="RMSE improvement over persistence model (%)",
    legend_title="Clark–West evidence",
)

fig.show()

,Region,Forecasts,First_Outcome_Month,Final_Outcome_Month,R0_RMSE,R1_RMSE,RMSE_Improvement_pct,R0_MAE,R1_MAE,MAE_Improvement_pct,...,Clark_West_Statistic,Clark_West_One_Sided_p,Minimum_Recursive_Direction,Maximum_Recursive_Direction,Negative_Direction_Share,Standardized_Direction_Effect,Clark_West_Holm_p,Clark_West_BH_FDR_p,CW_Holm_Significant_5pct,CW_BH_FDR_Significant_5pct
0,Others,30,2023-01-01,2026-07-01,1.3800,1.3435,2.6404,1.0505,1.0333,1.6371,...,1.8220,0.0394,-1.1850,-0.5915,1.0,-0.2826,0.1576,0.0557,False,False
1,Asia,30,2023-01-01,2026-07-01,20.4414,19.9686,2.3128,12.5091,12.2225,2.2910,...,1.8015,0.0410,-7.8701,-1.7238,1.0,-0.2231,0.1576,0.0557,False,False
2,North America,30,2023-01-01,2026-07-01,27.4854,27.1196,1.3309,20.1515,23.2514,-15.3831,...,1.7919,0.0418,-56.8515,-35.8300,1.0,-0.4027,0.1576,0.0557,False,False
3,Europe,30,2023-01-01,2026-07-01,20.9015,21.2102,-1.4772,17.2939,17.1380,0.9015,...,0.8510,0.2009,-30.8897,-18.2812,1.0,-0.3407,0.2009,0.2009,False,False


# Expected FOMC directiont​⟶ΔUSDt+1​.

In [46]:
# CELL 40: FOMC information and next-month broad USD changes

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------------------------------------
# Construct the next-calendar-month USD outcome BEFORE
# selecting FOMC meeting months
# -----------------------------------------------------------

usd_monthly = (
    etf_monthly
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

# Ensure the corrected broad USD series is used.
if "Broad_USD_Change" not in usd_monthly.columns:
    raise KeyError(
        "Corrected Broad_USD_Change is missing."
    )

usd_monthly[
    "Broad_USD_Change_Next_Month"
] = (
    usd_monthly[
        "Broad_USD_Change"
    ].shift(-1)
)

# Retain genuine FOMC meeting observations.
usd_event_df = (
    usd_monthly.loc[
        usd_monthly[
            "Fed_Meeting_Flag"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)

# -----------------------------------------------------------
# Nested forward USD specifications
# -----------------------------------------------------------

usd_model_specifications = {
    "U0_USD_Persistence": [
        "Broad_USD_Change",
    ],

    "U1_Direction": [
        "Broad_USD_Change",
        "Fed_Expected_Direction",
    ],

    "U2_Direction_Surprise": [
        "Broad_USD_Change",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
    ],

    "U3_Complete_Fed": [
        "Broad_USD_Change",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
        "Fed_Policy_Uncertainty",
    ],
}

all_usd_regressors = list(dict.fromkeys(
    variable
    for regressors in usd_model_specifications.values()
    for variable in regressors
))

# One common sample for all nested comparisons.
usd_forward_common = (
    usd_event_df[
        [
            "Month",
            "Broad_USD_Change_Next_Month",
        ] + all_usd_regressors
    ]
    .dropna()
    .sort_values("Month")
    .reset_index(drop=True)
)

print(
    "Forward USD-model observations:",
    len(usd_forward_common)
)

print(
    "FOMC-information period:",
    usd_forward_common["Month"].min(),
    "to",
    usd_forward_common["Month"].max()
)

print(
    "USD outcome period:",
    (
        usd_forward_common["Month"].min()
        + pd.offsets.MonthBegin(1)
    ),
    "to",
    (
        usd_forward_common["Month"].max()
        + pd.offsets.MonthBegin(1)
    )
)

print(
    "Excluded meeting months:",
    usd_event_df.loc[
        ~usd_event_df["Month"].isin(
            usd_forward_common["Month"]
        ),
        "Month"
    ].dt.strftime("%Y-%m").tolist()
)

# -----------------------------------------------------------
# Estimate nested models
# -----------------------------------------------------------

usd_forward_results = {}
usd_model_rows = []
usd_coefficient_rows = []

y_usd = (
    usd_forward_common[
        "Broad_USD_Change_Next_Month"
    ].astype(float)
)

for model_name, regressors in (
    usd_model_specifications.items()
):

    X_usd = sm.add_constant(
        usd_forward_common[
            regressors
        ].astype(float),
        has_constant="add",
    )

    usd_ols = sm.OLS(
        y_usd,
        X_usd,
    ).fit()

    usd_hac = (
        usd_ols.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    parameter_names = X_usd.columns.tolist()
    usd_hac.model.data.xnames = parameter_names

    parameters = np.asarray(
        usd_hac.params
    )

    standard_errors = np.asarray(
        usd_hac.bse
    )

    pvalues = np.asarray(
        usd_hac.pvalues
    )

    for position, variable in enumerate(
        parameter_names
    ):

        usd_coefficient_rows.append({
            "Model": model_name,
            "Variable": variable,
            "Coefficient":
                parameters[position],
            "HAC_SE":
                standard_errors[position],
            "HAC_p":
                pvalues[position],
        })

    included_fed_variables = [
        variable
        for variable in [
            "Fed_Expected_Direction",
            "Fed_Surprise_Signed",
            "Fed_Policy_Uncertainty",
        ]
        if variable in regressors
    ]

    if included_fed_variables:

        fed_block_test = usd_hac.wald_test(
            [
                f"{variable} = 0"
                for variable
                in included_fed_variables
            ],
            use_f=False,
            scalar=True,
        )

        fed_block_statistic = float(
            np.asarray(
                fed_block_test.statistic
            ).reshape(-1)[0]
        )

        fed_block_p = float(
            np.asarray(
                fed_block_test.pvalue
            ).reshape(-1)[0]
        )

    else:
        fed_block_statistic = np.nan
        fed_block_p = np.nan

    usd_model_rows.append({
        "Model": model_name,
        "Fed_Variables":
            len(included_fed_variables),
        "Parameters": int(
            usd_ols.df_model + 1
        ),
        "N": int(usd_ols.nobs),
        "R2": usd_ols.rsquared,
        "Adjusted_R2":
            usd_ols.rsquared_adj,
        "AIC": usd_ols.aic,
        "BIC": usd_ols.bic,
        "Fed_Block_Wald":
            fed_block_statistic,
        "Fed_Block_p":
            fed_block_p,
    })

    usd_forward_results[model_name] = {
        "OLS": usd_ols,
        "HAC": usd_hac,
        "Regressors": regressors,
    }

usd_model_comparison = pd.DataFrame(
    usd_model_rows
)

usd_forward_coefficients = pd.DataFrame(
    usd_coefficient_rows
)

usd_model_comparison[
    "Delta_Adjusted_R2"
] = usd_model_comparison[
    "Adjusted_R2"
].diff()

usd_model_comparison["Delta_AIC"] = (
    usd_model_comparison["AIC"].diff()
)

usd_model_comparison["Delta_BIC"] = (
    usd_model_comparison["BIC"].diff()
)

display(
    usd_model_comparison.round(4)
)

# -----------------------------------------------------------
# Coefficient and p-value comparisons
# -----------------------------------------------------------

usd_focus_variables = [
    "Broad_USD_Change",
    "Fed_Expected_Direction",
    "Fed_Surprise_Signed",
    "Fed_Policy_Uncertainty",
]

usd_coefficient_comparison = (
    usd_forward_coefficients.loc[
        usd_forward_coefficients[
            "Variable"
        ].isin(usd_focus_variables)
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="Coefficient",
    )
    .reindex(
        usd_model_specifications.keys()
    )
)

usd_pvalue_comparison = (
    usd_forward_coefficients.loc[
        usd_forward_coefficients[
            "Variable"
        ].isin(usd_focus_variables)
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="HAC_p",
    )
    .reindex(
        usd_model_specifications.keys()
    )
)

print("\nForward USD coefficients")
display(
    usd_coefficient_comparison.round(4)
)

print("\nForward USD HAC p-values")
display(
    usd_pvalue_comparison.round(4)
)

# -----------------------------------------------------------
# Plot model fit
# -----------------------------------------------------------

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Adjusted R²",
        "Information Criteria",
    ],
    horizontal_spacing=0.15,
)

fig.add_trace(
    go.Bar(
        x=usd_model_comparison["Model"],
        y=usd_model_comparison["Adjusted_R2"],
        name="Adjusted R²",
        marker_color="#2a6fbb",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=usd_model_comparison["Model"],
        y=usd_model_comparison["AIC"],
        mode="lines+markers",
        name="AIC",
        line=dict(color="#d62728"),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=usd_model_comparison["Model"],
        y=usd_model_comparison["BIC"],
        mode="lines+markers",
        name="BIC",
        line=dict(color="#2ca02c"),
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title=(
        "FOMC Information and "
        "Next-Month Broad USD Changes"
    ),
    template="plotly_white",
    width=1350,
    height=620,
)

fig.update_xaxes(tickangle=30)
fig.show()

Forward USD-model observations: 60
FOMC-information period: 2019-03-01 00:00:00 to 2026-06-01 00:00:00
USD outcome period: 2019-04-01 00:00:00 to 2026-07-01 00:00:00
Excluded meeting months: ['2019-01', '2026-07']


,Model,Fed_Variables,Parameters,N,R2,Adjusted_R2,AIC,BIC,Fed_Block_Wald,Fed_Block_p,Delta_Adjusted_R2,Delta_AIC,Delta_BIC
0,U0_USD_Persistence,0,2,60,0.2100,0.1964,186.8326,191.0213,NaN,NaN,NaN,NaN,NaN
1,U1_Direction,1,3,60,0.2101,0.1824,188.8247,195.1078,0.0094,0.9227,-0.0140,1.9921,4.0865
2,U2_Direction_Surprise,2,4,60,0.2735,0.2346,185.8092,194.1866,3.5595,0.1687,0.0521,-3.0156,-0.9212
3,U3_Complete_Fed,3,5,60,0.2884,0.2367,186.5605,197.0322,9.2007,0.0267,0.0021,0.7513,2.8456



Forward USD coefficients


Variable,Broad_USD_Change,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed
Model,,,,
U0_USD_Persistence,0.3780,NaN,NaN,NaN
U1_Direction,0.3782,0.0323,NaN,NaN
U2_Direction_Surprise,0.3808,0.1318,NaN,-0.8015
U3_Complete_Fed,0.3915,0.2474,0.765,-0.7733



Forward USD HAC p-values


Variable,Broad_USD_Change,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed
Model,,,,
U0_USD_Persistence,0.0,NaN,NaN,NaN
U1_Direction,0.0,0.9230,NaN,NaN
U2_Direction_Surprise,0.0,0.6691,NaN,0.0660
U3_Complete_Fed,0.0,0.4515,0.2434,0.0871


In [47]:
# CELL 41: Out-of-sample tests for the forward USD channel

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from scipy import stats
import pandas as pd
import numpy as np
import plotly.express as px

usd_forecast_specifications = {
    "U0_Persistence": [
        "Broad_USD_Change",
    ],
    "U1_Direction": [
        "Broad_USD_Change",
        "Fed_Expected_Direction",
    ],
    "U2_Direction_Surprise": [
        "Broad_USD_Change",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
    ],
    "U3_Complete_Fed": [
        "Broad_USD_Change",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
        "Fed_Policy_Uncertainty",
    ],
}

usd_initial_training_window = 30

usd_recursive_rows = []

for forecast_position in range(
    usd_initial_training_window,
    len(usd_forward_common)
):

    training_data = usd_forward_common.iloc[
        :forecast_position
    ].copy()

    test_data = usd_forward_common.iloc[
        [forecast_position]
    ].copy()

    actual = float(
        test_data[
            "Broad_USD_Change_Next_Month"
        ].iloc[0]
    )

    meeting_month = test_data[
        "Month"
    ].iloc[0]

    forecast_row = {
        "Meeting_Month": meeting_month,
        "Outcome_Month": (
            meeting_month
            + pd.offsets.MonthBegin(1)
        ),
        "Actual_USD_Change": actual,
        "Training_Observations":
            len(training_data),
    }

    for model_name, regressors in (
        usd_forecast_specifications.items()
    ):

        X_train = sm.add_constant(
            training_data[
                regressors
            ].astype(float),
            has_constant="add",
        )

        X_test = sm.add_constant(
            test_data[
                regressors
            ].astype(float),
            has_constant="add",
        ).reindex(
            columns=X_train.columns,
            fill_value=1.0,
        )

        recursive_model = sm.OLS(
            training_data[
                "Broad_USD_Change_Next_Month"
            ].astype(float),
            X_train,
        ).fit()

        forecast = float(
            recursive_model.predict(
                X_test
            ).iloc[0]
        )

        forecast_row[
            f"{model_name}_Forecast"
        ] = forecast

        forecast_row[
            f"{model_name}_Error"
        ] = actual - forecast

        if (
            "Fed_Expected_Direction"
            in recursive_model.params.index
        ):
            forecast_row[
                f"{model_name}_Direction_Coefficient"
            ] = recursive_model.params[
                "Fed_Expected_Direction"
            ]

        if (
            "Fed_Surprise_Signed"
            in recursive_model.params.index
        ):
            forecast_row[
                f"{model_name}_Surprise_Coefficient"
            ] = recursive_model.params[
                "Fed_Surprise_Signed"
            ]

    usd_recursive_rows.append(
        forecast_row
    )

usd_recursive_forecasts = pd.DataFrame(
    usd_recursive_rows
)

print(
    "Pseudo-out-of-sample USD forecasts:",
    len(usd_recursive_forecasts)
)

print(
    "First outcome month:",
    usd_recursive_forecasts[
        "Outcome_Month"
    ].min()
)

print(
    "Final outcome month:",
    usd_recursive_forecasts[
        "Outcome_Month"
    ].max()
)

# -----------------------------------------------------------
# Forecast metrics
# -----------------------------------------------------------

actual = usd_recursive_forecasts[
    "Actual_USD_Change"
].to_numpy()

usd_metric_rows = []

for model_name in usd_forecast_specifications:

    forecast = usd_recursive_forecasts[
        f"{model_name}_Forecast"
    ].to_numpy()

    error = actual - forecast

    usd_metric_rows.append({
        "Model": model_name,
        "Forecasts": len(actual),
        "RMSE": np.sqrt(
            np.mean(error ** 2)
        ),
        "MAE": np.mean(
            np.abs(error)
        ),
        "Bias": np.mean(error),
        "Forecast_Actual_Correlation":
            np.corrcoef(
                actual,
                forecast,
            )[0, 1],
        "Direction_Accuracy":
            np.mean(
                np.sign(actual)
                == np.sign(forecast)
            ),
    })

usd_forecast_metrics = pd.DataFrame(
    usd_metric_rows
)

benchmark_rmse = usd_forecast_metrics.loc[
    usd_forecast_metrics["Model"]
    == "U0_Persistence",
    "RMSE",
].iloc[0]

benchmark_mae = usd_forecast_metrics.loc[
    usd_forecast_metrics["Model"]
    == "U0_Persistence",
    "MAE",
].iloc[0]

usd_forecast_metrics[
    "RMSE_Improvement_vs_U0_pct"
] = (
    (
        benchmark_rmse
        - usd_forecast_metrics["RMSE"]
    )
    / benchmark_rmse
    * 100
)

usd_forecast_metrics[
    "MAE_Improvement_vs_U0_pct"
] = (
    (
        benchmark_mae
        - usd_forecast_metrics["MAE"]
    )
    / benchmark_mae
    * 100
)

# -----------------------------------------------------------
# Clark-West tests against persistence-only model
# -----------------------------------------------------------

benchmark_forecast = (
    usd_recursive_forecasts[
        "U0_Persistence_Forecast"
    ].to_numpy()
)

benchmark_error = (
    actual - benchmark_forecast
)

cw_rows = []

for model_name in [
    "U1_Direction",
    "U2_Direction_Surprise",
    "U3_Complete_Fed",
]:

    competing_forecast = (
        usd_recursive_forecasts[
            f"{model_name}_Forecast"
        ].to_numpy()
    )

    competing_error = (
        actual - competing_forecast
    )

    adjusted_loss = (
        benchmark_error ** 2
        - (
            competing_error ** 2
            - (
                benchmark_forecast
                - competing_forecast
            ) ** 2
        )
    )

    cw_regression = sm.OLS(
        adjusted_loss,
        np.ones(len(adjusted_loss)),
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 2,
            "use_correction": True,
        },
    )

    cw_statistic = float(
        np.asarray(
            cw_regression.tvalues
        ).reshape(-1)[0]
    )

    cw_p_value = float(
        1 - stats.t.cdf(
            cw_statistic,
            df=cw_regression.df_resid,
        )
    )

    cw_rows.append({
        "Model": model_name,
        "Clark_West_Statistic":
            cw_statistic,
        "Clark_West_One_Sided_p":
            cw_p_value,
    })

usd_clark_west = pd.DataFrame(
    cw_rows
)

usd_clark_west[
    "Holm_Adjusted_p"
] = multipletests(
    usd_clark_west[
        "Clark_West_One_Sided_p"
    ],
    method="holm",
)[1]

usd_clark_west[
    "BH_FDR_Adjusted_p"
] = multipletests(
    usd_clark_west[
        "Clark_West_One_Sided_p"
    ],
    method="fdr_bh",
)[1]

usd_clark_west[
    "Holm_Reject_5pct"
] = (
    usd_clark_west[
        "Holm_Adjusted_p"
    ] < 0.05
)

usd_clark_west[
    "BH_FDR_Reject_5pct"
] = (
    usd_clark_west[
        "BH_FDR_Adjusted_p"
    ] < 0.05
)

usd_forecast_metrics = (
    usd_forecast_metrics
    .merge(
        usd_clark_west,
        on="Model",
        how="left",
    )
)

display(
    usd_forecast_metrics.round(4)
)

# -----------------------------------------------------------
# Recursive coefficient stability
# -----------------------------------------------------------

coefficient_stability_rows = []

for model_name in [
    "U1_Direction",
    "U2_Direction_Surprise",
    "U3_Complete_Fed",
]:

    direction_column = (
        f"{model_name}_Direction_Coefficient"
    )

    direction_values = (
        usd_recursive_forecasts[
            direction_column
        ]
    )

    stability_row = {
        "Model": model_name,
        "Minimum_Direction_Coefficient":
            direction_values.min(),
        "Maximum_Direction_Coefficient":
            direction_values.max(),
        "Median_Direction_Coefficient":
            direction_values.median(),
        "Positive_Direction_Share":
            (direction_values > 0).mean(),
    }

    surprise_column = (
        f"{model_name}_Surprise_Coefficient"
    )

    if surprise_column in (
        usd_recursive_forecasts.columns
    ):

        surprise_values = (
            usd_recursive_forecasts[
                surprise_column
            ]
        )

        stability_row.update({
            "Minimum_Surprise_Coefficient":
                surprise_values.min(),
            "Maximum_Surprise_Coefficient":
                surprise_values.max(),
            "Median_Surprise_Coefficient":
                surprise_values.median(),
            "Negative_Surprise_Share":
                (surprise_values < 0).mean(),
        })

    coefficient_stability_rows.append(
        stability_row
    )

usd_recursive_coefficient_stability = (
    pd.DataFrame(
        coefficient_stability_rows
    )
)

display(
    usd_recursive_coefficient_stability.round(4)
)

# -----------------------------------------------------------
# Plot RMSE improvement
# -----------------------------------------------------------

usd_plot = (
    usd_forecast_metrics.loc[
        usd_forecast_metrics["Model"]
        != "U0_Persistence"
    ]
    .copy()
)

usd_plot["Forecast_Evidence"] = np.where(
    usd_plot["Holm_Reject_5pct"].fillna(False),
    "Holm significant",
    np.where(
        usd_plot[
            "BH_FDR_Reject_5pct"
        ].fillna(False),
        "FDR significant",
        "Not significant",
    ),
)

fig = px.bar(
    usd_plot,
    x="Model",
    y="RMSE_Improvement_vs_U0_pct",
    color="Forecast_Evidence",
    text="RMSE_Improvement_vs_U0_pct",
    hover_data={
        "MAE_Improvement_vs_U0_pct": ":.3f",
        "Direction_Accuracy": ":.3f",
        "Clark_West_One_Sided_p": ":.4f",
        "Holm_Adjusted_p": ":.4f",
    },
    color_discrete_map={
        "Holm significant": "#2ca02c",
        "FDR significant": "#ffbf00",
        "Not significant": "#d62728",
    },
    title=(
        "Out-of-Sample Value of FOMC Information "
        "for Next-Month Broad USD Changes"
    ),
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=620,
    xaxis_title="Forward USD model",
    yaxis_title="RMSE improvement over USD persistence (%)",
    legend_title="Clark–West evidence",
)

fig.show()

Pseudo-out-of-sample USD forecasts: 30
First outcome month: 2023-01-01 00:00:00
Final outcome month: 2026-07-01 00:00:00


,Model,Forecasts,RMSE,MAE,Bias,Forecast_Actual_Correlation,Direction_Accuracy,RMSE_Improvement_vs_U0_pct,MAE_Improvement_vs_U0_pct,Clark_West_Statistic,Clark_West_One_Sided_p,Holm_Adjusted_p,BH_FDR_Adjusted_p,Holm_Reject_5pct,BH_FDR_Reject_5pct
0,U0_Persistence,30,1.1564,0.9768,0.0333,0.3584,0.6000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN
1,U1_Direction,30,1.1691,0.9821,0.0100,0.3363,0.6000,-1.0978,-0.5336,-1.4146,0.9161,0.9161,0.9161,False,False
2,U2_Direction_Surprise,30,1.1444,0.9171,-0.0085,0.3938,0.6333,1.0354,6.1167,0.7170,0.2396,0.7187,0.3780,False,False
3,U3_Complete_Fed,30,1.1691,0.9563,-0.1607,0.3732,0.6000,-1.1055,2.1052,0.6765,0.2520,0.7187,0.3780,False,False


,Model,Minimum_Direction_Coefficient,Maximum_Direction_Coefficient,Median_Direction_Coefficient,Positive_Direction_Share,Minimum_Surprise_Coefficient,Maximum_Surprise_Coefficient,Median_Surprise_Coefficient,Negative_Surprise_Share
0,U1_Direction,-0.1883,0.239,-0.0451,0.2000,NaN,NaN,NaN,NaN
1,U2_Direction_Surprise,-0.0946,0.234,0.0254,0.5667,-0.9231,-0.3231,-0.7498,1.0
2,U3_Complete_Fed,0.0061,0.420,0.1457,1.0000,-0.8437,-0.2517,-0.6827,1.0


/tmp/ipykernel_1468/1989431164.py:448: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/tmp/ipykernel_1468/1989431164.py:453: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [48]:
# CELL 42: FOMC information and meeting-month broad USD changes

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------------------------------------
# Construct the calendar lag before filtering meeting months
# -----------------------------------------------------------

usd_contemporaneous_monthly = (
    usd_monthly
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

usd_contemporaneous_monthly[
    "Broad_USD_Change_L1_Calendar"
] = (
    usd_contemporaneous_monthly[
        "Broad_USD_Change"
    ].shift(1)
)

usd_contemporaneous_events = (
    usd_contemporaneous_monthly.loc[
        usd_contemporaneous_monthly[
            "Fed_Meeting_Flag"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)

usd_contemporaneous_specifications = {
    "C0_USD_Persistence": [
        "Broad_USD_Change_L1_Calendar",
    ],
    "C1_Direction": [
        "Broad_USD_Change_L1_Calendar",
        "Fed_Expected_Direction",
    ],
    "C2_Direction_Surprise": [
        "Broad_USD_Change_L1_Calendar",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
    ],
    "C3_Complete_Fed": [
        "Broad_USD_Change_L1_Calendar",
        "Fed_Expected_Direction",
        "Fed_Surprise_Signed",
        "Fed_Policy_Uncertainty",
    ],
}

all_contemporaneous_usd_regressors = list(
    dict.fromkeys(
        variable
        for regressors
        in usd_contemporaneous_specifications.values()
        for variable in regressors
    )
)

usd_contemporaneous_common = (
    usd_contemporaneous_events[
        [
            "Month",
            "Broad_USD_Change",
        ] + all_contemporaneous_usd_regressors
    ]
    .dropna()
    .sort_values("Month")
    .reset_index(drop=True)
)

print(
    "Contemporaneous USD observations:",
    len(usd_contemporaneous_common)
)

print(
    "Period:",
    usd_contemporaneous_common["Month"].min(),
    "to",
    usd_contemporaneous_common["Month"].max()
)

print(
    "Excluded meeting months:",
    usd_contemporaneous_events.loc[
        ~usd_contemporaneous_events["Month"].isin(
            usd_contemporaneous_common["Month"]
        ),
        "Month"
    ].dt.strftime("%Y-%m").tolist()
)

# -----------------------------------------------------------
# Estimate nested models
# -----------------------------------------------------------

usd_contemporaneous_results = {}
usd_contemporaneous_rows = []
usd_contemporaneous_coefficient_rows = []

y_current_usd = (
    usd_contemporaneous_common[
        "Broad_USD_Change"
    ].astype(float)
)

for model_name, regressors in (
    usd_contemporaneous_specifications.items()
):

    X_current_usd = sm.add_constant(
        usd_contemporaneous_common[
            regressors
        ].astype(float),
        has_constant="add",
    )

    current_usd_ols = sm.OLS(
        y_current_usd,
        X_current_usd,
    ).fit()

    current_usd_hac = (
        current_usd_ols.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    parameter_names = (
        X_current_usd.columns.tolist()
    )

    current_usd_hac.model.data.xnames = (
        parameter_names
    )

    parameters = np.asarray(
        current_usd_hac.params
    )

    standard_errors = np.asarray(
        current_usd_hac.bse
    )

    pvalues = np.asarray(
        current_usd_hac.pvalues
    )

    for position, variable in enumerate(
        parameter_names
    ):

        usd_contemporaneous_coefficient_rows.append({
            "Model": model_name,
            "Variable": variable,
            "Coefficient":
                parameters[position],
            "HAC_SE":
                standard_errors[position],
            "HAC_p":
                pvalues[position],
        })

    included_fed_variables = [
        variable
        for variable in [
            "Fed_Expected_Direction",
            "Fed_Surprise_Signed",
            "Fed_Policy_Uncertainty",
        ]
        if variable in regressors
    ]

    if included_fed_variables:

        fed_block_test = (
            current_usd_hac.wald_test(
                [
                    f"{variable} = 0"
                    for variable
                    in included_fed_variables
                ],
                use_f=False,
                scalar=True,
            )
        )

        fed_block_statistic = float(
            np.asarray(
                fed_block_test.statistic
            ).reshape(-1)[0]
        )

        fed_block_p = float(
            np.asarray(
                fed_block_test.pvalue
            ).reshape(-1)[0]
        )

    else:
        fed_block_statistic = np.nan
        fed_block_p = np.nan

    usd_contemporaneous_rows.append({
        "Model": model_name,
        "Fed_Variables":
            len(included_fed_variables),
        "Parameters": int(
            current_usd_ols.df_model + 1
        ),
        "N": int(current_usd_ols.nobs),
        "R2":
            current_usd_ols.rsquared,
        "Adjusted_R2":
            current_usd_ols.rsquared_adj,
        "AIC":
            current_usd_ols.aic,
        "BIC":
            current_usd_ols.bic,
        "Fed_Block_Wald":
            fed_block_statistic,
        "Fed_Block_p":
            fed_block_p,
    })

    usd_contemporaneous_results[
        model_name
    ] = {
        "OLS": current_usd_ols,
        "HAC": current_usd_hac,
        "Regressors": regressors,
    }

usd_contemporaneous_comparison = pd.DataFrame(
    usd_contemporaneous_rows
)

usd_contemporaneous_coefficients = pd.DataFrame(
    usd_contemporaneous_coefficient_rows
)

usd_contemporaneous_comparison[
    "Delta_Adjusted_R2"
] = (
    usd_contemporaneous_comparison[
        "Adjusted_R2"
    ].diff()
)

usd_contemporaneous_comparison[
    "Delta_AIC"
] = (
    usd_contemporaneous_comparison["AIC"].diff()
)

usd_contemporaneous_comparison[
    "Delta_BIC"
] = (
    usd_contemporaneous_comparison["BIC"].diff()
)

display(
    usd_contemporaneous_comparison.round(4)
)

# -----------------------------------------------------------
# Coefficient comparison
# -----------------------------------------------------------

current_usd_focus_variables = [
    "Broad_USD_Change_L1_Calendar",
    "Fed_Expected_Direction",
    "Fed_Surprise_Signed",
    "Fed_Policy_Uncertainty",
]

current_usd_coefficient_comparison = (
    usd_contemporaneous_coefficients.loc[
        usd_contemporaneous_coefficients[
            "Variable"
        ].isin(current_usd_focus_variables)
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="Coefficient",
    )
    .reindex(
        usd_contemporaneous_specifications.keys()
    )
)

current_usd_pvalue_comparison = (
    usd_contemporaneous_coefficients.loc[
        usd_contemporaneous_coefficients[
            "Variable"
        ].isin(current_usd_focus_variables)
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="HAC_p",
    )
    .reindex(
        usd_contemporaneous_specifications.keys()
    )
)

print("\nMeeting-month USD coefficients")
display(
    current_usd_coefficient_comparison.round(4)
)

print("\nMeeting-month USD HAC p-values")
display(
    current_usd_pvalue_comparison.round(4)
)

# -----------------------------------------------------------
# Plot information criteria
# -----------------------------------------------------------

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Adjusted R²",
        "AIC and BIC",
    ],
)

fig.add_trace(
    go.Bar(
        x=usd_contemporaneous_comparison["Model"],
        y=usd_contemporaneous_comparison["Adjusted_R2"],
        name="Adjusted R²",
        marker_color="#2a6fbb",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=usd_contemporaneous_comparison["Model"],
        y=usd_contemporaneous_comparison["AIC"],
        mode="lines+markers",
        name="AIC",
        line=dict(color="#d62728"),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=usd_contemporaneous_comparison["Model"],
        y=usd_contemporaneous_comparison["BIC"],
        mode="lines+markers",
        name="BIC",
        line=dict(color="#2ca02c"),
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title=(
        "FOMC Information and "
        "Meeting-Month Broad USD Changes"
    ),
    template="plotly_white",
    width=1350,
    height=620,
)

fig.update_xaxes(tickangle=30)
fig.show()

Contemporaneous USD observations: 61
Period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00
Excluded meeting months: ['2019-01']


,Model,Fed_Variables,Parameters,N,R2,Adjusted_R2,AIC,BIC,Fed_Block_Wald,Fed_Block_p,Delta_Adjusted_R2,Delta_AIC,Delta_BIC
0,C0_USD_Persistence,0,2,61,0.1148,0.0998,219.4180,223.6398,NaN,NaN,NaN,NaN,NaN
1,C1_Direction,1,3,61,0.1152,0.0846,221.3934,227.7260,0.0141,0.9054,-0.0152,1.9753,4.0862
2,C2_Direction_Surprise,2,4,61,0.1165,0.0700,223.2978,231.7413,0.2071,0.9016,-0.0146,1.9044,4.0153
3,C3_Complete_Fed,3,5,61,0.1356,0.0739,223.9666,234.5209,1.0809,0.7817,0.0038,0.6688,2.7797



Meeting-month USD coefficients


Variable,Broad_USD_Change_L1_Calendar,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed
Model,,,,
C0_USD_Persistence,0.3438,NaN,NaN,NaN
C1_Direction,0.3467,0.0736,NaN,NaN
C2_Direction_Surprise,0.3497,0.0577,NaN,0.1441
C3_Complete_Fed,0.3667,-0.0842,-1.0343,0.1065



Meeting-month USD HAC p-values


Variable,Broad_USD_Change_L1_Calendar,Fed_Expected_Direction,Fed_Policy_Uncertainty,Fed_Surprise_Signed
Model,,,,
C0_USD_Persistence,0.0055,NaN,NaN,NaN
C1_Direction,0.0044,0.9058,NaN,NaN
C2_Direction_Surprise,0.0037,0.9263,NaN,0.6573
C3_Complete_Fed,0.0047,0.8944,0.3458,0.7412


In [49]:
# CELL 43: Meeting-based Fed–ETF–gold mediation analysis

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------------------------------------
# Construct forward gold and control variables on the full
# regular monthly dataset before selecting meeting months
# -----------------------------------------------------------

mediation_monthly = (
    etf_monthly
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

mediation_monthly[
    "Gold_Return_Next_Month"
] = (
    mediation_monthly[
        "Gold_Return"
    ].shift(-1)
)

mediation_monthly[
    "CB_IMF_Change_Next_Month"
] = (
    mediation_monthly[
        "CB_IMF_Change"
    ].shift(-1)
)

mediation_monthly[
    "Real10Y_Change_Next_Month"
] = (
    mediation_monthly[
        "Real10Y_Change_pp"
    ].shift(-1)
)

mediation_monthly[
    "VIX_Change_Next_Month"
] = (
    mediation_monthly[
        "VIX_Change"
    ].shift(-1)
)

mediation_monthly[
    "Broad_USD_Change_Next_Month"
] = (
    mediation_monthly[
        "Broad_USD_Change"
    ].shift(-1)
)

# ETF_Flow_Next_Month was already constructed earlier.
if (
    "ETF_Flow_Next_Month"
    not in mediation_monthly.columns
):
    mediation_monthly[
        "ETF_Flow_Next_Month"
    ] = (
        mediation_monthly[
            "ETF_Flow"
        ].shift(-1)
    )

# -----------------------------------------------------------
# Retain genuine meeting months
# -----------------------------------------------------------

mediation_events = (
    mediation_monthly.loc[
        mediation_monthly[
            "Fed_Meeting_Flag"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)

# -----------------------------------------------------------
# Define nested gold-return equations
# -----------------------------------------------------------

gold_mediation_specifications = {
    # Gold-return persistence only
    "G0_Gold_Persistence": [
        "Gold_Return",
    ],

    # Total predictive relationship from Fed expectations
    "G1_Fed_Direction": [
        "Gold_Return",
        "Fed_Expected_Direction",
    ],

    # ETF association without the Fed-direction variable
    "G2_ETF_Channel": [
        "Gold_Return",
        "ETF_Flow_Next_Month",
    ],

    # Basic mediation/path equation
    "G3_Fed_and_ETF": [
        "Gold_Return",
        "Fed_Expected_Direction",
        "ETF_Flow_Next_Month",
    ],

    # Add central-bank demand and outcome-month financial controls
    "G4_Controlled_Mediation": [
        "Gold_Return",
        "Fed_Expected_Direction",
        "ETF_Flow_Next_Month",
        "CB_IMF_Change_Next_Month",
        "Broad_USD_Change_Next_Month",
        "Real10Y_Change_Next_Month",
        "VIX_Change_Next_Month",
    ],
}

all_mediation_regressors = list(dict.fromkeys(
    variable
    for regressors
    in gold_mediation_specifications.values()
    for variable in regressors
))

# One common sample for valid model comparison.
gold_mediation_common = (
    mediation_events[
        [
            "Month",
            "Gold_Return_Next_Month",
        ] + all_mediation_regressors
    ]
    .dropna()
    .sort_values("Month")
    .reset_index(drop=True)
)

print(
    "Mediation observations:",
    len(gold_mediation_common)
)

print(
    "FOMC-information period:",
    gold_mediation_common["Month"].min(),
    "to",
    gold_mediation_common["Month"].max()
)

print(
    "Gold-return outcome period:",
    (
        gold_mediation_common["Month"].min()
        + pd.offsets.MonthBegin(1)
    ),
    "to",
    (
        gold_mediation_common["Month"].max()
        + pd.offsets.MonthBegin(1)
    )
)

print(
    "Excluded meeting months:",
    mediation_events.loc[
        ~mediation_events["Month"].isin(
            gold_mediation_common["Month"]
        ),
        "Month"
    ].dt.strftime("%Y-%m").tolist()
)

# -----------------------------------------------------------
# Estimate the gold-return equations
# -----------------------------------------------------------

gold_mediation_results = {}
gold_mediation_rows = []
gold_mediation_coefficient_rows = []

y_gold_forward = (
    gold_mediation_common[
        "Gold_Return_Next_Month"
    ].astype(float)
)

for model_name, regressors in (
    gold_mediation_specifications.items()
):

    X_gold = sm.add_constant(
        gold_mediation_common[
            regressors
        ].astype(float),
        has_constant="add",
    )

    gold_ols = sm.OLS(
        y_gold_forward,
        X_gold,
    ).fit()

    gold_hac = (
        gold_ols.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    parameter_names = X_gold.columns.tolist()
    gold_hac.model.data.xnames = parameter_names

    parameters = np.asarray(
        gold_hac.params
    )

    standard_errors = np.asarray(
        gold_hac.bse
    )

    pvalues = np.asarray(
        gold_hac.pvalues
    )

    for position, variable in enumerate(
        parameter_names
    ):

        gold_mediation_coefficient_rows.append({
            "Model": model_name,
            "Variable": variable,
            "Coefficient":
                parameters[position],
            "HAC_SE":
                standard_errors[position],
            "HAC_p":
                pvalues[position],
        })

    gold_mediation_rows.append({
        "Model": model_name,
        "Parameters": int(
            gold_ols.df_model + 1
        ),
        "N": int(gold_ols.nobs),
        "R2":
            gold_ols.rsquared,
        "Adjusted_R2":
            gold_ols.rsquared_adj,
        "AIC":
            gold_ols.aic,
        "BIC":
            gold_ols.bic,
    })

    gold_mediation_results[
        model_name
    ] = {
        "OLS": gold_ols,
        "HAC": gold_hac,
        "Regressors": regressors,
    }

gold_mediation_comparison = pd.DataFrame(
    gold_mediation_rows
)

gold_mediation_coefficients = pd.DataFrame(
    gold_mediation_coefficient_rows
)

gold_mediation_comparison[
    "Delta_Adjusted_R2"
] = (
    gold_mediation_comparison[
        "Adjusted_R2"
    ].diff()
)

gold_mediation_comparison[
    "Delta_AIC"
] = (
    gold_mediation_comparison[
        "AIC"
    ].diff()
)

gold_mediation_comparison[
    "Delta_BIC"
] = (
    gold_mediation_comparison[
        "BIC"
    ].diff()
)

display(
    gold_mediation_comparison.round(4)
)

# -----------------------------------------------------------
# Focused coefficient comparison
# -----------------------------------------------------------

mediation_focus_variables = [
    "Gold_Return",
    "Fed_Expected_Direction",
    "ETF_Flow_Next_Month",
    "CB_IMF_Change_Next_Month",
    "Broad_USD_Change_Next_Month",
    "Real10Y_Change_Next_Month",
    "VIX_Change_Next_Month",
]

mediation_coefficient_comparison = (
    gold_mediation_coefficients.loc[
        gold_mediation_coefficients[
            "Variable"
        ].isin(mediation_focus_variables)
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="Coefficient",
    )
    .reindex(
        gold_mediation_specifications.keys()
    )
)

mediation_pvalue_comparison = (
    gold_mediation_coefficients.loc[
        gold_mediation_coefficients[
            "Variable"
        ].isin(mediation_focus_variables)
    ]
    .pivot(
        index="Model",
        columns="Variable",
        values="HAC_p",
    )
    .reindex(
        gold_mediation_specifications.keys()
    )
)

print("\nGold mediation/path coefficients")
display(
    mediation_coefficient_comparison.round(4)
)

print("\nGold mediation/path HAC p-values")
display(
    mediation_pvalue_comparison.round(4)
)

# -----------------------------------------------------------
# Preliminary path-effect calculation
# -----------------------------------------------------------

# Path a:
# Fed direction_t -> ETF flow_(t+1)
path_a = global_direction_coefficient

# Path b from the basic mediation equation.
g3_hac = gold_mediation_results[
    "G3_Fed_and_ETF"
]["HAC"]

g3_names = gold_mediation_results[
    "G3_Fed_and_ETF"
]["OLS"].model.exog_names

path_b_position = g3_names.index(
    "ETF_Flow_Next_Month"
)

direct_effect_position = g3_names.index(
    "Fed_Expected_Direction"
)

path_b = float(
    np.asarray(
        g3_hac.params
    )[path_b_position]
)

direct_effect = float(
    np.asarray(
        g3_hac.params
    )[direct_effect_position]
)

indirect_effect = path_a * path_b
total_decomposed_effect = (
    direct_effect + indirect_effect
)

preliminary_path_decomposition = pd.DataFrame({
    "Path": [
        "a: Fed direction → next-month ETF flow",
        "b: next-month ETF flow → next-month gold return",
        "Indirect effect: a × b",
        "Direct Fed effect conditional on ETF",
        "Decomposed total effect",
    ],
    "Estimate": [
        path_a,
        path_b,
        indirect_effect,
        direct_effect,
        total_decomposed_effect,
    ],
})

display(
    preliminary_path_decomposition.round(4)
)

# -----------------------------------------------------------
# Plot fit comparison
# -----------------------------------------------------------

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Adjusted R²",
        "AIC and BIC",
    ],
)

fig.add_trace(
    go.Bar(
        x=gold_mediation_comparison["Model"],
        y=gold_mediation_comparison["Adjusted_R2"],
        name="Adjusted R²",
        marker_color="#2a6fbb",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=gold_mediation_comparison["Model"],
        y=gold_mediation_comparison["AIC"],
        mode="lines+markers",
        name="AIC",
        line=dict(color="#d62728"),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=gold_mediation_comparison["Model"],
        y=gold_mediation_comparison["BIC"],
        mode="lines+markers",
        name="BIC",
        line=dict(color="#2ca02c"),
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title=(
        "Meeting-Based Fed–ETF–Gold "
        "Path Analysis"
    ),
    template="plotly_white",
    width=1400,
    height=650,
)

fig.update_xaxes(tickangle=30)
fig.show()

Mediation observations: 61
FOMC-information period: 2019-01-01 00:00:00 to 2026-06-01 00:00:00
Gold-return outcome period: 2019-02-01 00:00:00 to 2026-07-01 00:00:00
Excluded meeting months: ['2026-07']


,Model,Parameters,N,R2,Adjusted_R2,AIC,BIC,Delta_Adjusted_R2,Delta_AIC,Delta_BIC
0,G0_Gold_Persistence,2,61,0.2201,0.2069,315.7699,319.9917,NaN,NaN,NaN
1,G1_Fed_Direction,3,61,0.2609,0.2354,314.4969,320.8295,0.0285,-1.2730,0.8379
2,G2_ETF_Channel,3,61,0.4677,0.4493,294.4747,300.8073,0.2139,-20.0222,-20.0222
3,G3_Fed_and_ETF,4,61,0.4824,0.4551,294.7648,303.2083,0.0058,0.2901,2.4010
4,G4_Controlled_Mediation,8,61,0.5202,0.4569,298.1311,315.0181,0.0017,3.3663,11.8098



Gold mediation/path coefficients


Variable,Broad_USD_Change_Next_Month,CB_IMF_Change_Next_Month,ETF_Flow_Next_Month,Fed_Expected_Direction,Gold_Return,Real10Y_Change_Next_Month,VIX_Change_Next_Month
Model,,,,,,,
G0_Gold_Persistence,NaN,NaN,NaN,NaN,0.4164,NaN,NaN
G1_Fed_Direction,NaN,NaN,NaN,-1.8971,0.3623,NaN,NaN
G2_ETF_Channel,NaN,NaN,0.0308,NaN,0.2786,NaN,NaN
G3_Fed_and_ETF,NaN,NaN,0.0364,1.4259,0.2940,NaN,NaN
G4_Controlled_Mediation,-0.5442,0.0032,0.0347,1.1108,0.2750,0.598,0.0162



Gold mediation/path HAC p-values


Variable,Broad_USD_Change_Next_Month,CB_IMF_Change_Next_Month,ETF_Flow_Next_Month,Fed_Expected_Direction,Gold_Return,Real10Y_Change_Next_Month,VIX_Change_Next_Month
Model,,,,,,,
G0_Gold_Persistence,NaN,NaN,NaN,NaN,0.0000,NaN,NaN
G1_Fed_Direction,NaN,NaN,NaN,0.0631,0.0000,NaN,NaN
G2_ETF_Channel,NaN,NaN,0.0,NaN,0.0019,NaN,NaN
G3_Fed_and_ETF,NaN,NaN,0.0,0.0929,0.0015,NaN,NaN
G4_Controlled_Mediation,0.0172,0.7108,0.0,0.1815,0.0073,0.7305,0.7817


,Path,Estimate
0,a: Fed direction → next-month ETF flow,-68.0747
1,b: next-month ETF flow → next-month gold return,0.0364
2,Indirect effect: a × b,-2.4768
3,Direct Fed effect conditional on ETF,1.4259
4,Decomposed total effect,-1.0508


In [50]:
# CELL 44: Aligned Fed → ETF → gold temporal mediation

import statsmodels.api as sm
import pandas as pd
import numpy as np

# -----------------------------------------------------------
# Construct strictly pre-meeting controls
# -----------------------------------------------------------

aligned_monthly = (
    mediation_monthly
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

aligned_monthly[
    "Gold_Return_Previous_Month"
] = (
    aligned_monthly["Gold_Return"].shift(1)
)

aligned_monthly[
    "ETF_Flow_Previous_Month"
] = (
    aligned_monthly["ETF_Flow"].shift(1)
)

aligned_events = (
    aligned_monthly.loc[
        aligned_monthly[
            "Fed_Meeting_Flag"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)

# Same observations and same baseline controls in every equation.
aligned_variables = [
    "Month",
    "Fed_Expected_Direction",
    "ETF_Flow_Next_Month",
    "Gold_Return_Next_Month",
    "Gold_Return_Previous_Month",
    "ETF_Flow_Previous_Month",
]

aligned_mediation_df = (
    aligned_events[
        aligned_variables
    ]
    .dropna()
    .sort_values("Month")
    .reset_index(drop=True)
)

print(
    "Aligned mediation observations:",
    len(aligned_mediation_df)
)

print(
    "Treatment period:",
    aligned_mediation_df["Month"].min(),
    "to",
    aligned_mediation_df["Month"].max()
)

print(
    "Outcome period:",
    (
        aligned_mediation_df["Month"].min()
        + pd.offsets.MonthBegin(1)
    ),
    "to",
    (
        aligned_mediation_df["Month"].max()
        + pd.offsets.MonthBegin(1)
    )
)

baseline_controls = [
    "Gold_Return_Previous_Month",
    "ETF_Flow_Previous_Month",
]

# -----------------------------------------------------------
# Equation M:
# Fed direction_t → ETF flow_(t+1)
# -----------------------------------------------------------

mediator_regressors = [
    "Fed_Expected_Direction",
] + baseline_controls

X_mediator = sm.add_constant(
    aligned_mediation_df[
        mediator_regressors
    ].astype(float),
    has_constant="add",
)

y_mediator = aligned_mediation_df[
    "ETF_Flow_Next_Month"
].astype(float)

mediator_ols = sm.OLS(
    y_mediator,
    X_mediator,
).fit()

mediator_hac = (
    mediator_ols.get_robustcov_results(
        cov_type="HAC",
        maxlags=2,
        use_correction=True,
    )
)

mediator_names = X_mediator.columns.tolist()
mediator_hac.model.data.xnames = mediator_names

# -----------------------------------------------------------
# Equation Y-total:
# Total Fed-direction effect on next-month gold return
# -----------------------------------------------------------

total_regressors = [
    "Fed_Expected_Direction",
] + baseline_controls

X_total = sm.add_constant(
    aligned_mediation_df[
        total_regressors
    ].astype(float),
    has_constant="add",
)

y_gold = aligned_mediation_df[
    "Gold_Return_Next_Month"
].astype(float)

total_ols = sm.OLS(
    y_gold,
    X_total,
).fit()

total_hac = (
    total_ols.get_robustcov_results(
        cov_type="HAC",
        maxlags=2,
        use_correction=True,
    )
)

total_names = X_total.columns.tolist()
total_hac.model.data.xnames = total_names

# -----------------------------------------------------------
# Equation Y-direct:
# ETF mediator plus Fed direction and identical controls
# -----------------------------------------------------------

direct_regressors = [
    "Fed_Expected_Direction",
    "ETF_Flow_Next_Month",
] + baseline_controls

X_direct = sm.add_constant(
    aligned_mediation_df[
        direct_regressors
    ].astype(float),
    has_constant="add",
)

direct_ols = sm.OLS(
    y_gold,
    X_direct,
).fit()

direct_hac = (
    direct_ols.get_robustcov_results(
        cov_type="HAC",
        maxlags=2,
        use_correction=True,
    )
)

direct_names = X_direct.columns.tolist()
direct_hac.model.data.xnames = direct_names

# -----------------------------------------------------------
# Extract aligned path coefficients
# -----------------------------------------------------------

def named_result(
    robust_result,
    names,
    variable,
):

    position = names.index(variable)

    parameters = np.asarray(
        robust_result.params
    )

    standard_errors = np.asarray(
        robust_result.bse
    )

    pvalues = np.asarray(
        robust_result.pvalues
    )

    return {
        "Coefficient":
            float(parameters[position]),
        "Robust_SE":
            float(standard_errors[position]),
        "Robust_p":
            float(pvalues[position]),
    }

path_a_result = named_result(
    mediator_hac,
    mediator_names,
    "Fed_Expected_Direction",
)

total_effect_result = named_result(
    total_hac,
    total_names,
    "Fed_Expected_Direction",
)

path_b_result = named_result(
    direct_hac,
    direct_names,
    "ETF_Flow_Next_Month",
)

direct_effect_result = named_result(
    direct_hac,
    direct_names,
    "Fed_Expected_Direction",
)

path_a = path_a_result["Coefficient"]
path_b = path_b_result["Coefficient"]
total_effect = total_effect_result["Coefficient"]
direct_effect = direct_effect_result["Coefficient"]

indirect_effect = path_a * path_b
decomposed_total = (
    direct_effect + indirect_effect
)

decomposition_error = (
    total_effect - decomposed_total
)

aligned_path_results = pd.DataFrame({
    "Effect": [
        "Path a: Fed direction → ETF flow",
        "Path b: ETF flow → gold return",
        "Indirect effect: a × b",
        "Direct effect: c-prime",
        "Total effect: c",
        "Direct plus indirect",
        "Decomposition error",
    ],
    "Estimate": [
        path_a,
        path_b,
        indirect_effect,
        direct_effect,
        total_effect,
        decomposed_total,
        decomposition_error,
    ],
    "HAC_SE": [
        path_a_result["Robust_SE"],
        path_b_result["Robust_SE"],
        np.nan,
        direct_effect_result["Robust_SE"],
        total_effect_result["Robust_SE"],
        np.nan,
        np.nan,
    ],
    "HAC_p": [
        path_a_result["Robust_p"],
        path_b_result["Robust_p"],
        np.nan,
        direct_effect_result["Robust_p"],
        total_effect_result["Robust_p"],
        np.nan,
        np.nan,
    ],
})

display(
    aligned_path_results.round(6)
)

print(
    "Absolute decomposition error:",
    abs(decomposition_error)
)

# -----------------------------------------------------------
# Moving-block bootstrap for the indirect effect
# -----------------------------------------------------------

bootstrap_replications = 3000
block_length = 4
random_generator = np.random.default_rng(1857)

number_observations = len(
    aligned_mediation_df
)

bootstrap_results = []

def circular_block_indices(
    sample_size,
    block_size,
    rng,
):

    required_blocks = int(
        np.ceil(sample_size / block_size)
    )

    starting_points = rng.integers(
        0,
        sample_size,
        size=required_blocks,
    )

    indices = []

    for start in starting_points:

        block = [
            (start + offset) % sample_size
            for offset in range(block_size)
        ]

        indices.extend(block)

    return np.asarray(
        indices[:sample_size]
    )

for replication in range(
    bootstrap_replications
):

    bootstrap_indices = circular_block_indices(
        number_observations,
        block_length,
        random_generator,
    )

    bootstrap_data = (
        aligned_mediation_df.iloc[
            bootstrap_indices
        ]
        .reset_index(drop=True)
    )

    try:
        # Bootstrap mediator equation.
        X_m_boot = sm.add_constant(
            bootstrap_data[
                mediator_regressors
            ].astype(float),
            has_constant="add",
        )

        m_boot = sm.OLS(
            bootstrap_data[
                "ETF_Flow_Next_Month"
            ].astype(float),
            X_m_boot,
        ).fit()

        # Bootstrap total-effect equation.
        X_t_boot = sm.add_constant(
            bootstrap_data[
                total_regressors
            ].astype(float),
            has_constant="add",
        )

        t_boot = sm.OLS(
            bootstrap_data[
                "Gold_Return_Next_Month"
            ].astype(float),
            X_t_boot,
        ).fit()

        # Bootstrap direct/mediated equation.
        X_d_boot = sm.add_constant(
            bootstrap_data[
                direct_regressors
            ].astype(float),
            has_constant="add",
        )

        d_boot = sm.OLS(
            bootstrap_data[
                "Gold_Return_Next_Month"
            ].astype(float),
            X_d_boot,
        ).fit()

        a_boot = m_boot.params[
            "Fed_Expected_Direction"
        ]

        b_boot = d_boot.params[
            "ETF_Flow_Next_Month"
        ]

        direct_boot = d_boot.params[
            "Fed_Expected_Direction"
        ]

        total_boot = t_boot.params[
            "Fed_Expected_Direction"
        ]

        bootstrap_results.append({
            "Path_a": a_boot,
            "Path_b": b_boot,
            "Indirect_Effect":
                a_boot * b_boot,
            "Direct_Effect":
                direct_boot,
            "Total_Effect":
                total_boot,
        })

    except Exception:
        continue

mediation_bootstrap = pd.DataFrame(
    bootstrap_results
)

print(
    "Successful bootstrap replications:",
    len(mediation_bootstrap)
)

# -----------------------------------------------------------
# Percentile confidence intervals
# -----------------------------------------------------------

bootstrap_summary_rows = []

for variable, point_estimate in {
    "Path_a": path_a,
    "Path_b": path_b,
    "Indirect_Effect": indirect_effect,
    "Direct_Effect": direct_effect,
    "Total_Effect": total_effect,
}.items():

    bootstrap_values = (
        mediation_bootstrap[variable]
        .dropna()
    )

    lower_95 = bootstrap_values.quantile(
        0.025
    )

    upper_95 = bootstrap_values.quantile(
        0.975
    )

    # Two-sided bootstrap sign-based p-value.
    probability_nonpositive = (
        bootstrap_values <= 0
    ).mean()

    probability_nonnegative = (
        bootstrap_values >= 0
    ).mean()

    bootstrap_p = min(
        1.0,
        2 * min(
            probability_nonpositive,
            probability_nonnegative,
        ),
    )

    bootstrap_summary_rows.append({
        "Effect": variable,
        "Point_Estimate":
            point_estimate,
        "Bootstrap_SE":
            bootstrap_values.std(ddof=1),
        "CI_Lower_95pct":
            lower_95,
        "CI_Upper_95pct":
            upper_95,
        "Bootstrap_p":
            bootstrap_p,
        "Significant_5pct":
            (
                lower_95 > 0
                or upper_95 < 0
            ),
    })

mediation_bootstrap_summary = pd.DataFrame(
    bootstrap_summary_rows
)

display(
    mediation_bootstrap_summary.round(4)
)

Aligned mediation observations: 60
Treatment period: 2019-03-01 00:00:00 to 2026-06-01 00:00:00
Outcome period: 2019-04-01 00:00:00 to 2026-07-01 00:00:00


,Effect,Estimate,HAC_SE,HAC_p
0,Path a: Fed direction → ETF flow,-73.469576,14.523321,0.000005
1,Path b: ETF flow → gold return,0.044039,0.007057,0.000000
2,Indirect effect: a × b,-3.235525,NaN,NaN
3,Direct effect: c-prime,0.780374,1.110040,0.485012
4,Total effect: c,-2.455151,1.263067,0.056951
5,Direct plus indirect,-2.455151,NaN,NaN
6,Decomposition error,-0.000000,NaN,NaN


Absolute decomposition error: 1.021405182655144e-14
Successful bootstrap replications: 3000


,Effect,Point_Estimate,Bootstrap_SE,CI_Lower_95pct,CI_Upper_95pct,Bootstrap_p,Significant_5pct
0,Path_a,-73.4696,16.7035,-107.7651,-41.8221,0.0007,True
1,Path_b,0.0440,0.0077,0.0282,0.0588,0.0007,True
2,Indirect_Effect,-3.2355,0.7788,-4.7598,-1.7146,0.0013,True
3,Direct_Effect,0.7804,1.2898,-2.3177,2.9214,0.5407,False
4,Total_Effect,-2.4552,1.3986,-5.4940,0.3253,0.0733,False


In [51]:
# CELL 45: Robustness of the temporal mediation pathway

import statsmodels.api as sm
import pandas as pd
import numpy as np

# -----------------------------------------------------------
# Construct additional strictly lagged controls
# -----------------------------------------------------------

robust_mediation_monthly = (
    mediation_monthly
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

lagged_control_mapping = {
    "Gold_Return":
        "Gold_Return_Previous_Month",
    "ETF_Flow":
        "ETF_Flow_Previous_Month",
    "CB_IMF_Change":
        "CB_IMF_Change_Previous_Month",
    "Broad_USD_Change":
        "Broad_USD_Change_Previous_Month",
    "Real10Y_Change_pp":
        "Real10Y_Change_Previous_Month",
    "VIX_Change":
        "VIX_Change_Previous_Month",
    "CPI_MoM":
        "CPI_MoM_Previous_Month",
    "HY_OAS_Change_pp":
        "HY_OAS_Change_Previous_Month",
    "Brent_Return_pct":
        "Brent_Return_Previous_Month",
}

for source, target in lagged_control_mapping.items():

    robust_mediation_monthly[target] = (
        robust_mediation_monthly[source].shift(1)
    )

robust_mediation_events = (
    robust_mediation_monthly.loc[
        robust_mediation_monthly[
            "Fed_Meeting_Flag"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)

# -----------------------------------------------------------
# Alternative pre-treatment control sets
# -----------------------------------------------------------

mediation_robustness_specifications = {
    "S0_Baseline": [
        "Gold_Return_Previous_Month",
        "ETF_Flow_Previous_Month",
    ],

    "S1_Core_Macro": [
        "Gold_Return_Previous_Month",
        "ETF_Flow_Previous_Month",
        "Broad_USD_Change_Previous_Month",
        "Real10Y_Change_Previous_Month",
        "VIX_Change_Previous_Month",
    ],

    "S2_Core_Macro_CB": [
        "Gold_Return_Previous_Month",
        "ETF_Flow_Previous_Month",
        "Broad_USD_Change_Previous_Month",
        "Real10Y_Change_Previous_Month",
        "VIX_Change_Previous_Month",
        "CB_IMF_Change_Previous_Month",
    ],

    "S3_Expanded_PreTreatment": [
        "Gold_Return_Previous_Month",
        "ETF_Flow_Previous_Month",
        "Broad_USD_Change_Previous_Month",
        "Real10Y_Change_Previous_Month",
        "VIX_Change_Previous_Month",
        "CB_IMF_Change_Previous_Month",
        "CPI_MoM_Previous_Month",
        "HY_OAS_Change_Previous_Month",
        "Brent_Return_Previous_Month",
    ],
}

# -----------------------------------------------------------
# Function for aligned mediation estimation
# -----------------------------------------------------------

def estimate_aligned_paths(
    data,
    controls,
    specification_name,
):

    required_variables = [
        "Month",
        "Fed_Expected_Direction",
        "ETF_Flow_Next_Month",
        "Gold_Return_Next_Month",
    ] + controls

    estimation_data = (
        data[required_variables]
        .dropna()
        .sort_values("Month")
        .reset_index(drop=True)
    )

    treatment = "Fed_Expected_Direction"
    mediator = "ETF_Flow_Next_Month"
    outcome = "Gold_Return_Next_Month"

    # Mediator equation
    mediator_regressors = [
        treatment
    ] + controls

    X_m = sm.add_constant(
        estimation_data[
            mediator_regressors
        ].astype(float),
        has_constant="add",
    )

    mediator_model = sm.OLS(
        estimation_data[mediator].astype(float),
        X_m,
    ).fit()

    mediator_hac = (
        mediator_model.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    mediator_names = X_m.columns.tolist()

    # Total-effect equation
    total_regressors = [
        treatment
    ] + controls

    X_total = sm.add_constant(
        estimation_data[
            total_regressors
        ].astype(float),
        has_constant="add",
    )

    total_model = sm.OLS(
        estimation_data[outcome].astype(float),
        X_total,
    ).fit()

    total_hac = (
        total_model.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    total_names = X_total.columns.tolist()

    # Direct/mediated equation
    direct_regressors = [
        treatment,
        mediator,
    ] + controls

    X_direct = sm.add_constant(
        estimation_data[
            direct_regressors
        ].astype(float),
        has_constant="add",
    )

    direct_model = sm.OLS(
        estimation_data[outcome].astype(float),
        X_direct,
    ).fit()

    direct_hac = (
        direct_model.get_robustcov_results(
            cov_type="HAC",
            maxlags=2,
            use_correction=True,
        )
    )

    direct_names = X_direct.columns.tolist()

    def extract(
        result,
        names,
        variable,
    ):

        position = names.index(variable)

        return (
            float(
                np.asarray(
                    result.params
                )[position]
            ),
            float(
                np.asarray(
                    result.pvalues
                )[position]
            ),
        )

    path_a, path_a_p = extract(
        mediator_hac,
        mediator_names,
        treatment,
    )

    path_b, path_b_p = extract(
        direct_hac,
        direct_names,
        mediator,
    )

    direct_effect, direct_p = extract(
        direct_hac,
        direct_names,
        treatment,
    )

    total_effect, total_p = extract(
        total_hac,
        total_names,
        treatment,
    )

    indirect_effect = path_a * path_b

    return {
        "Specification": specification_name,
        "N": len(estimation_data),
        "First_Month":
            estimation_data["Month"].min(),
        "Final_Month":
            estimation_data["Month"].max(),
        "Controls": len(controls),
        "Path_a": path_a,
        "Path_a_HAC_p": path_a_p,
        "Path_b": path_b,
        "Path_b_HAC_p": path_b_p,
        "Indirect_Effect":
            indirect_effect,
        "Indirect_Effect_for_0_10":
            indirect_effect * 0.10,
        "Direct_Effect":
            direct_effect,
        "Direct_HAC_p":
            direct_p,
        "Total_Effect":
            total_effect,
        "Total_HAC_p":
            total_p,
        "Direct_Plus_Indirect":
            direct_effect + indirect_effect,
        "Decomposition_Error":
            total_effect
            - (
                direct_effect
                + indirect_effect
            ),
        "Outcome_Adjusted_R2":
            direct_model.rsquared_adj,
        "Outcome_AIC":
            direct_model.aic,
        "Outcome_BIC":
            direct_model.bic,
        "Estimation_Data":
            estimation_data,
        "Mediator_Regressors":
            mediator_regressors,
        "Direct_Regressors":
            direct_regressors,
        "Total_Regressors":
            total_regressors,
    }

# -----------------------------------------------------------
# Full-period specifications
# -----------------------------------------------------------

mediation_robustness_results = []

for specification_name, controls in (
    mediation_robustness_specifications.items()
):

    result = estimate_aligned_paths(
        robust_mediation_events,
        controls,
        specification_name,
    )

    mediation_robustness_results.append(
        result
    )

# -----------------------------------------------------------
# Regime sensitivity using the core-macro specification
# -----------------------------------------------------------

core_controls = (
    mediation_robustness_specifications[
        "S1_Core_Macro"
    ]
)

# Remove only the exceptional March 2020 meeting.
without_march_2020 = (
    robust_mediation_events.loc[
        robust_mediation_events["Month"]
        != pd.Timestamp("2020-03-01")
    ]
    .copy()
)

mediation_robustness_results.append(
    estimate_aligned_paths(
        without_march_2020,
        core_controls,
        "S4_Core_Excluding_March_2020",
    )
)

# Remove the broader COVID shock period.
without_covid = (
    robust_mediation_events.loc[
        ~robust_mediation_events["Month"].between(
            pd.Timestamp("2020-03-01"),
            pd.Timestamp("2021-06-01"),
        )
    ]
    .copy()
)

mediation_robustness_results.append(
    estimate_aligned_paths(
        without_covid,
        core_controls,
        "S5_Core_Excluding_COVID",
    )
)

# -----------------------------------------------------------
# Display compact results
# -----------------------------------------------------------

mediation_robustness_table = pd.DataFrame([
    {
        key: value
        for key, value in result.items()
        if key not in [
            "Estimation_Data",
            "Mediator_Regressors",
            "Direct_Regressors",
            "Total_Regressors",
        ]
    }
    for result in mediation_robustness_results
])

mediation_robustness_table[
    "Indirect_Sign_Stable"
] = (
    mediation_robustness_table[
        "Indirect_Effect"
    ] < 0
)

display(
    mediation_robustness_table.round(4)
)

# -----------------------------------------------------------
# Plot indirect-effect estimates
# -----------------------------------------------------------

plot_table = (
    mediation_robustness_table.copy()
)

fig = px.bar(
    plot_table,
    x="Specification",
    y="Indirect_Effect",
    color="Indirect_Sign_Stable",
    text="Indirect_Effect",
    hover_data={
        "N": True,
        "Path_a": ":.4f",
        "Path_a_HAC_p": ":.4f",
        "Path_b": ":.4f",
        "Path_b_HAC_p": ":.4f",
        "Direct_Effect": ":.4f",
        "Direct_HAC_p": ":.4f",
        "Total_Effect": ":.4f",
        "Total_HAC_p": ":.4f",
        "Indirect_Effect_for_0_10": ":.4f",
    },
    color_discrete_map={
        True: "#2ca02c",
        False: "#d62728",
    },
    title=(
        "Robustness of the Fed–ETF–Gold "
        "Temporal Indirect Effect"
    ),
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    width=1250,
    height=650,
    xaxis_title="Mediation specification",
    yaxis_title=(
        "Indirect effect on next-month "
        "gold return (percentage points)"
    ),
)

fig.update_xaxes(tickangle=25)
fig.show()

,Specification,N,First_Month,Final_Month,Controls,Path_a,Path_a_HAC_p,Path_b,Path_b_HAC_p,Indirect_Effect,...,Direct_Effect,Direct_HAC_p,Total_Effect,Total_HAC_p,Direct_Plus_Indirect,Decomposition_Error,Outcome_Adjusted_R2,Outcome_AIC,Outcome_BIC,Indirect_Sign_Stable
0,S0_Baseline,60,2019-03-01,2026-06-01,2,-73.4696,0.0000,0.0440,0.0,-3.2355,...,0.7804,0.4850,-2.4552,0.0570,-2.4552,-0.0,0.3584,301.2748,311.7465,True
1,S1_Core_Macro,60,2019-03-01,2026-06-01,5,-64.0121,0.0000,0.0450,0.0,-2.8809,...,0.6385,0.5855,-2.2424,0.1067,-2.2424,0.0,0.3423,305.3896,322.1444,True
2,S2_Core_Macro_CB,60,2019-03-01,2026-06-01,6,-62.0888,0.0000,0.0444,0.0,-2.7592,...,0.6508,0.5808,-2.1084,0.1338,-2.1084,-0.0,0.3317,307.1886,326.0377,True
3,S3_Expanded_PreTreatment,60,2019-03-01,2026-06-01,9,-55.7796,0.0001,0.0476,0.0,-2.6570,...,1.0136,0.3885,-1.6434,0.2791,-1.6434,0.0,0.3246,310.1869,335.3190,True
4,S4_Core_Excluding_March_2020,59,2019-03-01,2026-06-01,5,-57.4226,0.0001,0.0459,0.0,-2.6367,...,0.5474,0.6467,-2.0894,0.1557,-2.0894,-0.0,0.3320,301.2766,317.8969,True
5,S5_Core_Excluding_COVID,49,2019-03-01,2026-06-01,5,-54.3390,0.0002,0.0486,0.0,-2.6400,...,0.4782,0.7077,-2.1618,0.1538,-2.1618,-0.0,0.3093,254.3887,269.5232,True


In [52]:
# CELL 46: Moving-block bootstrap for S1_Core_Macro

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.express as px

# -----------------------------------------------------------
# Retrieve the preferred controlled specification
# -----------------------------------------------------------

s1_result = next(
    result
    for result in mediation_robustness_results
    if result["Specification"] == "S1_Core_Macro"
)

s1_data = (
    s1_result["Estimation_Data"]
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

s1_mediator_regressors = (
    s1_result["Mediator_Regressors"]
)

s1_direct_regressors = (
    s1_result["Direct_Regressors"]
)

s1_total_regressors = (
    s1_result["Total_Regressors"]
)

treatment_variable = (
    "Fed_Expected_Direction"
)

mediator_variable = (
    "ETF_Flow_Next_Month"
)

outcome_variable = (
    "Gold_Return_Next_Month"
)

print(
    "Controlled mediation observations:",
    len(s1_data)
)

print(
    "Period:",
    s1_data["Month"].min(),
    "to",
    s1_data["Month"].max()
)

# -----------------------------------------------------------
# Point estimates
# -----------------------------------------------------------

point_path_a = s1_result["Path_a"]
point_path_b = s1_result["Path_b"]

point_indirect = (
    s1_result["Indirect_Effect"]
)

point_direct = (
    s1_result["Direct_Effect"]
)

point_total = (
    s1_result["Total_Effect"]
)

print(
    "Controlled indirect-effect estimate:",
    round(point_indirect, 4)
)

print(
    "Effect for a 0.10 tightening shift:",
    round(point_indirect * 0.10, 4),
    "gold-return percentage points",
)

# -----------------------------------------------------------
# Circular moving-block bootstrap
# -----------------------------------------------------------

def circular_block_sample_indices(
    sample_size,
    block_length,
    random_generator,
):

    blocks_required = int(
        np.ceil(
            sample_size / block_length
        )
    )

    starting_positions = (
        random_generator.integers(
            low=0,
            high=sample_size,
            size=blocks_required,
        )
    )

    sampled_indices = []

    for starting_position in (
        starting_positions
    ):

        block_indices = [
            (
                starting_position
                + offset
            ) % sample_size
            for offset in range(
                block_length
            )
        ]

        sampled_indices.extend(
            block_indices
        )

    return np.asarray(
        sampled_indices[:sample_size]
    )


def estimate_bootstrap_paths(
    bootstrap_data,
):

    # Mediator equation
    X_mediator_bootstrap = (
        sm.add_constant(
            bootstrap_data[
                s1_mediator_regressors
            ].astype(float),
            has_constant="add",
        )
    )

    mediator_bootstrap_model = sm.OLS(
        bootstrap_data[
            mediator_variable
        ].astype(float),
        X_mediator_bootstrap,
    ).fit()

    # Total-effect equation
    X_total_bootstrap = (
        sm.add_constant(
            bootstrap_data[
                s1_total_regressors
            ].astype(float),
            has_constant="add",
        )
    )

    total_bootstrap_model = sm.OLS(
        bootstrap_data[
            outcome_variable
        ].astype(float),
        X_total_bootstrap,
    ).fit()

    # Direct/mediated equation
    X_direct_bootstrap = (
        sm.add_constant(
            bootstrap_data[
                s1_direct_regressors
            ].astype(float),
            has_constant="add",
        )
    )

    direct_bootstrap_model = sm.OLS(
        bootstrap_data[
            outcome_variable
        ].astype(float),
        X_direct_bootstrap,
    ).fit()

    path_a_bootstrap = (
        mediator_bootstrap_model.params[
            treatment_variable
        ]
    )

    path_b_bootstrap = (
        direct_bootstrap_model.params[
            mediator_variable
        ]
    )

    direct_effect_bootstrap = (
        direct_bootstrap_model.params[
            treatment_variable
        ]
    )

    total_effect_bootstrap = (
        total_bootstrap_model.params[
            treatment_variable
        ]
    )

    return {
        "Path_a":
            path_a_bootstrap,
        "Path_b":
            path_b_bootstrap,
        "Indirect_Effect":
            (
                path_a_bootstrap
                * path_b_bootstrap
            ),
        "Direct_Effect":
            direct_effect_bootstrap,
        "Total_Effect":
            total_effect_bootstrap,
    }


bootstrap_block_lengths = [
    3,
    4,
    6,
]

replications_per_block_length = 2000

controlled_bootstrap_rows = []

for block_length in bootstrap_block_lengths:

    random_generator = (
        np.random.default_rng(
            1857 + block_length
        )
    )

    for replication in range(
        replications_per_block_length
    ):

        sampled_indices = (
            circular_block_sample_indices(
                sample_size=len(s1_data),
                block_length=block_length,
                random_generator=random_generator,
            )
        )

        bootstrap_data = (
            s1_data.iloc[
                sampled_indices
            ]
            .reset_index(drop=True)
        )

        try:
            bootstrap_estimates = (
                estimate_bootstrap_paths(
                    bootstrap_data
                )
            )

            controlled_bootstrap_rows.append({
                "Block_Length":
                    block_length,
                "Replication":
                    replication,
                **bootstrap_estimates,
            })

        except Exception:
            continue

controlled_mediation_bootstrap = pd.DataFrame(
    controlled_bootstrap_rows
)

successful_replications = (
    controlled_mediation_bootstrap
    .groupby("Block_Length")
    .size()
    .rename("Successful_Replications")
    .reset_index()
)

display(successful_replications)

# -----------------------------------------------------------
# Summarize bootstrap distributions
# -----------------------------------------------------------

point_estimates = {
    "Path_a": point_path_a,
    "Path_b": point_path_b,
    "Indirect_Effect": point_indirect,
    "Direct_Effect": point_direct,
    "Total_Effect": point_total,
}

controlled_bootstrap_summary_rows = []

for block_length in bootstrap_block_lengths:

    block_results = (
        controlled_mediation_bootstrap.loc[
            controlled_mediation_bootstrap[
                "Block_Length"
            ] == block_length
        ]
    )

    for effect, point_estimate in (
        point_estimates.items()
    ):

        bootstrap_values = (
            block_results[effect]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna()
        )

        lower_95 = bootstrap_values.quantile(
            0.025
        )

        upper_95 = bootstrap_values.quantile(
            0.975
        )

        probability_nonpositive = (
            bootstrap_values <= 0
        ).mean()

        probability_nonnegative = (
            bootstrap_values >= 0
        ).mean()

        bootstrap_p = min(
            1.0,
            2 * min(
                probability_nonpositive,
                probability_nonnegative,
            ),
        )

        controlled_bootstrap_summary_rows.append({
            "Block_Length":
                block_length,
            "Effect":
                effect,
            "Point_Estimate":
                point_estimate,
            "Bootstrap_Mean":
                bootstrap_values.mean(),
            "Bootstrap_SE":
                bootstrap_values.std(ddof=1),
            "CI_Lower_95pct":
                lower_95,
            "CI_Upper_95pct":
                upper_95,
            "Bootstrap_p":
                bootstrap_p,
            "Significant_5pct":
                (
                    lower_95 > 0
                    or upper_95 < 0
                ),
            "Replications":
                len(bootstrap_values),
        })

controlled_bootstrap_summary = pd.DataFrame(
    controlled_bootstrap_summary_rows
)

display(
    controlled_bootstrap_summary.round(4)
)

# -----------------------------------------------------------
# Focus specifically on the indirect effect
# -----------------------------------------------------------

controlled_indirect_summary = (
    controlled_bootstrap_summary.loc[
        controlled_bootstrap_summary[
            "Effect"
        ] == "Indirect_Effect"
    ]
    .reset_index(drop=True)
)

controlled_indirect_summary[
    "Effect_for_0_10_Direction_Shift"
] = (
    controlled_indirect_summary[
        "Point_Estimate"
    ] * 0.10
)

display(
    controlled_indirect_summary.round(4)
)

# -----------------------------------------------------------
# Plot the indirect-effect distributions
# -----------------------------------------------------------

indirect_plot_data = (
    controlled_mediation_bootstrap[
        [
            "Block_Length",
            "Indirect_Effect",
        ]
    ]
    .copy()
)

indirect_plot_data[
    "Block_Length_Label"
] = (
    "Block length "
    + indirect_plot_data[
        "Block_Length"
    ].astype(str)
)

fig = px.histogram(
    indirect_plot_data,
    x="Indirect_Effect",
    color="Block_Length_Label",
    facet_row="Block_Length_Label",
    nbins=55,
    opacity=0.75,
    marginal="box",
    title=(
        "Bootstrap Distribution of the Controlled "
        "Fed–ETF–Gold Indirect Effect"
    ),
)

fig.add_vline(
    x=0,
    line_color="red",
    line_dash="dash",
)

fig.update_layout(
    template="plotly_white",
    width=1150,
    height=950,
    showlegend=False,
)

fig.update_xaxes(
    title_text=(
        "Indirect effect on next-month "
        "gold return"
    )
)

fig.show()

Controlled mediation observations: 60
Period: 2019-03-01 00:00:00 to 2026-06-01 00:00:00
Controlled indirect-effect estimate: -2.8809
Effect for a 0.10 tightening shift: -0.2881 gold-return percentage points


,Block_Length,Successful_Replications
0,3,2000
1,4,2000
2,6,2000


,Block_Length,Effect,Point_Estimate,Bootstrap_Mean,Bootstrap_SE,CI_Lower_95pct,CI_Upper_95pct,Bootstrap_p,Significant_5pct,Replications
0,3,Path_a,-64.0121,-65.2987,16.7244,-99.6245,-32.3569,0.000,True,2000
1,3,Path_b,0.0450,0.0453,0.0075,0.0305,0.0603,0.000,True,2000
2,3,Indirect_Effect,-2.8809,-2.9443,0.8441,-4.5788,-1.2578,0.000,True,2000
3,3,Direct_Effect,0.6385,0.4909,1.3378,-2.4585,2.8920,0.634,False,2000
4,3,Total_Effect,-2.2424,-2.4535,1.5239,-5.6099,0.3116,0.080,False,2000
5,4,Path_a,-64.0121,-65.6946,16.6722,-100.3039,-34.9355,0.001,True,2000
6,4,Path_b,0.0450,0.0457,0.0079,0.0303,0.0615,0.000,True,2000
7,4,Indirect_Effect,-2.8809,-2.9737,0.8106,-4.7024,-1.4147,0.001,True,2000
8,4,Direct_Effect,0.6385,0.4555,1.4190,-2.6838,2.9623,0.677,False,2000
9,4,Total_Effect,-2.2424,-2.5182,1.5027,-5.5985,0.2036,0.071,False,2000


,Block_Length,Effect,Point_Estimate,Bootstrap_Mean,Bootstrap_SE,CI_Lower_95pct,CI_Upper_95pct,Bootstrap_p,Significant_5pct,Replications,Effect_for_0_10_Direction_Shift
0,3,Indirect_Effect,-2.8809,-2.9443,0.8441,-4.5788,-1.2578,0.000,True,2000,-0.2881
1,4,Indirect_Effect,-2.8809,-2.9737,0.8106,-4.7024,-1.4147,0.001,True,2000,-0.2881
2,6,Indirect_Effect,-2.8809,-3.0067,0.8054,-4.7821,-1.5289,0.001,True,2000,-0.2881


In [53]:
# CELL 47: Generate expanding-window, leakage-free ETF predictions
# for use in the second-stage gold forecasting equation

import statsmodels.api as sm
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# -----------------------------------------------------------
# Use the controlled mediation sample
# -----------------------------------------------------------

system_variables = [
    "Month",
    "Fed_Expected_Direction",
    "ETF_Flow_Next_Month",
    "Gold_Return_Next_Month",
    "Gold_Return_Previous_Month",
    "ETF_Flow_Previous_Month",
    "Broad_USD_Change_Previous_Month",
    "Real10Y_Change_Previous_Month",
    "VIX_Change_Previous_Month",
]

two_stage_system_df = (
    s1_data[system_variables]
    .dropna()
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

print(
    "Two-stage system observations:",
    len(two_stage_system_df)
)

print(
    "Meeting-information period:",
    two_stage_system_df["Month"].min(),
    "to",
    two_stage_system_df["Month"].max()
)

print(
    "Outcome period:",
    (
        two_stage_system_df["Month"].min()
        + pd.offsets.MonthBegin(1)
    ),
    "to",
    (
        two_stage_system_df["Month"].max()
        + pd.offsets.MonthBegin(1)
    )
)

# -----------------------------------------------------------
# First-stage predictors
#
# Fed direction is measured for meeting month t.
# Every other predictor is taken from calendar month t-1.
# -----------------------------------------------------------

first_stage_predictors = [
    "Fed_Expected_Direction",
    "Gold_Return_Previous_Month",
    "ETF_Flow_Previous_Month",
    "Broad_USD_Change_Previous_Month",
    "Real10Y_Change_Previous_Month",
    "VIX_Change_Previous_Month",
]

first_stage_outcome = (
    "ETF_Flow_Next_Month"
)

# Require a sufficiently large initial estimation sample
# relative to the seven first-stage parameters, including
# the constant.
first_stage_initial_window = 18

if (
    first_stage_initial_window
    <= len(first_stage_predictors) + 2
):
    raise ValueError(
        "The initial first-stage window is too small."
    )

# -----------------------------------------------------------
# Recursive first-stage predictions
# -----------------------------------------------------------

recursive_etf_stage_rows = []

for forecast_position in range(
    first_stage_initial_window,
    len(two_stage_system_df)
):

    # Training observations strictly precede the forecast event.
    first_stage_training = (
        two_stage_system_df.iloc[
            :forecast_position
        ].copy()
    )

    first_stage_test = (
        two_stage_system_df.iloc[
            [forecast_position]
        ].copy()
    )

    X_first_train = sm.add_constant(
        first_stage_training[
            first_stage_predictors
        ].astype(float),
        has_constant="add",
    )

    X_first_test = sm.add_constant(
        first_stage_test[
            first_stage_predictors
        ].astype(float),
        has_constant="add",
    ).reindex(
        columns=X_first_train.columns,
        fill_value=1.0,
    )

    y_first_train = (
        first_stage_training[
            first_stage_outcome
        ].astype(float)
    )

    first_stage_model = sm.OLS(
        y_first_train,
        X_first_train,
    ).fit()

    predicted_etf_flow = float(
        first_stage_model.predict(
            X_first_test
        ).iloc[0]
    )

    event_month = first_stage_test[
        "Month"
    ].iloc[0]

    final_training_event_month = (
        first_stage_training[
            "Month"
        ].max()
    )

    recursive_etf_stage_rows.append({
        "Row_Position":
            forecast_position,
        "Event_Month":
            event_month,
        "Outcome_Month":
            event_month
            + pd.offsets.MonthBegin(1),
        "First_Stage_Training_N":
            len(first_stage_training),
        "First_Stage_Final_Training_Month":
            final_training_event_month,
        "Actual_ETF_Flow_Next_Month":
            float(
                first_stage_test[
                    "ETF_Flow_Next_Month"
                ].iloc[0]
            ),
        "Predicted_ETF_Flow_Next_Month":
            predicted_etf_flow,
        "ETF_Forecast_Error":
            float(
                first_stage_test[
                    "ETF_Flow_Next_Month"
                ].iloc[0]
            ) - predicted_etf_flow,
        "Gold_Return_Next_Month":
            float(
                first_stage_test[
                    "Gold_Return_Next_Month"
                ].iloc[0]
            ),
        "Gold_Return_Previous_Month":
            float(
                first_stage_test[
                    "Gold_Return_Previous_Month"
                ].iloc[0]
            ),
        "Fed_Expected_Direction":
            float(
                first_stage_test[
                    "Fed_Expected_Direction"
                ].iloc[0]
            ),
        "Recursive_Direction_Coefficient":
            float(
                first_stage_model.params[
                    "Fed_Expected_Direction"
                ]
            ),
    })

recursive_etf_stage = pd.DataFrame(
    recursive_etf_stage_rows
)

# -----------------------------------------------------------
# Explicit no-leakage validation
# -----------------------------------------------------------

recursive_etf_stage[
    "Chronology_Valid"
] = (
    recursive_etf_stage[
        "First_Stage_Final_Training_Month"
    ]
    < recursive_etf_stage[
        "Event_Month"
    ]
)

if not recursive_etf_stage[
    "Chronology_Valid"
].all():

    raise ValueError(
        "Future-data leakage detected in the first stage."
    )

print(
    "Recursive first-stage predictions:",
    len(recursive_etf_stage)
)

print(
    "First predicted ETF outcome month:",
    recursive_etf_stage[
        "Outcome_Month"
    ].min()
)

print(
    "Final predicted ETF outcome month:",
    recursive_etf_stage[
        "Outcome_Month"
    ].max()
)

print(
    "All first-stage chronology checks passed:",
    recursive_etf_stage[
        "Chronology_Valid"
    ].all()
)

# -----------------------------------------------------------
# First-stage forecast performance
# -----------------------------------------------------------

actual_etf = recursive_etf_stage[
    "Actual_ETF_Flow_Next_Month"
].to_numpy()

predicted_etf = recursive_etf_stage[
    "Predicted_ETF_Flow_Next_Month"
].to_numpy()

etf_stage_errors = (
    actual_etf - predicted_etf
)

first_stage_performance = pd.DataFrame({
    "Forecasts": [
        len(recursive_etf_stage)
    ],
    "RMSE": [
        np.sqrt(
            np.mean(
                etf_stage_errors ** 2
            )
        )
    ],
    "MAE": [
        np.mean(
            np.abs(
                etf_stage_errors
            )
        )
    ],
    "Bias": [
        np.mean(
            etf_stage_errors
        )
    ],
    "Forecast_Actual_Correlation": [
        np.corrcoef(
            actual_etf,
            predicted_etf,
        )[0, 1]
    ],
    "Flow_Sign_Accuracy": [
        np.mean(
            np.sign(actual_etf)
            == np.sign(predicted_etf)
        )
    ],
    "Negative_Direction_Coefficient_Share": [
        (
            recursive_etf_stage[
                "Recursive_Direction_Coefficient"
            ] < 0
        ).mean()
    ],
})

display(
    first_stage_performance.round(4)
)

# -----------------------------------------------------------
# Plot leakage-free ETF predictions
# -----------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=recursive_etf_stage[
            "Outcome_Month"
        ],
        y=recursive_etf_stage[
            "Actual_ETF_Flow_Next_Month"
        ],
        mode="lines+markers",
        name="Actual ETF flow",
        line=dict(
            color="black",
            width=3,
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=recursive_etf_stage[
            "Outcome_Month"
        ],
        y=recursive_etf_stage[
            "Predicted_ETF_Flow_Next_Month"
        ],
        mode="lines+markers",
        name="Recursive ETF forecast",
        line=dict(
            color="#1f77b4",
            width=2,
        ),
    )
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title=(
        "Leakage-Free Expanding-Window Forecasts "
        "of Next-Month Gold ETF Demand"
    ),
    template="plotly_white",
    width=1300,
    height=650,
    xaxis_title="ETF-flow outcome month",
    yaxis_title="ETF net demand (tonnes)",
)

fig.show()

display(
    recursive_etf_stage.head(10).round(4)
)

Two-stage system observations: 60
Meeting-information period: 2019-03-01 00:00:00 to 2026-06-01 00:00:00
Outcome period: 2019-04-01 00:00:00 to 2026-07-01 00:00:00
Recursive first-stage predictions: 42
First predicted ETF outcome month: 2021-07-01 00:00:00
Final predicted ETF outcome month: 2026-07-01 00:00:00
All first-stage chronology checks passed: True


,Forecasts,RMSE,MAE,Bias,Forecast_Actual_Correlation,Flow_Sign_Accuracy,Negative_Direction_Coefficient_Share
0,42,43.5321,35.0872,-1.6011,0.5845,0.8095,1.0


,Row_Position,Event_Month,Outcome_Month,First_Stage_Training_N,First_Stage_Final_Training_Month,Actual_ETF_Flow_Next_Month,Predicted_ETF_Flow_Next_Month,ETF_Forecast_Error,Gold_Return_Next_Month,Gold_Return_Previous_Month,Fed_Expected_Direction,Recursive_Direction_Coefficient,Chronology_Valid
0,18,2021-06-01,2021-07-01,18,2021-04-01,11.1,37.8816,-26.7816,-1.46,5.13,0.1356,-165.7630,True
1,19,2021-07-01,2021-08-01,19,2021-06-01,-22.3,-16.7763,-5.5237,-1.25,-0.85,0.2647,-169.0391,True
2,20,2021-09-01,2021-10-01,20,2021-07-01,-25.5,4.5963,-30.0963,0.10,-1.25,0.1343,-171.8287,True
3,21,2021-11-01,2021-12-01,21,2021-09-01,-6.4,25.0267,-31.4267,-1.72,0.10,0.1775,-176.5006,True
4,22,2021-12-01,2022-01-01,22,2021-11-01,46.3,20.9515,25.3485,1.43,2.53,0.0437,-181.6079,True
5,23,2022-01-01,2022-02-01,23,2021-12-01,35.3,9.3506,25.9494,2.22,-1.72,0.1808,-176.3961,True
6,24,2022-03-01,2022-04-01,24,2022-01-01,42.9,16.6036,26.2964,-0.57,2.22,0.0855,-171.3835,True
7,25,2022-05-01,2022-06-01,25,2022-03-01,-28.5,-152.5978,124.0978,-0.65,-0.57,0.8762,-165.9312,True
8,26,2022-06-01,2022-07-01,26,2022-05-01,-81.0,-67.6887,-13.3113,-5.66,-4.57,0.8888,-122.6850,True
9,27,2022-07-01,2022-08-01,27,2022-06-01,-50.9,-65.3495,14.4495,1.84,-0.65,0.9009,-127.3024,True


In [54]:
# CELL 48: Leakage-free two-stage gold-return forecasting system

import statsmodels.api as sm
from scipy import stats
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# -----------------------------------------------------------
# STEP 1: Generate a matching no-Fed ETF forecast
# -----------------------------------------------------------

no_fed_first_stage_predictors = [
    "Gold_Return_Previous_Month",
    "ETF_Flow_Previous_Month",
    "Broad_USD_Change_Previous_Month",
    "Real10Y_Change_Previous_Month",
    "VIX_Change_Previous_Month",
]

with_fed_first_stage_predictors = [
    "Fed_Expected_Direction",
] + no_fed_first_stage_predictors

first_stage_comparison_rows = []

for forecast_position in range(
    first_stage_initial_window,
    len(two_stage_system_df)
):

    training_data = (
        two_stage_system_df.iloc[
            :forecast_position
        ].copy()
    )

    test_data = (
        two_stage_system_df.iloc[
            [forecast_position]
        ].copy()
    )

    y_etf_train = training_data[
        "ETF_Flow_Next_Month"
    ].astype(float)

    event_month = test_data[
        "Month"
    ].iloc[0]

    forecast_record = {
        "Row_Position":
            forecast_position,
        "Event_Month":
            event_month,
        "Outcome_Month":
            event_month
            + pd.offsets.MonthBegin(1),
        "Actual_ETF_Flow_Next_Month":
            float(
                test_data[
                    "ETF_Flow_Next_Month"
                ].iloc[0]
            ),
        "Gold_Return_Next_Month":
            float(
                test_data[
                    "Gold_Return_Next_Month"
                ].iloc[0]
            ),
        "Gold_Return_Previous_Month":
            float(
                test_data[
                    "Gold_Return_Previous_Month"
                ].iloc[0]
            ),
        "Fed_Expected_Direction":
            float(
                test_data[
                    "Fed_Expected_Direction"
                ].iloc[0]
            ),
        "Final_First_Stage_Training_Month":
            training_data[
                "Month"
            ].max(),
    }

    for first_stage_name, predictors in {
        "ETF_NoFed":
            no_fed_first_stage_predictors,
        "ETF_WithFed":
            with_fed_first_stage_predictors,
    }.items():

        X_train = sm.add_constant(
            training_data[
                predictors
            ].astype(float),
            has_constant="add",
        )

        X_test = sm.add_constant(
            test_data[
                predictors
            ].astype(float),
            has_constant="add",
        ).reindex(
            columns=X_train.columns,
            fill_value=1.0,
        )

        model = sm.OLS(
            y_etf_train,
            X_train,
        ).fit()

        forecast_record[
            f"{first_stage_name}_Forecast"
        ] = float(
            model.predict(
                X_test
            ).iloc[0]
        )

    first_stage_comparison_rows.append(
        forecast_record
    )

two_stage_generated_data = pd.DataFrame(
    first_stage_comparison_rows
)

# Explicit chronology check.
two_stage_generated_data[
    "First_Stage_Chronology_Valid"
] = (
    two_stage_generated_data[
        "Final_First_Stage_Training_Month"
    ]
    < two_stage_generated_data[
        "Event_Month"
    ]
)

if not two_stage_generated_data[
    "First_Stage_Chronology_Valid"
].all():

    raise ValueError(
        "First-stage future leakage detected."
    )

print(
    "Generated historical ETF forecasts:",
    len(two_stage_generated_data)
)

print(
    "All first-stage chronology checks passed:",
    two_stage_generated_data[
        "First_Stage_Chronology_Valid"
    ].all()
)

# -----------------------------------------------------------
# Compare the two first-stage ETF forecasts
# -----------------------------------------------------------

actual_etf = two_stage_generated_data[
    "Actual_ETF_Flow_Next_Month"
].to_numpy()

first_stage_metric_rows = []

for forecast_variable, model_label in {
    "ETF_NoFed_Forecast":
        "ETF forecast without Fed",
    "ETF_WithFed_Forecast":
        "ETF forecast with Fed direction",
}.items():

    forecast_values = (
        two_stage_generated_data[
            forecast_variable
        ].to_numpy()
    )

    errors = actual_etf - forecast_values

    first_stage_metric_rows.append({
        "Model": model_label,
        "RMSE": np.sqrt(
            np.mean(errors ** 2)
        ),
        "MAE": np.mean(
            np.abs(errors)
        ),
        "Bias": np.mean(errors),
        "Forecast_Actual_Correlation":
            np.corrcoef(
                actual_etf,
                forecast_values,
            )[0, 1],
        "Flow_Sign_Accuracy":
            np.mean(
                np.sign(actual_etf)
                == np.sign(forecast_values)
            ),
    })

two_stage_first_stage_metrics = (
    pd.DataFrame(
        first_stage_metric_rows
    )
)

display(
    two_stage_first_stage_metrics.round(4)
)

# -----------------------------------------------------------
# STEP 2: Recursive gold-return equations
# -----------------------------------------------------------

# Number of generated historical ETF forecasts required before
# estimating the first second-stage gold equation.
second_stage_initial_window = 18

if (
    second_stage_initial_window
    >= len(two_stage_generated_data)
):
    raise ValueError(
        "Second-stage initial window is too large."
    )

second_stage_specifications = {
    # Gold persistence benchmark
    "H0_Gold_Persistence": [
        "Gold_Return_Previous_Month",
    ],

    # Direct Fed-information benchmark
    "H1_Direct_Fed": [
        "Gold_Return_Previous_Month",
        "Fed_Expected_Direction",
    ],

    # ETF forecast constructed without Fed direction
    "H2_ETF_NoFed_System": [
        "Gold_Return_Previous_Month",
        "ETF_NoFed_Forecast",
    ],

    # Proposed mediated forecasting system
    "H3_ETF_Fed_System": [
        "Gold_Return_Previous_Month",
        "ETF_WithFed_Forecast",
    ],
}

second_stage_forecast_rows = []

for forecast_position in range(
    second_stage_initial_window,
    len(two_stage_generated_data)
):

    gold_training_data = (
        two_stage_generated_data.iloc[
            :forecast_position
        ].copy()
    )

    gold_test_data = (
        two_stage_generated_data.iloc[
            [forecast_position]
        ].copy()
    )

    actual_gold_return = float(
        gold_test_data[
            "Gold_Return_Next_Month"
        ].iloc[0]
    )

    event_month = gold_test_data[
        "Event_Month"
    ].iloc[0]

    second_stage_record = {
        "Event_Month":
            event_month,
        "Outcome_Month":
            gold_test_data[
                "Outcome_Month"
            ].iloc[0],
        "Second_Stage_Training_N":
            len(gold_training_data),
        "Final_Second_Stage_Training_Month":
            gold_training_data[
                "Event_Month"
            ].max(),
        "Actual_Gold_Return":
            actual_gold_return,
        "ETF_NoFed_Forecast":
            float(
                gold_test_data[
                    "ETF_NoFed_Forecast"
                ].iloc[0]
            ),
        "ETF_WithFed_Forecast":
            float(
                gold_test_data[
                    "ETF_WithFed_Forecast"
                ].iloc[0]
            ),
    }

    y_gold_train = gold_training_data[
        "Gold_Return_Next_Month"
    ].astype(float)

    for model_name, predictors in (
        second_stage_specifications.items()
    ):

        X_gold_train = sm.add_constant(
            gold_training_data[
                predictors
            ].astype(float),
            has_constant="add",
        )

        X_gold_test = sm.add_constant(
            gold_test_data[
                predictors
            ].astype(float),
            has_constant="add",
        ).reindex(
            columns=X_gold_train.columns,
            fill_value=1.0,
        )

        gold_model = sm.OLS(
            y_gold_train,
            X_gold_train,
        ).fit()

        gold_forecast = float(
            gold_model.predict(
                X_gold_test
            ).iloc[0]
        )

        second_stage_record[
            f"{model_name}_Forecast"
        ] = gold_forecast

        second_stage_record[
            f"{model_name}_Error"
        ] = (
            actual_gold_return
            - gold_forecast
        )

        if (
            "ETF_WithFed_Forecast"
            in gold_model.params.index
        ):
            second_stage_record[
                "Recursive_Predicted_ETF_Coefficient"
            ] = gold_model.params[
                "ETF_WithFed_Forecast"
            ]

    second_stage_forecast_rows.append(
        second_stage_record
    )

two_stage_gold_forecasts = pd.DataFrame(
    second_stage_forecast_rows
)

two_stage_gold_forecasts[
    "Second_Stage_Chronology_Valid"
] = (
    two_stage_gold_forecasts[
        "Final_Second_Stage_Training_Month"
    ]
    < two_stage_gold_forecasts[
        "Event_Month"
    ]
)

if not two_stage_gold_forecasts[
    "Second_Stage_Chronology_Valid"
].all():

    raise ValueError(
        "Second-stage future leakage detected."
    )

print(
    "\nTrue two-stage gold forecasts:",
    len(two_stage_gold_forecasts)
)

print(
    "First gold outcome month:",
    two_stage_gold_forecasts[
        "Outcome_Month"
    ].min()
)

print(
    "Final gold outcome month:",
    two_stage_gold_forecasts[
        "Outcome_Month"
    ].max()
)

print(
    "All second-stage chronology checks passed:",
    two_stage_gold_forecasts[
        "Second_Stage_Chronology_Valid"
    ].all()
)

# -----------------------------------------------------------
# Gold forecast metrics
# -----------------------------------------------------------

actual_gold = two_stage_gold_forecasts[
    "Actual_Gold_Return"
].to_numpy()

gold_forecast_metric_rows = []

for model_name in second_stage_specifications:

    forecasts = two_stage_gold_forecasts[
        f"{model_name}_Forecast"
    ].to_numpy()

    errors = actual_gold - forecasts

    gold_forecast_metric_rows.append({
        "Model": model_name,
        "Forecasts":
            len(actual_gold),
        "RMSE":
            np.sqrt(
                np.mean(errors ** 2)
            ),
        "MAE":
            np.mean(
                np.abs(errors)
            ),
        "Bias":
            np.mean(errors),
        "Forecast_Actual_Correlation":
            np.corrcoef(
                actual_gold,
                forecasts,
            )[0, 1],
        "Return_Sign_Accuracy":
            np.mean(
                np.sign(actual_gold)
                == np.sign(forecasts)
            ),
    })

two_stage_gold_metrics = pd.DataFrame(
    gold_forecast_metric_rows
)

benchmark_rmse = two_stage_gold_metrics.loc[
    two_stage_gold_metrics["Model"]
    == "H0_Gold_Persistence",
    "RMSE",
].iloc[0]

benchmark_mae = two_stage_gold_metrics.loc[
    two_stage_gold_metrics["Model"]
    == "H0_Gold_Persistence",
    "MAE",
].iloc[0]

two_stage_gold_metrics[
    "RMSE_Improvement_vs_H0_pct"
] = (
    (
        benchmark_rmse
        - two_stage_gold_metrics["RMSE"]
    )
    / benchmark_rmse
    * 100
)

two_stage_gold_metrics[
    "MAE_Improvement_vs_H0_pct"
] = (
    (
        benchmark_mae
        - two_stage_gold_metrics["MAE"]
    )
    / benchmark_mae
    * 100
)

display(
    two_stage_gold_metrics.round(4)
)

# -----------------------------------------------------------
# HAC loss-difference tests
#
# Positive mean loss difference means the comparison model
# has lower squared error than the reference model.
# -----------------------------------------------------------

forecast_comparisons = [
    {
        "Comparison":
            "H1 Direct Fed vs H0 persistence",
        "Reference":
            "H0_Gold_Persistence",
        "Alternative":
            "H1_Direct_Fed",
    },
    {
        "Comparison":
            "H2 no-Fed ETF system vs H0 persistence",
        "Reference":
            "H0_Gold_Persistence",
        "Alternative":
            "H2_ETF_NoFed_System",
    },
    {
        "Comparison":
            "H3 Fed-ETF system vs H0 persistence",
        "Reference":
            "H0_Gold_Persistence",
        "Alternative":
            "H3_ETF_Fed_System",
    },
    {
        "Comparison":
            "H3 Fed-ETF system vs H1 direct Fed",
        "Reference":
            "H1_Direct_Fed",
        "Alternative":
            "H3_ETF_Fed_System",
    },
    {
        "Comparison":
            "H3 Fed-ETF system vs H2 no-Fed ETF system",
        "Reference":
            "H2_ETF_NoFed_System",
        "Alternative":
            "H3_ETF_Fed_System",
    },
]

loss_test_rows = []

for comparison in forecast_comparisons:

    reference_error = (
        two_stage_gold_forecasts[
            f"{comparison['Reference']}_Error"
        ].to_numpy()
    )

    alternative_error = (
        two_stage_gold_forecasts[
            f"{comparison['Alternative']}_Error"
        ].to_numpy()
    )

    loss_difference = (
        reference_error ** 2
        - alternative_error ** 2
    )

    loss_model = sm.OLS(
        loss_difference,
        np.ones(len(loss_difference)),
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 2,
            "use_correction": True,
        },
    )

    test_statistic = float(
        np.asarray(
            loss_model.tvalues
        ).reshape(-1)[0]
    )

    one_sided_p = float(
        1 - stats.t.cdf(
            test_statistic,
            df=loss_model.df_resid,
        )
    )

    loss_test_rows.append({
        "Comparison":
            comparison["Comparison"],
        "Mean_Squared_Loss_Improvement":
            np.mean(loss_difference),
        "HAC_t":
            test_statistic,
        "One_Sided_p":
            one_sided_p,
        "Improvement_Significant_5pct":
            one_sided_p < 0.05,
        "Improvement_Significant_10pct":
            one_sided_p < 0.10,
    })

two_stage_loss_tests = pd.DataFrame(
    loss_test_rows
)

display(
    two_stage_loss_tests.round(4)
)

# -----------------------------------------------------------
# Recursive second-stage coefficient stability
# -----------------------------------------------------------

predicted_etf_coefficients = (
    two_stage_gold_forecasts[
        "Recursive_Predicted_ETF_Coefficient"
    ]
)

second_stage_coefficient_summary = pd.DataFrame({
    "Minimum_ETF_Coefficient": [
        predicted_etf_coefficients.min()
    ],
    "Maximum_ETF_Coefficient": [
        predicted_etf_coefficients.max()
    ],
    "Median_ETF_Coefficient": [
        predicted_etf_coefficients.median()
    ],
    "Positive_Coefficient_Share": [
        (
            predicted_etf_coefficients > 0
        ).mean()
    ],
})

display(
    second_stage_coefficient_summary.round(4)
)

# -----------------------------------------------------------
# Plot actual and predicted gold returns
# -----------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=two_stage_gold_forecasts[
            "Outcome_Month"
        ],
        y=two_stage_gold_forecasts[
            "Actual_Gold_Return"
        ],
        mode="lines+markers",
        name="Actual gold return",
        line=dict(
            color="black",
            width=3,
        ),
    )
)

plot_colors = {
    "H0_Gold_Persistence": "#7f7f7f",
    "H1_Direct_Fed": "#d62728",
    "H2_ETF_NoFed_System": "#ff7f0e",
    "H3_ETF_Fed_System": "#1f77b4",
}

for model_name in second_stage_specifications:

    fig.add_trace(
        go.Scatter(
            x=two_stage_gold_forecasts[
                "Outcome_Month"
            ],
            y=two_stage_gold_forecasts[
                f"{model_name}_Forecast"
            ],
            mode="lines+markers",
            name=model_name,
            line=dict(
                color=plot_colors[model_name],
                dash=(
                    "solid"
                    if model_name
                    == "H3_ETF_Fed_System"
                    else "dot"
                ),
            ),
        )
    )

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title=(
        "Leakage-Free Two-Stage Forecasts "
        "of Next-Month Gold Returns"
    ),
    template="plotly_white",
    width=1400,
    height=700,
    xaxis_title="Gold-return outcome month",
    yaxis_title="Gold monthly return (%)",
)

fig.show()

Generated historical ETF forecasts: 42
All first-stage chronology checks passed: True


,Model,RMSE,MAE,Bias,Forecast_Actual_Correlation,Flow_Sign_Accuracy
0,ETF forecast without Fed,43.5281,33.7028,-10.0685,0.4666,0.6429
1,ETF forecast with Fed direction,43.5321,35.0872,-1.6011,0.5845,0.8095



True two-stage gold forecasts: 24
First gold outcome month: 2023-10-01 00:00:00
Final gold outcome month: 2026-07-01 00:00:00
All second-stage chronology checks passed: True


,Model,Forecasts,RMSE,MAE,Bias,Forecast_Actual_Correlation,Return_Sign_Accuracy,RMSE_Improvement_vs_H0_pct,MAE_Improvement_vs_H0_pct
0,H0_Gold_Persistence,24,4.3241,3.3658,1.7424,-0.1485,0.6667,0.0000,0.0000
1,H1_Direct_Fed,24,4.1421,3.3196,1.3804,0.0565,0.5833,4.2079,1.3723
2,H2_ETF_NoFed_System,24,4.3470,3.3850,1.7758,-0.1189,0.7083,-0.5316,-0.5708
3,H3_ETF_Fed_System,24,4.1695,3.3164,1.6354,0.1016,0.5833,3.5739,1.4671


,Comparison,Mean_Squared_Loss_Improvement,HAC_t,One_Sided_p,Improvement_Significant_5pct,Improvement_Significant_10pct
0,H1 Direct Fed vs H0 persistence,1.5404,1.0232,0.1584,False,False
1,H2 no-Fed ETF system vs H0 persistence,-0.1993,-0.3138,0.6217,False,False
2,H3 Fed-ETF system vs H0 persistence,1.3126,1.0583,0.1505,False,False
3,H3 Fed-ETF system vs H1 direct Fed,-0.2278,-0.5366,0.7016,False,False
4,H3 Fed-ETF system vs H2 no-Fed ETF system,1.5119,1.2203,0.1174,False,False


,Minimum_ETF_Coefficient,Maximum_ETF_Coefficient,Median_ETF_Coefficient,Positive_Coefficient_Share
0,-0.0134,0.0262,0.0063,0.6667


In [55]:
# CELL 49: First-stage and second-stage expanding-window sensitivity
# All specifications are evaluated on the same outcome months.

import statsmodels.api as sm
from scipy import stats
import pandas as pd
import numpy as np
import plotly.express as px

# -----------------------------------------------------------
# Preconditions
# -----------------------------------------------------------

if "two_stage_system_df" not in globals():
    raise NameError(
        "two_stage_system_df is unavailable. "
        "Run Cells 44–48 before Cell 49."
    )

sensitivity_source = (
    two_stage_system_df
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

required_columns = [
    "Month",
    "Fed_Expected_Direction",
    "ETF_Flow_Next_Month",
    "Gold_Return_Next_Month",
    "Gold_Return_Previous_Month",
    "ETF_Flow_Previous_Month",
    "Broad_USD_Change_Previous_Month",
    "Real10Y_Change_Previous_Month",
    "VIX_Change_Previous_Month",
]

missing_columns = [
    column
    for column in required_columns
    if column not in sensitivity_source.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

if sensitivity_source[required_columns].isna().any().any():
    raise ValueError(
        "The sensitivity source contains missing values."
    )

# -----------------------------------------------------------
# Window choices
#
# Baseline from Cell 48:
# first-stage window  = 18
# second-stage window = 18
# -----------------------------------------------------------

first_stage_windows = [15, 18, 21]
second_stage_windows = [12, 15, 18]

window_combinations = [
    (first_window, second_window)
    for first_window in first_stage_windows
    for second_window in second_stage_windows
]

# The original row position of the first gold forecast is:
# first-stage window + second-stage window.
#
# Use the largest starting position so every specification is
# evaluated on exactly the same outcome months.
common_start_position = max(
    first_window + second_window
    for first_window, second_window
    in window_combinations
)

if common_start_position >= len(sensitivity_source):
    raise ValueError(
        "The chosen windows leave no common evaluation sample."
    )

print(
    "Sensitivity-source observations:",
    len(sensitivity_source)
)

print(
    "Common evaluation begins at source-row position:",
    common_start_position
)

print(
    "Common evaluation event month:",
    sensitivity_source.loc[
        common_start_position,
        "Month"
    ]
)

print(
    "Common evaluation outcome month:",
    sensitivity_source.loc[
        common_start_position,
        "Month"
    ] + pd.offsets.MonthBegin(1)
)

print(
    "Common evaluation forecasts:",
    len(sensitivity_source)
    - common_start_position
)

# -----------------------------------------------------------
# First-stage specifications
# -----------------------------------------------------------

no_fed_predictors = [
    "Gold_Return_Previous_Month",
    "ETF_Flow_Previous_Month",
    "Broad_USD_Change_Previous_Month",
    "Real10Y_Change_Previous_Month",
    "VIX_Change_Previous_Month",
]

with_fed_predictors = [
    "Fed_Expected_Direction",
] + no_fed_predictors

# -----------------------------------------------------------
# Storage
# -----------------------------------------------------------

all_sensitivity_forecasts = []
sensitivity_metric_rows = []
sensitivity_loss_rows = []
sensitivity_coefficient_rows = []

# -----------------------------------------------------------
# Loop over first-stage windows
# -----------------------------------------------------------

for first_window in first_stage_windows:

    if first_window <= len(with_fed_predictors) + 2:
        raise ValueError(
            f"First-stage window {first_window} is too small."
        )

    generated_rows = []

    for forecast_position in range(
        first_window,
        len(sensitivity_source)
    ):

        train = sensitivity_source.iloc[
            :forecast_position
        ].copy()

        test = sensitivity_source.iloc[
            [forecast_position]
        ].copy()

        y_etf_train = train[
            "ETF_Flow_Next_Month"
        ].astype(float)

        record = {
            "Source_Row_Position":
                forecast_position,
            "Event_Month":
                test["Month"].iloc[0],
            "Outcome_Month":
                test["Month"].iloc[0]
                + pd.offsets.MonthBegin(1),
            "Actual_ETF_Flow_Next_Month":
                float(
                    test[
                        "ETF_Flow_Next_Month"
                    ].iloc[0]
                ),
            "Actual_Gold_Return_Next_Month":
                float(
                    test[
                        "Gold_Return_Next_Month"
                    ].iloc[0]
                ),
            "Gold_Return_Previous_Month":
                float(
                    test[
                        "Gold_Return_Previous_Month"
                    ].iloc[0]
                ),
            "Fed_Expected_Direction":
                float(
                    test[
                        "Fed_Expected_Direction"
                    ].iloc[0]
                ),
            "First_Stage_Final_Training_Month":
                train["Month"].max(),
        }

        first_stage_models = {
            "ETF_NoFed_Forecast":
                no_fed_predictors,
            "ETF_WithFed_Forecast":
                with_fed_predictors,
        }

        for forecast_name, predictors in (
            first_stage_models.items()
        ):

            X_train = sm.add_constant(
                train[predictors].astype(float),
                has_constant="add",
            )

            X_test = sm.add_constant(
                test[predictors].astype(float),
                has_constant="add",
            ).reindex(
                columns=X_train.columns,
                fill_value=1.0,
            )

            fitted_model = sm.OLS(
                y_etf_train,
                X_train,
            ).fit()

            record[forecast_name] = float(
                fitted_model.predict(
                    X_test
                ).iloc[0]
            )

            if (
                forecast_name
                == "ETF_WithFed_Forecast"
            ):
                record[
                    "Recursive_First_Stage_Direction_Coefficient"
                ] = float(
                    fitted_model.params[
                        "Fed_Expected_Direction"
                    ]
                )

        generated_rows.append(record)

    generated = pd.DataFrame(generated_rows)

    generated[
        "First_Stage_Chronology_Valid"
    ] = (
        generated[
            "First_Stage_Final_Training_Month"
        ]
        < generated["Event_Month"]
    )

    if not generated[
        "First_Stage_Chronology_Valid"
    ].all():
        raise ValueError(
            "First-stage future-data leakage detected."
        )

    # -------------------------------------------------------
    # Loop over second-stage windows
    # -------------------------------------------------------

    for second_window in second_stage_windows:

        if second_window >= len(generated):
            continue

        gold_forecast_rows = []

        second_stage_specs = {
            "H0_Gold_Persistence": [
                "Gold_Return_Previous_Month",
            ],
            "H1_Direct_Fed": [
                "Gold_Return_Previous_Month",
                "Fed_Expected_Direction",
            ],
            "H2_ETF_NoFed_System": [
                "Gold_Return_Previous_Month",
                "ETF_NoFed_Forecast",
            ],
            "H3_ETF_Fed_System": [
                "Gold_Return_Previous_Month",
                "ETF_WithFed_Forecast",
            ],
        }

        for generated_position in range(
            second_window,
            len(generated)
        ):

            gold_train = generated.iloc[
                :generated_position
            ].copy()

            gold_test = generated.iloc[
                [generated_position]
            ].copy()

            actual_gold = float(
                gold_test[
                    "Actual_Gold_Return_Next_Month"
                ].iloc[0]
            )

            forecast_record = {
                "First_Stage_Window":
                    first_window,
                "Second_Stage_Window":
                    second_window,
                "Source_Row_Position":
                    int(
                        gold_test[
                            "Source_Row_Position"
                        ].iloc[0]
                    ),
                "Event_Month":
                    gold_test[
                        "Event_Month"
                    ].iloc[0],
                "Outcome_Month":
                    gold_test[
                        "Outcome_Month"
                    ].iloc[0],
                "Actual_Gold_Return":
                    actual_gold,
                "Final_Second_Stage_Training_Month":
                    gold_train[
                        "Event_Month"
                    ].max(),
            }

            y_gold_train = gold_train[
                "Actual_Gold_Return_Next_Month"
            ].astype(float)

            for model_name, predictors in (
                second_stage_specs.items()
            ):

                X_gold_train = sm.add_constant(
                    gold_train[
                        predictors
                    ].astype(float),
                    has_constant="add",
                )

                X_gold_test = sm.add_constant(
                    gold_test[
                        predictors
                    ].astype(float),
                    has_constant="add",
                ).reindex(
                    columns=X_gold_train.columns,
                    fill_value=1.0,
                )

                gold_model = sm.OLS(
                    y_gold_train,
                    X_gold_train,
                ).fit()

                forecast = float(
                    gold_model.predict(
                        X_gold_test
                    ).iloc[0]
                )

                forecast_record[
                    f"{model_name}_Forecast"
                ] = forecast

                forecast_record[
                    f"{model_name}_Error"
                ] = actual_gold - forecast

                if (
                    model_name
                    == "H3_ETF_Fed_System"
                ):
                    forecast_record[
                        "Recursive_H3_ETF_Coefficient"
                    ] = float(
                        gold_model.params[
                            "ETF_WithFed_Forecast"
                        ]
                    )

            gold_forecast_rows.append(
                forecast_record
            )

        gold_forecasts = pd.DataFrame(
            gold_forecast_rows
        )

        gold_forecasts[
            "Second_Stage_Chronology_Valid"
        ] = (
            gold_forecasts[
                "Final_Second_Stage_Training_Month"
            ]
            < gold_forecasts["Event_Month"]
        )

        if not gold_forecasts[
            "Second_Stage_Chronology_Valid"
        ].all():
            raise ValueError(
                "Second-stage future-data leakage detected."
            )

        # Restrict every combination to the identical period.
        common_forecasts = (
            gold_forecasts.loc[
                gold_forecasts[
                    "Source_Row_Position"
                ] >= common_start_position
            ]
            .copy()
            .reset_index(drop=True)
        )

        if common_forecasts.empty:
            continue

        all_sensitivity_forecasts.append(
            common_forecasts
        )

        # ---------------------------------------------------
        # Forecast metrics
        # ---------------------------------------------------

        actual = common_forecasts[
            "Actual_Gold_Return"
        ].to_numpy()

        model_metrics = {}

        for model_name in second_stage_specs:

            forecast = common_forecasts[
                f"{model_name}_Forecast"
            ].to_numpy()

            error = actual - forecast

            model_metrics[model_name] = {
                "RMSE":
                    np.sqrt(
                        np.mean(error ** 2)
                    ),
                "MAE":
                    np.mean(
                        np.abs(error)
                    ),
                "Bias":
                    np.mean(error),
                "Correlation":
                    np.corrcoef(
                        actual,
                        forecast,
                    )[0, 1],
                "Sign_Accuracy":
                    np.mean(
                        np.sign(actual)
                        == np.sign(forecast)
                    ),
            }

        h0_rmse = model_metrics[
            "H0_Gold_Persistence"
        ]["RMSE"]

        h0_mae = model_metrics[
            "H0_Gold_Persistence"
        ]["MAE"]

        for model_name, metrics in (
            model_metrics.items()
        ):

            sensitivity_metric_rows.append({
                "First_Stage_Window":
                    first_window,
                "Second_Stage_Window":
                    second_window,
                "Model":
                    model_name,
                "Forecasts":
                    len(common_forecasts),
                "First_Outcome_Month":
                    common_forecasts[
                        "Outcome_Month"
                    ].min(),
                "Final_Outcome_Month":
                    common_forecasts[
                        "Outcome_Month"
                    ].max(),
                "RMSE":
                    metrics["RMSE"],
                "MAE":
                    metrics["MAE"],
                "Bias":
                    metrics["Bias"],
                "Forecast_Actual_Correlation":
                    metrics["Correlation"],
                "Return_Sign_Accuracy":
                    metrics["Sign_Accuracy"],
                "RMSE_Improvement_vs_H0_pct":
                    (
                        h0_rmse
                        - metrics["RMSE"]
                    ) / h0_rmse * 100,
                "MAE_Improvement_vs_H0_pct":
                    (
                        h0_mae
                        - metrics["MAE"]
                    ) / h0_mae * 100,
            })

        # ---------------------------------------------------
        # Decisive H3 versus H2 loss test
        # ---------------------------------------------------

        h2_error = common_forecasts[
            "H2_ETF_NoFed_System_Error"
        ].to_numpy()

        h3_error = common_forecasts[
            "H3_ETF_Fed_System_Error"
        ].to_numpy()

        loss_improvement = (
            h2_error ** 2
            - h3_error ** 2
        )

        loss_regression = sm.OLS(
            loss_improvement,
            np.ones(len(loss_improvement)),
        ).fit(
            cov_type="HAC",
            cov_kwds={
                "maxlags": min(
                    2,
                    len(loss_improvement) - 1
                ),
                "use_correction": True,
            },
        )

        hac_t = float(
            np.asarray(
                loss_regression.tvalues
            ).reshape(-1)[0]
        )

        one_sided_p = float(
            1 - stats.t.cdf(
                hac_t,
                df=loss_regression.df_resid,
            )
        )

        sensitivity_loss_rows.append({
            "First_Stage_Window":
                first_window,
            "Second_Stage_Window":
                second_window,
            "Forecasts":
                len(common_forecasts),
            "Mean_Squared_Loss_Improvement_H3_vs_H2":
                np.mean(loss_improvement),
            "HAC_t":
                hac_t,
            "One_Sided_p":
                one_sided_p,
            "H3_Beats_H2_5pct":
                one_sided_p < 0.05,
            "H3_Beats_H2_10pct":
                one_sided_p < 0.10,
        })

        # ---------------------------------------------------
        # Recursive coefficient stability
        # ---------------------------------------------------

        etf_coefficients = common_forecasts[
            "Recursive_H3_ETF_Coefficient"
        ]

        sensitivity_coefficient_rows.append({
            "First_Stage_Window":
                first_window,
            "Second_Stage_Window":
                second_window,
            "Minimum_H3_ETF_Coefficient":
                etf_coefficients.min(),
            "Maximum_H3_ETF_Coefficient":
                etf_coefficients.max(),
            "Median_H3_ETF_Coefficient":
                etf_coefficients.median(),
            "Positive_H3_ETF_Coefficient_Share":
                (etf_coefficients > 0).mean(),
        })

# -----------------------------------------------------------
# Assemble output tables
# -----------------------------------------------------------

window_sensitivity_forecasts = pd.concat(
    all_sensitivity_forecasts,
    ignore_index=True,
)

window_sensitivity_metrics = pd.DataFrame(
    sensitivity_metric_rows
)

window_sensitivity_loss_tests = pd.DataFrame(
    sensitivity_loss_rows
)

window_sensitivity_coefficients = pd.DataFrame(
    sensitivity_coefficient_rows
)

print(
    "\nAll first-stage chronology checks passed:",
    True
)

print(
    "All second-stage chronology checks passed:",
    window_sensitivity_forecasts[
        "Second_Stage_Chronology_Valid"
    ].all()
)

print(
    "Window combinations evaluated:",
    len(window_sensitivity_loss_tests)
)

# -----------------------------------------------------------
# Main H3 performance table
# -----------------------------------------------------------

h3_window_results = (
    window_sensitivity_metrics.loc[
        window_sensitivity_metrics["Model"]
        == "H3_ETF_Fed_System"
    ]
    .merge(
        window_sensitivity_loss_tests,
        on=[
            "First_Stage_Window",
            "Second_Stage_Window",
            "Forecasts",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        window_sensitivity_coefficients,
        on=[
            "First_Stage_Window",
            "Second_Stage_Window",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "RMSE",
            "First_Stage_Window",
            "Second_Stage_Window",
        ]
    )
    .reset_index(drop=True)
)

print(
    "\nH3 Fed–ETF system window sensitivity"
)

display(
    h3_window_results.round(4)
)

# -----------------------------------------------------------
# Complete model comparison
# -----------------------------------------------------------

print(
    "\nAll gold models across window combinations"
)

display(
    window_sensitivity_metrics.sort_values(
        [
            "First_Stage_Window",
            "Second_Stage_Window",
            "Model",
        ]
    ).round(4)
)

# -----------------------------------------------------------
# Sensitivity heatmaps
# -----------------------------------------------------------

rmse_heatmap_data = (
    h3_window_results.pivot(
        index="Second_Stage_Window",
        columns="First_Stage_Window",
        values="RMSE_Improvement_vs_H0_pct",
    )
)

fig = px.imshow(
    rmse_heatmap_data,
    text_auto=".2f",
    color_continuous_scale="RdYlGn",
    aspect="auto",
    labels={
        "x": "First-stage initial window",
        "y": "Second-stage initial window",
        "color": "H3 RMSE improvement (%)",
    },
    title=(
        "Fed–ETF Gold System: RMSE Improvement "
        "Across Initial Windows"
    ),
)

fig.update_layout(
    template="plotly_white",
    width=900,
    height=600,
)

fig.show()

pvalue_heatmap_data = (
    h3_window_results.pivot(
        index="Second_Stage_Window",
        columns="First_Stage_Window",
        values="One_Sided_p",
    )
)

fig = px.imshow(
    pvalue_heatmap_data,
    text_auto=".3f",
    color_continuous_scale="RdYlGn_r",
    aspect="auto",
    zmin=0,
    zmax=0.20,
    labels={
        "x": "First-stage initial window",
        "y": "Second-stage initial window",
        "color": "H3 vs H2 p-value",
    },
    title=(
        "Incremental Fed Information: "
        "H3 versus H2 Loss-Test p-values"
    ),
)

fig.update_layout(
    template="plotly_white",
    width=900,
    height=600,
)

fig.show()

Sensitivity-source observations: 60
Common evaluation begins at source-row position: 39
Common evaluation event month: 2024-01-01 00:00:00
Common evaluation outcome month: 2024-02-01 00:00:00
Common evaluation forecasts: 21

All first-stage chronology checks passed: True
All second-stage chronology checks passed: True
Window combinations evaluated: 9

H3 Fed–ETF system window sensitivity


,First_Stage_Window,Second_Stage_Window,Model,Forecasts,First_Outcome_Month,Final_Outcome_Month,RMSE,MAE,Bias,Forecast_Actual_Correlation,...,MAE_Improvement_vs_H0_pct,Mean_Squared_Loss_Improvement_H3_vs_H2,HAC_t,One_Sided_p,H3_Beats_H2_5pct,H3_Beats_H2_10pct,Minimum_H3_ETF_Coefficient,Maximum_H3_ETF_Coefficient,Median_H3_ETF_Coefficient,Positive_H3_ETF_Coefficient_Share
0,15,12,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.2848,3.4510,1.7017,0.1376,...,3.6949,2.5380,2.0135,0.0289,True,True,-0.0009,0.0266,0.0128,0.9048
1,15,15,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.2848,3.4510,1.7017,0.1376,...,3.6949,2.5380,2.0135,0.0289,True,True,-0.0009,0.0266,0.0128,0.9048
2,15,18,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.2848,3.4510,1.7017,0.1376,...,3.6949,2.5380,2.0135,0.0289,True,True,-0.0009,0.0266,0.0128,0.9048
3,21,12,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.3552,3.6709,1.4100,-0.0866,...,2.7115,1.2079,1.2010,0.1219,False,False,-0.0078,0.0262,0.0104,0.8571
4,21,15,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.3552,3.6709,1.4100,-0.0866,...,2.7115,1.2079,1.2010,0.1219,False,False,-0.0078,0.0262,0.0104,0.8571
5,21,18,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.3552,3.6709,1.4100,-0.0866,...,2.7115,1.2079,1.2010,0.1219,False,False,-0.0078,0.0262,0.0104,0.8571
6,18,12,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.4212,3.6311,1.7390,-0.0169,...,2.5210,1.9035,1.4228,0.0851,False,True,-0.0104,0.0262,0.0098,0.7619
7,18,15,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.4212,3.6311,1.7390,-0.0169,...,2.5210,1.9035,1.4228,0.0851,False,True,-0.0104,0.0262,0.0098,0.7619
8,18,18,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.4212,3.6311,1.7390,-0.0169,...,2.5210,1.9035,1.4228,0.0851,False,True,-0.0104,0.0262,0.0098,0.7619



All gold models across window combinations


,First_Stage_Window,Second_Stage_Window,Model,Forecasts,First_Outcome_Month,Final_Outcome_Month,RMSE,MAE,Bias,Forecast_Actual_Correlation,Return_Sign_Accuracy,RMSE_Improvement_vs_H0_pct,MAE_Improvement_vs_H0_pct
0,15,12,H0_Gold_Persistence,21,2024-02-01,2026-07-01,4.5489,3.5834,2.0408,-0.1199,0.6667,0.0000,0.0000
1,15,12,H1_Direct_Fed,21,2024-02-01,2026-07-01,4.2830,3.5026,1.4823,0.0508,0.6667,5.8450,2.2549
2,15,12,H2_ETF_NoFed_System,21,2024-02-01,2026-07-01,4.5714,3.6147,2.1056,-0.0621,0.6190,-0.4947,-0.8710
3,15,12,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.2848,3.4510,1.7017,0.1376,0.6667,5.8053,3.6949
4,15,15,H0_Gold_Persistence,21,2024-02-01,2026-07-01,4.5489,3.5834,2.0408,-0.1199,0.6667,0.0000,0.0000
5,15,15,H1_Direct_Fed,21,2024-02-01,2026-07-01,4.2830,3.5026,1.4823,0.0508,0.6667,5.8450,2.2549
6,15,15,H2_ETF_NoFed_System,21,2024-02-01,2026-07-01,4.5714,3.6147,2.1056,-0.0621,0.6190,-0.4947,-0.8710
7,15,15,H3_ETF_Fed_System,21,2024-02-01,2026-07-01,4.2848,3.4510,1.7017,0.1376,0.6667,5.8053,3.6949
8,15,18,H0_Gold_Persistence,21,2024-02-01,2026-07-01,4.5489,3.5834,2.0408,-0.1199,0.6667,0.0000,0.0000
9,15,18,H1_Direct_Fed,21,2024-02-01,2026-07-01,4.2830,3.5026,1.4823,0.0508,0.6667,5.8450,2.2549


In [56]:
# CELL 50: Infeasible oracle-ETF benchmark
#
# Purpose:
# Determine whether weak gold-forecast performance comes from:
# 1. first-stage ETF forecast error, or
# 2. an unstable ETF-to-gold forecasting relationship.
#
# IMPORTANT:
# The oracle uses realized ETF flow in outcome month t+1.
# It is deliberately infeasible and must never be reported as a
# deployable real-time forecasting model.

import statsmodels.api as sm
from scipy import stats
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# -----------------------------------------------------------
# Preconditions
# -----------------------------------------------------------

if "two_stage_generated_data" not in globals():
    raise NameError(
        "two_stage_generated_data is unavailable. "
        "Run Cell 48 before Cell 50."
    )

oracle_source = (
    two_stage_generated_data
    .sort_values("Event_Month")
    .reset_index(drop=True)
    .copy()
)

required_oracle_columns = [
    "Event_Month",
    "Outcome_Month",
    "Gold_Return_Next_Month",
    "Gold_Return_Previous_Month",
    "Fed_Expected_Direction",
    "Actual_ETF_Flow_Next_Month",
    "ETF_NoFed_Forecast",
    "ETF_WithFed_Forecast",
]

missing_oracle_columns = [
    column
    for column in required_oracle_columns
    if column not in oracle_source.columns
]

if missing_oracle_columns:
    raise ValueError(
        f"Required oracle columns are missing: "
        f"{missing_oracle_columns}"
    )

if oracle_source[
    required_oracle_columns
].isna().any().any():
    raise ValueError(
        "The oracle source contains missing values."
    )

# Match the baseline Cell 48 gold-forecast horizon.
oracle_second_stage_initial_window = 18

if (
    oracle_second_stage_initial_window
    >= len(oracle_source)
):
    raise ValueError(
        "Oracle second-stage window is too large."
    )

# -----------------------------------------------------------
# Competing gold equations
# -----------------------------------------------------------

oracle_specifications = {
    # Feasible persistence benchmark
    "O0_Gold_Persistence": [
        "Gold_Return_Previous_Month",
    ],

    # Feasible direct-Fed benchmark
    "O1_Direct_Fed": [
        "Gold_Return_Previous_Month",
        "Fed_Expected_Direction",
    ],

    # Feasible two-stage model using the recursively forecast ETF flow
    "O2_Forecasted_ETF": [
        "Gold_Return_Previous_Month",
        "ETF_WithFed_Forecast",
    ],

    # Infeasible oracle: realized outcome-month ETF flow
    "O3_Oracle_Realized_ETF": [
        "Gold_Return_Previous_Month",
        "Actual_ETF_Flow_Next_Month",
    ],

    # Infeasible oracle allowing both realized ETF flow and Fed direction
    "O4_Oracle_ETF_and_Fed": [
        "Gold_Return_Previous_Month",
        "Actual_ETF_Flow_Next_Month",
        "Fed_Expected_Direction",
    ],
}

oracle_forecast_rows = []

# -----------------------------------------------------------
# Recursive expanding-window forecasts
# -----------------------------------------------------------

for forecast_position in range(
    oracle_second_stage_initial_window,
    len(oracle_source)
):

    training_data = (
        oracle_source.iloc[
            :forecast_position
        ]
        .copy()
    )

    test_data = (
        oracle_source.iloc[
            [forecast_position]
        ]
        .copy()
    )

    actual_gold = float(
        test_data[
            "Gold_Return_Next_Month"
        ].iloc[0]
    )

    event_month = test_data[
        "Event_Month"
    ].iloc[0]

    record = {
        "Event_Month":
            event_month,
        "Outcome_Month":
            test_data[
                "Outcome_Month"
            ].iloc[0],
        "Training_N":
            len(training_data),
        "Final_Training_Event_Month":
            training_data[
                "Event_Month"
            ].max(),
        "Actual_Gold_Return":
            actual_gold,
        "Actual_ETF_Flow_Next_Month":
            float(
                test_data[
                    "Actual_ETF_Flow_Next_Month"
                ].iloc[0]
            ),
        "Forecasted_ETF_Flow_Next_Month":
            float(
                test_data[
                    "ETF_WithFed_Forecast"
                ].iloc[0]
            ),
    }

    y_train = training_data[
        "Gold_Return_Next_Month"
    ].astype(float)

    for model_name, predictors in (
        oracle_specifications.items()
    ):

        X_train = sm.add_constant(
            training_data[
                predictors
            ].astype(float),
            has_constant="add",
        )

        X_test = sm.add_constant(
            test_data[
                predictors
            ].astype(float),
            has_constant="add",
        ).reindex(
            columns=X_train.columns,
            fill_value=1.0,
        )

        fitted_model = sm.OLS(
            y_train,
            X_train,
        ).fit()

        forecast = float(
            fitted_model.predict(
                X_test
            ).iloc[0]
        )

        record[
            f"{model_name}_Forecast"
        ] = forecast

        record[
            f"{model_name}_Error"
        ] = actual_gold - forecast

        if (
            model_name
            == "O2_Forecasted_ETF"
        ):
            record[
                "Forecasted_ETF_Coefficient"
            ] = float(
                fitted_model.params[
                    "ETF_WithFed_Forecast"
                ]
            )

        if (
            model_name
            == "O3_Oracle_Realized_ETF"
        ):
            record[
                "Oracle_ETF_Coefficient"
            ] = float(
                fitted_model.params[
                    "Actual_ETF_Flow_Next_Month"
                ]
            )

        if (
            model_name
            == "O4_Oracle_ETF_and_Fed"
        ):
            record[
                "Oracle_Conditional_ETF_Coefficient"
            ] = float(
                fitted_model.params[
                    "Actual_ETF_Flow_Next_Month"
                ]
            )

            record[
                "Oracle_Direct_Fed_Coefficient"
            ] = float(
                fitted_model.params[
                    "Fed_Expected_Direction"
                ]
            )

    oracle_forecast_rows.append(record)

oracle_gold_forecasts = pd.DataFrame(
    oracle_forecast_rows
)

# -----------------------------------------------------------
# Chronology validation
# -----------------------------------------------------------

oracle_gold_forecasts[
    "Historical_Training_Chronology_Valid"
] = (
    oracle_gold_forecasts[
        "Final_Training_Event_Month"
    ]
    < oracle_gold_forecasts[
        "Event_Month"
    ]
)

if not oracle_gold_forecasts[
    "Historical_Training_Chronology_Valid"
].all():
    raise ValueError(
        "Historical training leakage detected."
    )

print(
    "Oracle-comparison forecasts:",
    len(oracle_gold_forecasts)
)

print(
    "First outcome month:",
    oracle_gold_forecasts[
        "Outcome_Month"
    ].min()
)

print(
    "Final outcome month:",
    oracle_gold_forecasts[
        "Outcome_Month"
    ].max()
)

print(
    "Historical training chronology passed:",
    oracle_gold_forecasts[
        "Historical_Training_Chronology_Valid"
    ].all()
)

print(
    "\nWARNING: O3 and O4 deliberately use realized "
    "future ETF flows and are infeasible oracle benchmarks."
)

# -----------------------------------------------------------
# Forecast-performance metrics
# -----------------------------------------------------------

actual_gold = oracle_gold_forecasts[
    "Actual_Gold_Return"
].to_numpy()

oracle_metric_rows = []

for model_name in oracle_specifications:

    forecast = oracle_gold_forecasts[
        f"{model_name}_Forecast"
    ].to_numpy()

    error = actual_gold - forecast

    oracle_metric_rows.append({
        "Model":
            model_name,
        "Feasible_Real_Time":
            model_name
            not in [
                "O3_Oracle_Realized_ETF",
                "O4_Oracle_ETF_and_Fed",
            ],
        "Forecasts":
            len(actual_gold),
        "RMSE":
            np.sqrt(
                np.mean(error ** 2)
            ),
        "MAE":
            np.mean(
                np.abs(error)
            ),
        "Bias":
            np.mean(error),
        "Forecast_Actual_Correlation":
            np.corrcoef(
                actual_gold,
                forecast,
            )[0, 1],
        "Return_Sign_Accuracy":
            np.mean(
                np.sign(actual_gold)
                == np.sign(forecast)
            ),
    })

oracle_gold_metrics = pd.DataFrame(
    oracle_metric_rows
)

o0_rmse = oracle_gold_metrics.loc[
    oracle_gold_metrics["Model"]
    == "O0_Gold_Persistence",
    "RMSE",
].iloc[0]

o0_mae = oracle_gold_metrics.loc[
    oracle_gold_metrics["Model"]
    == "O0_Gold_Persistence",
    "MAE",
].iloc[0]

oracle_gold_metrics[
    "RMSE_Improvement_vs_O0_pct"
] = (
    (
        o0_rmse
        - oracle_gold_metrics["RMSE"]
    )
    / o0_rmse
    * 100
)

oracle_gold_metrics[
    "MAE_Improvement_vs_O0_pct"
] = (
    (
        o0_mae
        - oracle_gold_metrics["MAE"]
    )
    / o0_mae
    * 100
)

print(
    "\nOracle and feasible forecast performance"
)

display(
    oracle_gold_metrics.round(4)
)

# -----------------------------------------------------------
# HAC loss-difference tests
#
# Positive improvement means that the alternative model has
# smaller squared forecast error than the reference model.
# -----------------------------------------------------------

oracle_comparisons = [
    {
        "Comparison":
            "Forecasted ETF vs persistence",
        "Reference":
            "O0_Gold_Persistence",
        "Alternative":
            "O2_Forecasted_ETF",
    },
    {
        "Comparison":
            "Oracle ETF vs persistence",
        "Reference":
            "O0_Gold_Persistence",
        "Alternative":
            "O3_Oracle_Realized_ETF",
    },
    {
        "Comparison":
            "Oracle ETF vs forecasted ETF",
        "Reference":
            "O2_Forecasted_ETF",
        "Alternative":
            "O3_Oracle_Realized_ETF",
    },
    {
        "Comparison":
            "Oracle ETF plus Fed vs oracle ETF",
        "Reference":
            "O3_Oracle_Realized_ETF",
        "Alternative":
            "O4_Oracle_ETF_and_Fed",
    },
    {
        "Comparison":
            "Oracle ETF plus Fed vs direct Fed",
        "Reference":
            "O1_Direct_Fed",
        "Alternative":
            "O4_Oracle_ETF_and_Fed",
    },
]

oracle_loss_rows = []

for comparison in oracle_comparisons:

    reference_error = oracle_gold_forecasts[
        f"{comparison['Reference']}_Error"
    ].to_numpy()

    alternative_error = oracle_gold_forecasts[
        f"{comparison['Alternative']}_Error"
    ].to_numpy()

    loss_improvement = (
        reference_error ** 2
        - alternative_error ** 2
    )

    loss_regression = sm.OLS(
        loss_improvement,
        np.ones(len(loss_improvement)),
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 2,
            "use_correction": True,
        },
    )

    hac_t = float(
        np.asarray(
            loss_regression.tvalues
        ).reshape(-1)[0]
    )

    one_sided_p = float(
        1 - stats.t.cdf(
            hac_t,
            df=loss_regression.df_resid,
        )
    )

    oracle_loss_rows.append({
        "Comparison":
            comparison["Comparison"],
        "Mean_Squared_Loss_Improvement":
            np.mean(loss_improvement),
        "HAC_t":
            hac_t,
        "One_Sided_p":
            one_sided_p,
        "Improvement_Significant_5pct":
            one_sided_p < 0.05,
        "Improvement_Significant_10pct":
            one_sided_p < 0.10,
    })

oracle_loss_tests = pd.DataFrame(
    oracle_loss_rows
)

print(
    "\nOracle loss-difference comparisons"
)

display(
    oracle_loss_tests.round(4)
)

# -----------------------------------------------------------
# ETF coefficient-stability comparison
# -----------------------------------------------------------

forecasted_coefficients = oracle_gold_forecasts[
    "Forecasted_ETF_Coefficient"
]

oracle_coefficients = oracle_gold_forecasts[
    "Oracle_ETF_Coefficient"
]

oracle_coefficient_comparison = pd.DataFrame({
    "ETF_Input": [
        "Recursively forecasted ETF flow",
        "Oracle realized ETF flow",
    ],
    "Minimum_Coefficient": [
        forecasted_coefficients.min(),
        oracle_coefficients.min(),
    ],
    "Maximum_Coefficient": [
        forecasted_coefficients.max(),
        oracle_coefficients.max(),
    ],
    "Median_Coefficient": [
        forecasted_coefficients.median(),
        oracle_coefficients.median(),
    ],
    "Positive_Coefficient_Share": [
        (
            forecasted_coefficients > 0
        ).mean(),
        (
            oracle_coefficients > 0
        ).mean(),
    ],
})

print(
    "\nForecasted versus oracle ETF coefficient stability"
)

display(
    oracle_coefficient_comparison.round(4)
)

# -----------------------------------------------------------
# ETF first-stage error diagnostic
# -----------------------------------------------------------

actual_etf = oracle_gold_forecasts[
    "Actual_ETF_Flow_Next_Month"
]

forecasted_etf = oracle_gold_forecasts[
    "Forecasted_ETF_Flow_Next_Month"
]

oracle_etf_input_diagnostic = pd.DataFrame({
    "Forecasts": [
        len(oracle_gold_forecasts)
    ],
    "ETF_RMSE": [
        np.sqrt(
            np.mean(
                (
                    actual_etf
                    - forecasted_etf
                ) ** 2
            )
        )
    ],
    "ETF_MAE": [
        np.mean(
            np.abs(
                actual_etf
                - forecasted_etf
            )
        )
    ],
    "ETF_Forecast_Actual_Correlation": [
        np.corrcoef(
            actual_etf,
            forecasted_etf,
        )[0, 1]
    ],
    "ETF_Sign_Accuracy": [
        np.mean(
            np.sign(actual_etf)
            == np.sign(forecasted_etf)
        )
    ],
})

print(
    "\nFirst-stage ETF quality over the gold evaluation period"
)

display(
    oracle_etf_input_diagnostic.round(4)
)

# -----------------------------------------------------------
# Plot actual and competing gold forecasts
# -----------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=oracle_gold_forecasts[
            "Outcome_Month"
        ],
        y=oracle_gold_forecasts[
            "Actual_Gold_Return"
        ],
        mode="lines+markers",
        name="Actual gold return",
        line=dict(
            color="black",
            width=3,
        ),
    )
)

plot_specifications = {
    "O0_Gold_Persistence": {
        "name": "Persistence",
        "color": "#7f7f7f",
        "dash": "dot",
    },
    "O1_Direct_Fed": {
        "name": "Direct Fed",
        "color": "#d62728",
        "dash": "dot",
    },
    "O2_Forecasted_ETF": {
        "name": "Forecasted ETF",
        "color": "#1f77b4",
        "dash": "solid",
    },
    "O3_Oracle_Realized_ETF": {
        "name": "Oracle realized ETF",
        "color": "#2ca02c",
        "dash": "dash",
    },
}

for model_name, plotting_details in (
    plot_specifications.items()
):

    fig.add_trace(
        go.Scatter(
            x=oracle_gold_forecasts[
                "Outcome_Month"
            ],
            y=oracle_gold_forecasts[
                f"{model_name}_Forecast"
            ],
            mode="lines+markers",
            name=plotting_details["name"],
            line=dict(
                color=plotting_details["color"],
                dash=plotting_details["dash"],
                width=2,
            ),
        )
    )

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title=(
        "Feasible and Oracle Gold-Return Forecasts"
    ),
    subtitle=dict(
        text=(
            "The oracle uses realized next-month ETF flow "
            "and is not deployable"
        )
    ),
    template="plotly_white",
    width=1400,
    height=700,
    xaxis_title="Gold-return outcome month",
    yaxis_title="Gold monthly return (%)",
)

fig.show()

Oracle-comparison forecasts: 24
First outcome month: 2023-10-01 00:00:00
Final outcome month: 2026-07-01 00:00:00
Historical training chronology passed: True


Oracle and feasible forecast performance


,Model,Feasible_Real_Time,Forecasts,RMSE,MAE,Bias,Forecast_Actual_Correlation,Return_Sign_Accuracy,RMSE_Improvement_vs_O0_pct,MAE_Improvement_vs_O0_pct
0,O0_Gold_Persistence,True,24,4.3241,3.3658,1.7424,-0.1485,0.6667,0.0000,0.0000
1,O1_Direct_Fed,True,24,4.1421,3.3196,1.3804,0.0565,0.5833,4.2079,1.3723
2,O2_Forecasted_ETF,True,24,4.1695,3.3164,1.6354,0.1016,0.5833,3.5739,1.4671
3,O3_Oracle_Realized_ETF,False,24,3.4199,2.6817,0.6679,0.4757,0.6250,20.9101,20.3264
4,O4_Oracle_ETF_and_Fed,False,24,3.7562,3.0184,1.1683,0.4177,0.5833,13.1319,10.3213



Oracle loss-difference comparisons


,Comparison,Mean_Squared_Loss_Improvement,HAC_t,One_Sided_p,Improvement_Significant_5pct,Improvement_Significant_10pct
0,Forecasted ETF vs persistence,1.3126,1.0583,0.1505,False,False
1,Oracle ETF vs persistence,7.0018,1.9023,0.0349,True,True
2,Oracle ETF vs forecasted ETF,5.6892,2.0570,0.0256,True,True
3,Oracle ETF plus Fed vs oracle ETF,-2.4136,-1.6141,0.9399,False,False
4,Oracle ETF plus Fed vs direct Fed,3.0478,0.9087,0.1865,False,False



Forecasted versus oracle ETF coefficient stability


,ETF_Input,Minimum_Coefficient,Maximum_Coefficient,Median_Coefficient,Positive_Coefficient_Share
0,Recursively forecasted ETF flow,-0.0134,0.0262,0.0063,0.6667
1,Oracle realized ETF flow,0.0291,0.0508,0.0367,1.0000



First-stage ETF quality over the gold evaluation period


,Forecasts,ETF_RMSE,ETF_MAE,ETF_Forecast_Actual_Correlation,ETF_Sign_Accuracy
0,24,42.9063,35.0221,0.4518,0.8333


ValueError: Invalid property specified for object of type plotly.graph_objs.Layout: 'subtitle'

Did you mean "title"?

    Valid properties:
        activeselection
            :class:`plotly.graph_objects.layout.Activeselection`
            instance or dict with compatible properties
        activeshape
            :class:`plotly.graph_objects.layout.Activeshape`
            instance or dict with compatible properties
        annotations
            A tuple of
            :class:`plotly.graph_objects.layout.Annotation`
            instances or dicts with compatible properties
        annotationdefaults
            When used in a template (as
            layout.template.layout.annotationdefaults), sets the
            default property values to use for elements of
            layout.annotations
        autosize
            Determines whether or not a layout width or height that
            has been left undefined by the user is initialized on
            each relayout. Note that, regar

In [57]:
# CELL 50A: Corrected oracle forecast chart
# The installed Plotly version does not support layout.subtitle.

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=oracle_gold_forecasts["Outcome_Month"],
        y=oracle_gold_forecasts["Actual_Gold_Return"],
        mode="lines+markers",
        name="Actual gold return",
        line=dict(
            color="black",
            width=3,
        ),
    )
)

plot_specifications = {
    "O0_Gold_Persistence": {
        "name": "Persistence",
        "color": "#7f7f7f",
        "dash": "dot",
    },
    "O1_Direct_Fed": {
        "name": "Direct Fed",
        "color": "#d62728",
        "dash": "dot",
    },
    "O2_Forecasted_ETF": {
        "name": "Forecasted ETF",
        "color": "#1f77b4",
        "dash": "solid",
    },
    "O3_Oracle_Realized_ETF": {
        "name": "Oracle realized ETF",
        "color": "#2ca02c",
        "dash": "dash",
    },
}

for model_name, details in plot_specifications.items():

    fig.add_trace(
        go.Scatter(
            x=oracle_gold_forecasts["Outcome_Month"],
            y=oracle_gold_forecasts[
                f"{model_name}_Forecast"
            ],
            mode="lines+markers",
            name=details["name"],
            line=dict(
                color=details["color"],
                dash=details["dash"],
                width=2,
            ),
        )
    )

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title={
        "text": (
            "Feasible and Oracle Gold-Return Forecasts"
            "<br><sup>"
            "The oracle uses realized next-month ETF flow "
            "and is not deployable"
            "</sup>"
        ),
        "x": 0.5,
    },
    template="plotly_white",
    width=1400,
    height=700,
    xaxis_title="Gold-return outcome month",
    yaxis_title="Gold monthly return (%)",
    legend_title="Model",
    hovermode="x unified",
)

fig.show()

In [58]:
# CELL 51: Diagnose first-stage ETF magnitude-forecast errors
#
# This cell determines whether the forecasting weakness is caused by:
# - extreme ETF-flow observations;
# - asymmetric tightening/easing regimes;
# - systematic shrinkage toward the mean;
# - structural periods such as COVID or post-2022;
# - poor magnitude calibration despite good sign accuracy.

import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import (
    het_breuschpagan,
)
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# -----------------------------------------------------------
# Preconditions
# -----------------------------------------------------------

if "two_stage_generated_data" not in globals():
    raise NameError(
        "two_stage_generated_data is unavailable. "
        "Run Cell 48 before Cell 51."
    )

etf_error_df = (
    two_stage_generated_data
    .sort_values("Outcome_Month")
    .reset_index(drop=True)
    .copy()
)

required_columns = [
    "Event_Month",
    "Outcome_Month",
    "Actual_ETF_Flow_Next_Month",
    "ETF_NoFed_Forecast",
    "ETF_WithFed_Forecast",
    "Fed_Expected_Direction",
]

missing_columns = [
    column
    for column in required_columns
    if column not in etf_error_df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

# -----------------------------------------------------------
# Construct forecast errors and regimes
# -----------------------------------------------------------

etf_error_df["NoFed_Error"] = (
    etf_error_df["Actual_ETF_Flow_Next_Month"]
    - etf_error_df["ETF_NoFed_Forecast"]
)

etf_error_df["WithFed_Error"] = (
    etf_error_df["Actual_ETF_Flow_Next_Month"]
    - etf_error_df["ETF_WithFed_Forecast"]
)

etf_error_df["Absolute_NoFed_Error"] = (
    etf_error_df["NoFed_Error"].abs()
)

etf_error_df["Absolute_WithFed_Error"] = (
    etf_error_df["WithFed_Error"].abs()
)

etf_error_df["Squared_NoFed_Error"] = (
    etf_error_df["NoFed_Error"] ** 2
)

etf_error_df["Squared_WithFed_Error"] = (
    etf_error_df["WithFed_Error"] ** 2
)

etf_error_df["Actual_ETF_Absolute_Magnitude"] = (
    etf_error_df[
        "Actual_ETF_Flow_Next_Month"
    ].abs()
)

etf_error_df["Actual_ETF_Sign"] = np.select(
    [
        etf_error_df[
            "Actual_ETF_Flow_Next_Month"
        ] > 0,
        etf_error_df[
            "Actual_ETF_Flow_Next_Month"
        ] < 0,
    ],
    [
        "Inflow",
        "Outflow",
    ],
    default="Zero",
)

etf_error_df["Fed_Regime"] = np.select(
    [
        etf_error_df[
            "Fed_Expected_Direction"
        ] > 0.05,
        etf_error_df[
            "Fed_Expected_Direction"
        ] < -0.05,
    ],
    [
        "Expected tightening",
        "Expected easing",
    ],
    default="Approximately neutral",
)

etf_error_df["COVID_Period"] = np.where(
    etf_error_df["Event_Month"].between(
        pd.Timestamp("2020-03-01"),
        pd.Timestamp("2020-06-01"),
    ),
    "COVID shock",
    "Outside COVID shock",
)

etf_error_df["Post_2022_Period"] = np.where(
    etf_error_df["Event_Month"]
    >= pd.Timestamp("2022-01-01"),
    "2022 onward",
    "Before 2022",
)

# Tail threshold is calculated over the recursive forecast sample.
tail_threshold = etf_error_df[
    "Actual_ETF_Absolute_Magnitude"
].quantile(0.75)

etf_error_df["ETF_Magnitude_Group"] = np.where(
    etf_error_df[
        "Actual_ETF_Absolute_Magnitude"
    ] >= tail_threshold,
    "Largest 25% of absolute flows",
    "Remaining 75% of flows",
)

print(
    "Recursive ETF forecasts:",
    len(etf_error_df)
)

print(
    "Forecast period:",
    etf_error_df["Outcome_Month"].min(),
    "to",
    etf_error_df["Outcome_Month"].max()
)

print(
    "Large-flow threshold:",
    round(tail_threshold, 4),
    "tonnes"
)

# -----------------------------------------------------------
# Overall comparison
# -----------------------------------------------------------

overall_rows = []

for forecast_column, error_column, label in [
    (
        "ETF_NoFed_Forecast",
        "NoFed_Error",
        "Without Fed direction",
    ),
    (
        "ETF_WithFed_Forecast",
        "WithFed_Error",
        "With Fed direction",
    ),
]:

    actual = etf_error_df[
        "Actual_ETF_Flow_Next_Month"
    ].to_numpy()

    forecast = etf_error_df[
        forecast_column
    ].to_numpy()

    error = etf_error_df[
        error_column
    ].to_numpy()

    overall_rows.append({
        "Model":
            label,
        "N":
            len(error),
        "RMSE":
            np.sqrt(np.mean(error ** 2)),
        "MAE":
            np.mean(np.abs(error)),
        "Median_Absolute_Error":
            np.median(np.abs(error)),
        "Bias":
            np.mean(error),
        "Forecast_Actual_Correlation":
            np.corrcoef(actual, forecast)[0, 1],
        "Sign_Accuracy":
            np.mean(
                np.sign(actual)
                == np.sign(forecast)
            ),
        "Actual_Standard_Deviation":
            np.std(actual, ddof=1),
        "Forecast_Standard_Deviation":
            np.std(forecast, ddof=1),
        "Forecast_to_Actual_SD_Ratio":
            (
                np.std(forecast, ddof=1)
                / np.std(actual, ddof=1)
            ),
    })

etf_error_overall = pd.DataFrame(overall_rows)

print("\nOverall first-stage performance")

display(
    etf_error_overall.round(4)
)

# -----------------------------------------------------------
# Calibration regressions
#
# Ideal calibration:
# intercept = 0 and slope = 1.
# A slope materially above 1 suggests that forecast magnitude
# is excessively compressed toward its mean.
# -----------------------------------------------------------

calibration_rows = []

for forecast_column, label in [
    (
        "ETF_NoFed_Forecast",
        "Without Fed direction",
    ),
    (
        "ETF_WithFed_Forecast",
        "With Fed direction",
    ),
]:

    X_calibration = sm.add_constant(
        etf_error_df[
            [forecast_column]
        ].astype(float),
        has_constant="add",
    )

    calibration_model = sm.OLS(
        etf_error_df[
            "Actual_ETF_Flow_Next_Month"
        ].astype(float),
        X_calibration,
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 2,
            "use_correction": True,
        },
    )

    intercept = float(
        calibration_model.params["const"]
    )

    slope = float(
        calibration_model.params[
            forecast_column
        ]
    )

    slope_minus_one_test = (
        calibration_model.t_test(
            f"{forecast_column} = 1"
        )
    )

    calibration_rows.append({
        "Model":
            label,
        "Calibration_Intercept":
            intercept,
        "Intercept_HAC_p":
            float(
                calibration_model.pvalues[
                    "const"
                ]
            ),
        "Calibration_Slope":
            slope,
        "Slope_HAC_p":
            float(
                calibration_model.pvalues[
                    forecast_column
                ]
            ),
        "Slope_Equals_One_t":
            float(
                np.asarray(
                    slope_minus_one_test.tvalue
                ).reshape(-1)[0]
            ),
        "Slope_Equals_One_p":
            float(
                np.asarray(
                    slope_minus_one_test.pvalue
                ).reshape(-1)[0]
            ),
        "Calibration_R2":
            calibration_model.rsquared,
    })

etf_calibration_results = pd.DataFrame(
    calibration_rows
)

print("\nETF forecast calibration")

display(
    etf_calibration_results.round(4)
)

# -----------------------------------------------------------
# Error performance by economically relevant regimes
# -----------------------------------------------------------

def grouped_forecast_diagnostics(
    dataframe,
    group_variable,
):
    rows = []

    for group_value, group in dataframe.groupby(
        group_variable,
        observed=True,
    ):

        actual = group[
            "Actual_ETF_Flow_Next_Month"
        ].to_numpy()

        no_fed_error = group[
            "NoFed_Error"
        ].to_numpy()

        with_fed_error = group[
            "WithFed_Error"
        ].to_numpy()

        rows.append({
            "Grouping_Variable":
                group_variable,
            "Group":
                group_value,
            "N":
                len(group),
            "Mean_Actual_ETF_Flow":
                np.mean(actual),
            "Mean_Absolute_Actual_Flow":
                np.mean(np.abs(actual)),
            "NoFed_RMSE":
                np.sqrt(
                    np.mean(no_fed_error ** 2)
                ),
            "WithFed_RMSE":
                np.sqrt(
                    np.mean(with_fed_error ** 2)
                ),
            "Fed_RMSE_Improvement_pct":
                (
                    np.sqrt(
                        np.mean(no_fed_error ** 2)
                    )
                    - np.sqrt(
                        np.mean(with_fed_error ** 2)
                    )
                )
                / np.sqrt(
                    np.mean(no_fed_error ** 2)
                )
                * 100,
            "NoFed_MAE":
                np.mean(
                    np.abs(no_fed_error)
                ),
            "WithFed_MAE":
                np.mean(
                    np.abs(with_fed_error)
                ),
            "Fed_MAE_Improvement_pct":
                (
                    np.mean(
                        np.abs(no_fed_error)
                    )
                    - np.mean(
                        np.abs(with_fed_error)
                    )
                )
                / np.mean(
                    np.abs(no_fed_error)
                )
                * 100,
            "WithFed_Bias":
                np.mean(with_fed_error),
            "WithFed_Sign_Accuracy":
                np.mean(
                    np.sign(actual)
                    == np.sign(
                        group[
                            "ETF_WithFed_Forecast"
                        ].to_numpy()
                    )
                ),
        })

    return pd.DataFrame(rows)

regime_tables = []

for grouping_variable in [
    "ETF_Magnitude_Group",
    "Actual_ETF_Sign",
    "Fed_Regime",
    "COVID_Period",
    "Post_2022_Period",
]:
    regime_tables.append(
        grouped_forecast_diagnostics(
            etf_error_df,
            grouping_variable,
        )
    )

etf_regime_diagnostics = pd.concat(
    regime_tables,
    ignore_index=True,
)

print("\nForecast performance across regimes")

display(
    etf_regime_diagnostics.round(4)
)

# -----------------------------------------------------------
# Concentration of total squared error
# -----------------------------------------------------------

etf_error_ranked = (
    etf_error_df
    .sort_values(
        "Squared_WithFed_Error",
        ascending=False,
    )
    .reset_index(drop=True)
    .copy()
)

etf_error_ranked[
    "Cumulative_Squared_Error_Share"
] = (
    etf_error_ranked[
        "Squared_WithFed_Error"
    ].cumsum()
    / etf_error_ranked[
        "Squared_WithFed_Error"
    ].sum()
)

total_squared_error = etf_error_ranked[
    "Squared_WithFed_Error"
].sum()

largest_3_error_share = (
    etf_error_ranked[
        "Squared_WithFed_Error"
    ].head(3).sum()
    / total_squared_error
)

largest_5_error_share = (
    etf_error_ranked[
        "Squared_WithFed_Error"
    ].head(5).sum()
    / total_squared_error
)

largest_10_error_share = (
    etf_error_ranked[
        "Squared_WithFed_Error"
    ].head(10).sum()
    / total_squared_error
)

error_concentration = pd.DataFrame({
    "Quantity": [
        "Largest 3 errors' share of total squared error",
        "Largest 5 errors' share of total squared error",
        "Largest 10 errors' share of total squared error",
    ],
    "Share": [
        largest_3_error_share,
        largest_5_error_share,
        largest_10_error_share,
    ],
})

print("\nSquared-error concentration")

display(
    error_concentration.round(4)
)

print("\nLargest first-stage ETF forecast errors")

display(
    etf_error_ranked[
        [
            "Event_Month",
            "Outcome_Month",
            "Actual_ETF_Flow_Next_Month",
            "ETF_WithFed_Forecast",
            "WithFed_Error",
            "Absolute_WithFed_Error",
            "Fed_Expected_Direction",
            "Fed_Regime",
            "COVID_Period",
            "Post_2022_Period",
            "Cumulative_Squared_Error_Share",
        ]
    ].head(12).round(4)
)

# -----------------------------------------------------------
# Test whether absolute error increases with actual magnitude
# -----------------------------------------------------------

magnitude_error_X = sm.add_constant(
    etf_error_df[
        ["Actual_ETF_Absolute_Magnitude"]
    ].astype(float),
    has_constant="add",
)

magnitude_error_model = sm.OLS(
    etf_error_df[
        "Absolute_WithFed_Error"
    ].astype(float),
    magnitude_error_X,
).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 2,
        "use_correction": True,
    },
)

magnitude_error_test = pd.DataFrame({
    "Test": [
        "Absolute forecast error increases with "
        "absolute realized ETF flow"
    ],
    "Coefficient": [
        magnitude_error_model.params[
            "Actual_ETF_Absolute_Magnitude"
        ]
    ],
    "HAC_SE": [
        magnitude_error_model.bse[
            "Actual_ETF_Absolute_Magnitude"
        ]
    ],
    "HAC_t": [
        magnitude_error_model.tvalues[
            "Actual_ETF_Absolute_Magnitude"
        ]
    ],
    "HAC_p": [
        magnitude_error_model.pvalues[
            "Actual_ETF_Absolute_Magnitude"
        ]
    ],
})

print("\nMagnitude-error relationship")

display(
    magnitude_error_test.round(4)
)

# -----------------------------------------------------------
# Breusch-Pagan test for forecast-error heteroskedasticity
# -----------------------------------------------------------

bp_X = sm.add_constant(
    etf_error_df[
        [
            "ETF_WithFed_Forecast",
            "Fed_Expected_Direction",
        ]
    ].astype(float),
    has_constant="add",
)

bp_result = het_breuschpagan(
    etf_error_df[
        "WithFed_Error"
    ].astype(float),
    bp_X,
)

etf_error_heteroskedasticity = pd.DataFrame({
    "Test": [
        "Breusch-Pagan LM",
        "Breusch-Pagan F",
    ],
    "Statistic": [
        bp_result[0],
        bp_result[2],
    ],
    "p_value": [
        bp_result[1],
        bp_result[3],
    ],
    "Reject_5pct": [
        bp_result[1] < 0.05,
        bp_result[3] < 0.05,
    ],
})

print("\nForecast-error heteroskedasticity")

display(
    etf_error_heteroskedasticity.round(4)
)

# -----------------------------------------------------------
# Actual-versus-forecast scatter
# -----------------------------------------------------------

fig = px.scatter(
    etf_error_df,
    x="ETF_WithFed_Forecast",
    y="Actual_ETF_Flow_Next_Month",
    color="Fed_Regime",
    size="Absolute_WithFed_Error",
    hover_data=[
        "Event_Month",
        "Outcome_Month",
        "WithFed_Error",
    ],
    title=(
        "Calibration of Recursive ETF-Flow Forecasts"
    ),
    labels={
        "ETF_WithFed_Forecast":
            "Forecasted next-month ETF flow",
        "Actual_ETF_Flow_Next_Month":
            "Realized next-month ETF flow",
    },
    template="plotly_white",
)

axis_min = min(
    etf_error_df[
        "ETF_WithFed_Forecast"
    ].min(),
    etf_error_df[
        "Actual_ETF_Flow_Next_Month"
    ].min(),
)

axis_max = max(
    etf_error_df[
        "ETF_WithFed_Forecast"
    ].max(),
    etf_error_df[
        "Actual_ETF_Flow_Next_Month"
    ].max(),
)

fig.add_trace(
    go.Scatter(
        x=[axis_min, axis_max],
        y=[axis_min, axis_max],
        mode="lines",
        name="Perfect calibration",
        line=dict(
            color="black",
            dash="dash",
        ),
    )
)

fig.update_layout(
    width=1000,
    height=700,
)

fig.show()

# -----------------------------------------------------------
# Actual, forecast and errors through time
# -----------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=etf_error_df["Outcome_Month"],
        y=etf_error_df[
            "Actual_ETF_Flow_Next_Month"
        ],
        mode="lines+markers",
        name="Actual ETF flow",
        line=dict(
            color="black",
            width=3,
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=etf_error_df["Outcome_Month"],
        y=etf_error_df[
            "ETF_WithFed_Forecast"
        ],
        mode="lines+markers",
        name="ETF forecast with Fed direction",
        line=dict(
            color="#1f77b4",
            width=2,
        ),
    )
)

fig.add_trace(
    go.Bar(
        x=etf_error_df["Outcome_Month"],
        y=etf_error_df["WithFed_Error"],
        name="Forecast error",
        marker_color="rgba(214,39,40,0.35)",
        yaxis="y2",
    )
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title=(
        "Recursive ETF Forecasts and Magnitude Errors"
    ),
    template="plotly_white",
    width=1400,
    height=750,
    xaxis_title="ETF-flow outcome month",
    yaxis=dict(
        title="ETF flow (tonnes)",
    ),
    yaxis2=dict(
        title="Forecast error (tonnes)",
        overlaying="y",
        side="right",
        showgrid=False,
    ),
    hovermode="x unified",
)

fig.show()

Recursive ETF forecasts: 42
Forecast period: 2021-07-01 00:00:00 to 2026-07-01 00:00:00
Large-flow threshold: 50.475 tonnes

Overall first-stage performance


,Model,N,RMSE,MAE,Median_Absolute_Error,Bias,Forecast_Actual_Correlation,Sign_Accuracy,Actual_Standard_Deviation,Forecast_Standard_Deviation,Forecast_to_Actual_SD_Ratio
0,Without Fed direction,42,43.5281,33.7028,27.7412,-10.0685,0.4666,0.6429,47.7134,29.7619,0.6238
1,With Fed direction,42,43.5321,35.0872,31.0405,-1.6011,0.5845,0.8095,47.7134,48.8642,1.0241



ETF forecast calibration


,Model,Calibration_Intercept,Intercept_HAC_p,Calibration_Slope,Slope_HAC_p,Slope_Equals_One_t,Slope_Equals_One_p,Calibration_R2
0,Without Fed direction,-5.0320,0.5285,0.7481,0.0,-1.5289,0.1263,0.2177
1,With Fed direction,3.3463,0.6617,0.5708,0.0,-3.2467,0.0012,0.3417



Forecast performance across regimes


,Grouping_Variable,Group,N,Mean_Actual_ETF_Flow,Mean_Absolute_Actual_Flow,NoFed_RMSE,WithFed_RMSE,Fed_RMSE_Improvement_pct,NoFed_MAE,WithFed_MAE,Fed_MAE_Improvement_pct,WithFed_Bias,WithFed_Sign_Accuracy
0,ETF_Magnitude_Group,Largest 25% of absolute flows,11,19.9909,74.1364,61.4925,41.7478,32.1091,49.2671,35.1598,28.6343,5.7388,0.9091
1,ETF_Magnitude_Group,Remaining 75% of flows,31,6.3532,28.1790,35.0036,44.1479,-26.1239,28.1800,35.0614,-24.4195,-4.2056,0.7742
2,Actual_ETF_Sign,Inflow,24,43.8729,43.8729,34.3616,39.3197,-14.4290,25.6415,32.7034,-27.5409,6.9895,0.9167
3,Actual_ETF_Sign,Outflow,18,-35.3389,35.3389,53.3542,48.5838,8.9411,44.4513,38.2656,13.9156,-13.0553,0.6667
4,Fed_Regime,Approximately neutral,8,50.1125,50.1125,23.9548,23.5828,1.5528,18.4736,16.9949,8.0044,16.8313,1.0000
5,Fed_Regime,Expected easing,6,53.7833,53.7833,52.6591,55.9337,-6.2184,37.5337,52.4211,-39.6644,-19.0983,1.0000
6,Fed_Regime,Expected tightening,28,-10.9554,34.4803,45.6496,44.8693,1.7093,37.2331,36.5420,1.8562,-3.1181,0.7143
7,COVID_Period,Outside COVID shock,42,9.9250,40.2155,43.5281,43.5321,-0.0090,33.7028,35.0872,-4.1076,-1.6011,0.8095
8,Post_2022_Period,2022 onward,37,11.1797,42.6338,42.5306,45.4134,-6.7782,32.0637,36.6077,-14.1717,0.0333,0.8378
9,Post_2022_Period,Before 2022,5,0.6400,22.3200,50.2988,25.6271,49.0503,45.8320,23.8354,47.9940,-13.6960,0.6000


... (4 outputs left out)


Magnitude-error relationship


,Test,Coefficient,HAC_SE,HAC_t,HAC_p
0,Absolute forecast error increases with absolut...,0.0028,0.1423,0.02,0.984



Forecast-error heteroskedasticity


,Test,Statistic,p_value,Reject_5pct
0,Breusch-Pagan LM,7.0618,0.0293,True
1,Breusch-Pagan F,3.9414,0.0276,True


In [59]:
# CELL 52: Recursive ETF magnitude-model tournament
#
# Models:
# B0: Full OLS baseline
# B1: Parsimonious OLS
# B2: Full Ridge with nested time-series tuning
# B3: Full Huber robust regression
# B4: Asymmetric Ridge with separate tightening/easing signals
#
# All forecasts are generated recursively with no future leakage.

import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# -----------------------------------------------------------
# Preconditions
# -----------------------------------------------------------

if "two_stage_system_df" not in globals():
    raise NameError(
        "two_stage_system_df is unavailable. "
        "Run Cells 44–48 before Cell 52."
    )

model_source = (
    two_stage_system_df
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

required_columns = [
    "Month",
    "ETF_Flow_Next_Month",
    "Fed_Expected_Direction",
    "Gold_Return_Previous_Month",
    "ETF_Flow_Previous_Month",
    "Broad_USD_Change_Previous_Month",
    "Real10Y_Change_Previous_Month",
    "VIX_Change_Previous_Month",
]

missing_columns = [
    column
    for column in required_columns
    if column not in model_source.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if model_source[required_columns].isna().any().any():
    raise ValueError(
        "Model source contains missing values."
    )

# -----------------------------------------------------------
# Asymmetric Fed-direction variables
# -----------------------------------------------------------

model_source["Expected_Tightening"] = (
    model_source[
        "Fed_Expected_Direction"
    ].clip(lower=0)
)

model_source["Expected_Easing"] = (
    -model_source[
        "Fed_Expected_Direction"
    ].clip(upper=0)
)

# Positive Expected_Easing means greater expected easing.
# Therefore, tightening and easing coefficients need not be
# constrained to be negatives of one another.

# -----------------------------------------------------------
# Predictor sets
# -----------------------------------------------------------

full_predictors = [
    "Fed_Expected_Direction",
    "Gold_Return_Previous_Month",
    "ETF_Flow_Previous_Month",
    "Broad_USD_Change_Previous_Month",
    "Real10Y_Change_Previous_Month",
    "VIX_Change_Previous_Month",
]

parsimonious_predictors = [
    "Fed_Expected_Direction",
    "ETF_Flow_Previous_Month",
]

asymmetric_predictors = [
    "Expected_Tightening",
    "Expected_Easing",
    "Gold_Return_Previous_Month",
    "ETF_Flow_Previous_Month",
    "Broad_USD_Change_Previous_Month",
    "Real10Y_Change_Previous_Month",
    "VIX_Change_Previous_Month",
]

initial_window = 18

if initial_window <= len(asymmetric_predictors) + 2:
    raise ValueError(
        "Initial window is too small for the asymmetric model."
    )

ridge_alpha_grid = np.array([
    0.01,
    0.10,
    1.00,
    10.00,
    100.00,
])

# -----------------------------------------------------------
# Expanding inner validation for Ridge alpha
# -----------------------------------------------------------

def choose_ridge_alpha(
    X,
    y,
    alpha_grid,
):
    """
    Select Ridge alpha using expanding-window one-step-ahead
    validation entirely inside the current outer training set.
    """

    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    n_observations = len(X)
    inner_initial = max(
        len(X.columns) + 3,
        int(np.floor(n_observations * 0.55)),
    )

    if inner_initial >= n_observations:
        return 10.0

    alpha_losses = {}

    for alpha in alpha_grid:

        squared_errors = []

        for inner_position in range(
            inner_initial,
            n_observations
        ):

            X_inner_train = X.iloc[
                :inner_position
            ]

            y_inner_train = y.iloc[
                :inner_position
            ]

            X_inner_test = X.iloc[
                [inner_position]
            ]

            inner_pipeline = Pipeline([
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "regressor",
                    Ridge(alpha=float(alpha)),
                ),
            ])

            inner_pipeline.fit(
                X_inner_train,
                y_inner_train,
            )

            inner_prediction = float(
                inner_pipeline.predict(
                    X_inner_test
                )[0]
            )

            inner_error = (
                float(y.iloc[inner_position])
                - inner_prediction
            )

            squared_errors.append(
                inner_error ** 2
            )

        alpha_losses[float(alpha)] = (
            np.mean(squared_errors)
        )

    return min(
        alpha_losses,
        key=alpha_losses.get,
    )

# -----------------------------------------------------------
# Recursive model tournament
# -----------------------------------------------------------

forecast_rows = []

for forecast_position in range(
    initial_window,
    len(model_source)
):

    train = model_source.iloc[
        :forecast_position
    ].copy()

    test = model_source.iloc[
        [forecast_position]
    ].copy()

    y_train = train[
        "ETF_Flow_Next_Month"
    ].astype(float)

    actual_etf = float(
        test[
            "ETF_Flow_Next_Month"
        ].iloc[0]
    )

    event_month = test[
        "Month"
    ].iloc[0]

    record = {
        "Event_Month":
            event_month,
        "Outcome_Month":
            event_month
            + pd.offsets.MonthBegin(1),
        "Training_N":
            len(train),
        "Final_Training_Month":
            train["Month"].max(),
        "Actual_ETF_Flow":
            actual_etf,
    }

    # -------------------------------------------------------
    # B0: Full OLS baseline
    # -------------------------------------------------------

    X_b0_train = sm.add_constant(
        train[full_predictors].astype(float),
        has_constant="add",
    )

    X_b0_test = sm.add_constant(
        test[full_predictors].astype(float),
        has_constant="add",
    ).reindex(
        columns=X_b0_train.columns,
        fill_value=1.0,
    )

    b0_model = sm.OLS(
        y_train,
        X_b0_train,
    ).fit()

    record["B0_Full_OLS_Forecast"] = float(
        b0_model.predict(
            X_b0_test
        ).iloc[0]
    )

    record["B0_Direction_Coefficient"] = float(
        b0_model.params[
            "Fed_Expected_Direction"
        ]
    )

    # -------------------------------------------------------
    # B1: Parsimonious OLS
    # -------------------------------------------------------

    X_b1_train = sm.add_constant(
        train[
            parsimonious_predictors
        ].astype(float),
        has_constant="add",
    )

    X_b1_test = sm.add_constant(
        test[
            parsimonious_predictors
        ].astype(float),
        has_constant="add",
    ).reindex(
        columns=X_b1_train.columns,
        fill_value=1.0,
    )

    b1_model = sm.OLS(
        y_train,
        X_b1_train,
    ).fit()

    record[
        "B1_Parsimonious_OLS_Forecast"
    ] = float(
        b1_model.predict(
            X_b1_test
        ).iloc[0]
    )

    record["B1_Direction_Coefficient"] = float(
        b1_model.params[
            "Fed_Expected_Direction"
        ]
    )

    # -------------------------------------------------------
    # B2: Full Ridge
    # -------------------------------------------------------

    X_b2_train = (
        train[full_predictors]
        .astype(float)
        .reset_index(drop=True)
    )

    X_b2_test = (
        test[full_predictors]
        .astype(float)
    )

    selected_b2_alpha = choose_ridge_alpha(
        X_b2_train,
        y_train.reset_index(drop=True),
        ridge_alpha_grid,
    )

    b2_pipeline = Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "regressor",
            Ridge(
                alpha=selected_b2_alpha
            ),
        ),
    ])

    b2_pipeline.fit(
        X_b2_train,
        y_train,
    )

    record["B2_Full_Ridge_Forecast"] = float(
        b2_pipeline.predict(
            X_b2_test
        )[0]
    )

    record["B2_Selected_Alpha"] = (
        selected_b2_alpha
    )

    b2_direction_position = (
        full_predictors.index(
            "Fed_Expected_Direction"
        )
    )

    record[
        "B2_Standardized_Direction_Coefficient"
    ] = float(
        b2_pipeline.named_steps[
            "regressor"
        ].coef_[b2_direction_position]
    )

    # -------------------------------------------------------
    # B3: Full Huber robust regression
    # -------------------------------------------------------

    X_b3_train = (
        train[full_predictors]
        .astype(float)
    )

    X_b3_test = (
        test[full_predictors]
        .astype(float)
    )

    b3_pipeline = Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "regressor",
            HuberRegressor(
                epsilon=1.35,
                alpha=0.01,
                max_iter=10000,
            ),
        ),
    ])

    b3_pipeline.fit(
        X_b3_train,
        y_train,
    )

    record["B3_Full_Huber_Forecast"] = float(
        b3_pipeline.predict(
            X_b3_test
        )[0]
    )

    record[
        "B3_Standardized_Direction_Coefficient"
    ] = float(
        b3_pipeline.named_steps[
            "regressor"
        ].coef_[b2_direction_position]
    )

    # -------------------------------------------------------
    # B4: Asymmetric Ridge
    # -------------------------------------------------------

    X_b4_train = (
        train[asymmetric_predictors]
        .astype(float)
        .reset_index(drop=True)
    )

    X_b4_test = (
        test[asymmetric_predictors]
        .astype(float)
    )

    selected_b4_alpha = choose_ridge_alpha(
        X_b4_train,
        y_train.reset_index(drop=True),
        ridge_alpha_grid,
    )

    b4_pipeline = Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "regressor",
            Ridge(
                alpha=selected_b4_alpha
            ),
        ),
    ])

    b4_pipeline.fit(
        X_b4_train,
        y_train,
    )

    record[
        "B4_Asymmetric_Ridge_Forecast"
    ] = float(
        b4_pipeline.predict(
            X_b4_test
        )[0]
    )

    record["B4_Selected_Alpha"] = (
        selected_b4_alpha
    )

    b4_coefficients = (
        b4_pipeline.named_steps[
            "regressor"
        ].coef_
    )

    record[
        "B4_Standardized_Tightening_Coefficient"
    ] = float(
        b4_coefficients[
            asymmetric_predictors.index(
                "Expected_Tightening"
            )
        ]
    )

    record[
        "B4_Standardized_Easing_Coefficient"
    ] = float(
        b4_coefficients[
            asymmetric_predictors.index(
                "Expected_Easing"
            )
        ]
    )

    forecast_rows.append(record)

etf_magnitude_forecasts = pd.DataFrame(
    forecast_rows
)

# -----------------------------------------------------------
# No-leakage check
# -----------------------------------------------------------

etf_magnitude_forecasts[
    "Chronology_Valid"
] = (
    etf_magnitude_forecasts[
        "Final_Training_Month"
    ]
    < etf_magnitude_forecasts[
        "Event_Month"
    ]
)

if not etf_magnitude_forecasts[
    "Chronology_Valid"
].all():
    raise ValueError(
        "Future-data leakage detected."
    )

print(
    "Recursive forecasts:",
    len(etf_magnitude_forecasts)
)

print(
    "Outcome period:",
    etf_magnitude_forecasts[
        "Outcome_Month"
    ].min(),
    "to",
    etf_magnitude_forecasts[
        "Outcome_Month"
    ].max()
)

print(
    "All chronology checks passed:",
    etf_magnitude_forecasts[
        "Chronology_Valid"
    ].all()
)

# -----------------------------------------------------------
# Forecast-performance table
# -----------------------------------------------------------

forecast_models = {
    "B0_Full_OLS":
        "B0_Full_OLS_Forecast",
    "B1_Parsimonious_OLS":
        "B1_Parsimonious_OLS_Forecast",
    "B2_Full_Ridge":
        "B2_Full_Ridge_Forecast",
    "B3_Full_Huber":
        "B3_Full_Huber_Forecast",
    "B4_Asymmetric_Ridge":
        "B4_Asymmetric_Ridge_Forecast",
}

actual = etf_magnitude_forecasts[
    "Actual_ETF_Flow"
].to_numpy()

metric_rows = []

for model_name, forecast_column in (
    forecast_models.items()
):

    forecast = etf_magnitude_forecasts[
        forecast_column
    ].to_numpy()

    error = actual - forecast

    metric_rows.append({
        "Model":
            model_name,
        "Forecasts":
            len(actual),
        "RMSE":
            np.sqrt(
                np.mean(error ** 2)
            ),
        "MAE":
            np.mean(
                np.abs(error)
            ),
        "Median_Absolute_Error":
            np.median(
                np.abs(error)
            ),
        "Bias":
            np.mean(error),
        "Forecast_Actual_Correlation":
            np.corrcoef(
                actual,
                forecast,
            )[0, 1],
        "Flow_Sign_Accuracy":
            np.mean(
                np.sign(actual)
                == np.sign(forecast)
            ),
        "Forecast_Standard_Deviation":
            np.std(
                forecast,
                ddof=1,
            ),
    })

etf_magnitude_metrics = pd.DataFrame(
    metric_rows
)

baseline_rmse = etf_magnitude_metrics.loc[
    etf_magnitude_metrics["Model"]
    == "B0_Full_OLS",
    "RMSE",
].iloc[0]

baseline_mae = etf_magnitude_metrics.loc[
    etf_magnitude_metrics["Model"]
    == "B0_Full_OLS",
    "MAE",
].iloc[0]

etf_magnitude_metrics[
    "RMSE_Improvement_vs_B0_pct"
] = (
    (
        baseline_rmse
        - etf_magnitude_metrics["RMSE"]
    )
    / baseline_rmse
    * 100
)

etf_magnitude_metrics[
    "MAE_Improvement_vs_B0_pct"
] = (
    (
        baseline_mae
        - etf_magnitude_metrics["MAE"]
    )
    / baseline_mae
    * 100
)

etf_magnitude_metrics = (
    etf_magnitude_metrics
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("\nETF magnitude-model performance")

display(
    etf_magnitude_metrics.round(4)
)

# -----------------------------------------------------------
# HAC loss tests against B0
# -----------------------------------------------------------

baseline_error = (
    actual
    - etf_magnitude_forecasts[
        "B0_Full_OLS_Forecast"
    ].to_numpy()
)

loss_test_rows = []

for model_name, forecast_column in (
    forecast_models.items()
):

    if model_name == "B0_Full_OLS":
        continue

    alternative_error = (
        actual
        - etf_magnitude_forecasts[
            forecast_column
        ].to_numpy()
    )

    loss_improvement = (
        baseline_error ** 2
        - alternative_error ** 2
    )

    loss_model = sm.OLS(
        loss_improvement,
        np.ones(len(loss_improvement)),
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 2,
            "use_correction": True,
        },
    )

    hac_t = float(
        np.asarray(
            loss_model.tvalues
        ).reshape(-1)[0]
    )

    one_sided_p = float(
        1 - stats.t.cdf(
            hac_t,
            df=loss_model.df_resid,
        )
    )

    loss_test_rows.append({
        "Alternative_Model":
            model_name,
        "Mean_Squared_Loss_Improvement":
            np.mean(loss_improvement),
        "HAC_t":
            hac_t,
        "One_Sided_p":
            one_sided_p,
    })

etf_magnitude_loss_tests = pd.DataFrame(
    loss_test_rows
)

# Multiple-testing corrections
holm_result = multipletests(
    etf_magnitude_loss_tests[
        "One_Sided_p"
    ],
    alpha=0.05,
    method="holm",
)

fdr_result = multipletests(
    etf_magnitude_loss_tests[
        "One_Sided_p"
    ],
    alpha=0.05,
    method="fdr_bh",
)

etf_magnitude_loss_tests[
    "Holm_Adjusted_p"
] = holm_result[1]

etf_magnitude_loss_tests[
    "BH_FDR_Adjusted_p"
] = fdr_result[1]

etf_magnitude_loss_tests[
    "Holm_Reject_5pct"
] = holm_result[0]

etf_magnitude_loss_tests[
    "BH_FDR_Reject_5pct"
] = fdr_result[0]

print("\nLoss-improvement tests against full OLS")

display(
    etf_magnitude_loss_tests
    .sort_values("One_Sided_p")
    .round(4)
)

# -----------------------------------------------------------
# Recursive coefficient and tuning summaries
# -----------------------------------------------------------

coefficient_summary = pd.DataFrame({
    "Model": [
        "B0 Full OLS",
        "B1 Parsimonious OLS",
        "B2 Full Ridge",
        "B3 Full Huber",
    ],
    "Minimum_Direction_Coefficient": [
        etf_magnitude_forecasts[
            "B0_Direction_Coefficient"
        ].min(),
        etf_magnitude_forecasts[
            "B1_Direction_Coefficient"
        ].min(),
        etf_magnitude_forecasts[
            "B2_Standardized_Direction_Coefficient"
        ].min(),
        etf_magnitude_forecasts[
            "B3_Standardized_Direction_Coefficient"
        ].min(),
    ],
    "Maximum_Direction_Coefficient": [
        etf_magnitude_forecasts[
            "B0_Direction_Coefficient"
        ].max(),
        etf_magnitude_forecasts[
            "B1_Direction_Coefficient"
        ].max(),
        etf_magnitude_forecasts[
            "B2_Standardized_Direction_Coefficient"
        ].max(),
        etf_magnitude_forecasts[
            "B3_Standardized_Direction_Coefficient"
        ].max(),
    ],
    "Median_Direction_Coefficient": [
        etf_magnitude_forecasts[
            "B0_Direction_Coefficient"
        ].median(),
        etf_magnitude_forecasts[
            "B1_Direction_Coefficient"
        ].median(),
        etf_magnitude_forecasts[
            "B2_Standardized_Direction_Coefficient"
        ].median(),
        etf_magnitude_forecasts[
            "B3_Standardized_Direction_Coefficient"
        ].median(),
    ],
    "Negative_Coefficient_Share": [
        (
            etf_magnitude_forecasts[
                "B0_Direction_Coefficient"
            ] < 0
        ).mean(),
        (
            etf_magnitude_forecasts[
                "B1_Direction_Coefficient"
            ] < 0
        ).mean(),
        (
            etf_magnitude_forecasts[
                "B2_Standardized_Direction_Coefficient"
            ] < 0
        ).mean(),
        (
            etf_magnitude_forecasts[
                "B3_Standardized_Direction_Coefficient"
            ] < 0
        ).mean(),
    ],
})

print("\nDirection-coefficient stability")

display(
    coefficient_summary.round(4)
)

alpha_summary = pd.DataFrame({
    "Model": [
        "B2 Full Ridge",
        "B4 Asymmetric Ridge",
    ],
    "Minimum_Selected_Alpha": [
        etf_magnitude_forecasts[
            "B2_Selected_Alpha"
        ].min(),
        etf_magnitude_forecasts[
            "B4_Selected_Alpha"
        ].min(),
    ],
    "Median_Selected_Alpha": [
        etf_magnitude_forecasts[
            "B2_Selected_Alpha"
        ].median(),
        etf_magnitude_forecasts[
            "B4_Selected_Alpha"
        ].median(),
    ],
    "Maximum_Selected_Alpha": [
        etf_magnitude_forecasts[
            "B2_Selected_Alpha"
        ].max(),
        etf_magnitude_forecasts[
            "B4_Selected_Alpha"
        ].max(),
    ],
})

print("\nNested Ridge tuning summary")

display(
    alpha_summary.round(4)
)

# -----------------------------------------------------------
# Performance chart
# -----------------------------------------------------------

fig = px.bar(
    etf_magnitude_metrics,
    x="Model",
    y="RMSE",
    color="RMSE_Improvement_vs_B0_pct",
    color_continuous_scale="RdYlGn",
    text="RMSE",
    title=(
        "Recursive ETF Magnitude-Forecast Tournament"
    ),
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
)

fig.update_layout(
    width=1100,
    height=650,
    xaxis_title="First-stage model",
    yaxis_title="ETF-flow RMSE (tonnes)",
)

fig.show()

# -----------------------------------------------------------
# Actual and model forecasts
# -----------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=etf_magnitude_forecasts[
            "Outcome_Month"
        ],
        y=etf_magnitude_forecasts[
            "Actual_ETF_Flow"
        ],
        mode="lines+markers",
        name="Actual ETF flow",
        line=dict(
            color="black",
            width=3,
        ),
    )
)

forecast_colors = {
    "B0_Full_OLS_Forecast": "#7f7f7f",
    "B1_Parsimonious_OLS_Forecast": "#1f77b4",
    "B2_Full_Ridge_Forecast": "#2ca02c",
    "B3_Full_Huber_Forecast": "#ff7f0e",
    "B4_Asymmetric_Ridge_Forecast": "#9467bd",
}

for forecast_column, color in (
    forecast_colors.items()
):

    fig.add_trace(
        go.Scatter(
            x=etf_magnitude_forecasts[
                "Outcome_Month"
            ],
            y=etf_magnitude_forecasts[
                forecast_column
            ],
            mode="lines",
            name=forecast_column.replace(
                "_Forecast",
                "",
            ),
            line=dict(
                color=color,
                width=2,
                dash=(
                    "dot"
                    if forecast_column
                    == "B0_Full_OLS_Forecast"
                    else "solid"
                ),
            ),
        )
    )

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title=(
        "Actual and Recursive ETF-Flow Forecasts"
    ),
    template="plotly_white",
    width=1400,
    height=750,
    xaxis_title="ETF-flow outcome month",
    yaxis_title="ETF net demand (tonnes)",
    hovermode="x unified",
)

fig.show()

Recursive forecasts: 42
Outcome period: 2021-07-01 00:00:00 to 2026-07-01 00:00:00
All chronology checks passed: True

ETF magnitude-model performance


,Model,Forecasts,RMSE,MAE,Median_Absolute_Error,Bias,Forecast_Actual_Correlation,Flow_Sign_Accuracy,Forecast_Standard_Deviation,RMSE_Improvement_vs_B0_pct,MAE_Improvement_vs_B0_pct
0,B1_Parsimonious_OLS,42,35.4015,28.1925,21.0599,0.0667,0.6888,0.7857,42.2195,18.6772,19.6502
1,B4_Asymmetric_Ridge,42,38.6781,31.8735,28.9560,-6.7102,0.5942,0.7619,32.0568,11.1504,9.1592
2,B2_Full_Ridge,42,39.2892,31.5788,28.4125,-4.5718,0.6051,0.7857,39.6763,9.7465,9.9991
3,B0_Full_OLS,42,43.5321,35.0872,31.0405,-1.6011,0.5845,0.8095,48.8642,0.0000,0.0000
4,B3_Full_Huber,42,48.5251,35.7064,28.8621,-2.0032,0.5602,0.8095,55.8140,-11.4698,-1.7648



Loss-improvement tests against full OLS


,Alternative_Model,Mean_Squared_Loss_Improvement,HAC_t,One_Sided_p,Holm_Adjusted_p,BH_FDR_Adjusted_p,Holm_Reject_5pct,BH_FDR_Reject_5pct
0,B1_Parsimonious_OLS,641.7740,2.0131,0.0254,0.1014,0.0528,False,False
1,B2_Full_Ridge,351.3975,1.9943,0.0264,0.1014,0.0528,False,False
3,B4_Asymmetric_Ridge,399.0495,1.1506,0.1283,0.2566,0.1710,False,False
2,B3_Full_Huber,-459.6467,-0.9060,0.8149,0.8149,0.8149,False,False



Direction-coefficient stability


,Model,Minimum_Direction_Coefficient,Maximum_Direction_Coefficient,Median_Direction_Coefficient,Negative_Coefficient_Share
0,B0 Full OLS,-181.6079,-60.9057,-90.1335,1.0
1,B1 Parsimonious OLS,-134.7851,-71.7295,-84.5720,1.0
2,B2 Full Ridge,-48.9084,-21.5368,-24.9253,1.0
3,B3 Full Huber,-51.9545,-22.8049,-35.1832,1.0



Nested Ridge tuning summary


,Model,Minimum_Selected_Alpha,Median_Selected_Alpha,Maximum_Selected_Alpha
0,B2 Full Ridge,0.1,10.0,10.0
1,B4 Asymmetric Ridge,10.0,10.0,100.0


In [60]:
# CELL 53: Pass the improved parsimonious ETF forecast
# into the leakage-free gold forecasting system
#
# Critical comparisons:
# G4 vs G3: Does Fed information add value through ETF forecasts?
# G4 vs G2: Does first-stage parsimony improve gold forecasts?
# G4 vs G1: Does the two-stage system beat direct Fed?
# G4 vs G0: Does it beat gold persistence?

import statsmodels.api as sm
from scipy import stats
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# -----------------------------------------------------------
# Preconditions
# -----------------------------------------------------------

required_objects = [
    "two_stage_system_df",
    "etf_magnitude_forecasts",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Required objects are unavailable: {missing_objects}. "
        "Run Cells 48 and 52 first."
    )

source = (
    two_stage_system_df
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

initial_first_stage_window = 18

# -----------------------------------------------------------
# Construct the matched parsimonious no-Fed ETF forecast
#
# Fed version:
# ETF(t+1) ~ ETF(t-1) + FedDirection(t)
#
# No-Fed version:
# ETF(t+1) ~ ETF(t-1)
# -----------------------------------------------------------

matched_forecast_rows = []

for position in range(
    initial_first_stage_window,
    len(source)
):

    train = source.iloc[:position].copy()
    test = source.iloc[[position]].copy()

    y_train = train[
        "ETF_Flow_Next_Month"
    ].astype(float)

    record = {
        "Event_Month":
            test["Month"].iloc[0],
        "Outcome_Month":
            test["Month"].iloc[0]
            + pd.offsets.MonthBegin(1),
        "Final_First_Stage_Training_Month":
            train["Month"].max(),
        "Actual_ETF_Flow":
            float(
                test[
                    "ETF_Flow_Next_Month"
                ].iloc[0]
            ),
        "Gold_Return_Next_Month":
            float(
                test[
                    "Gold_Return_Next_Month"
                ].iloc[0]
            ),
        "Gold_Return_Previous_Month":
            float(
                test[
                    "Gold_Return_Previous_Month"
                ].iloc[0]
            ),
        "Fed_Expected_Direction":
            float(
                test[
                    "Fed_Expected_Direction"
                ].iloc[0]
            ),
    }

    first_stage_specs = {
        "Parsimonious_NoFed_ETF_Forecast": [
            "ETF_Flow_Previous_Month",
        ],
        "Parsimonious_Fed_ETF_Forecast": [
            "ETF_Flow_Previous_Month",
            "Fed_Expected_Direction",
        ],
        "Full_OLS_ETF_Forecast": [
            "Fed_Expected_Direction",
            "Gold_Return_Previous_Month",
            "ETF_Flow_Previous_Month",
            "Broad_USD_Change_Previous_Month",
            "Real10Y_Change_Previous_Month",
            "VIX_Change_Previous_Month",
        ],
    }

    for forecast_name, predictors in (
        first_stage_specs.items()
    ):

        X_train = sm.add_constant(
            train[predictors].astype(float),
            has_constant="add",
        )

        X_test = sm.add_constant(
            test[predictors].astype(float),
            has_constant="add",
        ).reindex(
            columns=X_train.columns,
            fill_value=1.0,
        )

        model = sm.OLS(
            y_train,
            X_train,
        ).fit()

        record[forecast_name] = float(
            model.predict(X_test).iloc[0]
        )

        if (
            forecast_name
            == "Parsimonious_Fed_ETF_Forecast"
        ):
            record[
                "Recursive_Fed_Direction_Coefficient"
            ] = float(
                model.params[
                    "Fed_Expected_Direction"
                ]
            )

    matched_forecast_rows.append(record)

improved_etf_system_data = pd.DataFrame(
    matched_forecast_rows
)

improved_etf_system_data[
    "First_Stage_Chronology_Valid"
] = (
    improved_etf_system_data[
        "Final_First_Stage_Training_Month"
    ]
    < improved_etf_system_data[
        "Event_Month"
    ]
)

if not improved_etf_system_data[
    "First_Stage_Chronology_Valid"
].all():
    raise ValueError(
        "First-stage future-data leakage detected."
    )

# Confirm that the reconstructed B1 forecast matches Cell 52.
b1_check = (
    improved_etf_system_data[
        [
            "Event_Month",
            "Parsimonious_Fed_ETF_Forecast",
        ]
    ]
    .merge(
        etf_magnitude_forecasts[
            [
                "Event_Month",
                "B1_Parsimonious_OLS_Forecast",
            ]
        ],
        on="Event_Month",
        how="inner",
        validate="one_to_one",
    )
)

maximum_b1_difference = np.max(
    np.abs(
        b1_check[
            "Parsimonious_Fed_ETF_Forecast"
        ]
        - b1_check[
            "B1_Parsimonious_OLS_Forecast"
        ]
    )
)

print(
    "Maximum difference from Cell 52 B1 forecasts:",
    maximum_b1_difference
)

if maximum_b1_difference > 1e-8:
    raise ValueError(
        "Reconstructed B1 forecasts do not match Cell 52."
    )

# -----------------------------------------------------------
# First-stage comparison
# -----------------------------------------------------------

first_stage_model_map = {
    "Full OLS with Fed":
        "Full_OLS_ETF_Forecast",
    "Parsimonious without Fed":
        "Parsimonious_NoFed_ETF_Forecast",
    "Parsimonious with Fed":
        "Parsimonious_Fed_ETF_Forecast",
}

actual_etf = improved_etf_system_data[
    "Actual_ETF_Flow"
].to_numpy()

first_stage_rows = []

for label, forecast_column in (
    first_stage_model_map.items()
):

    forecast = improved_etf_system_data[
        forecast_column
    ].to_numpy()

    error = actual_etf - forecast

    first_stage_rows.append({
        "Model":
            label,
        "Forecasts":
            len(error),
        "RMSE":
            np.sqrt(
                np.mean(error ** 2)
            ),
        "MAE":
            np.mean(
                np.abs(error)
            ),
        "Bias":
            np.mean(error),
        "Forecast_Actual_Correlation":
            np.corrcoef(
                actual_etf,
                forecast,
            )[0, 1],
        "Flow_Sign_Accuracy":
            np.mean(
                np.sign(actual_etf)
                == np.sign(forecast)
            ),
    })

improved_first_stage_metrics = pd.DataFrame(
    first_stage_rows
)

print("\nMatched first-stage comparison")

display(
    improved_first_stage_metrics.round(4)
)

# -----------------------------------------------------------
# Recursive second-stage gold equations
# -----------------------------------------------------------

second_stage_initial_window = 18

gold_specifications = {
    "G0_Gold_Persistence": [
        "Gold_Return_Previous_Month",
    ],
    "G1_Direct_Fed": [
        "Gold_Return_Previous_Month",
        "Fed_Expected_Direction",
    ],
    "G2_Full_OLS_ETF_System": [
        "Gold_Return_Previous_Month",
        "Full_OLS_ETF_Forecast",
    ],
    "G3_Parsimonious_NoFed_ETF_System": [
        "Gold_Return_Previous_Month",
        "Parsimonious_NoFed_ETF_Forecast",
    ],
    "G4_Parsimonious_Fed_ETF_System": [
        "Gold_Return_Previous_Month",
        "Parsimonious_Fed_ETF_Forecast",
    ],
}

gold_forecast_rows = []

for position in range(
    second_stage_initial_window,
    len(improved_etf_system_data)
):

    train = improved_etf_system_data.iloc[
        :position
    ].copy()

    test = improved_etf_system_data.iloc[
        [position]
    ].copy()

    y_train = train[
        "Gold_Return_Next_Month"
    ].astype(float)

    actual_gold = float(
        test[
            "Gold_Return_Next_Month"
        ].iloc[0]
    )

    record = {
        "Event_Month":
            test["Event_Month"].iloc[0],
        "Outcome_Month":
            test["Outcome_Month"].iloc[0],
        "Training_N":
            len(train),
        "Final_Training_Event_Month":
            train["Event_Month"].max(),
        "Actual_Gold_Return":
            actual_gold,
    }

    for model_name, predictors in (
        gold_specifications.items()
    ):

        X_train = sm.add_constant(
            train[predictors].astype(float),
            has_constant="add",
        )

        X_test = sm.add_constant(
            test[predictors].astype(float),
            has_constant="add",
        ).reindex(
            columns=X_train.columns,
            fill_value=1.0,
        )

        model = sm.OLS(
            y_train,
            X_train,
        ).fit()

        forecast = float(
            model.predict(X_test).iloc[0]
        )

        record[
            f"{model_name}_Forecast"
        ] = forecast

        record[
            f"{model_name}_Error"
        ] = actual_gold - forecast

        if (
            model_name
            == "G4_Parsimonious_Fed_ETF_System"
        ):
            record[
                "Recursive_Improved_ETF_Coefficient"
            ] = float(
                model.params[
                    "Parsimonious_Fed_ETF_Forecast"
                ]
            )

    gold_forecast_rows.append(record)

improved_gold_forecasts = pd.DataFrame(
    gold_forecast_rows
)

improved_gold_forecasts[
    "Second_Stage_Chronology_Valid"
] = (
    improved_gold_forecasts[
        "Final_Training_Event_Month"
    ]
    < improved_gold_forecasts[
        "Event_Month"
    ]
)

if not improved_gold_forecasts[
    "Second_Stage_Chronology_Valid"
].all():
    raise ValueError(
        "Second-stage future-data leakage detected."
    )

print(
    "\nGold forecasts:",
    len(improved_gold_forecasts)
)

print(
    "Outcome period:",
    improved_gold_forecasts[
        "Outcome_Month"
    ].min(),
    "to",
    improved_gold_forecasts[
        "Outcome_Month"
    ].max()
)

print(
    "All chronology checks passed:",
    improved_gold_forecasts[
        "Second_Stage_Chronology_Valid"
    ].all()
)

# -----------------------------------------------------------
# Gold forecast metrics
# -----------------------------------------------------------

actual_gold = improved_gold_forecasts[
    "Actual_Gold_Return"
].to_numpy()

gold_metric_rows = []

for model_name in gold_specifications:

    forecast = improved_gold_forecasts[
        f"{model_name}_Forecast"
    ].to_numpy()

    error = actual_gold - forecast

    gold_metric_rows.append({
        "Model":
            model_name,
        "Forecasts":
            len(error),
        "RMSE":
            np.sqrt(
                np.mean(error ** 2)
            ),
        "MAE":
            np.mean(
                np.abs(error)
            ),
        "Bias":
            np.mean(error),
        "Forecast_Actual_Correlation":
            np.corrcoef(
                actual_gold,
                forecast,
            )[0, 1],
        "Return_Sign_Accuracy":
            np.mean(
                np.sign(actual_gold)
                == np.sign(forecast)
            ),
    })

improved_gold_metrics = pd.DataFrame(
    gold_metric_rows
)

g0_rmse = improved_gold_metrics.loc[
    improved_gold_metrics["Model"]
    == "G0_Gold_Persistence",
    "RMSE",
].iloc[0]

g0_mae = improved_gold_metrics.loc[
    improved_gold_metrics["Model"]
    == "G0_Gold_Persistence",
    "MAE",
].iloc[0]

improved_gold_metrics[
    "RMSE_Improvement_vs_G0_pct"
] = (
    (
        g0_rmse
        - improved_gold_metrics["RMSE"]
    )
    / g0_rmse
    * 100
)

improved_gold_metrics[
    "MAE_Improvement_vs_G0_pct"
] = (
    (
        g0_mae
        - improved_gold_metrics["MAE"]
    )
    / g0_mae
    * 100
)

print("\nImproved gold-system performance")

display(
    improved_gold_metrics
    .sort_values("RMSE")
    .round(4)
)

# -----------------------------------------------------------
# HAC loss comparisons
# -----------------------------------------------------------

gold_comparisons = [
    {
        "Comparison":
            "G4 improved Fed-ETF vs G0 persistence",
        "Reference":
            "G0_Gold_Persistence",
        "Alternative":
            "G4_Parsimonious_Fed_ETF_System",
    },
    {
        "Comparison":
            "G4 improved Fed-ETF vs G1 direct Fed",
        "Reference":
            "G1_Direct_Fed",
        "Alternative":
            "G4_Parsimonious_Fed_ETF_System",
    },
    {
        "Comparison":
            "G4 improved Fed-ETF vs G2 full ETF system",
        "Reference":
            "G2_Full_OLS_ETF_System",
        "Alternative":
            "G4_Parsimonious_Fed_ETF_System",
    },
    {
        "Comparison":
            "G4 improved Fed-ETF vs G3 matched no-Fed ETF",
        "Reference":
            "G3_Parsimonious_NoFed_ETF_System",
        "Alternative":
            "G4_Parsimonious_Fed_ETF_System",
    },
]

loss_rows = []

for comparison in gold_comparisons:

    reference_error = improved_gold_forecasts[
        f"{comparison['Reference']}_Error"
    ].to_numpy()

    alternative_error = improved_gold_forecasts[
        f"{comparison['Alternative']}_Error"
    ].to_numpy()

    loss_improvement = (
        reference_error ** 2
        - alternative_error ** 2
    )

    loss_model = sm.OLS(
        loss_improvement,
        np.ones(len(loss_improvement)),
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 2,
            "use_correction": True,
        },
    )

    hac_t = float(
        np.asarray(
            loss_model.tvalues
        ).reshape(-1)[0]
    )

    one_sided_p = float(
        1 - stats.t.cdf(
            hac_t,
            df=loss_model.df_resid,
        )
    )

    loss_rows.append({
        "Comparison":
            comparison["Comparison"],
        "Mean_Squared_Loss_Improvement":
            np.mean(loss_improvement),
        "HAC_t":
            hac_t,
        "One_Sided_p":
            one_sided_p,
        "Significant_5pct":
            one_sided_p < 0.05,
        "Significant_10pct":
            one_sided_p < 0.10,
    })

improved_gold_loss_tests = pd.DataFrame(
    loss_rows
)

print("\nGold loss-improvement tests")

display(
    improved_gold_loss_tests.round(4)
)

# -----------------------------------------------------------
# Improved second-stage ETF coefficient stability
# -----------------------------------------------------------

improved_etf_coefficients = (
    improved_gold_forecasts[
        "Recursive_Improved_ETF_Coefficient"
    ]
)

improved_etf_coefficient_summary = pd.DataFrame({
    "Minimum_ETF_Coefficient": [
        improved_etf_coefficients.min()
    ],
    "Maximum_ETF_Coefficient": [
        improved_etf_coefficients.max()
    ],
    "Median_ETF_Coefficient": [
        improved_etf_coefficients.median()
    ],
    "Positive_Coefficient_Share": [
        (
            improved_etf_coefficients > 0
        ).mean()
    ],
})

print(
    "\nImproved predicted-ETF coefficient stability"
)

display(
    improved_etf_coefficient_summary.round(4)
)

# -----------------------------------------------------------
# Plot gold forecasts
# -----------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=improved_gold_forecasts[
            "Outcome_Month"
        ],
        y=improved_gold_forecasts[
            "Actual_Gold_Return"
        ],
        mode="lines+markers",
        name="Actual gold return",
        line=dict(
            color="black",
            width=3,
        ),
    )
)

plot_models = {
    "G0_Gold_Persistence": {
        "name": "Gold persistence",
        "color": "#7f7f7f",
        "dash": "dot",
    },
    "G1_Direct_Fed": {
        "name": "Direct Fed",
        "color": "#d62728",
        "dash": "dot",
    },
    "G2_Full_OLS_ETF_System": {
        "name": "Original full ETF system",
        "color": "#ff7f0e",
        "dash": "dash",
    },
    "G3_Parsimonious_NoFed_ETF_System": {
        "name": "Parsimonious ETF without Fed",
        "color": "#9467bd",
        "dash": "dash",
    },
    "G4_Parsimonious_Fed_ETF_System": {
        "name": "Improved parsimonious Fed-ETF",
        "color": "#1f77b4",
        "dash": "solid",
    },
}

for model_name, details in plot_models.items():

    fig.add_trace(
        go.Scatter(
            x=improved_gold_forecasts[
                "Outcome_Month"
            ],
            y=improved_gold_forecasts[
                f"{model_name}_Forecast"
            ],
            mode="lines+markers",
            name=details["name"],
            line=dict(
                color=details["color"],
                dash=details["dash"],
                width=2,
            ),
        )
    )

fig.add_hline(
    y=0,
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title=(
        "Leakage-Free Gold Forecasts Using "
        "the Improved ETF Magnitude Model"
    ),
    template="plotly_white",
    width=1400,
    height=750,
    xaxis_title="Gold-return outcome month",
    yaxis_title="Gold monthly return (%)",
    hovermode="x unified",
)

fig.show()

Maximum difference from Cell 52 B1 forecasts: 1.8189894035458565e-12

Matched first-stage comparison


,Model,Forecasts,RMSE,MAE,Bias,Forecast_Actual_Correlation,Flow_Sign_Accuracy
0,Full OLS with Fed,42,43.5321,35.0872,-1.6011,0.5845,0.8095
1,Parsimonious without Fed,42,42.1386,33.6064,-9.8570,0.4954,0.6905
2,Parsimonious with Fed,42,35.4015,28.1925,0.0667,0.6888,0.7857



Gold forecasts: 24
Outcome period: 2023-10-01 00:00:00 to 2026-07-01 00:00:00
All chronology checks passed: True

Improved gold-system performance


,Model,Forecasts,RMSE,MAE,Bias,Forecast_Actual_Correlation,Return_Sign_Accuracy,RMSE_Improvement_vs_G0_pct,MAE_Improvement_vs_G0_pct
4,G4_Parsimonious_Fed_ETF_System,24,4.0171,3.2414,1.3373,0.1672,0.5833,7.0982,3.6946
1,G1_Direct_Fed,24,4.1421,3.3196,1.3804,0.0565,0.5833,4.2079,1.3723
2,G2_Full_OLS_ETF_System,24,4.1695,3.3164,1.6354,0.1016,0.5833,3.5739,1.4671
0,G0_Gold_Persistence,24,4.3241,3.3658,1.7424,-0.1485,0.6667,0.0000,0.0000
3,G3_Parsimonious_NoFed_ETF_System,24,4.5120,3.4788,1.6307,-0.3241,0.6667,-4.3467,-3.3581



Gold loss-improvement tests


,Comparison,Mean_Squared_Loss_Improvement,HAC_t,One_Sided_p,Significant_5pct,Significant_10pct
0,G4 improved Fed-ETF vs G0 persistence,2.5602,1.4238,0.0840,False,True
1,G4 improved Fed-ETF vs G1 direct Fed,1.0198,2.3978,0.0125,True,True
2,G4 improved Fed-ETF vs G2 full ETF system,1.2476,2.0452,0.0262,True,True
3,G4 improved Fed-ETF vs G3 matched no-Fed ETF,4.2209,1.8640,0.0376,True,True



Improved predicted-ETF coefficient stability


,Minimum_ETF_Coefficient,Maximum_ETF_Coefficient,Median_ETF_Coefficient,Positive_Coefficient_Share
0,-0.0125,0.0375,0.013,0.7083


In [61]:
# CELL 54: Multiplicity correction and nested adaptive
# first-stage model selection
#
# At each date, the ETF model is selected exclusively from
# forecast errors observed before that date.

import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np
import plotly.express as px

# -----------------------------------------------------------
# Preconditions
# -----------------------------------------------------------

required_objects = [
    "etf_magnitude_forecasts",
    "improved_etf_system_data",
    "improved_gold_loss_tests",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Required objects are unavailable: {missing_objects}. "
        "Run Cells 52 and 53 first."
    )

# -----------------------------------------------------------
# PART 1: Correct Cell 53 gold-test p-values
# -----------------------------------------------------------

corrected_gold_loss_tests = (
    improved_gold_loss_tests.copy()
)

holm = multipletests(
    corrected_gold_loss_tests["One_Sided_p"],
    alpha=0.05,
    method="holm",
)

fdr = multipletests(
    corrected_gold_loss_tests["One_Sided_p"],
    alpha=0.05,
    method="fdr_bh",
)

corrected_gold_loss_tests[
    "Holm_Adjusted_p"
] = holm[1]

corrected_gold_loss_tests[
    "BH_FDR_Adjusted_p"
] = fdr[1]

corrected_gold_loss_tests[
    "Holm_Reject_5pct"
] = holm[0]

corrected_gold_loss_tests[
    "BH_FDR_Reject_5pct"
] = fdr[0]

print(
    "Multiplicity-corrected Cell 53 comparisons"
)

display(
    corrected_gold_loss_tests
    .sort_values("One_Sided_p")
    .round(4)
)

# -----------------------------------------------------------
# PART 2: Assemble all recursive ETF candidates
# -----------------------------------------------------------

candidate_columns = {
    "B0_Full_OLS":
        "B0_Full_OLS_Forecast",
    "B1_Parsimonious_OLS":
        "B1_Parsimonious_OLS_Forecast",
    "B2_Full_Ridge":
        "B2_Full_Ridge_Forecast",
    "B3_Full_Huber":
        "B3_Full_Huber_Forecast",
    "B4_Asymmetric_Ridge":
        "B4_Asymmetric_Ridge_Forecast",
}

adaptive_source = (
    etf_magnitude_forecasts[
        [
            "Event_Month",
            "Outcome_Month",
            "Actual_ETF_Flow",
        ]
        + list(candidate_columns.values())
    ]
    .merge(
        improved_etf_system_data[
            [
                "Event_Month",
                "Gold_Return_Next_Month",
                "Gold_Return_Previous_Month",
                "Fed_Expected_Direction",
                "Parsimonious_NoFed_ETF_Forecast",
            ]
        ],
        on="Event_Month",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("Event_Month")
    .reset_index(drop=True)
)

# -----------------------------------------------------------
# PART 3: Prequential ETF-model selection
#
# Six historical recursive forecasts are required before
# the first model-selection decision.
# -----------------------------------------------------------

selection_burn_in = 6
adaptive_rows = []

for position in range(
    selection_burn_in,
    len(adaptive_source)
):

    historical_data = adaptive_source.iloc[
        :position
    ].copy()

    current_data = adaptive_source.iloc[
        [position]
    ].copy()

    historical_actual = historical_data[
        "Actual_ETF_Flow"
    ].to_numpy()

    historical_model_performance = {}

    for model_name, forecast_column in (
        candidate_columns.items()
    ):

        historical_forecast = historical_data[
            forecast_column
        ].to_numpy()

        historical_errors = (
            historical_actual
            - historical_forecast
        )

        historical_model_performance[
            model_name
        ] = np.sqrt(
            np.mean(
                historical_errors ** 2
            )
        )

    selected_model = min(
        historical_model_performance,
        key=historical_model_performance.get,
    )

    selected_forecast_column = (
        candidate_columns[selected_model]
    )

    adaptive_rows.append({
        "Original_Position":
            position,
        "Event_Month":
            current_data[
                "Event_Month"
            ].iloc[0],
        "Outcome_Month":
            current_data[
                "Outcome_Month"
            ].iloc[0],
        "Final_ETF_Selection_Month":
            historical_data[
                "Event_Month"
            ].max(),
        "ETF_Selection_History_N":
            len(historical_data),
        "Selected_ETF_Model":
            selected_model,
        "Selected_Model_Historical_RMSE":
            historical_model_performance[
                selected_model
            ],
        "Adaptive_ETF_Forecast":
            float(
                current_data[
                    selected_forecast_column
                ].iloc[0]
            ),
        "Fixed_B1_ETF_Forecast":
            float(
                current_data[
                    "B1_Parsimonious_OLS_Forecast"
                ].iloc[0]
            ),
        "NoFed_ETF_Forecast":
            float(
                current_data[
                    "Parsimonious_NoFed_ETF_Forecast"
                ].iloc[0]
            ),
        "Actual_ETF_Flow":
            float(
                current_data[
                    "Actual_ETF_Flow"
                ].iloc[0]
            ),
        "Gold_Return_Next_Month":
            float(
                current_data[
                    "Gold_Return_Next_Month"
                ].iloc[0]
            ),
        "Gold_Return_Previous_Month":
            float(
                current_data[
                    "Gold_Return_Previous_Month"
                ].iloc[0]
            ),
        "Fed_Expected_Direction":
            float(
                current_data[
                    "Fed_Expected_Direction"
                ].iloc[0]
            ),
    })

adaptive_etf_data = pd.DataFrame(
    adaptive_rows
)

adaptive_etf_data[
    "ETF_Selection_Chronology_Valid"
] = (
    adaptive_etf_data[
        "Final_ETF_Selection_Month"
    ]
    < adaptive_etf_data[
        "Event_Month"
    ]
)

if not adaptive_etf_data[
    "ETF_Selection_Chronology_Valid"
].all():
    raise ValueError(
        "Future leakage detected during ETF-model selection."
    )

print(
    "\nAdaptive ETF forecasts:",
    len(adaptive_etf_data)
)

print(
    "All ETF-selection chronology checks passed:",
    adaptive_etf_data[
        "ETF_Selection_Chronology_Valid"
    ].all()
)

# -----------------------------------------------------------
# Selection frequencies
# -----------------------------------------------------------

selection_frequency = (
    adaptive_etf_data[
        "Selected_ETF_Model"
    ]
    .value_counts()
    .rename_axis("Selected_ETF_Model")
    .reset_index(name="Selections")
)

selection_frequency[
    "Selection_Share"
] = (
    selection_frequency["Selections"]
    / selection_frequency["Selections"].sum()
)

print("\nPrequential ETF-model selection frequency")

display(
    selection_frequency.round(4)
)

# -----------------------------------------------------------
# Adaptive ETF forecast performance
# -----------------------------------------------------------

adaptive_actual = adaptive_etf_data[
    "Actual_ETF_Flow"
].to_numpy()

adaptive_etf_metric_rows = []

for label, forecast_column in {
    "Adaptive historical-RMSE selection":
        "Adaptive_ETF_Forecast",
    "Fixed B1 parsimonious Fed":
        "Fixed_B1_ETF_Forecast",
    "Matched parsimonious no-Fed":
        "NoFed_ETF_Forecast",
}.items():

    forecast = adaptive_etf_data[
        forecast_column
    ].to_numpy()

    error = adaptive_actual - forecast

    adaptive_etf_metric_rows.append({
        "Model":
            label,
        "N":
            len(error),
        "RMSE":
            np.sqrt(
                np.mean(error ** 2)
            ),
        "MAE":
            np.mean(
                np.abs(error)
            ),
        "Bias":
            np.mean(error),
        "Correlation":
            np.corrcoef(
                adaptive_actual,
                forecast,
            )[0, 1],
        "Sign_Accuracy":
            np.mean(
                np.sign(adaptive_actual)
                == np.sign(forecast)
            ),
    })

adaptive_etf_metrics = pd.DataFrame(
    adaptive_etf_metric_rows
)

print("\nAdaptive first-stage performance")

display(
    adaptive_etf_metrics.round(4)
)

# -----------------------------------------------------------
# PART 4: Recursive gold forecasts
#
# Adaptive data begin after six ETF forecast observations.
# Using 12 second-stage observations makes the first gold
# forecast correspond to the same original position as Cell 53:
#
# 6 + 12 = 18.
# -----------------------------------------------------------

adaptive_second_stage_window = 12

adaptive_gold_specs = {
    "A0_Gold_Persistence": [
        "Gold_Return_Previous_Month",
    ],
    "A1_Direct_Fed": [
        "Gold_Return_Previous_Month",
        "Fed_Expected_Direction",
    ],
    "A2_NoFed_ETF": [
        "Gold_Return_Previous_Month",
        "NoFed_ETF_Forecast",
    ],
    "A3_Fixed_B1_ETF": [
        "Gold_Return_Previous_Month",
        "Fixed_B1_ETF_Forecast",
    ],
    "A4_Adaptive_ETF": [
        "Gold_Return_Previous_Month",
        "Adaptive_ETF_Forecast",
    ],
}

adaptive_gold_rows = []

for position in range(
    adaptive_second_stage_window,
    len(adaptive_etf_data)
):

    train = adaptive_etf_data.iloc[
        :position
    ].copy()

    test = adaptive_etf_data.iloc[
        [position]
    ].copy()

    y_train = train[
        "Gold_Return_Next_Month"
    ].astype(float)

    actual_gold = float(
        test[
            "Gold_Return_Next_Month"
        ].iloc[0]
    )

    record = {
        "Event_Month":
            test["Event_Month"].iloc[0],
        "Outcome_Month":
            test["Outcome_Month"].iloc[0],
        "Training_N":
            len(train),
        "Final_Training_Event_Month":
            train["Event_Month"].max(),
        "Actual_Gold_Return":
            actual_gold,
        "Selected_ETF_Model":
            test[
                "Selected_ETF_Model"
            ].iloc[0],
    }

    for model_name, predictors in (
        adaptive_gold_specs.items()
    ):

        X_train = sm.add_constant(
            train[predictors].astype(float),
            has_constant="add",
        )

        X_test = sm.add_constant(
            test[predictors].astype(float),
            has_constant="add",
        ).reindex(
            columns=X_train.columns,
            fill_value=1.0,
        )

        model = sm.OLS(
            y_train,
            X_train,
        ).fit()

        forecast = float(
            model.predict(X_test).iloc[0]
        )

        record[
            f"{model_name}_Forecast"
        ] = forecast

        record[
            f"{model_name}_Error"
        ] = actual_gold - forecast

        if model_name == "A4_Adaptive_ETF":
            record[
                "Adaptive_ETF_Gold_Coefficient"
            ] = float(
                model.params[
                    "Adaptive_ETF_Forecast"
                ]
            )

    adaptive_gold_rows.append(record)

adaptive_gold_forecasts = pd.DataFrame(
    adaptive_gold_rows
)

adaptive_gold_forecasts[
    "Gold_Chronology_Valid"
] = (
    adaptive_gold_forecasts[
        "Final_Training_Event_Month"
    ]
    < adaptive_gold_forecasts[
        "Event_Month"
    ]
)

if not adaptive_gold_forecasts[
    "Gold_Chronology_Valid"
].all():
    raise ValueError(
        "Future leakage detected in the gold equation."
    )

print(
    "\nNested gold forecasts:",
    len(adaptive_gold_forecasts)
)

print(
    "Outcome period:",
    adaptive_gold_forecasts[
        "Outcome_Month"
    ].min(),
    "to",
    adaptive_gold_forecasts[
        "Outcome_Month"
    ].max()
)

print(
    "All gold chronology checks passed:",
    adaptive_gold_forecasts[
        "Gold_Chronology_Valid"
    ].all()
)

# -----------------------------------------------------------
# Gold forecast metrics
# -----------------------------------------------------------

actual_gold = adaptive_gold_forecasts[
    "Actual_Gold_Return"
].to_numpy()

adaptive_gold_metric_rows = []

for model_name in adaptive_gold_specs:

    forecast = adaptive_gold_forecasts[
        f"{model_name}_Forecast"
    ].to_numpy()

    error = actual_gold - forecast

    adaptive_gold_metric_rows.append({
        "Model":
            model_name,
        "Forecasts":
            len(error),
        "RMSE":
            np.sqrt(
                np.mean(error ** 2)
            ),
        "MAE":
            np.mean(
                np.abs(error)
            ),
        "Bias":
            np.mean(error),
        "Forecast_Actual_Correlation":
            np.corrcoef(
                actual_gold,
                forecast,
            )[0, 1],
        "Return_Sign_Accuracy":
            np.mean(
                np.sign(actual_gold)
                == np.sign(forecast)
            ),
    })

adaptive_gold_metrics = pd.DataFrame(
    adaptive_gold_metric_rows
)

a0_rmse = adaptive_gold_metrics.loc[
    adaptive_gold_metrics["Model"]
    == "A0_Gold_Persistence",
    "RMSE",
].iloc[0]

a0_mae = adaptive_gold_metrics.loc[
    adaptive_gold_metrics["Model"]
    == "A0_Gold_Persistence",
    "MAE",
].iloc[0]

adaptive_gold_metrics[
    "RMSE_Improvement_vs_A0_pct"
] = (
    (
        a0_rmse
        - adaptive_gold_metrics["RMSE"]
    )
    / a0_rmse
    * 100
)

adaptive_gold_metrics[
    "MAE_Improvement_vs_A0_pct"
] = (
    (
        a0_mae
        - adaptive_gold_metrics["MAE"]
    )
    / a0_mae
    * 100
)

print("\nNested adaptive gold-system performance")

display(
    adaptive_gold_metrics
    .sort_values("RMSE")
    .round(4)
)

# -----------------------------------------------------------
# Loss-difference tests
# -----------------------------------------------------------

adaptive_comparisons = [
    {
        "Comparison":
            "A4 adaptive ETF vs A0 persistence",
        "Reference":
            "A0_Gold_Persistence",
        "Alternative":
            "A4_Adaptive_ETF",
    },
    {
        "Comparison":
            "A4 adaptive ETF vs A1 direct Fed",
        "Reference":
            "A1_Direct_Fed",
        "Alternative":
            "A4_Adaptive_ETF",
    },
    {
        "Comparison":
            "A4 adaptive ETF vs A2 no-Fed ETF",
        "Reference":
            "A2_NoFed_ETF",
        "Alternative":
            "A4_Adaptive_ETF",
    },
    {
        "Comparison":
            "A4 adaptive ETF vs A3 fixed B1 ETF",
        "Reference":
            "A3_Fixed_B1_ETF",
        "Alternative":
            "A4_Adaptive_ETF",
    },
]

adaptive_loss_rows = []

for comparison in adaptive_comparisons:

    reference_error = adaptive_gold_forecasts[
        f"{comparison['Reference']}_Error"
    ].to_numpy()

    alternative_error = adaptive_gold_forecasts[
        f"{comparison['Alternative']}_Error"
    ].to_numpy()

    loss_improvement = (
        reference_error ** 2
        - alternative_error ** 2
    )

    loss_model = sm.OLS(
        loss_improvement,
        np.ones(len(loss_improvement)),
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 2,
            "use_correction": True,
        },
    )

    hac_t = float(
        np.asarray(
            loss_model.tvalues
        ).reshape(-1)[0]
    )

    one_sided_p = float(
        1 - stats.t.cdf(
            hac_t,
            df=loss_model.df_resid,
        )
    )

    adaptive_loss_rows.append({
        "Comparison":
            comparison["Comparison"],
        "Mean_Squared_Loss_Improvement":
            np.mean(loss_improvement),
        "HAC_t":
            hac_t,
        "One_Sided_p":
            one_sided_p,
    })

adaptive_gold_loss_tests = pd.DataFrame(
    adaptive_loss_rows
)

adaptive_holm = multipletests(
    adaptive_gold_loss_tests[
        "One_Sided_p"
    ],
    alpha=0.05,
    method="holm",
)

adaptive_fdr = multipletests(
    adaptive_gold_loss_tests[
        "One_Sided_p"
    ],
    alpha=0.05,
    method="fdr_bh",
)

adaptive_gold_loss_tests[
    "Holm_Adjusted_p"
] = adaptive_holm[1]

adaptive_gold_loss_tests[
    "BH_FDR_Adjusted_p"
] = adaptive_fdr[1]

adaptive_gold_loss_tests[
    "Holm_Reject_5pct"
] = adaptive_holm[0]

adaptive_gold_loss_tests[
    "BH_FDR_Reject_5pct"
] = adaptive_fdr[0]

print("\nNested adaptive gold loss tests")

display(
    adaptive_gold_loss_tests
    .sort_values("One_Sided_p")
    .round(4)
)

# -----------------------------------------------------------
# Adaptive ETF coefficient stability
# -----------------------------------------------------------

adaptive_coefficients = adaptive_gold_forecasts[
    "Adaptive_ETF_Gold_Coefficient"
]

adaptive_coefficient_summary = pd.DataFrame({
    "Minimum_Coefficient": [
        adaptive_coefficients.min()
    ],
    "Maximum_Coefficient": [
        adaptive_coefficients.max()
    ],
    "Median_Coefficient": [
        adaptive_coefficients.median()
    ],
    "Positive_Coefficient_Share": [
        (
            adaptive_coefficients > 0
        ).mean()
    ],
})

print("\nAdaptive predicted-ETF coefficient stability")

display(
    adaptive_coefficient_summary.round(4)
)

# -----------------------------------------------------------
# Selection path
# -----------------------------------------------------------

fig = px.scatter(
    adaptive_etf_data,
    x="Outcome_Month",
    y="Selected_ETF_Model",
    color="Selected_ETF_Model",
    title=(
        "Prequential First-Stage ETF Model Selection"
    ),
    labels={
        "Outcome_Month":
            "ETF-flow outcome month",
        "Selected_ETF_Model":
            "Historically selected model",
    },
    template="plotly_white",
)

fig.update_traces(
    marker=dict(size=11)
)

fig.update_layout(
    width=1200,
    height=600,
    showlegend=False,
)

fig.show()

Multiplicity-corrected Cell 53 comparisons


,Comparison,Mean_Squared_Loss_Improvement,HAC_t,One_Sided_p,Significant_5pct,Significant_10pct,Holm_Adjusted_p,BH_FDR_Adjusted_p,Holm_Reject_5pct,BH_FDR_Reject_5pct
1,G4 improved Fed-ETF vs G1 direct Fed,1.0198,2.3978,0.0125,True,True,0.0500,0.0500,False,False
2,G4 improved Fed-ETF vs G2 full ETF system,1.2476,2.0452,0.0262,True,True,0.0787,0.0501,False,False
3,G4 improved Fed-ETF vs G3 matched no-Fed ETF,4.2209,1.8640,0.0376,True,True,0.0787,0.0501,False,False
0,G4 improved Fed-ETF vs G0 persistence,2.5602,1.4238,0.0840,False,True,0.0840,0.0840,False,False



Adaptive ETF forecasts: 36
All ETF-selection chronology checks passed: True

Prequential ETF-model selection frequency


,Selected_ETF_Model,Selections,Selection_Share
0,B1_Parsimonious_OLS,29,0.8056
1,B4_Asymmetric_Ridge,7,0.1944



Adaptive first-stage performance


,Model,N,RMSE,MAE,Bias,Correlation,Sign_Accuracy
0,Adaptive historical-RMSE selection,36,37.6432,29.7810,0.1389,0.6864,0.8333
1,Fixed B1 parsimonious Fed,36,36.9848,29.2676,0.6778,0.6983,0.8056
2,Matched parsimonious no-Fed,36,42.4440,33.4283,-6.8289,0.5382,0.7222


... (2 outputs left out)


Nested adaptive gold loss tests


,Comparison,Mean_Squared_Loss_Improvement,HAC_t,One_Sided_p,Holm_Adjusted_p,BH_FDR_Adjusted_p,Holm_Reject_5pct,BH_FDR_Reject_5pct
1,A4 adaptive ETF vs A1 direct Fed,1.0599,2.6026,0.0080,0.0318,0.0318,True,True
2,A4 adaptive ETF vs A2 no-Fed ETF,3.3714,1.5676,0.0653,0.1959,0.1306,False,False
0,A4 adaptive ETF vs A0 persistence,2.3868,1.2665,0.1090,0.2180,0.1453,False,False
3,A4 adaptive ETF vs A3 fixed B1 ETF,-0.1443,-1.7660,0.9547,0.9547,0.9547,False,False



Adaptive predicted-ETF coefficient stability


,Minimum_Coefficient,Maximum_Coefficient,Median_Coefficient,Positive_Coefficient_Share
0,-0.0168,0.0384,0.0168,0.7917


In [62]:
# CELL 55: Freeze the final Fed → ETF → gold forecasting system

import statsmodels.api as sm
import pandas as pd
import numpy as np
import json
import hashlib
from datetime import datetime
from google.colab import files

# -----------------------------------------------------------
# Required objects
# -----------------------------------------------------------

required_objects = [
    "two_stage_system_df",
    "improved_etf_system_data",
    "improved_first_stage_metrics",
    "improved_gold_metrics",
    "corrected_gold_loss_tests",
    "adaptive_gold_metrics",
    "adaptive_gold_loss_tests",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Required objects are unavailable: {missing_objects}"
    )

# -----------------------------------------------------------
# Frozen specification
# -----------------------------------------------------------

FROZEN_MODEL_NAME = (
    "Parsimonious Fed-ETF-Gold Two-Stage System"
)

FROZEN_VERSION = "1.0"
FROZEN_DATE = "2026-09-03"

frozen_specification = {
    "Model_Name": FROZEN_MODEL_NAME,
    "Version": FROZEN_VERSION,
    "Freeze_Date": FROZEN_DATE,
    "Forecast_Frequency": "Monthly",
    "Information_Events": "FOMC meeting months",
    "First_Stage_Model": (
        "ETF_Flow_Next_Month ~ const + "
        "ETF_Flow_Previous_Month + "
        "Fed_Expected_Direction"
    ),
    "Second_Stage_Model": (
        "Gold_Return_Next_Month ~ const + "
        "Gold_Return_Previous_Month + "
        "Predicted_ETF_Flow_Next_Month"
    ),
    "First_Stage_Initial_Window": 18,
    "Second_Stage_Initial_Window": 18,
    "Estimation": "Expanding-window OLS",
    "Inference": "HAC, maximum lag 2",
    "Primary_First_Stage": "B1_Parsimonious_OLS",
    "Primary_Gold_System": (
        "G4_Parsimonious_Fed_ETF_System"
    ),
    "Adaptive_Model_Status": (
        "Robustness only; not selected"
    ),
    "Oracle_Model_Status": (
        "Infeasible upper-bound diagnostic only"
    ),
    "Broad_USD_Status": (
        "Excluded from frozen first-stage model"
    ),
    "Real10Y_Status": (
        "Excluded from frozen first-stage model"
    ),
    "VIX_Status": (
        "Excluded from frozen first-stage model"
    ),
    "Further_Model_Search": (
        "Frozen; requires a genuinely new evaluation sample"
    ),
}

frozen_specification_table = pd.DataFrame(
    {
        "Setting": list(
            frozen_specification.keys()
        ),
        "Frozen_Value": list(
            frozen_specification.values()
        ),
    }
)

# -----------------------------------------------------------
# Freeze the modelling data
# -----------------------------------------------------------

frozen_source_data = (
    two_stage_system_df
    .sort_values("Month")
    .reset_index(drop=True)
    .copy()
)

frozen_recursive_data = (
    improved_etf_system_data
    .sort_values("Event_Month")
    .reset_index(drop=True)
    .copy()
)

# Data fingerprints make future accidental changes detectable.
source_csv = frozen_source_data.to_csv(
    index=False,
    date_format="%Y-%m-%d",
)

recursive_csv = frozen_recursive_data.to_csv(
    index=False,
    date_format="%Y-%m-%d",
)

specification_json = json.dumps(
    frozen_specification,
    sort_keys=True,
)

source_fingerprint = hashlib.sha256(
    source_csv.encode("utf-8")
).hexdigest()

recursive_fingerprint = hashlib.sha256(
    recursive_csv.encode("utf-8")
).hexdigest()

specification_fingerprint = hashlib.sha256(
    specification_json.encode("utf-8")
).hexdigest()

fingerprint_table = pd.DataFrame({
    "Object": [
        "Frozen source data",
        "Frozen recursive forecast data",
        "Frozen model specification",
    ],
    "SHA256": [
        source_fingerprint,
        recursive_fingerprint,
        specification_fingerprint,
    ],
})

# -----------------------------------------------------------
# Estimate the final first-stage deployment equation
# -----------------------------------------------------------

frozen_first_stage_predictors = [
    "ETF_Flow_Previous_Month",
    "Fed_Expected_Direction",
]

X_first = sm.add_constant(
    frozen_source_data[
        frozen_first_stage_predictors
    ].astype(float),
    has_constant="add",
)

y_first = frozen_source_data[
    "ETF_Flow_Next_Month"
].astype(float)

frozen_first_stage_ols = sm.OLS(
    y_first,
    X_first,
).fit()

frozen_first_stage_hac = (
    frozen_first_stage_ols.get_robustcov_results(
        cov_type="HAC",
        maxlags=2,
        use_correction=True,
    )
)

first_stage_names = (
    frozen_first_stage_ols.params.index.tolist()
)

frozen_first_stage_coefficients = pd.DataFrame({
    "Variable":
        first_stage_names,
    "Coefficient":
        np.asarray(
            frozen_first_stage_hac.params
        ),
    "HAC_SE":
        np.asarray(
            frozen_first_stage_hac.bse
        ),
    "HAC_t":
        np.asarray(
            frozen_first_stage_hac.tvalues
        ),
    "HAC_p":
        np.asarray(
            frozen_first_stage_hac.pvalues
        ),
    "CI_Lower_95pct":
        np.asarray(
            frozen_first_stage_hac.conf_int()
        )[:, 0],
    "CI_Upper_95pct":
        np.asarray(
            frozen_first_stage_hac.conf_int()
        )[:, 1],
})

frozen_first_stage_coefficients[
    "Significant_5pct"
] = (
    frozen_first_stage_coefficients[
        "HAC_p"
    ] < 0.05
)

# -----------------------------------------------------------
# Estimate the final second-stage deployment equation
#
# Train on recursively generated ETF forecasts, not in-sample
# fitted ETF values. This matches the input available during
# actual forecasting.
# -----------------------------------------------------------

frozen_second_stage_predictors = [
    "Gold_Return_Previous_Month",
    "Parsimonious_Fed_ETF_Forecast",
]

X_second = sm.add_constant(
    frozen_recursive_data[
        frozen_second_stage_predictors
    ].astype(float),
    has_constant="add",
)

y_second = frozen_recursive_data[
    "Gold_Return_Next_Month"
].astype(float)

frozen_second_stage_ols = sm.OLS(
    y_second,
    X_second,
).fit()

frozen_second_stage_hac = (
    frozen_second_stage_ols.get_robustcov_results(
        cov_type="HAC",
        maxlags=2,
        use_correction=True,
    )
)

second_stage_names = (
    frozen_second_stage_ols.params.index.tolist()
)

frozen_second_stage_coefficients = pd.DataFrame({
    "Variable":
        second_stage_names,
    "Coefficient":
        np.asarray(
            frozen_second_stage_hac.params
        ),
    "HAC_SE":
        np.asarray(
            frozen_second_stage_hac.bse
        ),
    "HAC_t":
        np.asarray(
            frozen_second_stage_hac.tvalues
        ),
    "HAC_p":
        np.asarray(
            frozen_second_stage_hac.pvalues
        ),
    "CI_Lower_95pct":
        np.asarray(
            frozen_second_stage_hac.conf_int()
        )[:, 0],
    "CI_Upper_95pct":
        np.asarray(
            frozen_second_stage_hac.conf_int()
        )[:, 1],
})

frozen_second_stage_coefficients[
    "Significant_5pct"
] = (
    frozen_second_stage_coefficients[
        "HAC_p"
    ] < 0.05
)

# -----------------------------------------------------------
# Model summary
# -----------------------------------------------------------

frozen_model_summary = pd.DataFrame({
    "Equation": [
        "First stage: ETF flow",
        "Second stage: gold return",
    ],
    "Observations": [
        int(frozen_first_stage_ols.nobs),
        int(frozen_second_stage_ols.nobs),
    ],
    "R2": [
        frozen_first_stage_ols.rsquared,
        frozen_second_stage_ols.rsquared,
    ],
    "Adjusted_R2": [
        frozen_first_stage_ols.rsquared_adj,
        frozen_second_stage_ols.rsquared_adj,
    ],
    "AIC": [
        frozen_first_stage_ols.aic,
        frozen_second_stage_ols.aic,
    ],
    "BIC": [
        frozen_first_stage_ols.bic,
        frozen_second_stage_ols.bic,
    ],
})

# -----------------------------------------------------------
# Human-readable frozen equations
# -----------------------------------------------------------

first_params = (
    frozen_first_stage_ols.params
)

second_params = (
    frozen_second_stage_ols.params
)

frozen_equations = pd.DataFrame({
    "Equation": [
        "ETF forecast",
        "Gold-return forecast",
    ],
    "Frozen_Equation": [
        (
            f"ETF_next = "
            f"{first_params['const']:.6f} "
            f"+ ({first_params['ETF_Flow_Previous_Month']:.6f})"
            f"*ETF_previous "
            f"+ ({first_params['Fed_Expected_Direction']:.6f})"
            f"*Fed_direction"
        ),
        (
            f"Gold_next = "
            f"{second_params['const']:.6f} "
            f"+ ({second_params['Gold_Return_Previous_Month']:.6f})"
            f"*Gold_previous "
            f"+ ({second_params['Parsimonious_Fed_ETF_Forecast']:.6f})"
            f"*Predicted_ETF_next"
        ),
    ],
})

# -----------------------------------------------------------
# Deployment function
# -----------------------------------------------------------

def frozen_fed_etf_gold_forecast(
    etf_flow_previous_month,
    fed_expected_direction,
    gold_return_previous_month,
):
    """
    Produce the frozen two-stage monthly forecast.

    Inputs must correspond to information available at the
    forecast origin. No future realized ETF flow is used.
    """

    predicted_etf_next = (
        first_params["const"]
        + first_params[
            "ETF_Flow_Previous_Month"
        ] * etf_flow_previous_month
        + first_params[
            "Fed_Expected_Direction"
        ] * fed_expected_direction
    )

    predicted_gold_next = (
        second_params["const"]
        + second_params[
            "Gold_Return_Previous_Month"
        ] * gold_return_previous_month
        + second_params[
            "Parsimonious_Fed_ETF_Forecast"
        ] * predicted_etf_next
    )

    return {
        "Predicted_ETF_Flow_Next_Month":
            float(predicted_etf_next),
        "Predicted_Gold_Return_Next_Month":
            float(predicted_gold_next),
    }

# -----------------------------------------------------------
# Export frozen workbook
# -----------------------------------------------------------

output_file = (
    "Frozen_Fed_ETF_Gold_System_v1_0.xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="xlsxwriter",
    datetime_format="yyyy-mm-dd",
) as writer:

    frozen_specification_table.to_excel(
        writer,
        sheet_name="Frozen Specification",
        index=False,
    )

    frozen_equations.to_excel(
        writer,
        sheet_name="Frozen Equations",
        index=False,
    )

    frozen_model_summary.to_excel(
        writer,
        sheet_name="Model Summary",
        index=False,
    )

    frozen_first_stage_coefficients.to_excel(
        writer,
        sheet_name="ETF Coefficients",
        index=False,
    )

    frozen_second_stage_coefficients.to_excel(
        writer,
        sheet_name="Gold Coefficients",
        index=False,
    )

    improved_first_stage_metrics.to_excel(
        writer,
        sheet_name="ETF Validation",
        index=False,
    )

    improved_gold_metrics.to_excel(
        writer,
        sheet_name="Gold Validation",
        index=False,
    )

    corrected_gold_loss_tests.to_excel(
        writer,
        sheet_name="Corrected Loss Tests",
        index=False,
    )

    adaptive_gold_metrics.to_excel(
        writer,
        sheet_name="Adaptive Validation",
        index=False,
    )

    adaptive_gold_loss_tests.to_excel(
        writer,
        sheet_name="Adaptive Loss Tests",
        index=False,
    )

    frozen_recursive_data.to_excel(
        writer,
        sheet_name="Recursive Forecast Data",
        index=False,
    )

    frozen_source_data.to_excel(
        writer,
        sheet_name="Frozen Source Data",
        index=False,
    )

    fingerprint_table.to_excel(
        writer,
        sheet_name="Data Fingerprints",
        index=False,
    )

    workbook = writer.book

    header_format = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "border": 1,
    })

    number_format = workbook.add_format({
        "num_format": "0.0000",
    })

    for sheet_name, worksheet in (
        writer.sheets.items()
    ):
        worksheet.freeze_panes(1, 0)
        worksheet.autofilter(
            0,
            0,
            worksheet.dim_rowmax,
            worksheet.dim_colmax,
        )
        worksheet.set_row(
            0,
            None,
            header_format,
        )
        worksheet.set_column(
            0,
            worksheet.dim_colmax,
            22,
        )
        worksheet.set_column(
            1,
            worksheet.dim_colmax,
            16,
            number_format,
        )

print("Frozen model:", FROZEN_MODEL_NAME)
print("Version:", FROZEN_VERSION)
print("Frozen on:", FROZEN_DATE)
print("Workbook:", output_file)

print("\nFrozen equations")
display(frozen_equations)

print("\nFinal coefficient tables")
display(
    frozen_first_stage_coefficients.round(6)
)
display(
    frozen_second_stage_coefficients.round(6)
)

print("\nFingerprints")
display(fingerprint_table)

files.download(output_file)

ModuleNotFoundError: No module named 'xlsxwriter'

In [63]:
# CELL 55A: Export frozen system using openpyxl

from openpyxl.styles import (
    Font,
    PatternFill,
    Border,
    Side,
    Alignment,
)
from openpyxl.utils import get_column_letter
from google.colab import files
import pandas as pd

output_file = (
    "Frozen_Fed_ETF_Gold_System_v1_0.xlsx"
)

# -----------------------------------------------------------
# Export tables
# -----------------------------------------------------------

with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
    datetime_format="YYYY-MM-DD",
) as writer:

    frozen_specification_table.to_excel(
        writer,
        sheet_name="Frozen Specification",
        index=False,
    )

    frozen_equations.to_excel(
        writer,
        sheet_name="Frozen Equations",
        index=False,
    )

    frozen_model_summary.to_excel(
        writer,
        sheet_name="Model Summary",
        index=False,
    )

    frozen_first_stage_coefficients.to_excel(
        writer,
        sheet_name="ETF Coefficients",
        index=False,
    )

    frozen_second_stage_coefficients.to_excel(
        writer,
        sheet_name="Gold Coefficients",
        index=False,
    )

    improved_first_stage_metrics.to_excel(
        writer,
        sheet_name="ETF Validation",
        index=False,
    )

    improved_gold_metrics.to_excel(
        writer,
        sheet_name="Gold Validation",
        index=False,
    )

    corrected_gold_loss_tests.to_excel(
        writer,
        sheet_name="Corrected Loss Tests",
        index=False,
    )

    adaptive_gold_metrics.to_excel(
        writer,
        sheet_name="Adaptive Validation",
        index=False,
    )

    adaptive_gold_loss_tests.to_excel(
        writer,
        sheet_name="Adaptive Loss Tests",
        index=False,
    )

    frozen_recursive_data.to_excel(
        writer,
        sheet_name="Recursive Forecast Data",
        index=False,
    )

    frozen_source_data.to_excel(
        writer,
        sheet_name="Frozen Source Data",
        index=False,
    )

    fingerprint_table.to_excel(
        writer,
        sheet_name="Data Fingerprints",
        index=False,
    )

    workbook = writer.book

    # -------------------------------------------------------
    # Workbook formatting
    # -------------------------------------------------------

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78",
    )

    header_font = Font(
        bold=True,
        color="FFFFFF",
    )

    thin_blue_border = Border(
        bottom=Side(
            style="thin",
            color="5B9BD5",
        )
    )

    for worksheet in workbook.worksheets:

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        # Header formatting
        for cell in worksheet[1]:

            cell.fill = header_fill
            cell.font = header_font
            cell.border = thin_blue_border
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        worksheet.row_dimensions[1].height = 32

        # Automatic column widths
        for column_cells in worksheet.columns:

            column_letter = get_column_letter(
                column_cells[0].column
            )

            maximum_length = 0

            for cell in column_cells:

                if cell.value is None:
                    continue

                cell_length = len(
                    str(cell.value)
                )

                maximum_length = max(
                    maximum_length,
                    cell_length,
                )

                if (
                    isinstance(
                        cell.value,
                        (float, int),
                    )
                    and cell.row > 1
                ):
                    cell.number_format = "0.000000"

            worksheet.column_dimensions[
                column_letter
            ].width = min(
                max(maximum_length + 2, 12),
                55,
            )

        # Date formatting
        for row in worksheet.iter_rows(
            min_row=2
        ):
            for cell in row:

                if (
                    cell.is_date
                    or "Month" in str(
                        worksheet.cell(
                            row=1,
                            column=cell.column,
                        ).value
                    )
                ):
                    if cell.value is not None:
                        cell.number_format = (
                            "yyyy-mm-dd"
                        )

# -----------------------------------------------------------
# Verify the exported workbook
# -----------------------------------------------------------

verification_workbook = pd.ExcelFile(
    output_file,
    engine="openpyxl",
)

expected_sheets = [
    "Frozen Specification",
    "Frozen Equations",
    "Model Summary",
    "ETF Coefficients",
    "Gold Coefficients",
    "ETF Validation",
    "Gold Validation",
    "Corrected Loss Tests",
    "Adaptive Validation",
    "Adaptive Loss Tests",
    "Recursive Forecast Data",
    "Frozen Source Data",
    "Data Fingerprints",
]

missing_sheets = [
    sheet
    for sheet in expected_sheets
    if sheet
    not in verification_workbook.sheet_names
]

if missing_sheets:
    raise ValueError(
        f"Exported workbook is missing sheets: "
        f"{missing_sheets}"
    )

print(
    "Frozen model:",
    FROZEN_MODEL_NAME
)

print(
    "Version:",
    FROZEN_VERSION
)

print(
    "Frozen on:",
    FROZEN_DATE
)

print(
    "Workbook created:",
    output_file
)

print(
    "Workbook sheets:",
    verification_workbook.sheet_names
)

print(
    "Export verification passed:",
    len(missing_sheets) == 0
)

display(frozen_equations)

files.download(output_file)

Frozen model: Parsimonious Fed-ETF-Gold Two-Stage System
Version: 1.0
Frozen on: 2026-09-03
Workbook created: Frozen_Fed_ETF_Gold_System_v1_0.xlsx
Workbook sheets: ['Frozen Specification', 'Frozen Equations', 'Model Summary', 'ETF Coefficients', 'Gold Coefficients', 'ETF Validation', 'Gold Validation', 'Corrected Loss Tests', 'Adaptive Validation', 'Adaptive Loss Tests', 'Recursive Forecast Data', 'Frozen Source Data', 'Data Fingerprints']
Export verification passed: True


,Equation,Frozen_Equation
0,ETF forecast,ETF_next = 26.015295 + (0.319033)*ETF_previous...
1,Gold-return forecast,Gold_next = 1.798479 + (-0.181766)*Gold_previo...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>